In [1]:
import pandas as pd
import os
import json
import re
from datetime import datetime
import subprocess
import io
import contextlib


# === FUENTE SQL: INDICADORES HISTORICOS ===
SQL_INDICADORES = r"""
WITH BD_TOIND AS (
    SELECT
        CONCAT(
            emp.Alias,
            '_',
            CONVERT(char(8), CAST(fech.Fecha AS date), 112)
        ) AS CONCAT,
        ges.CEFContactoEfectivo,
        ges.IdDeudor,
        ges.PromesaDePago,
        ges.MontoPDP,
        ges.Observacion,
        tipif.Abreviatura,
        deuda.SaldoCapitalSoles,
        CAST(fech.Fecha AS date) AS Fecha,
        emp.Alias
    FROM fact.GestionHumana ges

    LEFT JOIN dim.Deudor deu
        ON ges.IdDeudor = deu.IdDeudor

    LEFT JOIN dim.Cartera car
        ON ges.IdCartera = car.IdCartera

    LEFT JOIN dim.Equipo equi
        ON ges.IdEquipo = equi.IdEquipo

    LEFT JOIN dim.Fecha fech
        ON ges.IdFechaGH = fech.IdFecha

    LEFT JOIN dim.Empleado emp
        ON ges.IdEmpleado = emp.IdEmpleado

    LEFT JOIN dim.Tipificacion tipif
        ON ges.IdTipificacion = tipif.IdTipificacion

    LEFT JOIN dim.Deuda deuda
        ON ges.IdDeudor = deuda.IdDeudor

    WHERE ges.TipoGestion = 'MANUAL'
      AND equi.Canal IN ('EXPERTIS', 'EXPERTIS BPO')
),

/* =========================================================
   ASISTENCIAS
   ========================================================= */

ASISTENCIAS_BASE AS (
    SELECT
        CONCAT(
            E.Alias,
            '_',
            CONVERT(char(8), A.IdFecha)
        ) AS CONCAT,

        A.IdFecha,
        E.Alias,
        A.HoraIngreso,
        A.Cargo,
        A.IdCondicionAsistencia,
        E.Vigencia,
        EQ.Canal,
        EQ.NombreEquipo AS Equipo

    FROM fact.Asistencia AS A

    LEFT JOIN dim.Empleado AS E
        ON A.IdEmpleado = E.IdEmpleado

    LEFT JOIN dim.Equipo AS EQ
        ON A.IdEquipo = EQ.IdEquipo
),

ASISTENCIAS_CALCULADAS AS (
    SELECT
        CONCAT,
        Vigencia,
        Canal,
        Equipo,

        CASE
            WHEN IdCondicionAsistencia = 3
                THEN 'SI'
            ELSE 'NO'
        END AS FALTA,

        /* 
           Devuelve la tardanza como fracción de un día,
           igual que Power Query.
        */
        CASE
            WHEN HoraIngreso IS NULL
                THEN 0

            WHEN HoraIngreso <= CAST('07:00:00' AS time)
                THEN 0

            ELSE
                CAST(
                    DATEDIFF(
                        SECOND,
                        CAST('07:00:00' AS time),
                        HoraIngreso
                    ) AS decimal(18,10)
                ) / 86400
        END AS TardanzaTiempo,

        IdCondicionAsistencia

    FROM ASISTENCIAS_BASE
),

IND_ASISTENCIAS AS (
    SELECT
        CONCAT,
        Vigencia,
        Canal,
        Equipo,
        FALTA,
        TardanzaTiempo,

        CASE
            WHEN TardanzaTiempo > 0
                 AND IdCondicionAsistencia <> 4
                THEN 'SI'
            ELSE 'NO'
        END AS TARDANZA

    FROM ASISTENCIAS_CALCULADAS
),

/* =========================================================
   INDICADORES DE GESTIÓN
   ========================================================= */

IND_CEFUnico AS (
    SELECT
        CONCAT,
        COUNT(DISTINCT IdDeudor) AS IND_CEFUnico
    FROM BD_TOIND
    WHERE CEFContactoEfectivo = 'SI'
    GROUP BY CONCAT
),

IND_PDP AS (
    SELECT
        CONCAT,
        COUNT(DISTINCT IdDeudor) AS IND_PDP
    FROM BD_TOIND
    WHERE PromesaDePago = 'SI'
    GROUP BY CONCAT
),

IND_MontoPDP AS (
    SELECT
        CONCAT,
        SUM(MontoPDP) AS MontoPDP
    FROM BD_TOIND
    WHERE PromesaDePago = 'SI'
    GROUP BY CONCAT
),

IND_DK_Base AS (
    SELECT
        CONCAT,
        IdDeudor,
        MAX(SaldoCapitalSoles) AS SaldoCapitalSoles
    FROM BD_TOIND
    WHERE Abreviatura = 'PPC'
      AND LOWER(Observacion) NOT LIKE '%ultima cuota%'
      AND LOWER(Observacion) NOT LIKE '%última cuota%'
    GROUP BY
        CONCAT,
        IdDeudor
),

IND_DK AS (
    SELECT
        CONCAT,
        SUM(SaldoCapitalSoles) AS DK
    FROM IND_DK_Base
    GROUP BY CONCAT
),

/* =========================================================
   ÚLTIMA GESTIÓN DEL DEUDOR
   ========================================================= */

UltimaGestion AS (
    SELECT
        fech.CodigoMes,
        ges.IdDeudor,
        deu.Documento_RUC,
        car.Nombre AS Cartera,
        fech.Fecha AS FechaGestion,
        ges.FechaCompromiso,
        emp.Alias AS Asesor,
        tipif.Abreviatura,

        ROW_NUMBER() OVER (
            PARTITION BY
                fech.CodigoMes,
                ges.IdDeudor,
                car.Nombre
            ORDER BY
                fech.Fecha DESC,
                ges.IdGestionHumana DESC
        ) AS rn

    FROM fact.GestionHumana ges

    LEFT JOIN dim.Deudor deu
        ON ges.IdDeudor = deu.IdDeudor

    LEFT JOIN dim.Cartera car
        ON ges.IdCartera = car.IdCartera

    LEFT JOIN dim.Fecha fech
        ON ges.IdFechaGH = fech.IdFecha

    LEFT JOIN dim.Empleado emp
        ON ges.IdEmpleado = emp.IdEmpleado

    LEFT JOIN dim.Tipificacion tipif
        ON ges.IdTipificacion = tipif.IdTipificacion

    WHERE fech.Fecha >= '2024-01-01'
      AND ges.TipoGestion = 'MANUAL'
      AND tipif.Abreviatura IN ('PAR', 'PPC', 'PPM')
),

GestionFinal AS (
    SELECT
        CodigoMes,
        IdDeudor,
        Documento_RUC,
        Cartera,
        FechaGestion,
        FechaCompromiso,
        Asesor,
        Abreviatura
    FROM UltimaGestion
    WHERE rn = 1
),

/* =========================================================
   PAGOS Y RECUPERO
   ========================================================= */

PagosAgrupados AS (
    SELECT
        fec2.CodigoMes,
        pag.IdDeudor,
        deu.Documento_RUC,
        car.Nombre AS Cartera,
        fec2.Fecha AS FechaReconocimiento,
        SUM(pag.Monto) AS MontoPago
    FROM fact.Pago pag

    LEFT JOIN dim.Deudor deu
        ON pag.IdDeudor = deu.IdDeudor

    LEFT JOIN dim.Cartera car
        ON pag.IdCartera = car.IdCartera

    LEFT JOIN dim.Fecha fec2
        ON pag.IdFechaReconocimiento = fec2.IdFecha

    GROUP BY
        fec2.CodigoMes,
        pag.IdDeudor,
        deu.Documento_RUC,
        car.Nombre,
        fec2.Fecha
),

IND_RECUPERO_DEUDOR AS (
    SELECT
        CONCAT(
            ges.Asesor,
            '_',
            CONVERT(char(8), CAST(ges.FechaGestion AS date), 112)
        ) AS CONCAT,

        ges.IdDeudor,
        SUM(pag.MontoPago) AS recupero
    FROM PagosAgrupados pag

    LEFT JOIN GestionFinal ges
        ON pag.IdDeudor = ges.IdDeudor
       AND pag.Cartera = ges.Cartera
       AND pag.CodigoMes = ges.CodigoMes

    WHERE ges.FechaGestion IS NOT NULL

    GROUP BY
        CONCAT(
            ges.Asesor,
            '_',
            CONVERT(char(8), CAST(ges.FechaGestion AS date), 112)
        ),
        ges.IdDeudor
),

IND_RECUPERO AS (
    SELECT
        CONCAT,
        SUM(recupero) AS recupero
    FROM IND_RECUPERO_DEUDOR
    GROUP BY CONCAT
),

/* =========================================================
   PAGO CONDONADO
   ========================================================= */

PagoCondonado_Base AS (
    SELECT
        b.CONCAT,
        b.IdDeudor,
        MAX(rec.recupero) AS recupero
    FROM BD_TOIND b

    LEFT JOIN IND_RECUPERO_DEUDOR rec
        ON b.CONCAT = rec.CONCAT
       AND b.IdDeudor = rec.IdDeudor

    WHERE b.Abreviatura = 'PPC'
      AND LOWER(b.Observacion) NOT LIKE '%ultima cuota%'
      AND LOWER(b.Observacion) NOT LIKE '%última cuota%'

    GROUP BY
        b.CONCAT,
        b.IdDeudor
),

PagoCondonado AS (
    SELECT
        CONCAT,
        SUM(recupero) AS PagoCondonado
    FROM PagoCondonado_Base
    GROUP BY CONCAT
),

/* =========================================================
   BASE FINAL
   ========================================================= */

BaseFinal AS (
    SELECT DISTINCT
        CONCAT,
        Fecha,
        Alias
    FROM BD_TOIND
    WHERE Fecha <> CAST(GETDATE() AS date)
),

/* =========================================================
   CONSOLIDACIÓN DE INDICADORES
   ========================================================= */

IndicadoresBase AS (
    SELECT
        bf.CONCAT,
        bf.Fecha AS IND_Fecha,
        bf.Alias AS IND_Alias,

        asi.FALTA AS IND_FALTA,
        asi.TardanzaTiempo AS IND_TardanzaTiempo,
        asi.TARDANZA AS IND_TARDE,

        cef.IND_CEFUnico,
        pdp.IND_PDP,
        mpdp.MontoPDP AS IND_MontoPDP,
        dk.DK AS IND_DK,
        rec.recupero AS IND_recupero,
        pc.PagoCondonado AS IND_PagoCondonado

    FROM BaseFinal bf

    LEFT JOIN IND_ASISTENCIAS asi
        ON bf.CONCAT = asi.CONCAT

    LEFT JOIN IND_CEFUnico cef
        ON bf.CONCAT = cef.CONCAT

    LEFT JOIN IND_PDP pdp
        ON bf.CONCAT = pdp.CONCAT

    LEFT JOIN IND_MontoPDP mpdp
        ON bf.CONCAT = mpdp.CONCAT

    LEFT JOIN IND_DK dk
        ON bf.CONCAT = dk.CONCAT

    LEFT JOIN IND_RECUPERO rec
        ON bf.CONCAT = rec.CONCAT

    LEFT JOIN PagoCondonado pc
        ON bf.CONCAT = pc.CONCAT
)

/* =========================================================
   RESULTADO FINAL
   ========================================================= */

SELECT
    ib.CONCAT,
    ib.IND_Fecha,
    ib.IND_Alias,

    ib.IND_FALTA,
    ib.IND_TardanzaTiempo,
    ib.IND_TARDE,

    ib.IND_CEFUnico,
    ib.IND_PDP,
    ib.IND_MontoPDP,
    ib.IND_DK,
    ib.IND_recupero,
    ib.IND_PagoCondonado,

    /* IND_Ticket_PDP =
       IND_MontoPDP / IND_PDP
    */
    CAST(ib.IND_MontoPDP AS decimal(18,6))
        / NULLIF(CAST(ib.IND_PDP AS decimal(18,6)), 0)
        AS IND_Ticket_PDP,

    /* IND_Calidad PDP =
       IND_recupero / IND_MontoPDP
    */
    CAST(ib.IND_recupero AS decimal(18,6))
        / NULLIF(CAST(ib.IND_MontoPDP AS decimal(18,6)), 0)
        AS [IND_Calidad PDP],

    /* IND_Cierre =
       IND_PDP / IND_CEFUnico
    */
    CAST(ib.IND_PDP AS decimal(18,6))
        / NULLIF(CAST(ib.IND_CEFUnico AS decimal(18,6)), 0)
        AS IND_Cierre,

    /* IND_Condonacion =
       IND_PagoCondonado / IND_DK
    */
    CAST(ib.IND_PagoCondonado AS decimal(18,6))
        / NULLIF(CAST(ib.IND_DK AS decimal(18,6)), 0)
        AS IND_Condonacion,

    /* IND_%puntualidad

       Si falta = SI          => 0
       Si supera 4 horas      => 0
       En los demás casos     => 1 - tardanza / 4 horas
    */
    CAST(
        CASE
            WHEN ib.IND_FALTA = 'SI'
                THEN 0

            WHEN ib.IND_TardanzaTiempo IS NULL
                THEN 1

            WHEN 1 - (
                ib.IND_TardanzaTiempo
                / CAST(0.166666666666667 AS decimal(18,15))
            ) < 0
                THEN 0

            ELSE 1 - (
                ib.IND_TardanzaTiempo
                / CAST(0.166666666666667 AS decimal(18,15))
            )
        END
        AS decimal(18,6)
    ) AS [IND_%puntualidad]

FROM IndicadoresBase ib

ORDER BY
    ib.IND_Fecha DESC,
    ib.CONCAT;
"""


def _escapar_valor_odbc(valor):
    """Escapa valores de credenciales para una cadena de conexión ODBC."""
    return "{" + str(valor).replace("}", "}}") + "}"


def crear_conexion_sql():
    """Solicita credenciales en ejecución; no las guarda en el notebook."""
    import pyodbc
    from getpass import getpass

    usuario = input("Usuario SQL: ").strip()
    password = getpass("Contraseña SQL: ")
    if not usuario or not password:
        raise ValueError("El usuario y la contraseña SQL son obligatorios.")

    cadena = (
        "DRIVER={ODBC Driver 18 for SQL Server};"
        "SERVER=192.168.100.02;"
        "DATABASE=BD_Expertis_DWH;"
        f"UID={_escapar_valor_odbc(usuario)};"
        f"PWD={_escapar_valor_odbc(password)};"
        "Encrypt=yes;"
        "TrustServerCertificate=yes;"
        "Connection Timeout=30;"
    )
    return pyodbc.connect(cadena)


def validar_indicadores_sql(df):
    """Normaliza el contrato mínimo de la Vista de Indicadores histórica."""
    if df is None or df.empty:
        raise ValueError("La consulta SQL de indicadores no devolvió filas.")

    df = df.copy()
    df.columns = [str(col).strip() for col in df.columns]
    columnas_por_mayuscula = {col.upper(): col for col in df.columns}
    obligatorias = {"IND_FECHA": "IND_Fecha", "IND_ALIAS": "IND_Alias"}
    faltantes = [nombre for nombre in obligatorias if nombre not in columnas_por_mayuscula]
    if faltantes:
        raise ValueError(f"Faltan columnas SQL obligatorias: {', '.join(faltantes)}")

    renombres = {
        columnas_por_mayuscula[normalizada]: canonica
        for normalizada, canonica in obligatorias.items()
        if columnas_por_mayuscula[normalizada] != canonica
    }
    df = df.rename(columns=renombres)
    df["IND_Fecha"] = pd.to_datetime(df["IND_Fecha"], errors="coerce")
    fechas_invalidas = int(df["IND_Fecha"].isna().sum())
    if fechas_invalidas:
        raise ValueError(f"IND_FECHA contiene {fechas_invalidas} valores no convertibles a fecha.")

    df["IND_Alias"] = df["IND_Alias"].fillna("").astype(str).str.strip()
    df = df[df["IND_Alias"] != ""].copy()
    if df.empty:
        raise ValueError("No quedaron filas con IND_ALIAS válido.")

    if "CONCAT" in df.columns:
        duplicados = int(df["CONCAT"].duplicated(keep=False).sum())
        if duplicados:
            print(f"  ⚠️ SQL indicadores: {duplicados} filas tienen CONCAT duplicado; se conservarán para conciliación.")

    return df


def leer_indicadores_desde_sql(conexion=None, consulta=SQL_INDICADORES):
    """Ejecuta la consulta histórica y devuelve un DataFrame validado."""
    conexion_propia = conexion is None
    if conexion_propia:
        conexion = crear_conexion_sql()
    try:
        df = pd.read_sql_query(consulta, conexion)
    finally:
        if conexion_propia:
            conexion.close()
    return validar_indicadores_sql(df)


def agrupar_indicadores_sql_por_mes(df):
    """Separa la tabla histórica en DataFrames compatibles, uno por MES_AÑO."""
    meses = {
        1: "ENERO", 2: "FEBRERO", 3: "MARZO", 4: "ABRIL",
        5: "MAYO", 6: "JUNIO", 7: "JULIO", 8: "AGOSTO",
        9: "SETIEMBRE", 10: "OCTUBRE", 11: "NOVIEMBRE", 12: "DICIEMBRE",
    }
    periodos = df["IND_Fecha"].dt.month.map(meses) + "_" + df["IND_Fecha"].dt.year.astype(str)
    return {periodo: grupo.copy() for periodo, grupo in df.groupby(periodos, sort=False)}



# === FUENTE SQL: INF HISTORICO ===
SQL_INF = r"""
SELECT
    E.[NombreCompleto],
    E.[Alias],
    E.[Documento],
    E.[Vigencia],
    EP.[IdFechaIngreso],
    EP.[IdFechaSalida],
    EP.[FechaNacimiento],
    EP.[Vigente],
    EP.[Cesado],
    EP.[DiasPermanencia],
    EP.[MesesPermanencia],
    CAST(NULL AS int) AS [RN]
FROM [BD_Expertis_DWH].[dim].[Empleado] AS E
LEFT JOIN [BD_Expertis_DWH].[fact].[EmpleadoPermanencia2] AS EP
    ON E.[IdEmpleado] = EP.[IdEmpleado]
WHERE EP.[IdFechaIngreso] IS NOT NULL;
"""


def validar_inf_sql(df):
    """Valida y ordena las 12 columnas que consumen los extractores actuales de INF."""
    if df is None or df.empty:
        raise ValueError("La consulta SQL de INF no devolvió filas.")

    columnas_requeridas = [
        "NombreCompleto", "Alias", "Documento", "Vigencia",
        "IdFechaIngreso", "IdFechaSalida", "FechaNacimiento",
        "Vigente", "Cesado", "DiasPermanencia", "MesesPermanencia", "RN",
    ]
    df = df.copy()
    df.columns = [str(col).strip() for col in df.columns]
    columnas_por_mayuscula = {col.upper(): col for col in df.columns}
    faltantes = [col for col in columnas_requeridas if col.upper() not in columnas_por_mayuscula]
    if faltantes:
        raise ValueError(f"Faltan columnas SQL de INF: {', '.join(faltantes)}")

    seleccion = [columnas_por_mayuscula[col.upper()] for col in columnas_requeridas]
    df = df.loc[:, seleccion]
    df.columns = columnas_requeridas
    return df


def convertir_inf_a_formato_legacy(df):
    """Replica read_excel(header=None) para no cambiar los extractores ni el HTML."""
    filas = [list(df.columns)] + df.astype(object).where(pd.notna(df), None).values.tolist()
    return pd.DataFrame(filas)


def leer_inf_desde_sql(conexion, consulta=SQL_INF):
    """Lee INF desde SQL y devuelve la estructura exacta esperada por el código actual."""
    df = pd.read_sql_query(consulta, conexion)
    return convertir_inf_a_formato_legacy(validar_inf_sql(df))



MESES_ES = [
    'ENERO', 'FEBRERO', 'MARZO', 'ABRIL', 'MAYO', 'JUNIO',
    'JULIO', 'AGOSTO', 'SETIEMBRE', 'OCTUBRE', 'NOVIEMBRE', 'DICIEMBRE'
]


def normalizar_clave_dimension(valor):
    """Normaliza equipos, segmentos y meses para cruzar DIM con METAS."""
    import unicodedata

    texto = '' if valor is None or pd.isna(valor) else str(valor).strip().upper()
    texto = unicodedata.normalize('NFKD', texto)
    texto = ''.join(c for c in texto if not unicodedata.combining(c))
    return re.sub(r'[^A-Z0-9]+', '', texto)


def normalizar_nombre_mes(valor):
    clave = normalizar_clave_dimension(valor)
    equivalencias = {
        'ENERO': 'ENERO', 'FEBRERO': 'FEBRERO', 'MARZO': 'MARZO',
        'ABRIL': 'ABRIL', 'MAYO': 'MAYO', 'JUNIO': 'JUNIO',
        'JULIO': 'JULIO', 'AGOSTO': 'AGOSTO',
        'SETIEMBRE': 'SETIEMBRE', 'SEPTIEMBRE': 'SETIEMBRE',
        'OCTUBRE': 'OCTUBRE', 'NOVIEMBRE': 'NOVIEMBRE',
        'DICIEMBRE': 'DICIEMBRE'
    }
    return equivalencias.get(clave, clave)


def numero_seguro(valor, default=0.0):
    if valor is None or pd.isna(valor):
        return default
    try:
        return float(valor)
    except (TypeError, ValueError):
        texto = str(valor).strip().replace('S/', '').replace('%', '').replace(',', '')
        try:
            return float(texto)
        except (TypeError, ValueError):
            return default


def texto_excel(valor):
    if valor is None or pd.isna(valor):
        return ''
    if isinstance(valor, pd.Timestamp):
        return valor.strftime('%Y-%m-%d')
    if isinstance(valor, float) and valor.is_integer():
        return str(int(valor))
    return str(valor).strip()


def cargar_metas_desde_excel(ruta_excel_fuentes):
    """Carga METAS completa. Meta_Asesor se usa para el cálculo individual."""
    df = pd.read_excel(ruta_excel_fuentes, sheet_name='METAS', header=0)
    requeridas = {'ANO', 'MES', 'EQUIPO', 'METAASESOR', 'SEGG'}
    headers = {normalizar_clave_dimension(col): col for col in df.columns}
    faltantes = sorted(requeridas - set(headers))
    if faltantes:
        raise ValueError(f"METAS no contiene las columnas requeridas: {faltantes}")

    df = df.copy()
    df['_ANIO'] = pd.to_numeric(df[headers['ANO']], errors='coerce').astype('Int64')
    df['_MES'] = df[headers['MES']].map(normalizar_nombre_mes)
    df['_EQUIPO'] = df[headers['EQUIPO']].map(normalizar_clave_dimension)
    df['_SEGMENTO'] = df[headers['SEGG']].map(normalizar_clave_dimension)
    df['_META_ASESOR'] = pd.to_numeric(df[headers['METAASESOR']], errors='coerce')
    print(f"METAS cargada: {len(df)} filas")
    return df


def construir_indice_metas(df_metas):
    exactas = {}
    generales = {}
    por_equipo = {}

    for _, row in df_metas.iterrows():
        if pd.isna(row.get('_ANIO')):
            continue
        anio = int(row['_ANIO'])
        mes = str(row.get('_MES', ''))
        equipo = str(row.get('_EQUIPO', ''))
        segmento = str(row.get('_SEGMENTO', ''))
        meta = numero_seguro(row.get('_META_ASESOR'), 0.0)
        if not mes or not equipo:
            continue

        clave_equipo = (anio, mes, equipo)
        por_equipo.setdefault(clave_equipo, []).append(meta)
        if segmento in {'', 'TODOS', 'TODO', 'GENERAL'}:
            generales[clave_equipo] = meta
        else:
            exactas[(anio, mes, equipo, segmento)] = meta

    return {'exactas': exactas, 'generales': generales, 'por_equipo': por_equipo}


def obtener_meta_asesor(indice_metas, periodo, supervisor, segmento):
    try:
        mes, anio = periodo.rsplit('_', 1)
        anio = int(anio)
    except (ValueError, TypeError):
        return 0.0

    mes = normalizar_nombre_mes(mes)
    equipo = normalizar_clave_dimension(supervisor)
    segmento = normalizar_clave_dimension(segmento)
    clave_exacta = (anio, mes, equipo, segmento)
    clave_equipo = (anio, mes, equipo)

    if clave_exacta in indice_metas['exactas']:
        return indice_metas['exactas'][clave_exacta]
    if clave_equipo in indice_metas['generales']:
        return indice_metas['generales'][clave_equipo]

    candidatas = indice_metas['por_equipo'].get(clave_equipo, [])
    if len(candidatas) == 1:
        return candidatas[0]
    return 0.0



def construir_base_historica_desde_indicadores(
    periodo, df_indicadores, mapas_dim_canales, indice_metas
):
    """Construye el mes solo con relaciones exactas existentes en DIM."""
    if df_indicadores is None or df_indicadores.empty:
        return []

    requeridas = {'IND_Alias', 'IND_recupero'}
    faltantes = requeridas - set(df_indicadores.columns)
    if faltantes:
        raise ValueError(f"Indicadores SQL no contiene: {sorted(faltantes)}")

    trabajo = df_indicadores.copy()
    trabajo['_ALIAS_TEXTO'] = trabajo['IND_Alias'].map(texto_excel)
    trabajo['_ALIAS_CLAVE'] = trabajo['_ALIAS_TEXTO'].map(normalizar_clave_asesor)
    trabajo['_RECUPERO'] = pd.to_numeric(trabajo['IND_recupero'], errors='coerce').fillna(0.0)
    trabajo = trabajo[trabajo['_ALIAS_CLAVE'] != '']

    relaciones = mapas_dim_canales.get('por_periodo_alias', {})
    ambiguos = mapas_dim_canales.get('ambiguos_periodo_alias', set())
    base = []
    sin_dim = 0
    dim_ambiguo = 0
    sin_estado = 0
    metas_no_encontradas = 0

    for clave_alias, grupo in trabajo.groupby('_ALIAS_CLAVE', sort=False):
        clave_dim = (periodo, clave_alias)
        if clave_dim in ambiguos:
            dim_ambiguo += 1
            continue
        dim_item = relaciones.get(clave_dim)
        if dim_item is None:
            sin_dim += 1
            continue

        estado = texto_excel(dim_item.get('estado')).upper()
        if not estado:
            sin_estado += 1
            continue

        alias = next((v for v in grupo['_ALIAS_TEXTO'].tolist() if v), clave_alias)
        supervisor = texto_excel(dim_item.get('supervisor'))
        segmento = texto_excel(dim_item.get('segmento'))
        recupero = float(grupo['_RECUPERO'].sum())
        meta = float(obtener_meta_asesor(indice_metas, periodo, supervisor, segmento))
        if meta <= 0 and estado == 'CALL':
            metas_no_encontradas += 1
        alcance = recupero / meta if meta > 0 else 0.0

        base.append({
            'alias_crr': alias,
            'dni': texto_excel(dim_item.get('dni')),
            'fecha_inicio': texto_excel(dim_item.get('fecha_inicio')),
            'supervisor': supervisor,
            'estado': estado,
            'segmento': segmento,
            'cartera': segmento,
            'canal': texto_excel(dim_item.get('canal')).upper(),
            'alcance': alcance,
            'recupero': recupero,
            'meta': meta,
            'q_alc': ''
        })

    if sin_dim:
        print(f"  ⚠️ {periodo}: {sin_dim} asesores SQL excluidos por no existir en DIM para ese periodo")
    if dim_ambiguo:
        print(f"  ⚠️ {periodo}: {dim_ambiguo} asesores excluidos por relaciones contradictorias en DIM")
    if sin_estado:
        print(f"  ⚠️ {periodo}: {sin_estado} asesores excluidos porque DIM no informa CONDICION")
    if metas_no_encontradas:
        print(f"  ⚠️ {periodo}: {metas_no_encontradas} asesores CALL sin Meta_Asesor compatible")
    return base



def asignar_quintiles_internos(base_asesores):
    """Asigna Q5 al 20 % superior de ALCANCE y Q1 al 20 % inferior."""
    for asesor in base_asesores:
        asesor['q_alc'] = ''

    evaluables = [
        asesor for asesor in base_asesores
        if str(asesor.get('estado', 'CALL')).strip().upper() == 'CALL'
        and numero_seguro(asesor.get('meta'), 0.0) > 0
    ]
    evaluables.sort(
        key=lambda item: (
            -numero_seguro(item.get('alcance'), 0.0),
            normalizar_clave_asesor(item.get('alias_crr', ''))
        )
    )

    total = len(evaluables)
    for indice, asesor in enumerate(evaluables):
        quintil = max(1, 5 - ((indice * 5) // total))
        asesor['q_alc'] = f'Q{quintil}'
    return base_asesores


def listar_hojas_periodo(ruta_excel_fuentes):
    patron = re.compile(
        r'^(ENERO|FEBRERO|MARZO|ABRIL|MAYO|JUNIO|JULIO|AGOSTO|'
        r'SETIEMBRE|SEPTIEMBRE|OCTUBRE|NOVIEMBRE|DICIEMBRE)_\d{4}$',
        re.IGNORECASE
    )
    with pd.ExcelFile(ruta_excel_fuentes) as libro:
        return [hoja.upper().replace('SEPTIEMBRE', 'SETIEMBRE') for hoja in libro.sheet_names if patron.match(hoja)]


def procesar_excel_y_actualizar_html():
    global df_metas

    base_path = r"C:\Users\Jorge Vasquez\Ranking"
    ruta_excel_fuentes = r"D:\NEXOS\CS_AVANCE_AS.xlsx"
    ruta_html = os.path.join(base_path, "index.html")

    print("Cargando DIM y METAS desde CS_AVANCE_AS.xlsx...")
    mapas_dim_canales = cargar_mapas_dim_canales(ruta_excel_fuentes)
    if not mapas_dim_canales.get('por_periodo_alias'):
        raise ValueError("La hoja DIM no produjo registros utilizables")
    df_metas = cargar_metas_desde_excel(ruta_excel_fuentes)
    indice_metas = construir_indice_metas(df_metas)

    hojas_periodo_excel = listar_hojas_periodo(ruta_excel_fuentes)
    nombre_mes_sistema = f"{MESES_ES[datetime.now().month - 1]}_{datetime.now().year}"
    if nombre_mes_sistema in hojas_periodo_excel:
        hoja_mes_vigente = nombre_mes_sistema
    elif hojas_periodo_excel:
        hoja_mes_vigente = max(hojas_periodo_excel, key=convertir_mes_a_fecha)
        print(f"⚠️ No existe {nombre_mes_sistema}; se usará {hoja_mes_vigente} como mes vigente")
    else:
        raise ValueError("No existe una hoja MES_AÑO para el mes vigente en CS_AVANCE_AS.xlsx")

    periodos_dim = set(mapas_dim_canales.get('periodos', set()))
    if hoja_mes_vigente not in periodos_dim:
        raise ValueError(
            f"DIM no contiene el periodo vigente {hoja_mes_vigente}; no se generará un periodo sin respaldo DIM"
        )

    indicadores_sql_por_mes = {}
    df_inf_sql = None
    conexion_sql = None
    try:
        conexion_sql = crear_conexion_sql()
    except Exception as error_conexion:
        print(f"⚠️ SQL Server no estuvo disponible: {error_conexion}")
        print("  Se continuará únicamente con la hoja del mes vigente; no se usará el Excel anterior.")

    if conexion_sql is not None:
        try:
            print("Cargando indicadores históricos desde SQL Server...")
            df_indicadores_sql = leer_indicadores_desde_sql(conexion_sql)
            indicadores_sql_por_mes = agrupar_indicadores_sql_por_mes(df_indicadores_sql)
            print(f"Indicadores SQL cargados: {len(df_indicadores_sql)} filas, {len(indicadores_sql_por_mes)} periodos")
        except Exception as error_indicadores:
            print(f"⚠️ No se pudieron cargar Indicadores SQL: {error_indicadores}")

        try:
            print("Cargando INF histórico desde SQL Server...")
            df_inf_sql = leer_inf_desde_sql(conexion_sql)
            print(f"INF SQL cargado: {max(len(df_inf_sql) - 1, 0)} filas")
        except Exception as error_inf:
            print(f"⚠️ No se pudo cargar INF SQL: {error_inf}")
        finally:
            conexion_sql.close()

    datos_cumpleaños_global = []
    datos_asesores_inf_global = []
    if df_inf_sql is not None:
        with contextlib.redirect_stdout(io.StringIO()):
            datos_cumpleaños_global = extraer_cumpleaños_de_inf(df_inf_sql)
            datos_asesores_inf_global = extraer_registro_asesores_inf(df_inf_sql)

    todos_los_datos = {}
    todos_los_asesores = set()
    datos_cumpleaños_por_mes = {}
    base_asesores_analisis = {}

    periodos_sql = set(indicadores_sql_por_mes)
    periodos_omitidos_sin_dim = sorted(periodos_sql - periodos_dim, key=convertir_mes_a_fecha)
    if periodos_omitidos_sin_dim:
        print(
            f"DIM excluye {len(periodos_omitidos_sin_dim)} periodos SQL sin cobertura: "
            f"{periodos_omitidos_sin_dim[0]} a {periodos_omitidos_sin_dim[-1]}"
        )

    periodos_historicos = sorted(
        (periodos_sql & periodos_dim) - {hoja_mes_vigente},
        key=convertir_mes_a_fecha
    )
    for periodo in periodos_historicos:
        print(f"Construyendo {periodo} desde Indicadores + DIM + METAS...")
        df_indicadores_mes = indicadores_sql_por_mes[periodo]
        base_mes = construir_base_historica_desde_indicadores(
            periodo, df_indicadores_mes, mapas_dim_canales, indice_metas
        )
        asignar_quintiles_internos(base_mes)
        with contextlib.redirect_stdout(io.StringIO()):
            resultado = extraer_datos_completos(
                pd.DataFrame(), periodo, mapas_dim_canales,
                df_indicadores=df_indicadores_mes,
                base_asesores=base_mes
            )
        todos_los_datos[periodo] = resultado['asesores']
        base_asesores_analisis[periodo] = base_mes
        datos_cumpleaños_por_mes[periodo] = datos_cumpleaños_global

    print(f"Cargando {hoja_mes_vigente} directamente desde CS_AVANCE_AS.xlsx...")
    df_mes_vigente = pd.read_excel(
        ruta_excel_fuentes, sheet_name=hoja_mes_vigente, header=None
    )
    base_mes_vigente = extraer_base_asesores_desde_b2(
        df_mes_vigente, hoja_mes_vigente, mapas_dim_canales
    )
    asignar_quintiles_internos(base_mes_vigente)
    with contextlib.redirect_stdout(io.StringIO()):
        resultado_vigente = extraer_datos_completos(
            df_mes_vigente, hoja_mes_vigente, mapas_dim_canales,
            df_indicadores=indicadores_sql_por_mes.get(hoja_mes_vigente),
            base_asesores=base_mes_vigente
        )
    todos_los_datos[hoja_mes_vigente] = resultado_vigente['asesores']
    base_asesores_analisis[hoja_mes_vigente] = base_mes_vigente
    datos_cumpleaños_por_mes[hoja_mes_vigente] = datos_cumpleaños_global

    meses_procesados = sorted(todos_los_datos, key=convertir_mes_a_fecha)
    años_unicos = sorted({periodo.rsplit('_', 1)[1] for periodo in meses_procesados}, reverse=True)
    meses_unicos = MESES_ES.copy()

    for asesores_mes in todos_los_datos.values():
        for asesor in asesores_mes:
            todos_los_asesores.add(asesor['nombre'])

    with contextlib.redirect_stdout(io.StringIO()):
        datos_por_supervisor = agrupar_por_supervisor(todos_los_datos)
    lista_supervisores = sorted({
        str(asesor.get('supervisor', 'Sin Supervisor')).strip() or 'Sin Supervisor'
        for asesores_mes in todos_los_datos.values()
        for asesor in asesores_mes
    })

    with contextlib.redirect_stdout(io.StringIO()):
        generar_html_modular(
            todos_los_datos=todos_los_datos,
            ruta_html=ruta_html,
            meses_unicos=meses_unicos,
            años_unicos=años_unicos,
            meses_validos=meses_procesados,
            lista_asesores=sorted(todos_los_asesores),
            datos_por_supervisor=datos_por_supervisor,
            lista_supervisores=lista_supervisores,
            datos_cumpleaños_por_mes=datos_cumpleaños_por_mes,
            base_asesores_analisis=base_asesores_analisis,
            datos_capa=[],
            datos_mundial={},
            datos_asesores_inf=datos_asesores_inf_global
        )
    print("Index Actualizado")







def extraer_base_asesores_desde_b2(df, nombre_hoja='', mapas_dim_canales=None):
    """Lee el mes vigente y solo permite enriquecimiento DIM del mismo periodo."""
    base = []
    mapas_dim_canales = mapas_dim_canales or {
        'por_periodo_alias': {}, 'ambiguos_periodo_alias': set()
    }

    if df.shape[0] < 3 or df.shape[1] < 2:
        return base

    headers = {}
    for col_idx in range(df.shape[1]):
        if pd.notna(df.iloc[1, col_idx]):
            headers[str(df.iloc[1, col_idx]).strip().upper()] = col_idx

    col_alias = headers.get('ASESOR_CRR')
    if col_alias is None:
        col_alias = headers.get('ASESOR')
    if col_alias is None:
        return base

    col_meta = headers.get('META')
    col_alcance = headers.get('ALCANCE')
    col_supervisor = headers.get('SUPER2')
    if col_supervisor is None:
        col_supervisor = headers.get('SUPER')
    col_recupero = headers.get('RECUPERO')
    col_dni = headers.get('DNI')
    col_fecha_inicio = headers.get('FECHA DE INICIO')
    col_estado = headers.get('ESTADO')
    col_segmento = headers.get('SEGMENTO')
    col_canal = headers.get('CANAL')

    relaciones = mapas_dim_canales.get('por_periodo_alias', {})
    ambiguos = mapas_dim_canales.get('ambiguos_periodo_alias', set())

    for i in range(2, len(df)):
        alias = texto_excel(df.iloc[i, col_alias]) if col_alias < df.shape[1] else ''
        if not alias:
            break

        clave_dim = (nombre_hoja, normalizar_clave_asesor(alias))
        dim_item = {} if clave_dim in ambiguos else (relaciones.get(clave_dim) or {})

        def valor_columna(columna):
            if columna is None or columna >= df.shape[1]:
                return None
            return df.iloc[i, columna]

        supervisor = texto_excel(valor_columna(col_supervisor)) or texto_excel(dim_item.get('supervisor'))
        canal = normalizar_canal(valor_columna(col_canal)) or texto_excel(dim_item.get('canal')).upper()
        segmento = texto_excel(valor_columna(col_segmento)) or texto_excel(dim_item.get('segmento'))
        dni = texto_excel(valor_columna(col_dni)) or texto_excel(dim_item.get('dni'))
        fecha_inicio = texto_excel(valor_columna(col_fecha_inicio)) or texto_excel(dim_item.get('fecha_inicio'))
        estado = texto_excel(valor_columna(col_estado)).upper() or texto_excel(dim_item.get('estado')).upper()

        base.append({
            'alias_crr': alias,
            'dni': dni,
            'fecha_inicio': fecha_inicio,
            'supervisor': supervisor,
            'estado': estado,
            'segmento': segmento,
            'cartera': segmento,
            'canal': canal,
            'alcance': numero_seguro(valor_columna(col_alcance), 0.0),
            'recupero': numero_seguro(valor_columna(col_recupero), 0.0),
            'meta': numero_seguro(valor_columna(col_meta), 0.0),
            'q_alc': ''
        })

    return base



def extraer_cumpleaños_de_inf(df_inf):
    """Extrae datos de cumpleaños específicamente de la hoja INF"""
    
    datos_cumpleaños = []

    def devolver_cumpleaños_unicos(registros):
        """Conserva un solo cumpleaños por asesor, sin alterar el histórico INF."""
        unicos = {}
        for item in registros:
            alias = str(item.get('alias') or item.get('nombre') or '').strip()
            clave = normalizar_clave_asesor(alias)
            if clave and clave not in unicos:
                unicos[clave] = item
        return list(unicos.values())

    def norm_header(valor):
        import unicodedata
        texto = '' if valor is None else str(valor)
        texto = unicodedata.normalize('NFKD', texto)
        texto = ''.join(c for c in texto if not unicodedata.combining(c))
        return texto.upper().strip().replace(' ', '')

    encabezados = {
        norm_header(valor): i
        for i, valor in enumerate(df_inf.iloc[0])
        if pd.notna(valor)
    } if len(df_inf) else {}

    col_fecha_nac = encabezados.get('FECHANACIMIENTO') or encabezados.get('FECNAC')
    if col_fecha_nac is not None:
        meses_nombre = {
            1: 'ENERO', 2: 'FEBRERO', 3: 'MARZO', 4: 'ABRIL',
            5: 'MAYO', 6: 'JUNIO', 7: 'JULIO', 8: 'AGOSTO',
            9: 'SETIEMBRE', 10: 'OCTUBRE', 11: 'NOVIEMBRE', 12: 'DICIEMBRE'
        }
        col_nombre_nuevo = encabezados.get('NOMBRECOMPLETO')
        col_alias_nuevo = encabezados.get('ALIAS')
        col_puesto_nuevo = encabezados.get('PUESTO')
        col_puesto_real_nuevo = encabezados.get('PUESTOREAL')
        col_vigente = encabezados.get('VIGENCIA') if encabezados.get('VIGENCIA') is not None else encabezados.get('VIGENTE')

        for idx in range(1, len(df_inf)):
            row = df_inf.iloc[idx]
            vigente = ''
            if col_vigente is not None and col_vigente < len(row):
                vigente = str(row[col_vigente]).strip().upper() if pd.notna(row[col_vigente]) else ''
                if vigente != 'SI':
                    continue

            fecha = pd.to_datetime(row[col_fecha_nac], errors='coerce', dayfirst=True)
            if pd.isna(fecha):
                continue

            nombre = str(row[col_nombre_nuevo]).strip() if col_nombre_nuevo is not None and col_nombre_nuevo < len(row) and pd.notna(row[col_nombre_nuevo]) else ''
            alias = str(row[col_alias_nuevo]).strip() if col_alias_nuevo is not None and col_alias_nuevo < len(row) and pd.notna(row[col_alias_nuevo]) else nombre
            puesto = str(row[col_puesto_nuevo]).strip().upper() if col_puesto_nuevo is not None and col_puesto_nuevo < len(row) and pd.notna(row[col_puesto_nuevo]) else 'ASESOR'
            puesto_real = str(row[col_puesto_real_nuevo]).strip() if col_puesto_real_nuevo is not None and col_puesto_real_nuevo < len(row) and pd.notna(row[col_puesto_real_nuevo]) else alias

            datos_cumpleaños.append({
                'alias': alias,
                'puesto_real': puesto_real,
                'nombre': nombre,
                'puesto': 'ASESOR',
                'vigencia': vigente,
                'mes': meses_nombre.get(int(fecha.month), ''),
                'dia': int(fecha.day)
            })

        return devolver_cumpleaños_unicos(datos_cumpleaños)
    
    # POSICIONES FIJAS - según tu estructura exacta
    col_alias = 0       # Columna A - Alias (se mantiene para referencia)
    col_puesto_real = 1 # Columna B - PUESTO REAL (NUEVO)
    col_nombre = 3      # Columna D - Nombre completo
    col_puesto = 4      # Columna E - PUESTO (ASESOR/STAFF)
    col_mes = 5         # Columna F - Mes
    col_dia = 6         # Columna G - Día
    
    # Saltar 2 encabezados: grupo y FEC_NAC
    fila_inicio = 2  # Fila 3 en Excel (0-indexed: 0=header1, 1=header2, 2=primer dato)
    
    for idx in range(fila_inicio, len(df_inf)):
        row = df_inf.iloc[idx]
        
        # Extracción directa
        alias = str(row[col_alias])
        puesto_real = str(row[col_puesto_real]) if col_puesto_real < len(row) else ""  # NUEVO
        nombre = str(row[col_nombre])
        puesto = str(row[col_puesto])  # Ya viene como "ASESOR" o "STAFF"
        mes = str(row[col_mes])        # Ya viene como "ENERO", "FEBRERO", etc.
        dia_valor = row[col_dia]
        if hasattr(dia_valor, 'day'):
            dia = int(dia_valor.day)
        else:
            fecha_dia = pd.to_datetime(dia_valor, errors='coerce', dayfirst=True)
            dia = int(fecha_dia.day) if pd.notna(fecha_dia) else int(dia_valor)
        
        # Crear registro - AGREGAR puesto_real
        cumple_data = {
            'alias': alias,
            'puesto_real': puesto_real,  # NUEVO
            'nombre': nombre,
            'puesto': puesto,
            'mes': mes,
            'dia': dia
        }
        
        datos_cumpleaños.append(cumple_data)
    
    return devolver_cumpleaños_unicos(datos_cumpleaños)

def extraer_registro_asesores_inf(df_inf):
    """Extrae el registro historico completo de asesores desde INF para la vista ASESOR."""
    if df_inf is None or len(df_inf) == 0:
        return []

    def norm_header(valor):
        import unicodedata
        texto = '' if valor is None else str(valor)
        texto = unicodedata.normalize('NFKD', texto)
        texto = ''.join(c for c in texto if not unicodedata.combining(c))
        return texto.upper().strip().replace(' ', '')

    def valor_texto(row, col):
        if col is None or col >= len(row) or pd.isna(row[col]):
            return ''
        valor = row[col]
        if isinstance(valor, float) and valor.is_integer():
            return str(int(valor))
        return str(valor).strip()

    def formatear_fecha_aaaammdd(valor):
        if valor is None or pd.isna(valor):
            return ''
        texto = str(valor).strip()
        if texto.endswith('.0'):
            texto = texto[:-2]
        texto = ''.join(ch for ch in texto if ch.isdigit())
        if len(texto) == 8:
            try:
                fecha = pd.to_datetime(texto, format='%Y%m%d', errors='coerce')
                if pd.notna(fecha):
                    return fecha.strftime('%d/%m/%Y')
            except Exception:
                pass
        fecha = pd.to_datetime(valor, errors='coerce', dayfirst=True)
        return fecha.strftime('%d/%m/%Y') if pd.notna(fecha) else ''

    headers = {
        norm_header(valor): i
        for i, valor in enumerate(df_inf.iloc[0])
        if pd.notna(valor)
    }

    columnas = {
        'nombre_completo': headers.get('NOMBRECOMPLETO'),
        'alias': headers.get('ALIAS'),
        'documento': headers.get('DOCUMENTO'),
        'vigencia': headers.get('VIGENCIA') if headers.get('VIGENCIA') is not None else headers.get('VIGENTE'),
        'id_fecha_ingreso': headers.get('IDFECHAINGRESO'),
        'id_fecha_salida': headers.get('IDFECHASALIDA'),
        'fecha_nacimiento': headers.get('FECHANACIMIENTO'),
        'vigente': headers.get('VIGENTE'),
        'cesado': headers.get('CESADO'),
        'dias_permanencia': headers.get('DIASPERMANENCIA'),
        'meses_permanencia': headers.get('MESESPERMANENCIA'),
        'rn': headers.get('RN')
    }

    registros = []
    for idx in range(1, len(df_inf)):
        row = df_inf.iloc[idx]
        alias = valor_texto(row, columnas['alias'])
        nombre = valor_texto(row, columnas['nombre_completo'])
        if not alias and not nombre:
            continue
        registros.append({
            'nombre_completo': nombre,
            'alias': alias,
            'documento': valor_texto(row, columnas['documento']),
            'vigencia': valor_texto(row, columnas['vigencia']).upper(),
            'id_fecha_ingreso': valor_texto(row, columnas['id_fecha_ingreso']),
            'fecha_ingreso': formatear_fecha_aaaammdd(row[columnas['id_fecha_ingreso']]) if columnas['id_fecha_ingreso'] is not None and columnas['id_fecha_ingreso'] < len(row) else '',
            'id_fecha_salida': valor_texto(row, columnas['id_fecha_salida']),
            'fecha_salida': formatear_fecha_aaaammdd(row[columnas['id_fecha_salida']]) if columnas['id_fecha_salida'] is not None and columnas['id_fecha_salida'] < len(row) else '',
            'fecha_nacimiento': formatear_fecha_aaaammdd(row[columnas['fecha_nacimiento']]) if columnas['fecha_nacimiento'] is not None and columnas['fecha_nacimiento'] < len(row) else '',
            'vigente': valor_texto(row, columnas['vigente']).upper(),
            'cesado': valor_texto(row, columnas['cesado']).upper(),
            'dias_permanencia': valor_texto(row, columnas['dias_permanencia']),
            'meses_permanencia': valor_texto(row, columnas['meses_permanencia']),
            'rn': valor_texto(row, columnas['rn'])
        })

    return registros

def agrupar_por_supervisor(todos_los_datos):
    """Agrupa supervisores desde la Vista Consolidada, sin leer la tabla de supervisores."""
    datos_por_supervisor = {}

    for mes, asesores in todos_los_datos.items():
        supervisores_mes = {}

        for asesor in asesores:
            supervisor = str(asesor.get('supervisor', 'Sin Supervisor')).strip() or 'Sin Supervisor'
            bucket = supervisores_mes.setdefault(supervisor, {
                'asesores': [],
                'recupero_supervisor': 0,
                'meta_super': 0,
                'interno_supervisor': 0,
                'cartera': 'No definida',
                'canal': str(asesor.get('canal', 'SURCO')).strip().upper() or 'SURCO',
                'datos_diarios_supervisor': {},
                'datos_diarios_interno_supervisor': {}
            })
            bucket['asesores'].append(asesor)
            bucket['recupero_supervisor'] += float(asesor.get('recupero') or 0)
            bucket['meta_super'] += float(asesor.get('meta') or 0)
            if bucket['cartera'] == 'No definida':
                cartera = str(asesor.get('cartera', '')).strip()
                if cartera and cartera != 'No definida':
                    bucket['cartera'] = cartera

        for supervisor, datos_super in supervisores_mes.items():
            if supervisor not in datos_por_supervisor:
                datos_por_supervisor[supervisor] = {}

            meta_super = datos_super['meta_super']
            recupero_supervisor = datos_super['recupero_supervisor']
            alcance_supervisor = (recupero_supervisor / meta_super) * 100 if meta_super > 0 else 0

            datos_por_supervisor[supervisor][mes] = {
                'asesores': datos_super['asesores'],
                'total_recupero': recupero_supervisor,
                'total_meta': meta_super,
                'meta_super': meta_super,
                'interno_supervisor': datos_super.get('interno_supervisor', 0),
                'porcentaje_promedio': alcance_supervisor,
                'porcentaje_vs_meta_super': alcance_supervisor,
                'cartera': datos_super['cartera'],
                'canal': datos_super.get('canal', 'SURCO'),
                'cantidad_asesores': len(datos_super['asesores']),
                'distribucion_categorias': {">100%": 0, ">70%": 0, ">40%": 0, ">0%": 0},
                'datos_diarios_supervisor': {},
                'datos_diarios_interno_supervisor': {},
                'alcance_acumulado_diario': {}
            }

            datos_mes = datos_por_supervisor[supervisor][mes]
            for asesor in datos_super['asesores']:
                clasificacion = asesor.get('clasificacion', '>0%')
                if clasificacion in datos_mes['distribucion_categorias']:
                    datos_mes['distribucion_categorias'][clasificacion] += 1

            datos_mes['clasificacion'] = determinar_clasificacion(alcance_supervisor)

    return datos_por_supervisor

def normalizar_clave_asesor(valor):
    """Normaliza nombres/alias para cruzar tablas aunque varien espacios o acentos."""
    import unicodedata
    texto = '' if valor is None else str(valor)
    texto = unicodedata.normalize('NFKD', texto)
    texto = ''.join(c for c in texto if not unicodedata.combining(c))
    return ' '.join(texto.upper().strip().split())


def normalizar_canal(valor):
    """Estandariza canales conocidos y conserva cualquier canal futuro."""
    if valor is None or pd.isna(valor):
        return ''
    texto = str(valor).upper().strip()
    if texto in {'', 'NAN', 'NONE'}:
        return ''
    if 'BPO' in texto:
        return 'BPO'
    if 'SURCO' in texto:
        return 'SURCO'
    return texto

def convertir_periodo_dim(valor):
    meses = {
        1: 'ENERO', 2: 'FEBRERO', 3: 'MARZO', 4: 'ABRIL',
        5: 'MAYO', 6: 'JUNIO', 7: 'JULIO', 8: 'AGOSTO',
        9: 'SETIEMBRE', 10: 'OCTUBRE', 11: 'NOVIEMBRE', 12: 'DICIEMBRE'
    }
    texto = str(valor or '').strip().upper()
    m = re.match(r'^(\d{4})[_-](\d{1,2})$', texto)
    if m:
        anio = m.group(1)
        mes = meses.get(int(m.group(2)), '')
        return f'{mes}_{anio}' if mes else ''
    if '_' in texto:
        return texto
    return ''



def cargar_mapas_dim_canales(ruta_excel_dim):
    """Carga DIM con claves estrictas por periodo y alias, sin fallback temporal."""
    mapas = {
        'por_periodo_alias': {},
        'periodos': set(),
        'ambiguos_periodo_alias': set()
    }

    try:
        df_dim = pd.read_excel(ruta_excel_dim, sheet_name='DIM', header=0)
    except Exception as error_dim:
        print(f"  ⚠️ No se pudo leer DIM desde {ruta_excel_dim}: {error_dim}")
        return mapas

    headers = {normalizar_clave_dimension(col): col for col in df_dim.columns}
    col_periodo = headers.get('FECHA') or headers.get('FECHAFORMAT')
    col_alias = headers.get('ASESOR') or headers.get('ALIAS')
    col_supervisor = headers.get('EQUIPO') or headers.get('SUPER') or headers.get('SUPER2')
    col_segmento = headers.get('SEGMENTO')
    col_canal = headers.get('CANAL')
    col_dni = headers.get('DNI') or headers.get('DOCUMENTO')
    col_fecha_inicio = headers.get('FECHADEINGRESO') or headers.get('FECHAINGRESO')
    col_estado = headers.get('CONDICION') or headers.get('ESTADO')

    if not col_periodo or not col_alias or not col_canal:
        print("  ⚠️ DIM requiere FECHA, ASESOR/ALIAS y CANAL")
        return mapas

    for _, row in df_dim.iterrows():
        periodo = convertir_periodo_dim(row.get(col_periodo, ''))
        alias = texto_excel(row.get(col_alias, ''))
        clave_alias = normalizar_clave_asesor(alias)
        canal = normalizar_canal(row.get(col_canal, ''))
        if not periodo or not clave_alias or not canal:
            continue

        mapas['periodos'].add(periodo)
        clave = (periodo, clave_alias)
        item = {
            'canal': canal,
            'supervisor': texto_excel(row.get(col_supervisor, '')) if col_supervisor else '',
            'segmento': texto_excel(row.get(col_segmento, '')) if col_segmento else '',
            'dni': texto_excel(row.get(col_dni, '')) if col_dni else '',
            'fecha_inicio': texto_excel(row.get(col_fecha_inicio, '')) if col_fecha_inicio else '',
            'estado': texto_excel(row.get(col_estado, '')).upper() if col_estado else ''
        }

        if clave in mapas['ambiguos_periodo_alias']:
            continue
        existente = mapas['por_periodo_alias'].get(clave)
        if existente is None:
            mapas['por_periodo_alias'][clave] = item
        elif existente != item:
            mapas['por_periodo_alias'].pop(clave, None)
            mapas['ambiguos_periodo_alias'].add(clave)

    print(
        f"DIM cargada: {len(df_dim)} filas, {len(mapas['periodos'])} periodos, "
        f"{len(mapas['por_periodo_alias'])} relaciones exactas"
    )
    if mapas['ambiguos_periodo_alias']:
        print(
            f"  ⚠️ DIM: {len(mapas['ambiguos_periodo_alias'])} relaciones periodo+asesor "
            "son contradictorias y serán excluidas"
        )
    return mapas



def extraer_indicadores_calidad(df, nombre_hoja=None):
    """Agrupa los indicadores IND_ diarios por asesor usando la hoja ya filtrada por periodo."""

    def norm_header(valor):
        import unicodedata
        texto = '' if valor is None else str(valor)
        texto = unicodedata.normalize('NFKD', texto)
        texto = ''.join(c for c in texto if not unicodedata.combining(c))
        texto = texto.upper().strip()
        texto = texto.replace('% ', '%')
        texto = texto.replace('  ', ' ')
        return texto

    # SQL entrega los encabezados en df.columns; Excel con header=None los trae como una fila.
    headers_columnas = [norm_header(col) for col in df.columns]
    encabezados_desde_columnas = (
        'IND_ALIAS' in headers_columnas
        and ('IND_FECHA' in headers_columnas or any(h.startswith('IND_') for h in headers_columnas))
    )

    fila_encabezados = None
    if not encabezados_desde_columnas:
        for idx_header in range(len(df)):
            headers_row = [norm_header(v) for v in df.iloc[idx_header]]
            if 'IND_ALIAS' in headers_row and ('IND_FECHA' in headers_row or any(h.startswith('IND_') for h in headers_row)):
                fila_encabezados = idx_header
                break
        if fila_encabezados is None:
            return {}

    def formatear_fecha_indicador(valor):
        if valor is None or pd.isna(valor):
            return ''
        try:
            if hasattr(valor, 'strftime'):
                return valor.strftime('%d/%m/%Y')
            if isinstance(valor, (int, float)) and not isinstance(valor, bool):
                fecha_num = pd.to_datetime(valor, unit='D', origin='1899-12-30', errors='coerce')
                if pd.notna(fecha_num):
                    return fecha_num.strftime('%d/%m/%Y')
            fecha = pd.to_datetime(valor, errors='coerce', dayfirst=True)
            if pd.notna(fecha):
                return fecha.strftime('%d/%m/%Y')
        except Exception:
            pass
        return str(valor).strip()

    if encabezados_desde_columnas:
        encabezados = {norm_header(valor): i for i, valor in enumerate(df.columns)}
        start_data_row = 0
    else:
        row_headers = df.iloc[fila_encabezados]
        encabezados = {}
        for i, valor in enumerate(row_headers):
            if pd.notna(valor):
                encabezados[norm_header(valor)] = i
        start_data_row = fila_encabezados + 1

    definiciones = {
        'puntualidad': ('IND_%puntualidad', ['IND_%PUNTUALIDAD']),
        'falta': ('IND_FALTA', ['IND_FALTA']),
        'tardanza': ('IND_TARDE', ['IND_TARDE']),
        'tardanza_tiempo': ('IND_TardanzaTiempo', ['IND_TARDANZATIEMPO', 'IND_TARDANZA TIEMPO']),
        'cef_unico': ('IND_CEFUnico', ['IND_CEFUNICO', 'IND_CEF UNICO']),
        'pdp': ('IND_PDP', ['IND_PDP']),
        'cierre': ('IND_Cierre', ['IND_CIERRE']),
        'monto_pdp': ('IND_MontoPDP', ['IND_MONTOPDP', 'IND_MONTO PDP']),
        'dk': ('IND_DK', ['IND_DK']),
        'pago_condonado': ('IND_PagoCondonado', ['IND_PAGOCONDONADO', 'IND_PAGO CONDONADO']),
        'condonacion': ('IND_Condonacion', ['IND_CONDONACION']),
        'recupero': ('IND_recupero', ['IND_RECUPERO']),
        'ticket_pdp': ('IND_Ticket_PDP', ['IND_TICKET_PDP', 'IND_TICKET PDP']),
        'calidad_pdp': ('IND_Calidad PDP', ['IND_CALIDAD PDP'])
    }

    indicadores_porcentaje = {'puntualidad', 'cierre', 'condonacion', 'calidad_pdp'}
    indicadores_binarios = {'falta', 'tardanza'}
    indicadores_tiempo = {'tardanza_tiempo'}

    columnas = {
        'alias': encabezados.get('IND_ALIAS'),
        'fecha': encabezados.get('IND_FECHA')
    }
    for key, (_, aliases) in definiciones.items():
        columnas[key] = next((encabezados[a] for a in aliases if a in encabezados), None)

    if columnas['alias'] is None:
        return {}

    def leer_valor_indicador(row, key):
        col = columnas.get(key)
        if col is None or col >= len(row) or pd.isna(row.iloc[col]):
            return None
        try:
            return float(row.iloc[col])
        except Exception:
            return None

    def leer_binario_indicador(row, key):
        col = columnas.get(key)
        if col is None or col >= len(row) or pd.isna(row.iloc[col]):
            return None
        texto = str(row.iloc[col]).strip().upper()
        if texto in {'SI', 'SÍ', 'S', 'YES', 'Y', 'TRUE', '1', 'X'}:
            return 1.0
        if texto in {'NO', 'N', 'FALSE', '0'}:
            return 0.0
        try:
            return 1.0 if float(row.iloc[col]) > 0 else 0.0
        except Exception:
            return None

    def normalizar_porcentaje(valor):
        if valor is None:
            return None
        num = float(valor)
        return num * 100.0 if abs(num) <= 2 else num

    acumulado = {}
    for idx in range(start_data_row, len(df)):
        row = df.iloc[idx]
        col_alias = columnas['alias']
        if col_alias >= len(row) or pd.isna(row.iloc[col_alias]):
            continue
        alias = str(row.iloc[col_alias]).strip()
        clave = normalizar_clave_asesor(alias)
        if not clave:
            continue
        item = acumulado.setdefault(clave, {
            'alias': alias,
            'dias_registrados': 0,
            'indicadores': {
                k: {'suma': 0.0, 'n': 0, 'max': 0.0}
                for k in definiciones
            },
            'detalle_diario': []
        })
        item['dias_registrados'] += 1
        col_fecha = columnas.get('fecha')
        fecha_raw = row.iloc[col_fecha] if col_fecha is not None and col_fecha < len(row) and pd.notna(row.iloc[col_fecha]) else None
        fecha_dia = formatear_fecha_indicador(fecha_raw) if fecha_raw is not None else str(item['dias_registrados'])
        fecha_orden = pd.to_datetime(fecha_raw, errors='coerce', dayfirst=True) if fecha_raw is not None else pd.NaT
        if pd.isna(fecha_orden) and isinstance(fecha_raw, (int, float)) and not isinstance(fecha_raw, bool):
            fecha_orden = pd.to_datetime(fecha_raw, unit='D', origin='1899-12-30', errors='coerce')
        detalle_dia = {
            'fecha': fecha_dia,
            'fecha_orden': fecha_orden.strftime('%Y-%m-%d') if pd.notna(fecha_orden) else '',
            'indicadores': {}
        }
        tiene_dato_dia = False
        for key in definiciones:
            if key in indicadores_binarios:
                valor = leer_binario_indicador(row, key)
                if valor is None:
                    continue
                valor = 1.0 if valor > 0 else 0.0
            else:
                valor = leer_valor_indicador(row, key)
                if valor is None:
                    continue
                if key in indicadores_porcentaje:
                    valor = normalizar_porcentaje(valor)
                valor = round(valor, 9) if key in indicadores_tiempo else round(valor, 2)
            item['indicadores'][key]['suma'] += valor
            item['indicadores'][key]['n'] += 1
            item['indicadores'][key]['max'] = max(item['indicadores'][key]['max'], valor)
            detalle_dia['indicadores'][key] = valor
            tiene_dato_dia = True
        if tiene_dato_dia:
            item['detalle_diario'].append(detalle_dia)

    resultado = {}
    for clave, item in acumulado.items():
        indicadores = {}
        for key, datos in item['indicadores'].items():
            if datos['n']:
                if key in indicadores_binarios:
                    valor_final = datos['max']
                elif key in indicadores_tiempo:
                    valor_final = datos['suma']
                else:
                    valor_final = datos['suma'] / datos['n']
                indicadores[key] = {
                    'valor': round(valor_final, 9) if key in indicadores_tiempo else round(valor_final, 2),
                    'registros': datos['n'],
                    'columna': definiciones[key][0]
                }
        resultado[clave] = {
            'alias': item['alias'],
            'dias_registrados': item['dias_registrados'],
            'indicadores': indicadores,
            'detalle_diario': sorted(item.get('detalle_diario', []), key=lambda d: d.get('fecha_orden') or '')
        }
    return resultado


def extraer_datos_completos(
    df, nombre_hoja, mapas_dim_canales=None, df_indicadores=None, base_asesores=None
):
    """Construye la estructura que consume generar_html_modular."""
    fuente_indicadores = df_indicadores if df_indicadores is not None else df
    indicadores_calidad_por_alias = extraer_indicadores_calidad(fuente_indicadores, nombre_hoja)
    if indicadores_calidad_por_alias:
        print(f"  Indicadores: {len(indicadores_calidad_por_alias)} asesores con data diaria")

    if base_asesores is None:
        base_asesores = extraer_base_asesores_desde_b2(df, nombre_hoja, mapas_dim_canales)

    base_asesores_por_clave = {
        normalizar_clave_asesor(item.get('alias_crr', '')): item
        for item in base_asesores
        if normalizar_clave_asesor(item.get('alias_crr', ''))
    }

    asesores_data = []
    for base_asesor_ref in base_asesores_por_clave.values():
        nombre_asesor = str(base_asesor_ref.get('alias_crr', '')).strip()
        if not nombre_asesor:
            continue

        estado_asesor = str(base_asesor_ref.get('estado', '')).strip().upper()
        canal_asesor = str(base_asesor_ref.get('canal', '')).strip().upper()
        if not estado_asesor or not canal_asesor:
            continue
        if estado_asesor != 'CALL':
            continue

        porcentaje_valor = convertir_porcentaje_excel(base_asesor_ref.get('alcance', 0))
        supervisor_nombre = str(base_asesor_ref.get('supervisor', '')).strip() or 'Sin Supervisor'
        asesor_data = {
            'nombre': nombre_asesor,
            'porcentaje': round(porcentaje_valor, 2),
            'clasificacion': determinar_clasificacion(round(porcentaje_valor, 2)),
            'q_alc': str(base_asesor_ref.get('q_alc', '')).strip(),
            'alias_crr': nombre_asesor,
            'dni': str(base_asesor_ref.get('dni', '')).strip(),
            'fecha_inicio': str(base_asesor_ref.get('fecha_inicio', '')).strip(),
            'supervisor': supervisor_nombre,
            'canal': canal_asesor,
            'recupero': float(base_asesor_ref.get('recupero') or 0),
            'meta': float(base_asesor_ref.get('meta') or 0),
            'datos_diarios_asesor': {}
        }

        clave_calidad_nombre = normalizar_clave_asesor(nombre_asesor)
        asesor_data['indicadores_calidad'] = indicadores_calidad_por_alias.get(clave_calidad_nombre, {})
        asesor_data['meta_super'] = float(base_asesor_ref.get('meta') or 0)
        asesor_data['recupero_supervisor'] = float(base_asesor_ref.get('recupero') or 0)
        asesor_data['alcance_supervisor'] = round(porcentaje_valor, 2)
        asesor_data['cartera'] = str(
            base_asesor_ref.get('cartera') or base_asesor_ref.get('segmento') or ''
        ).strip() or 'No definida'
        asesor_data['segmento'] = asesor_data['cartera']
        asesores_data.append(asesor_data)

    return {'asesores': asesores_data, 'supervisores': {}}


def determinar_clasificacion(porcentaje):
    """Determina la clasificación según el porcentaje"""
    if porcentaje > 100:
        return ">100%"
    elif porcentaje > 70:
        return ">70%"
    elif porcentaje > 40:
        return ">40%"
    else:
        return ">0%"

def convertir_porcentaje_excel(valor_crudo):
    """Convierte porcentajes del Excel: 0.0142 => 1.42 y '1.42%' => 1.42."""
    try:
        if valor_crudo is None or pd.isna(valor_crudo):
            return 0
        texto = str(valor_crudo).strip()
        if texto == '':
            return 0
        texto = texto.replace(',', '.')
        if texto.endswith('%'):
            return float(texto[:-1].strip())
        return float(texto) * 100
    except Exception:
        return 0

def generar_seccion_evaluacion(opciones_mes_periodo, opciones_año_periodo, opciones_asesores_periodo):
    return f"""
    
    <!-- CONTROLES (EVALUACIÓN) -->
    <div class="controles-grafica" id="controlesEvaluacionMeses">

        <button type="button" class="btn-comparacion activo" id="btnEval3Meses">
        3 MESES
        </button>
        
        <button type="button" class="btn-comparacion" id="btnEval6Meses">
        6 MESES
        </button>
        
        <button type="button" class="btn-comparacion" id="btnEval12Meses">
          12 MESES
        </button>

        <button type="button" class="btn-comparacion" id="btnEvalOtros">
          OTROS
        </button>

      <div class="grupo-checkbox-eval">
          <label class="checkbox-incluir-mes" id="wrapChkIncluirMesSeleccionadoEval">
            <input type="checkbox" id="chkIncluirMesSeleccionadoEval" />
            <span>INCLUIR MES SELECCIONADO</span>
          </label>
        
          <label class="checkbox-incluir-mes" id="wrapChkMostrarAsesoresNuevosEval">
            <input type="checkbox" id="chkMostrarAsesoresNuevosEval" checked />
            <span>MOSTRAR ASESORES NUEVOS</span>
          </label>
      </div>

      <button
        type="button"
        class="boton-busqueda"
        id="btnEvalExportarAnalisis"
        style="background: linear-gradient(135deg, #9b59b6, #8e44ad);"
      >
        📊 EXPORTAR ANÁLISIS
      </button>

      <button
          type="button"
          class="boton-busqueda"
          id="btnEvalExportarRecuperoMetas"
          style="background: linear-gradient(135deg, #27ae60, #2ecc71);"
        >
          💾 EXPORTAR RyM
      </button>

    </div>
    
    <!-- TARJETAS RÁPIDAS (EVALUACIÓN) -->
    <div style="padding: 15px 10px;">
        <h2 id="evalResumenTitulo" style="margin: 0 0 25px 0; color: #2c3e50; text-align: center; font-size: 1.8rem; font-weight: 700; letter-spacing: 0.5px;">
          Evaluación por Promedios
        </h2>
    
      <div class="clasificaciones" style="grid-template-columns: repeat(auto-fit, minmax(340px, 1fr));">
        
        <!-- >=70% -->
        <div class="clasificacion">
          <div class="clasificacion-header clasificacion-100">
            ✅ >= 70%
            <span style="font-weight:400; opacity:0.9;">(</span>
            <span id="cantidad-eval-ok" style="font-weight:800;">0</span>
            <span style="font-weight:400; opacity:0.9;">)</span>
          </div>
          <div class="asesores-lista" id="lista-eval-ok"></div>
        </div>
    
        <!-- <70% -->
        <div class="clasificacion">
          <div class="clasificacion-header clasificacion-0">
            ❌ &lt; 70%
            <span style="font-weight:400; opacity:0.9;">(</span>
            <span id="cantidad-eval-bad" style="font-weight:800;">0</span>
            <span style="font-weight:400; opacity:0.9;">)</span>
          </div>
          <div class="asesores-lista" id="lista-eval-bad"></div>
        </div>
    
      </div>
    
      <div id="evalResumenNota" style="text-align:center; font-size: 0.9rem; color:#666; margin-top: 10px;">
        —
      </div>

      <!-- MODAL DETALLE (EVALUACIÓN) -->

      <div id="modalEvalDetalle" class="modal-eval" style="display:none;">
          <div class="modal-eval-backdrop" data-close="1"></div>
        
          <div class="modal-eval-card" role="dialog" aria-modal="true" aria-labelledby="modalEvalTitulo">
            <div class="modal-eval-header">
              <div>
                <div id="modalEvalTitulo" style="font-size:1.15rem; font-weight:800; color:#2c3e50;">
                  Detalle de Evaluación
                </div>
                <div id="modalEvalSub" style="font-size:0.9rem; color:#666; margin-top:4px;">
                  —
                </div>
              </div>
        
              <button type="button" class="modal-eval-close" data-close="1">✕</button>
            </div>
        
            <div class="modal-eval-body">
              <div id="modalEvalReglas" style="background:#f6f8fb; border:1px solid #e6eaf0; padding:12px; border-radius:12px; color:#2c3e50;">
                —
              </div>
        
              <div style="margin-top:12px; overflow:auto;">
                <table class="tabla-eval-detalle">
                    <thead>
                      <tr>
                        <th>Mes</th>
                        <th>Estado</th>
                        <th style="text-align:right;">Meta</th>
                        <th style="text-align:right;">Recupero</th>
                        <th style="text-align:right;">Alcance</th>
                        <th>¿Incluido?</th>
                        <th><span class="eval-quintil-head-wrap">Quintil <button type="button" class="eval-quintil-extra-btn" id="btnEvalQuintilesExtra" onclick="toggleEvalQuintilesExtra()">+</button></span></th>
                        <th class="eval-quintil-extra-col oculta">Condonacion</th>
                        <th class="eval-quintil-extra-col oculta">Cierre</th>
                        <th class="eval-quintil-extra-col oculta">Calidad</th>
                        <th class="eval-quintil-extra-col oculta">Puntualidad</th>
                      </tr>
                    </thead>
                  <tbody id="modalEvalTbody">
                    <tr><td colspan="11" style="padding:14px; text-align:center; color:#666;">—</td></tr>
                  </tbody>
                </table>
              </div>
        
              <div id="modalEvalTotales" style="margin-top:12px; display:flex; gap:10px; flex-wrap:wrap;">
                —
              </div>
            </div>
          </div>
      </div>

      <!-- QUINTILES DE EVALUACION CALCULADOS DESDE LOS PROMEDIOS -->
      <div id="seccionEvalQuintil" style="margin-top:18px;">
      <div class="eval-quintil-toolbar">
        <button type="button" class="btn-exportar-quintiles" id="btnEvalExportarQuintiles">Exportar quintiles</button>
      </div>
      <div id="evalQuintilCards" style="display:grid; grid-template-columns: repeat(5, minmax(180px, 1fr)); gap:12px; align-items:start;">
        <div class="eval-quintil-card eval-quintil-q5" data-quintil="5" onclick="toggleEvalQuintil(5)">
          <div class="eval-quintil-title"><span>Q5</span><span class="eval-quintil-toggle" id="evalQuintilToggleQ5">+</span></div>
          <div class="eval-quintil-summary" id="evalQuintilResumenQ5"></div>
          <div class="eval-quintil-list colapsada" id="evalQuintilQ5" onclick="event.stopPropagation()"></div>
        </div>
        <div class="eval-quintil-card eval-quintil-q4" data-quintil="4" onclick="toggleEvalQuintil(4)">
          <div class="eval-quintil-title"><span>Q4</span><span class="eval-quintil-toggle" id="evalQuintilToggleQ4">+</span></div>
          <div class="eval-quintil-summary" id="evalQuintilResumenQ4"></div>
          <div class="eval-quintil-list colapsada" id="evalQuintilQ4" onclick="event.stopPropagation()"></div>
        </div>
        <div class="eval-quintil-card eval-quintil-q3" data-quintil="3" onclick="toggleEvalQuintil(3)">
          <div class="eval-quintil-title"><span>Q3</span><span class="eval-quintil-toggle" id="evalQuintilToggleQ3">+</span></div>
          <div class="eval-quintil-summary" id="evalQuintilResumenQ3"></div>
          <div class="eval-quintil-list colapsada" id="evalQuintilQ3" onclick="event.stopPropagation()"></div>
        </div>
        <div class="eval-quintil-card eval-quintil-q2" data-quintil="2" onclick="toggleEvalQuintil(2)">
          <div class="eval-quintil-title"><span>Q2</span><span class="eval-quintil-toggle" id="evalQuintilToggleQ2">+</span></div>
          <div class="eval-quintil-summary" id="evalQuintilResumenQ2"></div>
          <div class="eval-quintil-list colapsada" id="evalQuintilQ2" onclick="event.stopPropagation()"></div>
        </div>
        <div class="eval-quintil-card eval-quintil-q1" data-quintil="1" onclick="toggleEvalQuintil(1)">
          <div class="eval-quintil-title"><span>Q1</span><span class="eval-quintil-toggle" id="evalQuintilToggleQ1">+</span></div>
          <div class="eval-quintil-summary" id="evalQuintilResumenQ1"></div>
          <div class="eval-quintil-list colapsada" id="evalQuintilQ1" onclick="event.stopPropagation()"></div>
        </div>
      </div>
      <div id="evalQuintilNota" style="text-align:center; font-size:0.9rem; color:#666; margin-top:10px;">-</div>
      </div>

    </div>

    <!-- PERIODO NUEVOS -->
    <div class="seccion-busqueda">
        <h2>Evaluación de nuevos ingresos</h2>
        <div class="selectores-periodo-prueba" style="grid-template-columns: repeat(47, 1fr); align-items:end;">
            <div class="selector-item" style="grid-column: span 8;">
                <label for="porcentajeMinimo" class="selector-label">&#127919; % M&iacute;nimo</label>
                <input type="number" id="porcentajeMinimo" placeholder="Ej: 50" value="50" min="0" max="100" step="1" oninput="calcularPeriodoPrueba()">
            </div>

            <div class="selector-item" style="grid-column: span 13;">
              <label class="selector-label">&nbsp;</label>
              <label class="checkbox-incluir-mes" style="height:43px; display:flex; align-items:center; gap:8px; padding:0 12px; border:1px solid #ddd; border-radius:10px; background:#fff; font-weight:700; color:#2c3e50; cursor:pointer; user-select:none;">
                <input type="checkbox" id="chkOmitirPrimerMesPeriodo" onchange="calcularPeriodoPrueba()" />
                <span>OMITIR PRIMER MES</span>
              </label>
              <label class="checkbox-incluir-mes" style="height:43px; display:flex; align-items:center; gap:8px; padding:0 12px; border:1px solid #ddd; border-radius:10px; background:#fff; font-weight:700; color:#2c3e50; cursor:pointer; user-select:none; margin-top:8px;">
                <input type="checkbox" id="chkFiltrarPeriodoCumplido" onchange="calcularPeriodoPrueba()" />
                <span>FILTRAR PERIODO CUMPLIDO</span>
              </label>
            </div>

            <div class="selector-item" style="grid-column: span 13;">
              <label for="selectorFechaIngresoDesde" class="selector-label">&#127381; Desde</label>
              <input
                type="date"
                id="selectorFechaIngresoDesde"
                onchange="calcularPeriodoPrueba()"
                style="padding:10px 12px; border-radius:10px; border:1px solid #ddd; width:100%;"/>
            </div>
            
            <div class="selector-item" style="grid-column: span 13;">
              <label for="selectorFechaIngresoHasta" class="selector-label">&#127381; Hasta</label>
              <input
                type="date"
                id="selectorFechaIngresoHasta"
                onchange="calcularPeriodoPrueba()"
                style="padding:10px 12px; border-radius:10px; border:1px solid #ddd; width:100%;"/>
            </div>
        </div>

        <div id="resultadosPeriodo" style="display: block; margin-top: 30px;">
            <div id="tablaPeriodo"></div>
        </div>
    </div>
    </div>
    """

def generar_html_modular(todos_los_datos, ruta_html, meses_unicos, años_unicos, meses_validos,
                         lista_asesores, datos_por_supervisor, lista_supervisores, datos_cumpleaños_por_mes=None,
                         base_asesores_analisis=None, datos_capa=None, datos_mundial=None, datos_asesores_inf=None):
    """Genera el HTML con estructura modular de 4 secciones"""
    if base_asesores_analisis is None:
        base_asesores_analisis = {}
    if datos_cumpleaños_por_mes is None:
        datos_cumpleaños_por_mes = {}
    if datos_mundial is None:
        datos_mundial = {"asesores": [], "equipos": [], "paises": [], "banderas": {}}
    if datos_asesores_inf is None:
        datos_asesores_inf = []
    
    # Determinar mes y año actual más reciente
    mes_actual, año_actual = obtener_mes_año_actual(meses_validos)
    
    # Obtener datos del mes actual
    clave_actual = f"{mes_actual}_{año_actual}"
    asesores_data = todos_los_datos.get(clave_actual, [])
    asesores_data.sort(key=lambda x: x['porcentaje'], reverse=True)
    
    # Contar asesores por categoría
    contadores = {
        ">100%": len([a for a in asesores_data if a['clasificacion'] == ">100%"]),
        ">70%": len([a for a in asesores_data if a['clasificacion'] == ">70%"]),
        ">40%": len([a for a in asesores_data if a['clasificacion'] == ">40%"]),
        ">0%": len([a for a in asesores_data if a['clasificacion'] == ">0%"])
    }

    # Crear opciones de los selectores
    opciones_mes = ""
    for mes in meses_unicos:
        selected = "selected" if mes == mes_actual else ""
        opciones_mes += f'<option value="{mes}" {selected}>{mes.title()}</option>\n'
    
    opciones_año = ""
    for año in años_unicos:
        selected = "selected" if año == año_actual else ""
        opciones_año += f'<option value="{año}" {selected}>{año}</option>\n'

    
    # Generar opciones para selects de periodo de prueba
    opciones_mes_periodo = "".join(
        [f'<option value="{mes}">{mes.title()}</option>\n' for mes in meses_unicos]
    )
    
    opciones_año_periodo = "".join(
        [f'<option value="{año}">{año}</option>\n' for año in años_unicos]
    )
    
    # Datalist de asesores (usar la lista global, no solo mes actual)
    opciones_asesores_periodo = "".join(
        [f'<option value="{asesor}"></option>\n' for asesor in lista_asesores]
    )
    
    # Crear opciones para el autocompletado de asesores
    opciones_asesores = "".join([f'<option value="{asesor}">{asesor}</option>' for asesor in lista_asesores])
    
    # Crear opciones para el autocompletado y selector de supervisores
    opciones_supervisores = "".join([f'<option value="{supervisor}">' for supervisor in lista_supervisores])
    opciones_supervisores_select = "".join([f'<option value="{supervisor}">{supervisor}</option>' for supervisor in lista_supervisores])
    segmentos_supervisores = []
    opciones_segmentos_select = ""

    # Crear una versión serializable de los datos
    datos_para_js = {}
    for mes, asesores in todos_los_datos.items():
        datos_para_js[mes] = []
        for asesor in asesores:
            datos_asesor = {
                'nombre': asesor.get('nombre', ''),
                'porcentaje': asesor.get('porcentaje', 0),
                'clasificacion': asesor.get('clasificacion', '>0%'),
                'q_alc': asesor.get('q_alc', ''),
                'supervisor': asesor.get('supervisor', 'Sin Supervisor'),
                'canal': asesor.get('canal', 'SURCO'),
                'recupero': asesor.get('recupero', 0),
                'meta': asesor.get('meta', 0),
                'meta_super': asesor.get('meta_super', 0),
                'recupero_supervisor': asesor.get('recupero_supervisor', 0),
                'alcance_supervisor': asesor.get('alcance_supervisor', 0),
                'cartera': asesor.get('cartera', 'No definida'),
                'segmento': asesor.get('segmento', asesor.get('cartera', 'No definida')),
                'datos_diarios_asesor': asesor.get('datos_diarios_asesor', {}),
                'indicadores_calidad': asesor.get('indicadores_calidad', {}),
                'alias_crr': asesor.get('alias_crr', ''),
                'dni': asesor.get('dni', ''),
                'fecha_inicio': asesor.get('fecha_inicio', '')
            }
            datos_para_js[mes].append(datos_asesor)
    
    # Convertir a JSON
    datos_json_str = json.dumps(datos_para_js, default=str, ensure_ascii=False)
    datos_supervisores_json_str = json.dumps(datos_por_supervisor, default=str, ensure_ascii=False)
    base_asesores_json = json.dumps(base_asesores_analisis, ensure_ascii=False)
    datos_cumpleaños_json_str = json.dumps(datos_cumpleaños_por_mes, default=str, ensure_ascii=False)
    datos_mundial_json_str = json.dumps(datos_mundial, default=str, ensure_ascii=False)
    datos_asesores_inf_json_str = json.dumps(datos_asesores_inf, default=str, ensure_ascii=False)
    
    # ESCAPAR para JavaScript
    datos_json_str_escaped = datos_json_str.replace('\\', '\\\\').replace('`', '\\`').replace('${', '\\${')
    datos_supervisores_json_str_escaped = datos_supervisores_json_str.replace('\\', '\\\\').replace('`', '\\`').replace('${', '\\${')
    base_asesores_json_escaped = base_asesores_json.replace('\\', '\\\\').replace("'", "\\'")
    datos_cumpleaños_json_str_escaped = datos_cumpleaños_json_str.replace('\\', '\\\\').replace('`', '\\`').replace('${{', '\\${{')
    datos_mundial_json_str_escaped = datos_mundial_json_str.replace('\\', '\\\\').replace('`', '\\`').replace('${{', '\\${{')
    datos_asesores_inf_json_str_escaped = datos_asesores_inf_json_str.replace('\\', '\\\\').replace('`', '\\`').replace('${{', '\\${{')
    
    # Generar contenido para cada sección
    contenido_ranking = generar_seccion_ranking(asesores_data, contadores, opciones_asesores, opciones_supervisores, opciones_supervisores_select, opciones_segmentos_select, meses_unicos, años_unicos)
    contenido_evaluacion = generar_seccion_evaluacion(opciones_mes_periodo, opciones_año_periodo, opciones_asesores_periodo)

    ahora = datetime.now()
    
    meses_es = [
        "enero","febrero","marzo","abril","mayo","junio",
        "julio","agosto","setiembre","octubre","noviembre","diciembre"
    ]
    
    fecha_actualizacion = (
        f"{ahora.day:02d} de {meses_es[ahora.month-1]} "
        f"de {ahora.year} a las {ahora.strftime('%H:%M')}"
    )
    
    html_content = rf'''<!DOCTYPE html>
<html lang="es">
<head>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/exceljs/4.4.0/exceljs.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/FileSaver.js/2.0.5/FileSaver.min.js"></script>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Sistema de Gestión de Desempeño</title>
    <style>
        :root {{
            --verde-oscuro: #1b5e20;
            --verde-claro: #c8e6c9;
            --amarillo-oscuro: #FCCF10;
            --amarillo-claro: #FEF6D2;
            --naranja-oscuro: #e65100;
            --naranja-claro: #ffe0b2;
            --rojo-oscuro: #b71c1c;
            --rojo-claro: #ffcdd2;
            --azul-oscuro: #1565c0;
            --azul-claro: #e3f2fd;
            --gris: #f5f5f5;
            --azul-principal: #2c3e50;
            --azul-secundario: #34495e;
            --verde-principal: #27ae60;
            --naranja-principal: #e67e22;
            --purpura-principal: #9b59b6;
        }}
        
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        }}
        
        body {{
            background-color: #E7E9ED;
            color: #333;
            padding: 20px;
        }}
        
        .container {{
            max-width: 1400px;
            margin: 0 auto;
            background: transparent;
        }}
        
        /* ========== HEADER ========== */
        header {{
            text-align: center;
            margin-bottom: 30px;
            padding: 25px;
            background: linear-gradient(135deg, var(--azul-principal), var(--azul-secundario));
            color: white;
            border-radius: 15px;
            box-shadow: 0 6px 15px rgba(0, 0, 0, 0.15);
        }}
        
        h1 {{
            font-size: 2.8rem;
            margin-bottom: 15px;
            letter-spacing: 1px;
        }}
        
        .selectores-periodo {{
            margin: 25px 0;
            text-align: center;
        }}
        
        .selectores-periodo select {{
            padding: 12px 20px;
            font-size: 1.2rem;
            border: 2px solid #4a6491;
            border-radius: 10px;
            background-color: white;
            cursor: pointer;
            margin: 0 15px;
            min-width: 180px;
        }}
        
        .fecha-actualizacion {{
            font-size: 1.1rem;
            margin-bottom: 40px;
            opacity: 0.95;
            background: rgba(255, 255, 255, 0.1);
            padding: 10px 20px;
            border-radius: 25px;
            display: inline-block;
        }}
        
        .accesos-secundarios {{
            display: flex;
            justify-content: center;
            gap: 10px;
            margin: -22px 0 22px;
            flex-wrap: wrap;
        }}

        .boton-acceso-secundario {{
            border: 1px solid rgba(255,255,255,0.35);
            background: rgba(255,255,255,0.12);
            color: white;
            padding: 8px 14px;
            border-radius: 999px;
            font-size: 0.9rem;
            font-weight: 700;
            cursor: pointer;
            transition: all 0.2s ease;
        }}

        .boton-acceso-secundario:hover,
        .boton-acceso-secundario.activo {{
            background: rgba(255,255,255,0.24);
            transform: translateY(-1px);
        }}
        
        /* ========== MENÚ PRINCIPAL ========== */
        .menu-principal {{
            display: grid;
            grid-template-columns: repeat(4, minmax(210px, 1fr));
            gap: 10px;
            margin: 10px 0;
            max-width: 1300px;
            margin-left: auto;
            margin-right: auto;
            padding: 0 60px;
        }}
        
        .boton-modulo {{
            background: white;
            border: none;
            border-radius: 15px;
            padding: 20px 10px;
            cursor: pointer;
            transition: all 0.3s ease;
            display: flex;
            flex-direction: column;
            align-items: center;
            justify-content: center;
            box-shadow: 0 5px 15px rgba(0, 0, 0, 0.1);
            position: relative;
            overflow: hidden;
            width: 100%;
            max-width: 350px;
            margin: 0 auto;
            transform-origin: center;
            will-change: transform;
        }}

        .boton-modulo::before {{
            content: '';
            position: absolute;
            top: 0;
            left: 0;
            right: 0;
            height: 5px;
            background: linear-gradient(90deg, var(--verde-principal), var(--azul-principal), var(--naranja-principal), var(--purpura-principal));
        }}
        
        .boton-modulo:hover {{
            transform: translateY(-10px);
            box-shadow: 0 15px 30px rgba(0, 0, 0, 0.2);
        }}
        
        .boton-modulo.activo {{
            background: linear-gradient(135deg, var(--azul-principal), var(--azul-secundario));
            color: white;
            transform: translateY(-6px) scale(1.04);
            box-shadow: 0 18px 38px rgba(0, 0, 0, 0.28);
            animation: pulsoModuloActivo 1.8s ease-in-out infinite;
        }}

        .boton-modulo.activo::after {{
            content: '';
            position: absolute;
            inset: 8px;
            border-radius: 12px;
            border: 2px solid rgba(255, 255, 255, 0.45);
            pointer-events: none;
        }}
        
        .icono-modulo {{
            font-size: 3.5rem;
            margin-bottom: 20px;
        }}
        
        .titulo-modulo {{
            font-size: 1.8rem;
            font-weight: 700;
            margin-bottom: 10px;
        }}
        
        .descripcion-modulo {{
            font-size: 1rem;
            opacity: 0.8;
            text-align: center;
            max-width: 250px;
        }}
        
        /* COLORES POR MÓDULO */
        .boton-modulo.ranking {{
            border-top: 5px solid var(--verde-principal);
        }}
        
        .boton-modulo.ranking.activo {{
            background: linear-gradient(135deg, var(--verde-principal), #2ecc71);
        }}

        .boton-modulo.alcances {{
            border-top: 5px solid var(--purpura-principal);
        }}
        
        .boton-modulo.alcances.activo {{
            background: linear-gradient(135deg, var(--purpura-principal), #8e44ad);
        }}
        
        .boton-modulo.evaluacion {{
            border-top: 5px solid var(--naranja-principal);
        }}
        
        .boton-modulo.evaluacion.activo {{
            background: linear-gradient(135deg, var(--naranja-principal), #f39c12);
        }}

        .boton-modulo.calidad {{
            border-top: 5px solid var(--azul-principal);
        }}

        .boton-modulo.calidad.activo {{
            background: linear-gradient(135deg, var(--azul-principal), #3498db);
        }}

        .boton-modulo.gestiones {{
            border-top: 5px solid var(--purpura-principal);
        }}

        .boton-modulo.gestiones.activo {{
            background: linear-gradient(135deg, var(--purpura-principal), #8e44ad);
        }}

        .canales-alcances {{
            display: flex;
            justify-content: center;
            gap: 10px;
            flex-wrap: wrap;
            margin: 0 0 14px 0;
        }}

        .btn-canal-alcances {{
            border: 1px solid #d8dfe8;
            background: #ffffff;
            color: #233142;
            border-radius: 8px;
            padding: 10px 16px;
            font-weight: 900;
            cursor: pointer;
            box-shadow: 0 5px 14px rgba(16, 24, 40, 0.08);
            transition: transform 0.18s ease, box-shadow 0.18s ease, background 0.18s ease;
        }}

        .btn-canal-alcances:hover {{
            transform: translateY(-1px);
            box-shadow: 0 8px 18px rgba(16, 24, 40, 0.12);
        }}

        .btn-canal-alcances.activo {{
            background: linear-gradient(135deg, #155bb5, #3498db);
            color: #ffffff;
            border-color: transparent;
        }}

        .accesos-secundarios .canales-alcances {{
            margin: 0;
        }}

        .accesos-secundarios .btn-canal-alcances {{
            padding: 8px 14px;
            border-radius: 999px;
            box-shadow: none;
        }}

        /* ========== SECCIONES DE CONTENIDO ========== */
        .seccion-contenido {{
            display: none;
            background: #F8F9FA;
            padding: 10px;
            border-radius: 15px;
            box-shadow: 0 8px 30px rgba(0, 0, 0, 0.35);
            margin-top: 30px;
            animation: fadeIn 0.5s ease;
        }}
        
        .seccion-contenido.activa {{
            display: block;
        }}
        
        .panel-seccion-vacia {{
            padding: 36px 24px;
            text-align: center;
            color: #2c3e50;
            background: white;
            border: 1px solid #e6eaf0;
            border-radius: 12px;
        }}

        .panel-seccion-vacia h2 {{
            margin: 0 0 8px;
            font-size: 1.5rem;
        }}

        .panel-seccion-vacia p {{
            margin: 0;
            color: #667085;
        }}

        .calidad-dashboard {{ padding: 18px 10px 28px; }}
        .calidad-resumen {{ display: none; }}

        .calidad-stats-grid {{
            display: grid;
            grid-template-columns: repeat(4, minmax(180px, 1fr));
            gap: 15px;
            margin: 0 0 18px;
        }}

        .calidad-stat-card {{
            text-align: center;
            padding: 18px;
            border-radius: 10px;
            border: 2px solid var(--stat-color);
            background: var(--stat-bg);
        }}

        .calidad-stat-card .stat-label {{
            font-size: 0.95rem;
            color: var(--stat-color);
            margin-bottom: 6px;
            font-weight: 800;
        }}

        .calidad-stat-card .stat-value {{
            font-size: 1.55rem;
            font-weight: 900;
            color: var(--stat-color);
            white-space: nowrap;
        }}

        .calidad-stat-card.stat-dias .stat-value,
        .calidad-stat-card.stat-alcance .stat-value {{
            font-size: 2rem;
        }}

        .calidad-stat-card .stat-note {{
            font-size: 0.85rem;
            color: #666;
            margin-top: 5px;
        }}

        .stat-dias {{ --stat-color: #3498db; --stat-bg: #E8F4FD; }}
        .stat-meta {{ --stat-color: #9b59b6; --stat-bg: #F4ECF7; }}
        .stat-recupero {{ --stat-color: #27ae60; --stat-bg: #E8F6F3; }}
        .stat-alcance {{ --stat-color: #3498db; --stat-bg: #D1E9FB; }}

        .asesor-dashboard {{
            background: #ffffff;
            border: 1px solid #dfe5ee;
            border-radius: 8px;
            padding: 18px;
            box-shadow: 0 5px 16px rgba(16, 24, 40, 0.08);
        }}

        .asesor-topbar {{
            display: grid;
            grid-template-columns: minmax(280px, 1fr) auto;
            gap: 14px;
            align-items: end;
            margin-bottom: 16px;
        }}

        .asesor-titulo h2 {{
            margin: 0 0 4px;
            color: #233142;
            font-size: 1.55rem;
        }}

        .asesor-titulo p {{
            margin: 0;
            color: #667085;
            font-weight: 700;
        }}

        .asesor-search {{
            display: flex;
            gap: 8px;
            align-items: center;
            justify-content: flex-end;
        }}

        .asesor-search input {{
            min-width: 320px;
            height: 42px;
            border: 1px solid #cfd8e3;
            border-radius: 7px;
            padding: 0 12px;
            font: inherit;
            font-weight: 700;
            color: #233142;
        }}

        .asesor-search button,
        .asesor-rangos button {{
            height: 42px;
            border: 0;
            border-radius: 7px;
            padding: 0 14px;
            background: #233142;
            color: #ffffff;
            font-weight: 900;
            cursor: pointer;
        }}

        .asesor-rangos {{
            display: flex;
            gap: 8px;
            flex-wrap: wrap;
            margin: 0 0 14px;
        }}

        .asesor-rangos button {{
            background: #eef2f7;
            color: #344054;
            border: 1px solid #d6dde8;
        }}

        .asesor-rangos button.activo {{
            background: #1565c0;
            color: #ffffff;
            border-color: #1565c0;
        }}

        .asesor-ficha-grid {{
            display: grid;
            grid-template-columns: repeat(6, minmax(120px, 1fr));
            gap: 8px;
            margin-bottom: 12px;
        }}

        .asesor-ficha-card,
        .asesor-kpi-card {{
            border: 1px solid #e3e8ef;
            border-radius: 8px;
            background: #f8fafc;
            padding: 9px 10px;
            min-width: 0;
        }}

        .asesor-kpi-card {{
            display: grid;
            grid-template-columns: minmax(0, 1fr) auto 8px;
            column-gap: 9px;
            align-items: center;
        }}

        .asesor-kpi-main {{
            min-width: 0;
        }}

        .asesor-ficha-card span,
        .asesor-kpi-card span {{
            display: block;
            color: #667085;
            font-size: 0.68rem;
            font-weight: 800;
            text-transform: uppercase;
            margin-bottom: 3px;
            white-space: nowrap;
        }}

        .asesor-ficha-card strong,
        .asesor-kpi-card strong {{
            color: #233142;
            font-size: 0.9rem;
            font-weight: 900;
            line-height: 1.15;
            overflow-wrap: anywhere;
        }}

        .asesor-kpi-grid {{
            display: grid;
            grid-template-columns: repeat(5, minmax(150px, 1fr));
            gap: 10px;
            margin-bottom: 14px;
        }}

        .asesor-kpi-card strong {{
            font-size: 1.28rem;
        }}

        .asesor-panel-grid {{
            display: grid;
            grid-template-columns: minmax(420px, 1.15fr) minmax(300px, 0.55fr);
            gap: 14px;
            align-items: stretch;
            margin-bottom: 14px;
        }}

        .asesor-chart-card,
        .asesor-tendencia-card {{
            border: 1px solid #e3e8ef;
            border-radius: 8px;
            padding: 14px;
            background: #ffffff;
        }}

        .asesor-chart-wrap {{
            position: relative;
            width: 100%;
            height: 460px;
            min-height: 460px;
        }}

        .asesor-chart-card canvas,
        .asesor-metrica-chart canvas {{
            display: block;
            width: 100% !important;
            height: 100% !important;
        }}

        .asesor-tendencia-card h3 {{
            margin: 0 0 10px;
            color: #233142;
            font-size: 1.1rem;
        }}

        .asesor-tendencia-list {{
            display: grid;
            gap: 8px;
        }}

        .asesor-tendencia-item {{
            display: flex;
            justify-content: space-between;
            gap: 10px;
            border-bottom: 1px solid #eef2f7;
            padding-bottom: 8px;
            color: #344054;
            font-weight: 800;
        }}

        .asesor-tendencia-label {{
            display: inline-flex;
            align-items: center;
            gap: 8px;
            color: #233142;
            white-space: nowrap;
        }}

        .asesor-tendencia-item strong {{
            color: #233142;
        }}

        .asesor-tendencia-item strong.tendencia-mejora {{
            color: #16803a;
        }}

        .asesor-tendencia-item strong.tendencia-retroceso {{
            color: #c0392b;
        }}

        .asesor-analisis-final {{
            margin-top: 14px;
            border: 1px solid #dfe5ee;
            border-radius: 8px;
            padding: 12px;
            background: #f8fafc;
            display: grid;
            gap: 7px;
            cursor: pointer;
            user-select: none;
        }}

        .asesor-analisis-head {{
            display: grid;
            grid-template-columns: 1fr auto;
            align-items: center;
            gap: 10px;
        }}

        .asesor-analisis-title {{
            display: grid;
            gap: 4px;
        }}

        .asesor-analisis-toggle {{
            width: 28px;
            height: 28px;
            border-radius: 50%;
            display: inline-grid;
            place-items: center;
            background: rgba(35, 49, 66, 0.08);
            color: #233142;
            font-size: 1.05rem;
            font-weight: 950;
        }}

        .asesor-analisis-final span {{
            color: #667085;
            font-size: 0.76rem;
            font-weight: 900;
            text-transform: uppercase;
        }}

        .asesor-analisis-final strong {{
            font-size: 1.35rem;
            font-weight: 950;
            color: #233142;
        }}

        .asesor-analisis-final small {{
            color: #526071;
            font-weight: 800;
            line-height: 1.35;
        }}

        .asesor-analisis-detalle {{
            display: none;
            gap: 7px;
            cursor: default;
        }}

        .asesor-analisis-final.abierto .asesor-analisis-detalle {{
            display: grid;
        }}

        .asesor-analisis-list {{
            display: grid;
            gap: 6px;
            margin-top: 3px;
        }}

        .asesor-analisis-list div {{
            display: grid;
            grid-template-columns: 96px 1fr;
            gap: 8px;
            padding-top: 6px;
            border-top: 1px solid #e8eef5;
        }}

        .asesor-analisis-list b {{
            color: #233142;
            display: inline-flex;
            align-items: center;
            gap: 7px;
            white-space: nowrap;
        }}

        .asesor-metrica-dot {{
            width: 9px;
            height: 9px;
            border-radius: 50%;
            display: inline-block;
            flex: 0 0 9px;
        }}

        .asesor-analisis-list em {{
            color: #667085;
            font-style: normal;
            font-weight: 750;
        }}

        .asesor-analisis-productivo {{
            border-color: #bfe6cf;
            background: #f4fbf6;
        }}

        .asesor-analisis-decadente {{
            border-color: #f4c7c3;
            background: #fff7f6;
        }}

        .asesor-analisis-inestable {{
            border-color: #f3dfaa;
            background: #fffaf0;
        }}

        .asesor-kpi-percentil {{
            display: contents;
        }}

        .asesor-kpi-percentil span {{
            color: #233142;
            font-size: 1.28rem;
            font-weight: 900;
            line-height: 1.15;
            white-space: nowrap;
            align-self: center;
            justify-self: end;
        }}

        .asesor-kpi-percentil-bar {{
            width: 7px;
            height: 48px;
            border-radius: 999px;
            background: linear-gradient(0deg, #d64545 0%, #f3c74f 50%, #229954 100%);
            position: relative;
            overflow: hidden;
            align-self: center;
        }}

        .asesor-kpi-percentil-bar::after {{
            content: '';
            position: absolute;
            left: 0;
            right: 0;
            bottom: var(--pct, 0%);
            height: 3px;
            background: #233142;
            box-shadow: 0 0 0 1px rgba(255,255,255,0.7);
        }}

        .tabla-valor-comparado {{
            display: inline-flex;
            align-items: center;
            gap: 6px;
            font-weight: 850;
        }}

        .tabla-triangulo {{
            width: 0;
            height: 0;
            display: inline-block;
        }}

        .tabla-triangulo.arriba {{
            border-left: 5px solid transparent;
            border-right: 5px solid transparent;
            border-bottom: 9px solid #16803a;
        }}

        .tabla-triangulo.abajo {{
            border-left: 5px solid transparent;
            border-right: 5px solid transparent;
            border-top: 9px solid #c0392b;
        }}

        .asesor-tabla-wrap {{
            overflow: auto;
            border: 1px solid #dfe5ee;
            border-radius: 8px;
            max-height: 420px;
        }}

        .asesor-tabla {{
            width: 100%;
            border-collapse: separate;
            border-spacing: 0;
            min-width: 520px;
        }}

        .asesor-tabla th {{
            position: sticky;
            top: 0;
            background: #233142;
            color: #ffffff;
            padding: 10px;
            text-align: right;
            z-index: 2;
        }}

        .asesor-tabla th:first-child,
        .asesor-tabla td:first-child {{
            text-align: left;
        }}

        .asesor-tabla td {{
            padding: 9px 10px;
            border-bottom: 1px solid #edf1f6;
            text-align: right;
            font-weight: 700;
            color: #344054;
        }}

        .asesor-metricas-grid {{
            display: grid;
            gap: 14px;
            margin-top: 14px;
        }}

        .asesor-metrica-card {{
            border: 1px solid #dfe5ee;
            border-radius: 8px;
            background: #ffffff;
            padding: 14px;
        }}

        .asesor-metrica-head {{
            display: flex;
            justify-content: space-between;
            align-items: center;
            gap: 12px;
            margin-bottom: 12px;
        }}

        .asesor-metrica-head h3 {{
            margin: 0;
            color: #233142;
            font-size: 1.05rem;
            font-weight: 900;
        }}

        .asesor-metrica-head strong {{
            color: #155bb5;
            font-size: 1.1rem;
        }}

        .asesor-metrica-layout {{
            display: grid;
            grid-template-columns: minmax(300px, 0.9fr) minmax(360px, 1.1fr);
            gap: 14px;
            align-items: stretch;
        }}

        .asesor-metrica-chart {{
            position: relative;
            height: 260px;
            min-height: 260px;
        }}

        @media (max-width: 1180px) {{
            .asesor-ficha-grid {{
                grid-template-columns: repeat(3, minmax(140px, 1fr));
            }}
            .asesor-metrica-layout {{
                grid-template-columns: 1fr;
            }}
        }}

        .asesor-empty {{
            text-align: center;
            color: #667085;
            font-weight: 800;
            padding: 30px;
            border: 1px dashed #cfd8e3;
            border-radius: 8px;
            background: #f8fafc;
        }}

        .calidad-tabla-wrap {{
            background: #ffffff;
            border: 1px solid #dfe5ee;
            border-radius: 8px;
            overflow: auto;
            box-shadow: 0 5px 16px rgba(16, 24, 40, 0.08);
            max-height: 68vh;
        }}

        .calidad-quintiles {{
            display: grid;
            gap: 18px;
            margin: 14px 0;
        }}

        .calidad-quintil-bloque {{
            border: 1px solid #dfe5ee;
            border-radius: 8px;
            background: #ffffff;
            padding: 14px;
        }}

        .calidad-quintil-bloque h3 {{
            margin: 0 0 12px;
            color: #233142;
            font-size: 1.08rem;
            font-weight: 950;
        }}

        .calidad-quintil-cards {{
            display: grid;
            grid-template-columns: repeat(5, minmax(150px, 1fr));
            gap: 10px;
            align-items: start;
        }}

        .calidad-quintil-nota {{
            text-align: center;
            color: #667085;
            font-size: 0.86rem;
            font-weight: 750;
            margin-top: 10px;
        }}

        .calidad-tabla {{
            width: 100%;
            min-width: 1120px;
            border-collapse: separate;
            border-spacing: 0;
            font-size: 0.92rem;
        }}

        .calidad-tabla th {{
            position: sticky;
            top: 0;
            z-index: 8;
            background: #233142;
            color: #ffffff;
            text-align: left;
            height: 56px;
            padding: 7px 8px;
            font-weight: 800;
            white-space: nowrap;
            line-height: 1;
            border-right: 1px solid rgba(255,255,255,0.12);
            vertical-align: middle;
            box-shadow: 0 2px 0 rgba(16, 24, 40, 0.16);
        }}

        .calidad-tabla .indicador-principal-col,
        .calidad-tabla th.indicador-principal-col {{
            background: #233142;
            color: #ffffff;
            font-weight: 900;
            padding: 6px 8px;
        }}

        .indicador-head-actions {{
            display: grid;
            grid-template-columns: minmax(108px, 1fr) 34px;
            align-items: stretch;
            gap: 6px;
            height: 38px;
        }}

        .indicador-sort {{
            appearance: none;
            border: 1px solid color-mix(in srgb, var(--indicador-color, #526071), #ffffff 42%);
            border-radius: 6px;
            padding: 0 10px;
            min-width: 118px;
            height: 40px;
            background: var(--indicador-color, #526071);
            color: #ffffff;
            font: inherit;
            font-size: 1rem;
            font-weight: 1000;
            cursor: pointer;
            white-space: nowrap;
            transition: background 0.2s ease, border-color 0.2s ease, transform 0.2s ease;
        }}

        .indicador-sort::after {{
            content: "";
        }}

        .indicador-sort:hover,
        .indicador-sort.activo {{
            background: var(--indicador-color, #526071);
            border-color: rgba(255,255,255,0.48);
            filter: brightness(1.08);
            transform: translateY(-1px);
        }}

        .calidad-tabla .indicador-grupo {{
            text-align: center;
            min-width: 138px;
            background: #233142;
            color: #ffffff;
            font-size: 1rem;
            transition: background 0.24s ease, box-shadow 0.24s ease;
        }}

        .calidad-tabla .indicador-grupo.expandido {{
            text-align: center;
            box-shadow: inset 0 -3px 0 rgba(255,255,255,0.38);
        }}

        .calidad-tabla .indicador-grupo.expandido .indicador-toggle {{
            text-align: center;
            justify-content: center;
        }}

        .calidad-tabla .indicador-grupo.ordenado {{
            box-shadow: inset 0 -3px 0 rgba(255,255,255,0.38);
        }}

        .indicador-grupo.indicador-puntualidad {{ --indicador-color: #219653; }}
        .indicador-grupo.indicador-cierre {{ --indicador-color: #155bb5; }}
        .indicador-grupo.indicador-condonacion {{ --indicador-color: #8e44ad; }}
        .indicador-grupo.indicador-calidad_pdp {{ --indicador-color: #008c8c; }}
        .indicador-grupo .indicador-toggle {{ --toggle-color: var(--indicador-color, #526071); }}

        .calidad-tabla td {{
            padding: 9px 10px;
            border-bottom: 1px solid #edf1f6;
            color: #263445;
            vertical-align: middle;
            background: #ffffff;
            transition: background 0.22s ease, opacity 0.22s ease;
        }}

        .calidad-tabla tbody tr:nth-child(even) td {{
            background: #f9fbfd;
        }}

        .calidad-tabla tbody tr:hover td {{
            background: #eef6ff;
        }}

        .calidad-tabla th:first-child,
        .calidad-tabla td:first-child {{
            position: sticky;
            left: 0;
            z-index: 6;
            background-clip: padding-box;
        }}

        .calidad-tabla th:first-child {{
            z-index: 8;
            background: #233142;
        }}

        .calidad-tabla td:first-child {{
            font-weight: 800;
            color: #1f2d3d;
            background: #ffffff;
            min-width: 220px;
            box-shadow: 8px 0 12px rgba(16, 24, 40, 0.08);
        }}

        .calidad-tabla tbody tr:nth-child(even) td:first-child {{
            background: #f9fbfd;
        }}

        .calidad-tabla tbody tr:hover td:first-child,
        .calidad-main-row.abierta td:first-child {{
            background: #eef6ff !important;
        }}

        .calidad-tabla .col-num {{
            text-align: right;
            font-variant-numeric: tabular-nums;
        }}

        .calidad-tabla .calidad-bar-cell {{
            min-width: 150px;
        }}

        .barra-dato {{
            --valor: 0%;
            --bar-color: #b8c2cc;
            position: relative;
            display: flex;
            align-items: center;
            justify-content: flex-end;
            width: 100%;
            min-width: 118px;
            height: 28px;
            padding: 0 10px;
            border-radius: 7px;
            overflow: hidden;
            background: #f4f6f8;
            color: #1f2d3d;
            font-weight: 900;
            font-variant-numeric: tabular-nums;
            box-shadow: inset 0 0 0 1px rgba(16, 24, 40, 0.06);
        }}

        .barra-dato::before {{
            content: "";
            position: absolute;
            inset: 0 auto 0 0;
            width: var(--valor);
            background: var(--bar-color);
            opacity: 0.26;
        }}

        .barra-dato span {{
            position: relative;
            z-index: 1;
        }}

        .barra-dato.alcance {{ --bar-color: #f1c40f; }}
        .barra-dato.puntualidad {{ --bar-color: #27ae60; }}
        .barra-dato.cierre {{ --bar-color: #1565c0; }}
        .barra-dato.condonacion {{ --bar-color: #8e44ad; }}
        .barra-dato.calidad-pdp {{ --bar-color: #00a6a6; }}
        .barra-dato.neutral {{ --bar-color: #b8c2cc; color: #667085; }}


        .indicador-toggle {{
            --toggle-color: #526071;
            appearance: none;
            width: 34px;
            height: 40px;
            border: 1px solid color-mix(in srgb, var(--toggle-color), #ffffff 42%);
            border-radius: 6px;
            padding: 0;
            background: var(--toggle-color);
            color: #ffffff;
            font: inherit;
            font-size: 1.25rem;
            font-weight: 900;
            cursor: pointer;
            text-align: center;
            white-space: nowrap;
            line-height: 1;
            transition: background 0.2s ease, transform 0.2s ease;
        }}

        .indicador-toggle:hover {{
            filter: brightness(1.06);
            transform: translateY(-1px);
        }}

        .detalle-col {{
            display: none;
            opacity: 0;
            min-width: 92px;
            max-width: none;
            text-align: center;
            white-space: nowrap;
            font-size: 0.78rem;
            font-weight: 800;
            padding-left: 6px !important;
            padding-right: 6px !important;
        }}

        .detalle-col.visible {{
            display: table-cell;
            opacity: 1;
            animation: calidadColIn 0.22s ease both;
        }}

        .calidad-tabla td.detalle-col {{
            background: #f7fbff;
        }}

        .calidad-tabla td.detalle-puntualidad {{ background: #f4fbf6; }}
        .calidad-tabla td.detalle-cierre {{ background: #f3f7ff; }}
        .calidad-tabla td.detalle-condonacion {{ background: #faf5ff; }}
        .calidad-tabla td.detalle-calidad_pdp {{ background: #f1fbfa; }}

        .estado-ind {{
            display: inline-grid;
            place-items: center;
            width: 22px;
            height: 22px;
            border-radius: 50%;
            font-weight: 1000;
            line-height: 1;
        }}

        .estado-ok {{
            color: #16803a;
            background: #e8f7ee;
        }}

        .estado-x {{
            color: #c62828;
            background: #fdeaea;
        }}

        .hora-ind {{
            font-variant-numeric: tabular-nums;
            font-weight: 800;
        }}

        .hora-alerta {{
            color: #c62828;
            background: #fdeaea;
            border-radius: 5px;
            padding: 2px 6px;
        }}

        @keyframes calidadColIn {{
            from {{ opacity: 0; transform: translateX(-8px); }}
            to {{ opacity: 1; transform: translateX(0); }}
        }}

        .calidad-main-row {{
            cursor: pointer;
        }}

        .calidad-main-row td {{
            transition: background 0.22s ease, box-shadow 0.22s ease;
        }}

        .calidad-main-row.abierta td {{
            background: #edf8ff !important;
            box-shadow: inset 0 -1px 0 #cfe8ff;
        }}

        .calidad-asesor-toggle {{
            appearance: none;
            border: 0;
            background: transparent;
            color: inherit;
            font: inherit;
            font-weight: 900;
            display: inline-flex;
            align-items: center;
            gap: 8px;
            padding: 0;
            cursor: pointer;
            text-align: left;
        }}

        .calidad-chevron {{
            display: inline-grid;
            place-items: center;
            width: 18px;
            height: 18px;
            border-radius: 50%;
            background: #e7eef7;
            color: #233142;
            transition: transform 0.24s ease, background 0.24s ease;
            flex: 0 0 auto;
        }}

        .calidad-main-row.abierta .calidad-chevron {{
            transform: rotate(90deg);
            background: #bfe5ff;
        }}

        .calidad-dia-row {{
            display: none;
        }}

        .calidad-dia-row.abierta {{
            display: table-row;
        }}

        .calidad-dia-row td {{
            padding: 8px 14px;
            background: #fbfdff !important;
            border-bottom: 1px solid #edf3f8;
            color: #405064;
            animation: calidadDiaIn 0.26s ease both;
            animation-delay: var(--delay, 0ms);
        }}

        .calidad-main-row.abierta + .calidad-dia-row td {{
            border-top: 2px solid #9fb7d2;
        }}

        .calidad-dia-row.abierta:has(+ .calidad-main-row) td,
        .calidad-dia-row.abierta:last-child td {{
            border-bottom: 2px solid #9fb7d2;
        }}

        .calidad-dia-fecha {{
            position: sticky !important;
            left: 0;
            z-index: 1;
            min-width: 0 !important;
            font-weight: 900 !important;
            color: #344054 !important;
            background: #f4f8fc !important;
        }}

        .calidad-dia-fecha span {{
            display: inline-flex;
            align-items: center;
            min-height: 26px;
            padding: 4px 10px;
            border-radius: 6px;
            background: #e8f1fb;
            color: #233142;
            font-size: 0.9rem;
        }}

        .calidad-dia-valor {{
            font-size: 0.92rem;
            white-space: nowrap;
        }}

        .calidad-dia-row .dia-principal {{
            font-weight: 900;
            background: #f2f6fb !important;
        }}

        .calidad-dia-row .dia-puntualidad {{ background: #f7fcf9 !important; }}
        .calidad-dia-row .dia-cierre {{ background: #f6f9ff !important; }}
        .calidad-dia-row .dia-condonacion {{ background: #fbf7ff !important; }}
        .calidad-dia-row .dia-calidad_pdp {{ background: #f5fcfb !important; }}

        .calidad-dia-empty {{
            text-align: center !important;
            padding: 14px !important;
            color: #667085 !important;
            font-weight: 800 !important;
            background: #fbfdff !important;
        }}

        @keyframes calidadDiaIn {{
            from {{ opacity: 0; transform: translateY(-8px); }}
            to {{ opacity: 1; transform: translateY(0); }}
        }}

        .calidad-tabla-vacia {{
            text-align: center !important;
            color: #667085 !important;
            padding: 28px !important;
            font-weight: 700;
        }}

        
        @keyframes fadeIn {{
            from {{ opacity: 0; transform: translateY(20px); }}
            to {{ opacity: 1; transform: translateY(0); }}
        }}
        
        /* ========== ESTILOS ESPECÍFICOS POR SECCIÓN ========== */
        
        /* RANKING */

        .acciones-ranking{{
            display: flex;
            justify-content: center;
            gap: 15px;
            margin: 10px 0 25px;
            flex-wrap: wrap;
        }}
        
        .clasificaciones {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(280px, 1fr));
            gap: 25px;
            margin-bottom: 40px;
        }}
        
        .clasificacion {{
            border-radius: 12px;
            overflow: hidden;
            box-shadow: 0 5px 15px rgba(0, 0, 0, 0.1);
        }}
        
        .clasificacion-header {{
            padding: 20px;
            color: white;
            text-align: center;
            font-weight: bold;
            font-size: 1.3rem;
        }}
        
        .clasificacion-100 {{ background-color: var(--verde-oscuro); }}
        .clasificacion-70 {{ background-color: var(--amarillo-oscuro); }}
        .clasificacion-40 {{ background-color: var(--naranja-oscuro); }}
        .clasificacion-0 {{ background-color: var(--rojo-oscuro); }}
        
        .asesores-lista {{
            max-height: 400px;
            overflow-y: auto;
            background-color: white;
        }}
        
        .asesor-item {{
            padding: 15px;
            border-bottom: 1px solid #eee;
            display: flex;
            justify-content: space-between;
            align-items: center;
            transition: background-color 0.2s;
        }}
        
        .asesor-item:hover {{
            background-color: #f8f9fa;
        }}
        
        .asesor-nombre {{
            font-weight: 500;
            font-size: 1.1rem;
        }}
        
        .asesor-porcentaje {{
            font-weight: bold;
            padding: 6px 12px;
            border-radius: 20px;
            color: white;
            font-size: 1rem;
            min-width: 80px;
            text-align: center;
        }}
        
        .gradiente-100 {{ background: linear-gradient(to right, var(--verde-claro), white); }}
        .gradiente-70 {{ background: linear-gradient(to right, var(--amarillo-claro), white); }}
        .gradiente-40 {{ background: linear-gradient(to right, var(--naranja-claro), white); }}
        .gradiente-0 {{ background: linear-gradient(to right, var(--rojo-claro), white); }}
        
        .porcentaje-100 {{ background-color: var(--verde-oscuro); }}
        .porcentaje-70 {{ background-color: var(--amarillo-oscuro); }}
        .porcentaje-40 {{ background-color: var(--naranja-oscuro); }}
        .porcentaje-0 {{ background-color: var(--rojo-oscuro); }}
        
        .boton-supervisor.activo {{
            background: linear-gradient(135deg, #2ecc71, #27ae60);
            box-shadow: 0 0 0 3px white, 0 0 0 6px #2ecc71;
        }}
        
        /* Secciones de búsqueda */
        .seccion-busqueda {{
            background: white;
            padding: 30px;
            border-radius: 12px;
            box-shadow: 0 4px 10px rgba(0, 0, 0, 0.08);
            margin: 30px 0;
            border-left: 5px solid #3498db;
        }}
        
        .seccion-busqueda h2 {{
            color: #2c3e50;
            margin-bottom: 25px;
            text-align: center;
            font-size: 1.8rem;
        }}
        
        .controles-busqueda {{
            display: flex;
            gap: 15px;
            align-items: center;
            flex-wrap: wrap;
            justify-content: center;
            margin-bottom: 20px;
        }}
        
        .input-busqueda {{
            flex: 1;
            min-width: 300px;
            padding: 14px 20px;
            font-size: 1.1rem;
            border: 2px solid #ddd;
            border-radius: 10px;
            background-color: white;
            transition: border-color 0.3s;
        }}
        
        .input-busqueda:focus {{
            border-color: #3498db;
            outline: none;
            box-shadow: 0 0 0 3px rgba(52, 152, 219, 0.2);
        }}
        
        .selector-supervisor-asesores {{
            min-width: 260px;
            cursor: pointer;
        }}
        
        .boton-busqueda {{
            padding: 14px 30px;
            font-size: 1.1rem;
            background: var(--azul-principal);
            color: white;
            border: none;
            border-radius: 10px;
            cursor: pointer;
            transition: all 0.3s;
            font-weight: 600;
        }}
        
        .boton-busqueda:hover {{
            background: #0d47a1;
            transform: translateY(-2px);
            box-shadow: 0 4px 8px rgba(0,0,0,0.2);
        }}
        
        .boton-busqueda.verde {{
            background: var(--verde-principal);
        }}
        
        .boton-busqueda.rojo {{
            background: #e74c3c;
        }}
        
        .boton-busqueda.naranja {{
            background: var(--naranja-principal);
        }}
        
        /* Tags de selección */
        .elementos-seleccionados {{
            margin-top: 25px;
            display: flex;
            flex-wrap: wrap;
            gap: 12px;
        }}
        
        .tag-elemento {{
            background: var(--azul-claro);
            color: var(--azul-principal);
            padding: 10px 20px;
            border-radius: 25px;
            display: flex;
            align-items: center;
            gap: 10px;
            font-weight: 600;
            font-size: 1rem;
            box-shadow: 0 2px 5px rgba(0,0,0,0.1);
        }}
        
        .eliminar-elemento {{
            background: none;
            border: none;
            color: var(--azul-principal);
            cursor: pointer;
            font-weight: bold;
            padding: 0;
            width: 24px;
            height: 24px;
            border-radius: 50%;
            display: flex;
            align-items: center;
            justify-content: center;
            transition: all 0.2s;
        }}
        
        .eliminar-elemento:hover {{
            background: var(--azul-principal);
            color: white;
        }}
        
        /* Tablas de comparación */
        .tabla-comparacion {{
            width: 100%;
            border-collapse: collapse;
            margin-top: 20px;
            font-size: 0.95rem;
            box-shadow: 0 3px 10px rgba(0,0,0,0.08);
            border-radius: 10px;
            overflow: hidden;
            border: 1px solid #e0e0e0;
        }}
        
        .tabla-comparacion th {{
            background: var(--azul-principal);
            color: white;
            padding: 15px 10px;
            text-align: center;
            font-weight: 600;
            position: sticky;
            top: 0;
            z-index: 10;
        }}
        
        .tabla-comparacion td {{
            padding: 12px 8px;
            border-bottom: 1px solid #eee;
            text-align: center;
            vertical-align: middle;
        }}
        
        /* Estilo para fila de supervisor */
        .tabla-comparacion tr.fila-supervisor {{
            background-color: #f8f9fa;
            border-top: 1px dashed #ddd;
        }}
        
        .tabla-comparacion tr.fila-supervisor td {{
            padding: 6px 8px;
            font-size: 0.85em;
            color: #666;
            border-bottom: none;
        }}

        .tabla-comparacion tr:first-child td {{
            padding: 8px 8px !important;      /* Padding normal para fila principal */
        }}
        
        /* Estilo para separador entre asesores */

        .fila-supervisor-compacta {{
            background-color: #f8f9fa !important;
            height: 22px !important;
            line-height: 1 !important;
        }}
        
        .fila-supervisor-compacta td {{
            padding: 1px 8px 2px 8px !important;
            line-height: 1 !important;
            vertical-align: middle !important;
            border-bottom: none !important;
            font-size: 0.85em !important;
        }}
        
        .label-supervisor {{
            color: #666 !important;
            font-style: italic !important;
            padding-left: 19px !important;
            padding-right: 0 !important;
            text-align: left !important;
            width: 80px !important;
            min-width: 80px !important;
            max-width: 80px !important;
        }}
        
        /* Estilos para los textos de supervisor */
        .supervisor-normal {{
            display: inline-block;
            color: #3498db;
            font-weight: 500;
            white-space: nowrap;
            line-height: 1.2;
        }}
        
        .supervisor-jefe {{
            display: inline-block;
            color: #2ecc71;
            font-weight: 600;
            white-space: nowrap;
            line-height: 1.2;
        }}
        
        .supervisor-sin {{
            display: inline-block;
            color: #e74c3c;
            font-weight: 500;
            white-space: nowrap;
            line-height: 1.2;
        }}
        
        .supervisor-vacio {{
            color: #bbb !important;
            font-style: italic !important;
            line-height: 1.2 !important;
        }}
        
        /* Separador compacto entre asesores */
        .separador-compacto td {{
            padding: 4px 0 !important;
            background: linear-gradient(to right, #f0f2f5, #e8eaf1, #f0f2f5) !important;
            height: 6px !important;
            line-height: 1 !important;
            border: none !important;
        }}
        
        /* Asegurar consistencia en las celdas */
        .tabla-comparacion td {{
            padding: 10px 8px !important;
            line-height: 1.4 !important;
            vertical-align: middle !important;
            height: auto !important;
            min-height: 36px !important;
            box-sizing: border-box !important;
        }}
        
        /* Forzar consistencia en todas las filas */
        .tabla-comparacion tr {{
            height: auto !important;
            min-height: 36px !important;
            line-height: 1.4 !important;
        }}
        
        .tabla-comparacion tr.separador-asesor td {{
            padding: 5px 0;
            background: linear-gradient(to right, #f0f2f5, #e8eaf1, #f0f2f5);
            height: 10px;
            border: none;
        }}
        
        .tabla-comparacion tr:hover {{
            background-color: #f5f7fa;
        }}
        
        .tabla-comparacion tr.fila-supervisor:hover {{
            background-color: #f0f2f5;
        }}
        
        /* Contenedor con scroll para muchas columnas */
        .contenedor-tabla-scroll {{
            overflow-x: auto;
            max-width: 100%;
            margin: 20px 0;
            border: 1px solid #e0e0e0;
            border-radius: 8px;
        }}
        
        /* Badges para supervisores */
        .badge-supervisor {{
            display: inline-block;
            padding: 3px 10px;
            background-color: #e3f2fd;
            color: #1565c0;
            border-radius: 15px;
            font-size: 0.85em;
            font-weight: 500;
            border: 1px solid #bbdefb;
            white-space: nowrap;
          line-height: 1.08;
        }}
        
        .badge-supervisor-jefe {{
            background-color: #e8f5e9;
            color: #2e7d32;
            border-color: #c8e6c9;
        }}
        
        .badge-sin-supervisor {{
            background-color: #ffebee;
            color: #c62828;
            border-color: #ffcdd2;
        }}
        
        /* ========== RECUPEROS Y ALCANCES ========== */
        .grafica-container {{
            background: white;
            padding: 25px;
            border-radius: 12px;
            margin: 25px 0;
            box-shadow: 0 4px 15px rgba(0,0,0,0.1);
            border: 1px solid #e0e0e0;
        }}
        
        .grafica-container h3 {{
            color: var(--naranja-principal);
            margin-bottom: 20px;
            text-align: center;
            font-size: 1.6rem;
        }}
        
        .boton-exportar-excel {{
            background: linear-gradient(135deg, #21c45d, #0f9d58);
            color: white;
            border: none;
            padding: 15px 35px;
            font-size: 1.2rem;
            font-weight: 600;
            border-radius: 10px;
            cursor: pointer;
            transition: all 0.3s ease;
            box-shadow: 0 5px 15px rgba(15, 157, 88, 0.3);
            display: inline-flex;
            align-items: center;
            justify-content: center;
            gap: 12px;
            margin: 20px 0;
        }}
        
        .boton-exportar-excel:hover {{
            background: linear-gradient(135deg, #0f9d58, #0a7e46);
            transform: translateY(-3px);
            box-shadow: 0 8px 20px rgba(15, 157, 88, 0.4);
        }}
        
        /* ========== PROYECCIONES ========== */
        .panel-proyecciones {{
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 30px;
            border-radius: 15px;
            margin: 30px 0;
            box-shadow: 0 10px 30px rgba(102, 126, 234, 0.4);
        }}
        
        .controles-proyeccion {{
            display: flex;
            gap: 15px;
            align-items: center;
            flex-wrap: wrap;
            margin-bottom: 25px;
            justify-content: center;
        }}
        
        .input-proyeccion {{
            padding: 12px 20px;
            font-size: 1.1rem;
            border: 2px solid rgba(255,255,255,0.3);
            border-radius: 10px;
            background: rgba(255,255,255,0.1);
            color: white;
            min-width: 150px;
        }}
        
        .input-proyeccion::placeholder {{
            color: rgba(255,255,255,0.7);
        }}
        
        .estadistica-row {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
            gap: 15px;
            margin-top: 15px;
        }}
        
        .estadistica-item {{
            text-align: center;
            padding: 15px;
            border-radius: 8px;
            background: #f8f9fa;
        }}
        
        .estadistica-item.verde {{
            background: #e8f6f3;
            border: 2px solid #27ae60;
        }}
        
        .estadistica-item.naranja {{
            background: #fdebd0;
            border: 2px solid #e67e22;
        }}
        
        .estadistica-item.rojo {{
            background: #fadbd8;
            border: 2px solid #e74c3c;
        }}
        
        .estadistica-valor {{
            font-size: 2rem;
            font-weight: bold;
            margin: 10px 0;
        }}
        
        .estadistica-label {{
            font-size: 0.9rem;
            color: #666;
            text-transform: uppercase;
            letter-spacing: 1px;
        }}
        
        /* ========== FOOTER ========== */
        footer {{
            text-align: center;
            margin-top: 50px;
            padding: 25px;
            color: #666;
            font-size: 0.9rem;
            border-top: 1px solid #ddd;
        }}
        
        /* ========== RESPONSIVE ========== */
        @media (max-width: 1200px) {{
            .menu-principal {{
                grid-template-columns: repeat(4, minmax(210px, 1fr));
                max-width: 1000px;
            }}
        }}
        
        @media (max-width: 768px) {{
            .menu-principal {{
                grid-template-columns: 1fr;
                max-width: 500px;
            }}
            
            .selectores-periodo select {{
                display: block;
                margin: 10px auto;
                width: 80%;
            }}
            
            .controles-busqueda {{
                flex-direction: column;
            }}
            
            .input-busqueda {{
                width: 100%;
                min-width: auto;
            }}
        }}

        /* ========== CSS PARA BOTÓN DE 3 MESES ========== */
        .controles-grafica {{
            display: flex;
            justify-content: center;
            gap: 15px;
            padding: 15px;
        }}
            
        .controles-grafica .btn-comparacion {{
            padding: 10px 25px;
            border: 2px solid #3498db;
            background: white;
            color: #3498db;
            border-radius: 8px;
            cursor: pointer;
            font-weight: 600;
            font-size: 1rem;
            transition: all 0.3s;
            min-width: 100px;
        }}
            
        .controles-grafica .btn-comparacion:hover {{
            background: #3498db;
            color: white;
            transform: translateY(-3px);
            box-shadow: 0 5px 15px rgba(52, 152, 219, 0.3);
        }}
            
        .controles-grafica .btn-comparacion.activo {{
            background: #3498db;
            color: white;
            border-color: #2980b9;
            box-shadow: 0 0 0 3px white, 0 0 0 6px #3498db;
        }}

        /* ========== CHECKBOX "INCLUIR MES SELECCIONADO" ========== */
        .checkbox-incluir-mes {{
          display: flex;
          align-items: center;
          gap: 10px;
          cursor: pointer;
          font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
          font-weight: 700;
          font-size: 0.95rem;
          letter-spacing: 0.4px;
          text-transform: uppercase;
        
          color: #2c3e50;
        }}
        
        .checkbox-incluir-mes input[type="checkbox"] {{
          width: 18px;
          height: 18px;
          cursor: pointer;
        }}

        /* ========== ESTILOS PARA TABLA DE COMPARACIÓN DE SUPERVISORES ========== */
        .supervisor-name {{
            font-weight: bold !important;
            color: #2c3e50 !important;
            padding-left: 15px !important;
            background-color: #f8f9fa !important;
            border-left: 4px solid #3498db !important;
            /* MANTENER padding normal para fila principal */
            padding: 8px 8px 8px 15px !important;  /* Cambiado de padding-top/padding-bottom separados */
            font-size: 1em !important;
        }}
        
        .fila-equipo {{
            background-color: #ffffff !important;
            height: 26px !important;
            min-height: 26px !important;
            line-height: 1 !important;             /* AGREGADO: Anula line-height general */
        }}
        
        .fila-equipo:hover {{
            background-color: #f5f7fa !important;
        }}
        
        /* ESTILOS ESPECÍFICOS para celdas de filas de equipo - DEBEN ir DESPUÉS de reglas generales */
        .tabla-comparacion tr.fila-equipo td {{
            padding: 3px 8px !important;           /* Sobreescribe padding general */
            line-height: 1 !important;             /* Sobreescribe line-height general */
            min-height: 26px !important;           /* Sobreescribe min-height general */
            height: 26px !important;               /* Sobreescribe height general */
            vertical-align: middle !important;
            border-bottom: 1px solid #eee;
            text-align: center;
        }}
        
        .equipo-label {{
            padding-left: 30px !important;
            font-style: italic !important;
            color: #666 !important;
            background-color: #ffffff !important;
            text-align: left !important;
            width: 120px !important;
            min-width: 120px !important;
            max-width: 120px !important;
        }}
        
        .equipo-supervisor {{
            text-align: center !important;
            vertical-align: middle !important;
            background-color: #ffffff !important;
            font-size: 0.9em !important;
        }}
        
        .sin-equipo {{
            color: #bbb !important;
            font-style: italic !important;
            text-align: center !important;
            background-color: #ffffff !important;
            font-size: 0.9em !important;
        }}
        
        /* Mejorar el hover para filas de equipo */
        .fila-equipo:hover td {{
            background-color: #f5f7fa !important;
        }}
        
        /* Estilos para colores de segmento */
        .segmento-premier {{
            color: #8e44ad !important;
            background-color: #f4ecf7 !important;
        }}
        
        .segmento-empresarial {{
            color: #3498db !important;
            background-color: #ebf5fb !important;
        }}
        
        .segmento-masivo {{
            color: #27ae60 !important;
            background-color: #eafaf1 !important;
        }}
        
        .segmento-pyme {{
            color: #f39c12 !important;
            background-color: #fef5e7 !important;
        }}
        
        /* REGLA CRÍTICA: Sobreescribir estilos generales para filas específicas */
        .tabla-comparacion tr.fila-equipo {{
            height: 26px !important;
            min-height: 26px !important;
            line-height: 1 !important;
        }}
        
        /* Asegurar que las filas principales mantengan su altura normal */
        .tabla-comparacion tr:first-child:not(.fila-equipo) td {{
            padding: 8px 8px !important;
            min-height: 36px !important;
            line-height: 1.4 !important;
        }}

        /* ========== ESTILOS PARA TABLA DE COMPARACIÓN DE ASESORES ========== */
        .fila-secundaria-asesor {{
            background-color: #f8f9fa !important;
            height: 26px !important;
            min-height: 26px !important;
            line-height: 1 !important;
        }}
        
        /* Sobreescribir estilos generales para filas secundarias de asesores */
        .tabla-comparacion tr.fila-secundaria-asesor td {{
            padding: 3px 8px !important;           /* Altura reducida */
            line-height: 1 !important;             /* Altura reducida */
            min-height: 26px !important;           /* Altura reducida */
            height: 26px !important;               /* Altura reducida */
            vertical-align: middle !important;
            border-bottom: 1px solid #eee;
        }}
        
        .label-secundario-asesor {{
            color: #666 !important;
            font-style: italic !important;
            padding-left: 20px !important;
            text-align: left !important;
            width: 100px !important;
            min-width: 100px !important;
            max-width: 100px !important;
            background-color: #f8f9fa !important;
        }}
        
        .dato-secundario-asesor {{
            text-align: center !important;
            vertical-align: middle !important;
            background-color: #f8f9fa !important;
            font-size: 0.9em !important;
        }}
        
        /* Asegurar que las filas principales de asesores mantengan altura normal */
        .tabla-comparacion tr:first-child:not(.fila-secundaria-asesor) td {{
            padding: 8px 8px !important;
            min-height: 36px !important;
            line-height: 1.4 !important;
        }}
        
        /* Asegurar hover para filas secundarias */
        .fila-secundaria-asesor:hover td {{
            background-color: #f5f7fa !important;
        }}
        
        /* ========== ESTILOS MEJORADOS PARA PERIODO DE PRUEBA ========== */
        .selectores-periodo-prueba {{
          display: grid;
          grid-template-columns: repeat(100, 1fr);
          gap: 12px;
          align-items: end;
          margin-bottom: 15px;
        }}
        
        .selector-item {{
            display: flex;
            flex-direction: column;
            min-width: 0;
        }}
        
        .selector-label {{
            font-size: 0.9rem;
            font-weight: 600;
            color: #2c3e50;
            margin-bottom: 4px;
            display: flex;
            align-items: center;
            gap: 5px;
        }}
        
        .selectores-periodo-prueba select,
        .selectores-periodo-prueba input {{
            padding: 12px 15px;
            font-size: 1rem;
            border: 2px solid #ddd;
            border-radius: 8px;
            background-color: white;
            color: #333;
            transition: all 0.3s;
            width: 100%;
            height: 48px;
            min-width: 0;
        }}
        
        .selectores-periodo-prueba select:focus,
        .selectores-periodo-prueba input:focus {{
            border-color: #3498db;
            outline: none;
            box-shadow: 0 0 0 3px rgba(52, 152, 219, 0.2);
        }}
        
        .selectores-periodo-prueba select:hover,
        .selectores-periodo-prueba input:hover {{
            border-color: #3498db;
        }}
        
        /* Placeholder y opciones vacías */
        .selectores-periodo-prueba select option[value=""] {{
            color: #999;
            font-style: italic;
        }}
        
        .selectores-periodo-prueba input::placeholder {{
            color: #999;
            font-style: italic;
        }}
        
        /* Estilo para el input de porcentaje */
        #porcentajeMinimo {{
            background-color: #fffbf0;
            border-color: #f39c12;
        }}
        
        #porcentajeMinimo:focus {{
            border-color: #e67e22;
            box-shadow: 0 0 0 3px rgba(230, 126, 34, 0.2);
        }}
        
        /* Indicador visual para selects vacíos */
        .selectores-periodo-prueba select:invalid {{
            border-color: #ffcccb;
            background-color: #fff5f5;
        }}
        
        .selectores-periodo-prueba select:invalid + .selector-label::after {{
            content: " *";
            color: #e74c3c;
        }}

        /* ========== ESTILOS PARA DATALIST ========== */
        datalist {{
            position: absolute;
            background-color: white;
            border: 1px solid #ddd;
            border-radius: 8px;
            max-height: 200px;
            overflow-y: auto;
            box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);
            z-index: 1000;
        }}
        
        datalist option {{
            padding: 10px 15px;
            cursor: pointer;
            transition: background-color 0.2s;
        }}
        
        datalist option:hover {{
            background-color: #f8f9fa;
        }}
        
        /* Mejorar el input con datalist */
        .input-busqueda[list] {{
            background-image: url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='16' height='16' fill='%233498db' viewBox='0 0 16 16'%3E%3Cpath d='M7.247 11.14 2.451 5.658C1.885 5.013 2.345 4 3.204 4h9.592a1 1 0 0 1 .753 1.659l-4.796 5.48a1 1 0 0 1-1.506 0z'/%3E%3C/svg%3E");
            background-repeat: no-repeat;
            background-position: right 15px center;
            padding-right: 40px;
        }}

        /* ========== ESTILOS PARA TABLA DE PERIODO DE PRUEBA ========== */
        #tablaPeriodo {{
            margin-top: 20px;
            font-size: 0.95rem;
        }}
        
        #tablaPeriodo th {{
            background-color: #2c3e50;
            color: white;
            padding: 12px 10px;
            text-align: center;
            font-weight: 600;
            position: sticky;
            top: 0;
            z-index: 10;
            white-space: nowrap;
        }}
        
        #tablaPeriodo td {{
            padding: 10px 8px;
            border-bottom: 1px solid #eee;
            vertical-align: middle;
            text-align: center;
        }}
        
        #tablaPeriodo tr:hover {{
            background-color: #f5f7fa;
        }}
        
        /* Estilos específicos para columnas numéricas */
        #tablaPeriodo td:nth-child(2),
        #tablaPeriodo td:nth-child(3),
        #tablaPeriodo td:nth-child(6) {{
            font-family: 'Courier New', monospace;
            font-weight: 500;
            letter-spacing: 0.5px;
        }}
        
        /* Colores para estado */
        #tablaPeriodo td:nth-child(5) {{
            font-weight: 600;
        }}
        
        /* Alinear nombres de asesores a la izquierda */
        #tablaPeriodo td:first-child {{
            text-align: left;
            padding-left: 15px;
        }}
        
        /* Estilo para la fila de totales */
        #tablaPeriodo tr:last-child {{
            background-color: #e8f4fd !important;
            font-size: 1rem;
        }}
        
        #tablaPeriodo tr:last-child td {{
            padding: 14px 8px;
            border-top: 2px solid #2c3e50;
            border-bottom: 2px solid #2c3e50;
        }}
        
        /* Responsive para tabla de periodo de prueba */
        @media (max-width: 1200px) {{
            #resultadosPeriodo .contenedor-tabla-scroll {{
                max-width: 100%;
                overflow-x: auto;
            }}
            
            #tablaPeriodo {{
                min-width: 1000px;
            }}
        }}
        
        /* Mejorar visualización de números grandes */
        .numero-grande {{
            font-size: 1.05em;
            font-weight: 600;
        }}

        //styles para cambio de gráfica recupero//
        .btn-toggle-grafica {{
            padding: 10px 25px;
            border: none;
            border-radius: 25px;
            cursor: pointer;
            font-size: 1rem;
            font-weight: 600;
            transition: all 0.3s ease;
            min-width: 180px;
        }}
        
        .btn-toggle-grafica:hover {{
            transform: translateY(-3px);
            box-shadow: 0 6px 15px rgba(0, 0, 0, 0.2) !important;
        }}
        
        .btn-toggle-grafica.activo {{
            background: linear-gradient(135deg, #2ecc71, #27ae60) !important;
            box-shadow: 0 0 0 3px white, 0 0 0 6px #2ecc71 !important;
            transform: translateY(-2px);
        }}
        
        /* Contenedor de gráfica con posición relativa para los botones */
        .grafica-container {{
            position: relative;
            background: white;
            padding: 25px;
            border-radius: 12px;
            margin: 25px 0;
            box-shadow: 0 4px 15px rgba(0,0,0,0.1);
            border: 1px solid #e0e0e0;
        }}

        /* Contenedor de checkbox */
        .grupo-checkbox-eval {{
          display: flex;
          flex-direction: column;
          align-items: flex-start; /* ← CLAVE */
          gap: 6px;
        }}

        /* Estilos para los promedios */
        .promedio-card {{
            background: white;
            padding: 12px;
            border-radius: 6px;
            border-left: 4px solid #ccc;
            transition: transform 0.2s ease;
        }}
        
        .promedio-card:hover {{
            transform: translateY(-2px);
            box-shadow: 0 4px 8px rgba(0,0,0,0.1);
        }}

        .eval-quintil-card {{
            background: #fff;
            border-radius: 10px;
            border: 1px solid #e6eaf0;
            overflow: hidden;
            box-shadow: 0 4px 12px rgba(0,0,0,0.08);
            min-height: 132px;
            cursor: pointer;
            transform: translateY(0) scale(1);
            transition: transform 0.28s cubic-bezier(.2,.8,.2,1), box-shadow 0.28s ease, border-color 0.28s ease;
        }}

        .eval-quintil-toolbar {{
            display: flex;
            justify-content: flex-end;
            margin: 0 0 10px 0;
        }}

        .btn-exportar-quintiles {{
            border: 1px solid #d8dee8;
            background: #f8fafc;
            color: #34495e;
            border-radius: 8px;
            padding: 8px 12px;
            font-size: 0.86rem;
            font-weight: 800;
            cursor: pointer;
            box-shadow: 0 2px 6px rgba(44, 62, 80, 0.08);
            transition: background 0.2s ease, border-color 0.2s ease, transform 0.2s ease;
        }}

        .btn-exportar-quintiles:hover {{
            background: #eef5fb;
            border-color: #b8c7d9;
            transform: translateY(-1px);
        }}

        .eval-quintil-card:hover {{
            transform: translateY(-2px);
            box-shadow: 0 6px 16px rgba(0,0,0,0.12);
        }}

        .eval-quintil-card.quintil-abierto {{
            transform: translateY(-4px) scale(1.012);
            border-color: rgba(41, 182, 246, 0.48);
            box-shadow: 0 14px 30px rgba(44, 62, 80, 0.18);
        }}

        .eval-quintil-card.quintil-abierto:hover {{
            transform: translateY(-5px) scale(1.012);
        }}

        .eval-quintil-title {{
            color: #fff;
            font-size: 1.05rem;
            font-weight: 900;
            text-align: center;
            padding: 10px 10px;
            letter-spacing: 0.5px;
            display: flex;
            align-items: center;
            justify-content: space-between;
            gap: 8px;
        }}

        .eval-quintil-toggle {{
            width: 22px;
            height: 22px;
            border-radius: 50%;
            display: inline-flex;
            align-items: center;
            justify-content: center;
            background: rgba(255,255,255,0.22);
            font-size: 1rem;
            line-height: 1;
            transform: rotate(0deg) scale(1);
            transition: transform 0.32s cubic-bezier(.34,1.56,.64,1), background 0.25s ease;
        }}

        .eval-quintil-card.quintil-abierto .eval-quintil-toggle {{
            transform: rotate(180deg) scale(1.12);
            background: rgba(255,255,255,0.34);
        }}

        .eval-quintil-q1 .eval-quintil-title {{ background: #c62828; }}
        .eval-quintil-q2 .eval-quintil-title {{ background: #ef6c00; }}
        .eval-quintil-q3 .eval-quintil-title {{ background: #f9a825; color:#2c3e50; }}
        .eval-quintil-q4 .eval-quintil-title {{ background: #26a69a; }}
        .eval-quintil-q5 .eval-quintil-title {{ background: #29b6f6; }}

        .eval-quintil-summary {{
            padding: 12px 10px;
            display: grid;
            grid-template-columns: 1fr;
            gap: 8px;
            background: #fff;
        }}

        .eval-quintil-metric {{
            display: flex;
            align-items: center;
            justify-content: space-between;
            gap: 8px;
            padding: 8px 9px;
            background: #f8fafc;
            border: 1px solid #eef2f7;
            border-radius: 8px;
        }}

        .eval-quintil-metric-label {{
            font-size: 0.78rem;
            font-weight: 800;
            color: #667085;
            text-transform: uppercase;
        }}

        .eval-quintil-metric-value {{
            font-size: 0.9rem;
            font-weight: 900;
            color: #2c3e50;
            white-space: nowrap;
        }}

        .eval-quintil-list {{
            max-height: 360px;
            overflow: auto;
            padding: 8px;
            display: flex;
            flex-direction: column;
            gap: 6px;
            border-top: 1px solid #eef2f7;
            cursor: default;
            opacity: 1;
            transform: translateY(0);
            transform-origin: top center;
            pointer-events: auto;
            will-change: max-height, opacity, transform;
            transition: max-height 0.42s cubic-bezier(.2,.8,.2,1), opacity 0.24s ease 0.06s, transform 0.34s ease, padding 0.34s ease, border-color 0.25s ease;
        }}

        .eval-quintil-list.colapsada {{
            max-height: 0;
            padding-top: 0;
            padding-bottom: 0;
            opacity: 0;
            transform: translateY(-10px);
            pointer-events: none;
            border-top-color: transparent;
            transition: max-height 0.34s cubic-bezier(.4,0,.6,1), opacity 0.18s ease, transform 0.25s ease, padding 0.28s ease, border-color 0.2s ease;
        }}

        .eval-quintil-row {{
            display: grid;
            grid-template-columns: minmax(0, 1fr) auto;
            gap: 8px;
            align-items: center;
            padding: 8px 9px;
            background: #f8fafc;
            border: 1px solid #eef2f7;
            border-radius: 8px;
        }}

        .eval-quintil-card.quintil-abierto .eval-quintil-row {{
            animation: quintilFilaEntrada 0.34s cubic-bezier(.2,.8,.2,1) both;
            animation-delay: var(--quintil-delay, 0ms);
        }}

        @keyframes quintilFilaEntrada {{
            from {{ opacity: 0; transform: translateY(-8px) scale(0.985); }}
            to {{ opacity: 1; transform: translateY(0) scale(1); }}
        }}

        @media (prefers-reduced-motion: reduce) {{
            .eval-quintil-card,
            .eval-quintil-toggle,
            .eval-quintil-list,
            .eval-quintil-row {{
                transition: none !important;
                animation: none !important;
            }}
        }}

        .eval-quintil-asesor {{
            font-size: 0.85rem;
            font-weight: 800;
            color: #2c3e50;
            overflow: hidden;
            text-overflow: ellipsis;
            white-space: nowrap;
        }}

        .eval-quintil-alcance {{
            font-size: 0.85rem;
            font-weight: 900;
            color: #2980b9;
            white-space: nowrap;
        }}

        .quintil-barra {{
            min-width: 110px;
        }}

        .quintil-barra-track {{
            height: 22px;
            background: transparent;
            border: 0;
            border-radius: 4px;
            overflow: visible;
            position: relative;
        }}

        .quintil-barra-fill {{
            height: 100%;
            display: flex;
            align-items: center;
            padding-left: 8px;
            color: #fff;
            font-size: 0.82rem;
            font-weight: 900;
            line-height: 1;
            min-width: 24px;
            border-radius: 4px;
        }}

        .eval-quintil-head-wrap {{
            display: inline-flex;
            align-items: center;
            justify-content: center;
            gap: 8px;
        }}

        .eval-quintil-extra-btn {{
            width: 24px;
            height: 24px;
            border: 0;
            border-radius: 50%;
            background: #eef2f7;
            color: #233142;
            font-weight: 950;
            cursor: pointer;
            line-height: 1;
        }}

        .eval-quintil-extra-col.oculta {{ display: none; }}

        .eval-quintiles-extra-box {{
            display: grid;
            grid-template-columns: repeat(4, minmax(120px, 1fr));
            gap: 10px;
            padding: 10px;
            background: #f8fafc;
            border: 1px solid #e5e7eb;
            border-radius: 8px;
        }}

        .eval-quintil-extra-item {{ display: grid; gap: 5px; }}

        .eval-quintil-extra-label {{
            font-size: 0.72rem;
            font-weight: 900;
            color: #667085;
            text-transform: uppercase;
        }}

        .quintil-barra-fill.gris {{
            background: #8f98a3 !important;
            color: #fff !important;
        }}

        .tabla-eval-detalle .quintil-barra {{
            min-width: 54px;
            max-width: 68px;
            margin: 0 auto;
        }}

        .tabla-eval-detalle .quintil-barra-track {{
            height: 18px;
        }}

        .tabla-eval-detalle .quintil-barra-fill {{
            font-size: 0.72rem;
            min-width: 18px;
            padding-left: 5px;
        }}
        
        /* Estilos para etiqueta de evaluación */
        .etiqueta-evaluacion {{
            background: white;
            padding: 15px;
            border-radius: 6px;
            border: 2px solid #2ecc71;
            transition: all 0.3s ease;
        }}
        
        .etiqueta-evaluacion.inestable {{
            border-color: #e74c3c;
            background: linear-gradient(135deg, #fff5f5, #ffeaea);
        }}
        
        .etiqueta-evaluacion.estable {{
            border-color: #2ecc71;
            background: linear-gradient(135deg, #f5fff7, #eaffef);
        }}
        
        .etiqueta-evaluacion.mejora {{
            border-color: #3498db;
            background: linear-gradient(135deg, #f0f8ff, #e6f2ff);
        }}

        /* ========== BARRA DE FILTROS GLOBAL ========== */
        .barra-filtros-supervisores {{
            display: flex;
            flex-wrap: wrap;
            gap: 10px;
            justify-content: center;
            margin: 20px 0;
            padding: 15px;
            background: rgba(255, 255, 255, 0.1);
            border-radius: 12px;
        }}
        
        .filtro-supervisor {{
            padding: 10px 20px;
            background: linear-gradient(135deg, #3498db, #2980b9);
            color: white;
            border: none;
            border-radius: 25px;
            cursor: pointer;
            font-size: 0.95rem;
            font-weight: 600;
            transition: all 0.3s ease;
            box-shadow: 0 3px 8px rgba(0,0,0,0.2);
        }}
        
        .filtro-supervisor:hover {{
            background: linear-gradient(135deg, #2980b9, #1f6396);
            transform: translateY(-3px);
            box-shadow: 0 5px 12px rgba(0,0,0,0.3);
        }}
        
        .filtro-supervisor.activo {{
            background: linear-gradient(135deg, #2ecc71, #27ae60);
            box-shadow: 0 0 0 3px white, 0 0 0 6px #2ecc71;
        }}
        
        .filtro-supervisor.especial {{
            background: linear-gradient(135deg, #9b59b6, #8e44ad);
        }}
        
        .filtro-supervisor.especial:hover {{
            background: linear-gradient(135deg, #8e44ad, #7d3c98);
        }}

        .filtro-supervisor.accion-global {{
          background: linear-gradient(135deg, #9b59b6, #8e44ad);
          color: #fff;
          font-weight: 700;
          border: none;
        }}
        
        .filtro-supervisor.accion-global:hover {{
          background: linear-gradient(135deg, #8e44ad, #7d3c98);
          transform: translateY(-1px);
          box-shadow: 0 4px 10px rgba(142, 68, 173, 0.35);
        }}
        
        .filtro-supervisor.accion-global.activo {{
          background: linear-gradient(135deg, #9b59b6, #8e44ad);
          box-shadow: none;
        }}

        .modal-eval {{
          position: fixed;
          inset: 0;
          z-index: 9999;
        }}
        
        .modal-eval-backdrop {{
          position: absolute;
          inset: 0;
          background: rgba(0,0,0,0.45);
        }}
        
        .modal-eval-card {{
          position: relative;
          width: min(980px, calc(100vw - 30px));
          margin: 50px auto;
          background: #fff;
          border-radius: 16px;
          box-shadow: 0 20px 60px rgba(0,0,0,0.35);
          overflow: hidden;
        }}
        
        .modal-eval-header {{
          display:flex;
          align-items:flex-start;
          justify-content:space-between;
          gap: 10px;
          padding: 16px 18px;
          border-bottom: 1px solid #eef2f7;
        }}
        
        .modal-eval-close {{
          border:none;
          background:#f2f4f8;
          border-radius: 10px;
          cursor:pointer;
          padding: 10px 12px;
          font-weight: 900;
        }}
        
        .modal-eval-body {{
          padding: 16px 18px 18px;
        }}
        
        .tabla-eval-detalle {{
          width: 100%;
          border-collapse: collapse;
          min-width: 0;
          table-layout: fixed;
        }}

        .tabla-eval-detalle th,
        .tabla-eval-detalle td {{
          font-size: 0.82rem;
        }}

        .tabla-eval-detalle th:nth-child(1), .tabla-eval-detalle td:nth-child(1) {{ width: 11%; }}
        .tabla-eval-detalle th:nth-child(2), .tabla-eval-detalle td:nth-child(2) {{ width: 8%; }}
        .tabla-eval-detalle th:nth-child(3), .tabla-eval-detalle td:nth-child(3) {{ width: 10%; }}
        .tabla-eval-detalle th:nth-child(4), .tabla-eval-detalle td:nth-child(4) {{ width: 10%; }}
        .tabla-eval-detalle th:nth-child(5), .tabla-eval-detalle td:nth-child(5) {{ width: 8%; }}
        .tabla-eval-detalle th:nth-child(6), .tabla-eval-detalle td:nth-child(6) {{ width: 8%; }}
        .tabla-eval-detalle th:nth-child(7), .tabla-eval-detalle td:nth-child(7) {{ width: 9%; }}
        .tabla-eval-detalle th:nth-child(n+8), .tabla-eval-detalle td:nth-child(n+8) {{ width: 9%; }}
        
        .tabla-eval-detalle th {{
          background: #2c3e50;
          color: #fff;
          padding: 8px 6px;
          text-align: left;
          position: sticky;
          top: 0;
          z-index: 2;
        }}
        
        .tabla-eval-detalle td {{
          border-bottom: 1px solid #eef2f7;
          padding: 10px;
          vertical-align: middle;
        }}
        
        .tag-si {{
          display:inline-block;
          padding: 4px 10px;
          border-radius: 999px;
          background: #e8f6f3;
          color: #0b6b57;
          font-weight: 800;
        }}
        
        .tag-no {{
          display:inline-block;
          padding: 4px 10px;
          border-radius: 999px;
          background: #fdecea;
          color: #b42318;
          font-weight: 800;
        }}

        /* Filas informativas (no son alcances) */
        .fila-asesores {{
            background-color: #eef6ff;
        }}
        
        .fila-cartera {{
            background-color: #f3ecfa;
        }}
        
        /* Para que las celdas se vean ordenadas */
        .fila-asesores td,
        .fila-cartera td {{
            font-weight: 600;
            color: #2c3e50;
        }}

        /* ===== Scroll interno para modal Evaluación ===== */
        
        .modal-eval-card {{
          max-height: 90vh;
          display: flex;
          flex-direction: column;
        }}
        
        .modal-eval-body {{
          overflow-y: auto;
          flex: 1;
          padding-right: 6px; /* espacio para scrollbar */
        }}
        
        /* Scroll más elegante */
        .modal-eval-body::-webkit-scrollbar {{
          width: 8px;
        }}
        
        .modal-eval-body::-webkit-scrollbar-thumb {{
          background: rgba(0,0,0,0.25);
          border-radius: 6px;
        }}
        
        .modal-eval-body::-webkit-scrollbar-track {{
          background: transparent;
        }}

        /* ===================== TABLA COMPARATIVA 4M (ALCANCES) - UI PRO ===================== */
        #cardTablaComparativa4M {{
          padding: 10px 40px !important;
        }}
        
        #cardTablaComparativa4M .tabla4m-wrap {{
          margin-top: 12px;
          border: 1px solid #e6eaf0;
          border-radius: 14px;
          background: #fff;
          width: fit-content;
          max-width: 100%;
          max-height: none;
          overflow-y: visible;
          overflow-x: auto;
        }}
        
        #cardTablaComparativa4M .tabla-comparativa-4m {{
          width: 100%;
          border-collapse: separate;
          border-spacing: 0;
          min-width: 430px;
          table-layout: fixed;
          font-size: 0.76rem;
        }}
        
        #cardTablaComparativa4M .tabla-comparativa-4m th,
        #cardTablaComparativa4M .tabla-comparativa-4m td {{
          padding: 2px 5px;
          border-bottom: 1px solid #eef2f7;
          border-right: 1px solid #f1f4f8;
          text-align: center;
          vertical-align: middle;
          white-space: nowrap;
        }}
        
        #cardTablaComparativa4M .tabla-comparativa-4m thead th {{
          position: sticky;
          top: 0;
          z-index: 3;
          background: #0d47a1;
          color: #fff;
          font-weight: 800;
          letter-spacing: 0.2px;
        }}
        
        #cardTablaComparativa4M .tabla-comparativa-4m thead tr:nth-child(2) th {{
          top: 28px;
          background: #2c3e50;
          z-index: 2;
        }}
        
        #cardTablaComparativa4M .tabla-comparativa-4m thead tr:nth-child(3) th {{
          top: 56px;
          background: #1f618d;
          z-index: 2;
        }}
        
        #cardTablaComparativa4M .tabla-comparativa-4m thead th:first-child {{
          left: 0;
          z-index: 5;
        }}
        
        #cardTablaComparativa4M .tabla-comparativa-4m thead tr:nth-child(2) th:first-child {{
          left: 0;
          z-index: 6;
        }}
        
        #cardTablaComparativa4M .tabla-comparativa-4m thead tr:nth-child(3) th:first-child {{
          left: 0;
          z-index: 6;
        }}
        
        #cardTablaComparativa4M .tabla-comparativa-4m tbody td:first-child {{
          position: sticky;
          left: 0;
          z-index: 1;
          background: #f7f9fc;
          font-weight: 900;
          border-right: 1px solid #e6eaf0;
        }}
        
        #cardTablaComparativa4M .tabla-comparativa-4m tbody tr:nth-child(odd) td {{
          background: #fcfdff;
        }}
        
        #cardTablaComparativa4M .tabla-comparativa-4m tbody tr:hover td {{
          background: #eef6ff;
        }}
        
        #cardTablaComparativa4M .tabla-comparativa-4m td.valor-alto {{
          font-weight: 900;
        }}
        
        #cardTablaComparativa4M .tabla-comparativa-4m td.valor-vacio {{
          color: #9aa4b2;
        }}
        
        #cardTablaComparativa4M .tabla-comparativa-4m th:last-child,
        #cardTablaComparativa4M .tabla-comparativa-4m td:last-child {{
          border-right: none;
        }}

        #cardTablaComparativa4M .tabla-comparativa-4m th:first-child,
        #cardTablaComparativa4M .tabla-comparativa-4m td:first-child {{
          width: 64px;
          min-width: 64px;
        }}
        
        #cardTablaComparativa4M .tabla-comparativa-4m th:not(:first-child),
        #cardTablaComparativa4M .tabla-comparativa-4m td:not(:first-child) {{
          width: 92px;
          min-width: 92px;
        }}

        #cardTablaComparativa4M .tabla-comparativa-4m {{
          width: auto !important;
        }}
        
        #cardTablaComparativa4M .tabla4m-doble {{
          display: grid;
          grid-template-columns: minmax(0, 1fr) minmax(0, 1fr);
          gap: 12px;
          align-items: start;
          width: 100%;
        }}

        #cardTablaComparativa4M .tabla4m-doble > div {{
          min-width: 0;
        }}

        #cardTablaComparativa4M .tabla4m-titulo {{
          margin: 0 0 6px 0;
          font-size: 0.9rem;
          color: #2c3e50;
          font-weight: 900;
          text-align: center;
        }}
        
        #cardTablaComparativa4M .tabla4m-comparativo {{
          margin-top: 12px;
        }}

        #cardTablaComparativa4M .tabla4m-supervisores-grid {{
          display: flex;
          gap: 12px;
          margin-top: 14px;
          overflow-x: auto;
          padding-bottom: 8px;
          scroll-snap-type: x proximity;
        }}

        #cardTablaComparativa4M .tabla4m-supervisor-card {{
          flex: 0 0 520px;
          min-width: 520px;
          scroll-snap-align: start;
        }}

        #cardTablaComparativa4M .tabla4m-supervisor-card .tabla4m-titulo {{
          text-align: left;
          padding-left: 4px;
        }}
        
        #cardTablaComparativa4M td.is-clickable {{
          cursor: pointer;
        }}
        
        #cardTablaComparativa4M tr.is-selected td {{
          background: #dbeeff !important;
          font-weight: 900;
        }}

        /* Selectores alineados al ancho real de la tabla */
        #cardTablaComparativa4M .tabla4m-selectores {{
          display: grid;
          grid-template-columns: repeat(4, minmax(120px, 1fr));
          gap: 10px;
          margin-top: 14px;
          width: fit-content;
          max-width: 100%;
          padding-left: 64px;
        }}
        
        #cardTablaComparativa4M .tabla4m-selectores select {{
          width: 170px;
        }}

        .mini-chart-card {{
          background:#fff;
          border:1px solid #e6eaf0;
          border-radius:14px;
          padding:10px;
          cursor:pointer;
          box-shadow: 0 4px 12px rgba(0,0,0,0.06);
          transition: transform 0.15s ease, box-shadow 0.15s ease;
        }}
        
        .mini-chart-card:hover {{
          transform: translateY(-2px);
          box-shadow: 0 10px 20px rgba(0,0,0,0.12);
        }}
        
        .mini-title {{
          font-weight: 900;
          font-size: 0.85rem;
          color:#2c3e50;
          margin-bottom: 6px;
        }}
        
        .mini-chart-card canvas {{
          width: 100% !important;
          height: 150px !important;
          display:block;
        }}



    </style>
</head>
<body>
<div class="container">
    <!-- ========== HEADER ========== -->
    <header>
        <h1>🏆 SISTEMA DE GESTIÓN DE DESEMPEÑO</h1>
        <div class="selectores-periodo">
            <select id="selectorAño" onchange="actualizarSelectorMesSegunAñoHeader(); actualizarPeriodo()">
                {opciones_año}
            </select>
            <select id="selectorMes" onchange="actualizarPeriodo()">
                {opciones_mes}
            </select>
        </div>
        <div class="fecha-actualizacion">Última actualización: {fecha_actualizacion}</div>

        <div class="accesos-secundarios">
            <div class="canales-alcances" aria-label="Filtro de canal">
                <button type="button" class="btn-canal-alcances" data-canal-alcances="TODOS LOS CANALES" onclick="seleccionarCanalAlcances('TODOS LOS CANALES')">TODOS LOS CANALES</button>
                <button type="button" class="btn-canal-alcances activo" data-canal-alcances="SURCO" onclick="seleccionarCanalAlcances('SURCO')">SURCO</button>
                <button type="button" class="btn-canal-alcances" data-canal-alcances="BPO" onclick="seleccionarCanalAlcances('BPO')">BPO</button>
            </div>
        </div>

        <!-- ========== MENÚ PRINCIPAL ========== -->
        <div class="menu-principal">
            <button class="boton-modulo ranking activo" onclick="mostrarSeccion('ranking')">
                <div class="icono-modulo"></div>
                <div class="titulo-modulo">RANKING</div>
                <div class="descripcion-modulo">Clasificación y comparación de asesores</div>
            </button>

            <button class="boton-modulo calidad" onclick="mostrarSeccion('calidad')">
                <div class="icono-modulo"></div>
                <div class="titulo-modulo">INDICADORES</div>
                <div class="descripcion-modulo">Indicadores operativos y de calidad</div>
            </button>

            <button class="boton-modulo gestiones" onclick="mostrarSeccion('gestiones')">
                <div class="icono-modulo"></div>
                <div class="titulo-modulo">HISTÓRICO</div>
                <div class="descripcion-modulo">Registro y control histórico en operaciones por asesor</div>
            </button>

            <button class="boton-modulo evaluacion" onclick="mostrarSeccion('evaluacion')">
                <div class="icono-modulo"></div>
                <div class="titulo-modulo">EVALUACIÓN</div>
                <div class="descripcion-modulo">Análisis y clasificación de asesores por periodo y alcance promediado</div>
            </button>
        </div>
        <div class="barra-filtros-supervisores" id="barraFiltrosSupervisoresGlobal">
            <!-- Se generará dinámicamente -->
        </div>
    </header>
    <!-- ========== SECCIÓN RANKING ========== -->
    <section id="seccion-ranking" class="seccion-contenido activa">
        {contenido_ranking}
    </section>

    <!-- ========== SECCIÓN EVALUACIÓN ========== -->
    <!-- ========== SECCION INDICADORES ========== -->
    <section id="seccion-calidad" class="seccion-contenido">

        <div class="calidad-dashboard calidad-tabla-dashboard">
            <div id="calidadResumen" class="calidad-resumen"></div>
            <div class="calidad-tabla-wrap">
                <table class="calidad-tabla" id="tablaCalidadIndicadores">
                    <thead>
                        <tr class="calidad-head-principal">
                            <th>Asesor</th>
                            <th>Supervisor</th>
                            <th>Segmento</th>
                            <th class="indicador-grupo indicador-puntualidad" data-grupo-head="puntualidad">
                                <div class="indicador-head-actions">
                                    <button type="button" class="indicador-sort" data-sort-key="puntualidad" onclick="ordenarCalidadPor('puntualidad')">Puntualidad</button>
                                    <button type="button" class="indicador-toggle" data-grupo="puntualidad" onclick="toggleIndicadorDetalle('puntualidad')" aria-label="Mostrar columnas de puntualidad">+</button>
                                </div>
                            </th>
                            <th class="detalle-col detalle-puntualidad">Falta</th>
                            <th class="detalle-col detalle-puntualidad">Tardanza</th>
                            <th class="detalle-col detalle-puntualidad">Tardanza Tiempo</th>
                            <th class="indicador-grupo indicador-cierre" data-grupo-head="cierre">
                                <div class="indicador-head-actions">
                                    <button type="button" class="indicador-sort" data-sort-key="cierre" onclick="ordenarCalidadPor('cierre')">Cierre</button>
                                    <button type="button" class="indicador-toggle" data-grupo="cierre" onclick="toggleIndicadorDetalle('cierre')" aria-label="Mostrar columnas de cierre">+</button>
                                </div>
                            </th>
                            <th class="detalle-col detalle-cierre">PDP</th>
                            <th class="detalle-col detalle-cierre">CEF Unico</th>
                            <th class="indicador-grupo indicador-condonacion" data-grupo-head="condonacion">
                                <div class="indicador-head-actions">
                                    <button type="button" class="indicador-sort" data-sort-key="condonacion" onclick="ordenarCalidadPor('condonacion')">Condonación</button>
                                    <button type="button" class="indicador-toggle" data-grupo="condonacion" onclick="toggleIndicadorDetalle('condonacion')" aria-label="Mostrar columnas de condonación">+</button>
                                </div>
                            </th>
                            <th class="detalle-col detalle-condonacion">Monto PDP</th>
                            <th class="detalle-col detalle-condonacion">DK</th>
                            <th class="indicador-grupo indicador-calidad_pdp" data-grupo-head="calidad_pdp">
                                <div class="indicador-head-actions">
                                    <button type="button" class="indicador-sort" data-sort-key="calidad_pdp" onclick="ordenarCalidadPor('calidad_pdp')">Calidad PDP</button>
                                    <button type="button" class="indicador-toggle" data-grupo="calidad_pdp" onclick="toggleIndicadorDetalle('calidad_pdp')" aria-label="Mostrar columnas de calidad PDP">+</button>
                                </div>
                            </th>
                            <th class="detalle-col detalle-calidad_pdp">Recupero</th>
                            <th class="detalle-col detalle-calidad_pdp">Ticket PDP</th>
                        </tr>
                    </thead>
                    <tbody id="tbodyCalidadIndicadores">
                        <tr><td colspan="7" class="calidad-tabla-vacia">Selecciona un periodo para cargar indicadores.</td></tr>
                    </tbody>
                </table>
            </div>
            <div id="calidadQuintiles" class="calidad-quintiles"></div>
        </div>
    </section>

    <!-- ========== SECCION HISTORICO ========== -->
    <section id="seccion-gestiones" class="seccion-contenido">
        <div class="asesor-dashboard">
            <div class="asesor-topbar">
                <div class="asesor-titulo">
                    <h2>HISTÓRICO</h2>
                    <p>Registro y control histórico en operaciones por asesor</p>
                </div>
                <div class="asesor-search">
                    <input id="asesorSearchInput" list="asesorSearchDatalist" placeholder="Buscar asesor por alias" onkeydown="if(event.key === 'Enter') seleccionarAsesorHistoricoDesdeInput();">
                    <datalist id="asesorSearchDatalist"></datalist>
                    <button type="button" onclick="seleccionarAsesorHistoricoDesdeInput()">Buscar</button>
                </div>
            </div>
            <div class="asesor-rangos">
                <button type="button" data-rango-asesor="ALL" onclick="cambiarRangoAsesorHistorico('ALL')">Histórico</button>
                <button type="button" data-rango-asesor="12" onclick="cambiarRangoAsesorHistorico('12')">12 meses</button>
                <button type="button" data-rango-asesor="6" onclick="cambiarRangoAsesorHistorico('6')">6 meses</button>
                <button type="button" data-rango-asesor="3" onclick="cambiarRangoAsesorHistorico('3')">3 meses</button>
            </div>
            <div id="asesorFicha"></div>
            <div id="asesorKpis"></div>
            <div class="asesor-panel-grid">
                <div class="asesor-chart-card">
                    <div class="asesor-chart-wrap">
                        <canvas id="asesorHistoricoChart"></canvas>
                    </div>
                </div>
                <div id="asesorTendencias" class="asesor-tendencia-card"></div>
            </div>
            <div id="asesorHistoricoTabla"></div>
        </div>
    </section>

    <section id="seccion-evaluacion" class="seccion-contenido">
        {contenido_evaluacion}
    </section>
    
    <footer>
        <p>Sistema de seguimiento de metas - Actualización diaria según cortes</p>
    </footer>
</div>

<script>

    // ========== VARIABLES GLOBALES ==========
    const asesoresSeleccionados = new Set();
    const supervisoresSeleccionados = new Set();
    const asesoresSeleccionadosPeriodo = new Set();
    let estadoGlobal = {{
    mesActual: '',
    añoActual: '',
    periodoActual: ''
    }};
    let mesesAMostrar = 6;
    let modoPromedioPeriodo = false;
    let datosMeses = {datos_json_str_escaped};
    let datosSupervisores = {datos_supervisores_json_str_escaped};
    window.baseAsesoresAnalisis = JSON.parse('{base_asesores_json_escaped}');
    let datosCumpleaños = JSON.parse(`{datos_cumpleaños_json_str_escaped}`);
    let datosMundial = JSON.parse(`{datos_mundial_json_str_escaped}`);
    let datosAsesoresInf = JSON.parse(`{datos_asesores_inf_json_str_escaped}`);
    let mundialPaginaEquipos = 0;
    let chartTop10Modal = null;
    let chartCumpleañosModal = null;
    let chartInstance = null;
    let chartInstanceSupervisores = null;
    let chartAlcanceEquipos = null;
    let chartTendencia = null;
    let chartIncrementoTotal = null;
    let supervisorFiltroActual = 'TODOS';
    let vistaEvaluacion = '3M'; 
    let modoOtrosEvaluacion = false;
    let incluirMesSeleccionadoEval = false;
    let calidadOrdenActual = null;
    let calidadQuintilSeleccionado = 'calidad_pdp';
    let asesorSeleccionadoHistorico = '';
    let asesorRangoHistorico = 'ALL';
    let chartAsesorHistorico = null;
    let chartsMetricasAsesor = {{}};
    window.canalSeleccionadoGlobal = window.canalSeleccionadoGlobal || 'SURCO';

    function normalizarCanalDashboard(canal) {{
      const texto = String(canal || '').trim().toUpperCase();
      if (!texto || texto === 'TODOS' || texto === 'TODOS LOS CANALES') return 'TODOS LOS CANALES';
      if (texto.includes('BPO')) return 'BPO';
      if (texto.includes('SURCO')) return 'SURCO';
      return texto;
    }}

    function canalActivoDashboard() {{
      return normalizarCanalDashboard(window.canalSeleccionadoGlobal || 'SURCO');
    }}

    function sincronizarBotonesCanalAlcances() {{
      const canalActivo = canalActivoDashboard();
      window.canalSeleccionadoGlobal = canalActivo;
      window.canalAlcancesSeleccionado = canalActivo;
      document.querySelectorAll('[data-canal-alcances]').forEach(btn => {{
        const canalBoton = normalizarCanalDashboard(btn.getAttribute('data-canal-alcances'));
        btn.classList.toggle('activo', canalBoton === canalActivo);
        btn.setAttribute('aria-pressed', String(canalBoton === canalActivo));
      }});
    }}

    function asesorEnCanal(asesor) {{
      const canalActivo = canalActivoDashboard();
      if (canalActivo === 'TODOS LOS CANALES') return true;
      return normalizarCanalDashboard(asesor?.canal || '') === canalActivo;
    }}

    function filtrarAsesoresPorCanal(asesores) {{
      return (Array.isArray(asesores) ? asesores : []).filter(asesorEnCanal);
    }}

    function supervisorEnCanal(supervisor, periodoCompleto = '') {{
      const canalActivo = canalActivoDashboard();
      if (canalActivo === 'TODOS LOS CANALES') return true;
      const canalSup = normalizarCanalDashboard(datosSupervisores?.[supervisor]?.[periodoCompleto]?.canal || '');
      if (canalSup && canalSup !== 'TODOS LOS CANALES') return canalSup === canalActivo;
      const asesoresPeriodo = filtrarAsesoresPorCanal(Array.isArray(datosMeses?.[periodoCompleto]) ? datosMeses[periodoCompleto] : []);
      return asesoresPeriodo.some(a => String(a.supervisor || '').trim() === supervisor && asesorEnCanal(a));
    }}

    const calidadIndicadores = [
      {{ key: 'puntualidad', label: 'Puntualidad' }},
      {{ key: 'ticket_pdp', label: 'Ticket PDP' }},
      {{ key: 'calidad_pdp', label: 'Calidad PDP' }},
      {{ key: 'cierre', label: 'Cierre' }},
      {{ key: 'condonacion', label: 'Condonación' }}
    ];

    const indicadorDetalles = {{
      puntualidad: ['falta', 'tardanza', 'tardanza_tiempo'],
      ticket_pdp: ['monto_pdp', 'pdp'],
      calidad_pdp: ['recupero', 'monto_pdp'],
      cierre: ['pdp', 'cef_unico'],
      condonacion: ['pago_condonado', 'dk']
    }};

    const indicadorLabels = {{
      puntualidad: 'Puntualidad',
      falta: 'Falta',
      tardanza: 'Tardanza',
      tardanza_tiempo: 'Tardanza Tiempo',
      ticket_pdp: 'Ticket PDP',
      monto_pdp: 'Monto PDP',
      pdp: 'PDP',
      calidad_pdp: 'Calidad PDP',
      recupero: 'Recupero',
      cierre: 'Cierre',
      cef_unico: 'CEF Unico',
      condonacion: 'Condonación',
      pago_condonado: 'Pago Condonado',
      dk: 'DK'
    }};

    const columnasCantidadIndicadores = new Set([
      'cef_unico', 'pdp', 'monto_pdp', 'dk', 'pago_condonado', 'recupero', 'ticket_pdp'
    ]);

    const gruposDetalleAbiertos = new Set();

    function formatearPorcentajeCalidad(valor) {{
      const num = Number(valor);
      if (!Number.isFinite(num)) return '&mdash;';
      return `${{num.toFixed(1)}}%`;
    }}

    function formatearEstadoBinario(valor) {{
      const num = Number(valor);
      if (Number.isFinite(num) && num > 0) return '<span class="estado-ind estado-x">&#10005;</span>';
      return '<span class="estado-ind estado-ok">&#10003;</span>';
    }}

    function formatearHoraIndicador(valor, marcarRojo = false) {{
      const num = Number(valor);
      if (!Number.isFinite(num) || num <= 0) return '<span class="hora-ind">0:00:00</span>';
      const total = Math.round(num * 24 * 60 * 60);
      const horas = Math.floor(total / 3600);
      const minutos = Math.floor((total % 3600) / 60);
      const segundos = total % 60;
      const texto = `${{horas}}:${{String(minutos).padStart(2, '0')}}:${{String(segundos).padStart(2, '0')}}`;
      return `<span class="hora-ind ${{marcarRojo ? 'hora-alerta' : ''}}">${{texto}}</span>`;
    }}

    function formatearDatoIndicador(valor, key = '', contexto = null) {{
      if (key === 'tardanza' || key === 'falta') return formatearEstadoBinario(valor);
      if (key === 'tardanza_tiempo') {{
        const valorTiempo = Number(valor);
        return formatearHoraIndicador(valorTiempo, Number.isFinite(valorTiempo) && valorTiempo > 0);
      }}
      const num = Number(valor);
      if (!Number.isFinite(num)) return '&mdash;';
      if (columnasCantidadIndicadores.has(key)) return num.toLocaleString('es-PE', {{ maximumFractionDigits: 2 }});
      return formatearPorcentajeCalidad(num);
    }}

    function obtenerDatoCalidad(asesor, key) {{
      const calidad = asesor.indicadores_calidad || {{}};
      return calidad.indicadores?.[key] || null;
    }}

    function obtenerRegistrosCalidad(asesor) {{
      const calidad = asesor.indicadores_calidad || {{}};
      const registros = calidadIndicadores
        .map(ind => Number(calidad.indicadores?.[ind.key]?.registros || 0))
        .filter(n => Number.isFinite(n));
      return registros.length ? Math.max(...registros) : Number(calidad.dias_registrados || 0);
    }}

    const filasCalidadAbiertas = new Set();

    function obtenerColumnasCalidadVisibles() {{
      const columnas = [];
      calidadIndicadores.forEach(ind => {{
        columnas.push([ind.key, indicadorLabels[ind.key], 'principal']);
        if (gruposDetalleAbiertos.has(ind.key)) {{
          (indicadorDetalles[ind.key] || []).forEach(key => columnas.push([key, indicadorLabels[key] || key, ind.key]));
        }}
      }});
      return columnas;
    }}

    function actualizarColspansIndicadores() {{
      calidadIndicadores.forEach(ind => {{
        const abierto = gruposDetalleAbiertos.has(ind.key);
        const th = document.querySelector(`[data-grupo-head="${{ind.key}}"]`);
        if (th) {{
          th.classList.toggle('expandido', abierto);
        }}
      }});
    }}

    function claseDetalle(grupo) {{
      return gruposDetalleAbiertos.has(grupo) ? 'detalle-col visible' : 'detalle-col';
    }}

    function crearIdCalidad(asesor, index) {{
      const base = `${{asesor.nombre || asesor.alias_crr || 'asesor'}}-${{asesor.supervisor || ''}}-${{index}}`;
      return base.normalize('NFD').replace(/[\u0300-\u036f]/g, '').replace(/[^A-Za-z0-9_-]+/g, '-');
    }}

    function obtenerColspanCalidadVisible() {{
      return 3 + obtenerColumnasCalidadVisibles().length;
    }}

    function renderCeldasDiariasCalidad(dia) {{
      return obtenerColumnasCalidadVisibles()
        .map(([key, , grupo]) => `<td class="col-num calidad-dia-valor ${{grupo === 'principal' ? 'dia-principal' : 'dia-subcol dia-' + grupo}}">${{formatearDatoIndicador(dia.indicadores?.[key], key, dia.indicadores || {{}})}}</td>`)
        .join('');
    }}

    function renderDetalleDiarioCalidad(asesor, filaId, filaAbierta) {{
      const detalle = asesor.indicadores_calidad?.detalle_diario || [];
      if (!detalle.length) {{
        return `
          <tr class="calidad-dia-row ${{filaAbierta ? 'abierta' : ''}}" data-parent-calidad="${{filaId}}">
            <td colspan="${{obtenerColspanCalidadVisible()}}" class="calidad-dia-empty">Sin detalle diario para este asesor.</td>
          </tr>
        `;
      }}
      return detalle.map((dia, diaIndex) => `
        <tr class="calidad-dia-row ${{filaAbierta ? 'abierta' : ''}}" data-parent-calidad="${{filaId}}" style="--delay:${{Math.min(diaIndex, 10) * 18}}ms">
          <td colspan="3" class="calidad-dia-fecha"><span>${{dia.fecha || '&mdash;'}}</span></td>
          ${{renderCeldasDiariasCalidad(dia)}}
        </tr>
      `).join('');
    }}

    function toggleDetalleDiarioCalidad(filaId) {{
      if (filasCalidadAbiertas.has(filaId)) filasCalidadAbiertas.delete(filaId);
      else filasCalidadAbiertas.add(filaId);
      document.querySelectorAll(`[data-calidad-id="${{filaId}}"], [data-parent-calidad="${{filaId}}"]`).forEach(el => {{
        el.classList.toggle('abierta', filasCalidadAbiertas.has(filaId));
      }});
    }}

    function renderizarTablaCalidad(items, mensajeVacio = 'Sin indicadores para mostrar.') {{
      const tbody = document.getElementById('tbodyCalidadIndicadores');
      if (!tbody) return;
      if (!items.length) {{
        tbody.innerHTML = `<tr><td colspan="${{obtenerColspanCalidadVisible()}}" class="calidad-tabla-vacia">${{mensajeVacio}}</td></tr>`;
        return;
      }}

      const celdaBarra = (valorEntrada, tipo) => {{
        const valor = Number(valorEntrada);
        const ancho = Number.isFinite(valor) ? Math.max(0, Math.min(100, valor)) : 0;
        const clase = Number.isFinite(valor) ? tipo : 'neutral';
        return `<div class="barra-dato ${{clase}}" style="--valor:${{ancho.toFixed(2)}}%;"><span>${{formatearPorcentajeCalidad(valor)}}</span></div>`;
      }};

      const celdaDetalle = (asesor, key, grupo) => {{
        const dato = obtenerDatoCalidad(asesor, key);
        return `<td class="col-num ${{claseDetalle(grupo)}} detalle-${{grupo}}">${{formatearDatoIndicador(dato?.valor, key)}}</td>`;
      }};

      window.asesoresCalidadRenderizados = items;
      tbody.innerHTML = items.map((asesor, index) => {{
        const filaId = crearIdCalidad(asesor, index);
        const filaAbierta = filasCalidadAbiertas.has(filaId);
        const puntualidad = obtenerDatoCalidad(asesor, 'puntualidad');
        const cierre = obtenerDatoCalidad(asesor, 'cierre');
        const condonacion = obtenerDatoCalidad(asesor, 'condonacion');
        const calidadPdp = obtenerDatoCalidad(asesor, 'calidad_pdp');
        return `
          <tr class="calidad-main-row ${{filaAbierta ? 'abierta' : ''}}" data-calidad-id="${{filaId}}" data-asesor-index="${{index}}" onclick="toggleDetalleDiarioCalidad('${{filaId}}')">
            <td><button type="button" class="calidad-asesor-toggle"><span class="calidad-chevron">&rsaquo;</span>${{asesor.nombre || asesor.alias_crr || 'Sin nombre'}}</button></td>
            <td>${{asesor.supervisor || 'Sin Supervisor'}}</td>
            <td>${{asesor.cartera || asesor.segmento || 'No definida'}}</td>
            <td class="col-num calidad-bar-cell">${{celdaBarra(puntualidad?.valor, 'puntualidad')}}</td>
            ${{celdaDetalle(asesor, 'falta', 'puntualidad')}}
            ${{celdaDetalle(asesor, 'tardanza', 'puntualidad')}}
            ${{celdaDetalle(asesor, 'tardanza_tiempo', 'puntualidad')}}
            <td class="col-num calidad-bar-cell">${{celdaBarra(cierre?.valor, 'cierre')}}</td>
            ${{celdaDetalle(asesor, 'pdp', 'cierre')}}
            ${{celdaDetalle(asesor, 'cef_unico', 'cierre')}}
            <td class="col-num calidad-bar-cell">${{celdaBarra(condonacion?.valor, 'condonacion')}}</td>
            ${{celdaDetalle(asesor, 'monto_pdp', 'condonacion')}}
            ${{celdaDetalle(asesor, 'dk', 'condonacion')}}
            <td class="col-num calidad-bar-cell">${{celdaBarra(calidadPdp?.valor, 'calidad-pdp')}}</td>
            ${{celdaDetalle(asesor, 'recupero', 'calidad_pdp')}}
            ${{celdaDetalle(asesor, 'ticket_pdp', 'calidad_pdp')}}
          </tr>
          ${{renderDetalleDiarioCalidad(asesor, filaId, filaAbierta)}}
        `;
      }}).join('');
      actualizarBotonesOrdenCalidad();
    }}

    function sincronizarColumnasDetalle() {{
      actualizarColspansIndicadores();
      document.querySelectorAll('.indicador-toggle').forEach(btn => {{
        const grupo = btn.dataset.grupo;
        btn.classList.toggle('activo', gruposDetalleAbiertos.has(grupo));
        btn.textContent = gruposDetalleAbiertos.has(grupo) ? '-' : '+';
      }});
      document.querySelectorAll('.detalle-col').forEach(cell => {{
        const grupo = ['puntualidad', 'cierre', 'condonacion', 'calidad_pdp']
          .find(g => cell.classList.contains(`detalle-${{g}}`));
        cell.classList.toggle('visible', !!grupo && gruposDetalleAbiertos.has(grupo));
      }});
    }}

    function toggleIndicadorDetalle(grupo) {{
      if (gruposDetalleAbiertos.has(grupo)) gruposDetalleAbiertos.delete(grupo);
      else gruposDetalleAbiertos.add(grupo);
      sincronizarColumnasDetalle();
      if (Array.isArray(window.asesoresCalidadRenderizados)) {{
        renderizarTablaCalidad(window.asesoresCalidadRenderizados);
        sincronizarColumnasDetalle();
      }}
    }}

    function aplicarOrdenCalidad(items) {{
      if (!calidadOrdenActual) return items;
      return items.slice().sort((a, b) => {{
        const av = Number(obtenerDatoCalidad(a, calidadOrdenActual)?.valor);
        const bv = Number(obtenerDatoCalidad(b, calidadOrdenActual)?.valor);
        const an = Number.isFinite(av) ? av : -Infinity;
        const bn = Number.isFinite(bv) ? bv : -Infinity;
        return bn - an;
      }});
    }}

    function ordenarCalidadPor(key) {{
      calidadOrdenActual = key;
      const base = Array.isArray(window.asesoresCalidadBaseRender) ? window.asesoresCalidadBaseRender : window.asesoresCalidadRenderizados;
      if (!Array.isArray(base)) return;
      renderizarTablaCalidad(aplicarOrdenCalidad(base));
      sincronizarColumnasDetalle();
    }}

    function actualizarBotonesOrdenCalidad() {{
      document.querySelectorAll('.indicador-sort').forEach(btn => {{
        btn.classList.toggle('activo', btn.dataset.sortKey === calidadOrdenActual);
      }});
    }}

    function limpiarCalidad(mensaje = 'No hay indicadores para este periodo.') {{
      renderizarTablaCalidad([], mensaje);
      const resumen = document.getElementById('calidadResumen');
      if (resumen) resumen.textContent = mensaje;
      const stats = document.getElementById('calidadStatsAlcances');
      if (stats) stats.innerHTML = '';
      const quintiles = document.getElementById('calidadQuintiles');
      if (quintiles) quintiles.innerHTML = '';
      sincronizarColumnasDetalle();
    }}

    function obtenerSupervisoresParaResumen(periodoCompleto, supervisorFiltro = 'TODOS') {{
      if (supervisorFiltro && supervisorFiltro !== 'TODOS') {{
        const datos = datosSupervisores?.[supervisorFiltro]?.[periodoCompleto];
        return datos ? [{{ nombre: supervisorFiltro, datos }}] : [];
      }}
      return Object.keys(datosSupervisores || {{}})
        .filter(nombre => supervisorEnCanal(nombre, periodoCompleto))
        .map(nombre => ({{ nombre, datos: datosSupervisores?.[nombre]?.[periodoCompleto] }}))
        .filter(item => item.datos);
    }}

    function calcularResumenAlcanceIndicadores(periodoCompleto, supervisorFiltro = 'TODOS') {{
      const supervisores = obtenerSupervisoresParaResumen(periodoCompleto, supervisorFiltro);
      if (!supervisores.length) return null;

      let metaTotalMes = 0;
      let recuperoTotalActual = 0;
      supervisores.forEach(item => {{
        metaTotalMes += Number(item.datos.meta_super || 0);
        recuperoTotalActual += Number(item.datos.total_recupero || 0);
      }});

      const fechas = new Set();
      supervisores.forEach(item => {{
        Object.keys(item.datos.datos_diarios_supervisor || {{}}).forEach(fecha => fechas.add(fecha));
        if (!Object.keys(item.datos.datos_diarios_supervisor || {{}}).length) {{
          Object.keys(item.datos.alcance_acumulado_diario || {{}}).forEach(fecha => fechas.add(fecha));
        }}
      }});

      const fechasOrdenadas = Array.from(fechas).sort((a, b) => convertirFechaDiaria(a) - convertirFechaDiaria(b));
      const diasLaborables = 0;
      let diasTrabajados = 0;

      fechasOrdenadas.forEach((fecha, index) => {{
        let recuperoDiaTotal = 0;
        supervisores.forEach(item => {{
          const diario = item.datos.datos_diarios_supervisor || {{}};
          if (diario[fecha] === undefined) return;
          const actual = Number(diario[fecha] || 0);
          const anterior = index > 0 ? Number(diario[fechasOrdenadas[index - 1]] || 0) : 0;
          recuperoDiaTotal += index > 0 ? actual - anterior : actual;
        }});
        if (recuperoDiaTotal > 0) diasTrabajados++;
      }});

      const alcanceActual = metaTotalMes > 0 ? (recuperoTotalActual / metaTotalMes) * 100 : 0;
      return {{
        metaTotalMes,
        recuperoTotalActual,
        alcanceActual,
        diasLaborables,
        diasTrabajados
      }};
    }}

    function renderizarTarjetasIndicadores(periodoCompleto, supervisorFiltro = 'TODOS') {{
      const cont = document.getElementById('calidadStatsAlcances');
      if (!cont) return;
      const datos = calcularResumenAlcanceIndicadores(periodoCompleto, supervisorFiltro);
      if (!datos) {{
        cont.innerHTML = '';
        return;
      }}

      const formatoMoneda = (valor) => 'S/ ' + Number(valor || 0).toLocaleString('es-PE', {{ minimumFractionDigits: 0 }});
      const dl = Number(datos.diasLaborables || 0);
      const dt = Number(datos.diasTrabajados || 0);
      const dtCap = dl > 0 ? Math.min(dt, dl) : dt;
      const ratioTrabajados = dl > 0 ? `${{dtCap}}/${{dl}}` : '0/0';
      const porcentajeTiempo = dl > 0 ? ((dtCap / dl) * 100).toFixed(0) : 0;
      const colorAlcanceActual = datos.alcanceActual >= 100 ? '#27ae60' :
        datos.alcanceActual >= 70 ? '#f39c12' :
        datos.alcanceActual >= 40 ? '#e67e22' : '#e74c3c';
      const etiqueta = supervisorFiltro && supervisorFiltro !== 'TODOS' ? 'Supervisor seleccionado' : 'Todos los equipos';

      cont.innerHTML = `
        <div class="calidad-stats-grid">
          <div class="calidad-stat-card stat-dias">
            <div class="stat-label">DIAS TRABAJADOS</div>
            <div class="stat-value">${{ratioTrabajados}}</div>
            <div class="stat-note">${{porcentajeTiempo}}% del mes</div>
          </div>
          <div class="calidad-stat-card stat-meta">
            <div class="stat-label">META DEL MES</div>
            <div class="stat-value">${{formatoMoneda(datos.metaTotalMes)}}</div>
            <div class="stat-note">${{etiqueta}}</div>
          </div>
          <div class="calidad-stat-card stat-recupero">
            <div class="stat-label">RECUPERO ACTUAL</div>
            <div class="stat-value">${{formatoMoneda(datos.recuperoTotalActual)}}</div>
            <div class="stat-note">Total recuperado</div>
          </div>
          <div class="calidad-stat-card stat-alcance">
            <div class="stat-label">ALCANCE ACTUAL</div>
            <div class="stat-value" style="color:${{colorAlcanceActual}};">${{datos.alcanceActual.toFixed(2)}}%</div>
            <div class="stat-note">${{formatoMoneda(datos.recuperoTotalActual)}} / ${{formatoMoneda(datos.metaTotalMes)}}</div>
          </div>
        </div>
      `;
    }}

    function obtenerQuintilesPorIndicador(asesores, key) {{
      const items = (Array.isArray(asesores) ? asesores : [])
        .map(asesor => {{
          const nombre = String(asesor.nombre || asesor.alias_crr || '').trim();
          const qAlc = key === 'alcance' ? _normalizarQuintil(asesor?.q_alc) : null;
          const valor = key === 'alcance'
            ? Number(asesor?.porcentaje)
            : Number(obtenerDatoCalidad(asesor, key)?.valor);
          return {{ nombre, valor, qAlc }};
        }})
        .filter(item => item.nombre && Number.isFinite(item.valor))
        .sort((a, b) => b.valor - a.valor);

      const total = items.length;
      const grupos = {{ 1: [], 2: [], 3: [], 4: [], 5: [] }};
      items.forEach((item, index) => {{
        const q = key === 'alcance' && item.qAlc
          ? item.qAlc
          : (total <= 1 ? 5 : Math.max(1, Math.min(5, 5 - Math.floor((index * 5) / total))));
        grupos[q].push(item);
      }});
      return grupos;
    }}

    function _setCalidadQuintilAbierto(key, q, abierto) {{
      const lista = document.getElementById(`calidadQuintil_${{key}}_Q${{q}}`);
      const toggle = document.getElementById(`calidadQuintilToggle_${{key}}_Q${{q}}`);
      const tarjeta = lista?.closest('.eval-quintil-card');
      if (lista) {{
        lista.classList.toggle('colapsada', !abierto);
        lista.setAttribute('aria-hidden', String(!abierto));
      }}
      if (tarjeta) {{
        tarjeta.classList.toggle('quintil-abierto', abierto);
        tarjeta.setAttribute('aria-expanded', String(abierto));
      }}
      if (toggle) toggle.textContent = abierto ? '-' : '+';
    }}

    function toggleCalidadQuintil(key, q) {{
      const lista = document.getElementById(`calidadQuintil_${{key}}_Q${{q}}`);
      if (!lista) return;
      _setCalidadQuintilAbierto(key, q, lista.classList.contains('colapsada'));
    }}

    function renderizarQuintilesIndicadores(asesores, periodoCompleto, supervisorFiltro = 'TODOS') {{
      const cont = document.getElementById('calidadQuintiles');
      if (!cont) return;
      const indicadores = [
        {{ key: 'calidad_pdp', label: 'Calidad PDP' }},
        {{ key: 'cierre', label: 'Cierre' }},
        {{ key: 'condonacion', label: 'Condonación' }},
        {{ key: 'puntualidad', label: 'Puntualidad' }},
        {{ key: 'alcance', label: 'Alcance' }}
      ];
      if (!indicadores.some(ind => ind.key === calidadQuintilSeleccionado)) {{
        calidadQuintilSeleccionado = 'calidad_pdp';
      }}

      const ind = indicadores.find(item => item.key === calidadQuintilSeleccionado) || indicadores[0];
      const grupos = obtenerQuintilesPorIndicador(asesores, ind.key);
      const total = Object.values(grupos).reduce((acc, arr) => acc + arr.length, 0);
      const cards = [5, 4, 3, 2, 1].map(q => {{
        const items = grupos[q] || [];
        const valores = items.map(it => it.valor).filter(Number.isFinite);
        const promedio = valores.length ? valores.reduce((a, b) => a + b, 0) / valores.length : null;
        return `
          <div class="eval-quintil-card eval-quintil-q${{q}}" data-quintil="${{q}}" onclick="toggleCalidadQuintil('${{ind.key}}', ${{q}})">
            <div class="eval-quintil-title"><span>Q${{q}}</span><span class="eval-quintil-toggle" id="calidadQuintilToggle_${{ind.key}}_Q${{q}}">+</span></div>
            <div class="eval-quintil-summary">
              <div class="eval-quintil-metric">
                <span class="eval-quintil-metric-label">Asesores</span>
                <span class="eval-quintil-metric-value">${{items.length}}</span>
              </div>
              <div class="eval-quintil-metric">
                <span class="eval-quintil-metric-label">Promedio</span>
                <span class="eval-quintil-metric-value">${{promedio === null ? '-' : `${{promedio.toFixed(2)}}%`}}</span>
              </div>
            </div>
            <div class="eval-quintil-list colapsada" id="calidadQuintil_${{ind.key}}_Q${{q}}" onclick="event.stopPropagation()">
              ${{items.map((it, index) => `
                <div class="eval-quintil-row" style="--quintil-delay:${{Math.min(index, 10) * 32}}ms">
                  <div class="eval-quintil-asesor" title="${{it.nombre}}">${{it.nombre}}</div>
                  <div class="eval-quintil-alcance">${{it.valor.toFixed(2)}}%</div>
                </div>
              `).join('') || '<div style="padding:14px 8px; text-align:center; color:#666; font-size:0.85rem;">Sin registros</div>'}}
            </div>
          </div>
        `;
      }}).join('');

      const etiquetaSupervisor = supervisorFiltro && supervisorFiltro !== 'TODOS' ? supervisorFiltro : 'todos los equipos';
      const opciones = indicadores.map(item => `
        <option value="${{item.key}}" ${{item.key === ind.key ? 'selected' : ''}}>${{item.label}}</option>
      `).join('');
      cont.innerHTML = `
        <section class="calidad-quintil-bloque">
          <div class="eval-quintil-toolbar" style="justify-content:space-between; align-items:end; gap:12px; flex-wrap:wrap;">
            <h3 style="margin:0;">Quintiles de ${{ind.label}}</h3>
            <label style="display:flex; flex-direction:column; gap:5px; font-size:0.85rem; color:#5f6b7a; font-weight:700;">
              Ver quintil
              <select id="selectorCalidadQuintil" onchange="cambiarCalidadQuintil(this.value)" style="min-width:220px; padding:9px 12px; border:1px solid #d9e1ec; border-radius:8px; font-weight:700; color:#2c3e50; background:#fff;">
                ${{opciones}}
              </select>
            </label>
          </div>
          <div class="calidad-quintil-cards">${{cards}}</div>
          <div class="calidad-quintil-nota">${{total}} asesores evaluados · ${{String(periodoCompleto || '').replace('_', ' ')}} · ${{etiquetaSupervisor}}</div>
        </section>
      `;
    }}

    function cambiarCalidadQuintil(key) {{
      calidadQuintilSeleccionado = key || 'calidad_pdp';
      const base = Array.isArray(window.asesoresCalidadBaseRender) ? window.asesoresCalidadBaseRender : [];
      const mesSeleccionado = document.getElementById('selectorMes')?.value || '';
      const selectorAnioCalidad = document.getElementById('selectorAño') || document.querySelector('select[id^="selectorA"]');
      const anioSeleccionado = selectorAnioCalidad?.value || '';
      const periodoCompleto = `${{mesSeleccionado}}_${{anioSeleccionado}}`;
      renderizarQuintilesIndicadores(base, periodoCompleto, supervisorFiltroActual || 'TODOS');
    }}

    function actualizarCalidad(supervisorParam = '') {{
      const mesSeleccionado = document.getElementById('selectorMes')?.value;
      const selectorAnioCalidad = document.getElementById('selectorA\u00f1o') || document.querySelector('select[id^="selectorA"]');
      const anioSeleccionado = selectorAnioCalidad?.value;
      const periodoCompleto = `${{mesSeleccionado}}_${{anioSeleccionado}}`;
      const asesoresPeriodo = filtrarAsesoresPorCanal(Array.isArray(datosMeses?.[periodoCompleto]) ? datosMeses[periodoCompleto] : []);
      if (!asesoresPeriodo.length) {{
        limpiarCalidad('No hay asesores para el periodo seleccionado.');
        return;
      }}

      let supervisorFiltro = (supervisorParam || '').trim();
      if (!supervisorFiltro) supervisorFiltro = supervisorFiltroActual || 'TODOS';

      const asesoresFiltrados = supervisorFiltro && supervisorFiltro !== 'TODOS'
        ? asesoresPeriodo.filter(a => String(a.supervisor || '').trim() === supervisorFiltro)
        : asesoresPeriodo.slice();

      const asesoresConIndicadores = asesoresFiltrados
        .filter(asesor => calidadIndicadores.some(ind => Number.isFinite(Number(obtenerDatoCalidad(asesor, ind.key)?.valor))))
        .sort((a, b) => String(a.nombre || '').localeCompare(String(b.nombre || ''), 'es'));

      if (!asesoresConIndicadores.length) {{
        limpiarCalidad(`No hay indicadores para ${{supervisorFiltro || 'TODOS'}} en ${{periodoCompleto.replace('_', ' ')}}.`);
        return;
      }}

      window.asesoresCalidadBaseRender = asesoresConIndicadores;
      renderizarQuintilesIndicadores(asesoresConIndicadores, periodoCompleto, supervisorFiltro);
      renderizarTablaCalidad(aplicarOrdenCalidad(asesoresConIndicadores));
      sincronizarColumnasDetalle();

      const resumen = document.getElementById('calidadResumen');
      if (resumen) {{
        const etiquetaSupervisor = supervisorFiltro && supervisorFiltro !== 'TODOS' ? supervisorFiltro : 'todos los equipos';
        resumen.textContent = `Periodo ${{periodoCompleto.replace('_', ' ')}} - ${{etiquetaSupervisor}} - ${{asesoresConIndicadores.length}} asesores con indicadores.`;
      }}
    }}

    function normalizarTextoAsesor(valor) {{
      return String(valor || '')
        .normalize('NFD')
        .replace(/[\u0300-\u036f]/g, '')
        .toUpperCase()
        .trim()
        .replace(/\s+/g, ' ');
    }}

    function ordenarPeriodosAsc(periodos) {{
      const ordenMes = {{
        ENERO: 1, FEBRERO: 2, MARZO: 3, ABRIL: 4, MAYO: 5, JUNIO: 6,
        JULIO: 7, AGOSTO: 8, SETIEMBRE: 9, SEPTIEMBRE: 9, OCTUBRE: 10,
        NOVIEMBRE: 11, DICIEMBRE: 12
      }};
      return periodos.slice().sort((a, b) => {{
        const [ma, aa] = String(a).split('_');
        const [mb, ab] = String(b).split('_');
        return (Number(aa) * 100 + (ordenMes[ma] || 0)) - (Number(ab) * 100 + (ordenMes[mb] || 0));
      }});
    }}

    function inicializarVistaAsesor() {{
      const datalist = document.getElementById('asesorSearchDatalist');
      if (datalist && datalist.dataset.ready !== '1') {{
        const aliases = Array.from(new Set((datosAsesoresInf || [])
          .map(r => String(r.alias || '').trim())
          .filter(Boolean)))
          .sort((a, b) => a.localeCompare(b, 'es'));
        datalist.innerHTML = aliases.map(alias => `<option value="${{alias}}"></option>`).join('');
        datalist.dataset.ready = '1';
      }}

      const input = document.getElementById('asesorSearchInput');
      if (input && !asesorSeleccionadoHistorico) {{
        input.value = '';
      }}
      renderizarVistaAsesorHistorico();
    }}

    function seleccionarAsesorHistoricoDesdeInput() {{
      const input = document.getElementById('asesorSearchInput');
      const valor = String(input?.value || '').trim();
      if (!valor) return;
      const clave = normalizarTextoAsesor(valor);
      const match = (datosAsesoresInf || []).find(r => normalizarTextoAsesor(r.alias) === clave)
        || (datosAsesoresInf || []).find(r => normalizarTextoAsesor(r.alias).includes(clave))
        || (datosAsesoresInf || []).find(r => normalizarTextoAsesor(r.nombre_completo).includes(clave));
      asesorSeleccionadoHistorico = match?.alias || valor;
      if (input) input.value = asesorSeleccionadoHistorico;
      renderizarVistaAsesorHistorico();
    }}

    function cambiarRangoAsesorHistorico(rango) {{
      asesorRangoHistorico = rango;
      renderizarVistaAsesorHistorico();
    }}

    function obtenerRegistrosInfAsesor(alias) {{
      const clave = normalizarTextoAsesor(alias);
      return (datosAsesoresInf || [])
        .filter(r => normalizarTextoAsesor(r.alias) === clave)
        .sort((a, b) => {{
          const vigencia = (String(b.vigencia || b.vigente || '').toUpperCase() === 'SI') - (String(a.vigencia || a.vigente || '').toUpperCase() === 'SI');
          if (vigencia) return vigencia;
          return String(b.id_fecha_ingreso || '').localeCompare(String(a.id_fecha_ingreso || ''));
        }});
    }}

    function obtenerIndicadorAsesorPeriodo(asesor, key) {{
      return Number(asesor?.indicadores_calidad?.indicadores?.[key]?.valor);
    }}

    function obtenerHistoricoAsesor(alias) {{
      const clave = normalizarTextoAsesor(alias);
      return ordenarPeriodosAsc(Object.keys(datosMeses || {{}})).map(periodo => {{
        const asesor = (datosMeses[periodo] || []).find(a =>
          normalizarTextoAsesor(a.nombre) === clave ||
          normalizarTextoAsesor(a.alias_crr) === clave ||
          normalizarTextoAsesor(a.indicadores_calidad?.alias) === clave
        );
        if (!asesor) return null;
        return {{
          periodo,
          supervisor: asesor.supervisor || '',
          segmento: asesor.cartera || asesor.segmento || '',
          alcance: Number(asesor.porcentaje),
          recupero: Number(asesor.recupero || 0),
          meta: Number(asesor.meta || 0),
          puntualidad: obtenerIndicadorAsesorPeriodo(asesor, 'puntualidad'),
          cierre: obtenerIndicadorAsesorPeriodo(asesor, 'cierre'),
          condonacion: obtenerIndicadorAsesorPeriodo(asesor, 'condonacion'),
          calidad: obtenerIndicadorAsesorPeriodo(asesor, 'calidad_pdp')
        }};
      }}).filter(Boolean);
    }}

    function filtrarHistoricoAsesor(historico) {{
      if (asesorRangoHistorico === 'ALL') return historico;
      const n = Number(asesorRangoHistorico || 12);
      return historico.slice(Math.max(0, historico.length - n));
    }}

    function promedioValores(items, key) {{
      const vals = items.map(x => Number(x[key])).filter(Number.isFinite);
      return vals.length ? vals.reduce((a, b) => a + b, 0) / vals.length : null;
    }}

    function tendenciaTexto(items, key) {{
      const vals = items.map(x => Number(x[key])).filter(Number.isFinite);
      if (vals.length < 2) return 'Sin data suficiente';
      const delta = vals[vals.length - 1] - vals[0];
      if (delta > 1) return `Mejora (+${{delta.toFixed(1)}} pts)`;
      if (delta < -1) return `Retroceso (${{delta.toFixed(1)}} pts)`;
      return 'Sin variación relevante';
    }}

    function tendenciaClase(texto) {{
      const t = String(texto || '').toLowerCase();
      if (t.includes('mejora')) return 'tendencia-mejora';
      if (t.includes('retroceso')) return 'tendencia-retroceso';
      return '';
    }}

    function percentilColor(percentil) {{
      const p = Math.max(0, Math.min(100, Number(percentil) || 0));
      const hue = Math.round((p / 100) * 120);
      return `hsl(${{hue}}, 62%, 42%)`;
    }}

    function renderPercentilKpi(periodos, key, promedio) {{
      const pct = percentilPromedioAsesor(periodos, key, promedio);
      if (!Number.isFinite(pct)) return '<div class="asesor-kpi-percentil"><span>P-</span><div class="asesor-kpi-percentil-bar" style="--pct:0%; opacity:.35;"></div></div>';
      const pctClamp = Math.max(0, Math.min(100, pct));
      return `
        <div class="asesor-kpi-percentil">
          <span style="color:${{percentilColor(pctClamp)}}">P${{pctClamp.toFixed(0)}}</span>
          <div class="asesor-kpi-percentil-bar" style="--pct:${{pctClamp}}%;"></div>
        </div>
      `;
    }}

    function renderValorComparado(valor, promedio) {{
      const val = Number(valor);
      const prom = Number(promedio);
      if (!Number.isFinite(val)) return formatearPctAsesor(valor);
      if (!Number.isFinite(prom)) return formatearPctAsesor(val);
      const clase = val >= prom ? 'arriba' : 'abajo';
      const titulo = val >= prom ? 'Sobre el promedio del rango' : 'Bajo el promedio del rango';
      return `<span class="tabla-valor-comparado" title="${{titulo}}"><span>${{formatearPctAsesor(val)}}</span><i class="tabla-triangulo ${{clase}}"></i></span>`;
    }}

    function desviacionValores(vals) {{
      const nums = vals.filter(Number.isFinite);
      if (!nums.length) return null;
      const prom = nums.reduce((a, b) => a + b, 0) / nums.length;
      const varianza = nums.reduce((acc, n) => acc + Math.pow(n - prom, 2), 0) / nums.length;
      return Math.sqrt(varianza);
    }}

    function keyIndicadorMensualAsesor(key) {{
      return key === 'calidad' ? 'calidad_pdp' : key;
    }}

    function percentilPromedioAsesor(periodos, key, promedioAsesor) {{
      if (!Number.isFinite(promedioAsesor) || !periodos.length) return null;
      const porAsesor = new Map();
      periodos.forEach(periodo => {{
        (datosMeses?.[periodo] || []).forEach(a => {{
          const nombre = String(a.nombre || a.alias_crr || a.indicadores_calidad?.alias || '').trim();
          if (!nombre) return;
          const valor = key === 'alcance' ? Number(a.porcentaje) : obtenerIndicadorAsesorPeriodo(a, keyIndicadorMensualAsesor(key));
          if (!Number.isFinite(valor)) return;
          if (!porAsesor.has(nombre)) porAsesor.set(nombre, []);
          porAsesor.get(nombre).push(valor);
        }});
      }});
      const promedios = Array.from(porAsesor.values())
        .map(vals => vals.reduce((a, b) => a + b, 0) / vals.length)
        .filter(Number.isFinite)
        .sort((a, b) => a - b);
      if (!promedios.length) return null;
      const debajoOIgual = promedios.filter(v => v <= promedioAsesor).length;
      return (debajoOIgual / promedios.length) * 100;
    }}

    function promedioParesPeriodo(periodos, key) {{
      const vals = [];
      periodos.forEach(periodo => {{
        (datosMeses?.[periodo] || []).forEach(a => {{
          const valor = key === 'alcance' ? Number(a.porcentaje) : obtenerIndicadorAsesorPeriodo(a, keyIndicadorMensualAsesor(key));
          if (Number.isFinite(valor)) vals.push(valor);
        }});
      }});
      return vals.length ? vals.reduce((a, b) => a + b, 0) / vals.length : null;
    }}

    function analizarDesempenoAsesor(items) {{
      const metricas = [
        ['Alcance', 'alcance', '#f1c40f'],
        ['Cierre', 'cierre', '#155bb5'],
        ['Condonacion', 'condonacion', '#8e44ad'],
        ['Puntualidad', 'puntualidad', '#219653'],
        ['Calidad', 'calidad', '#008c8c']
      ];
      const periodos = items.map(r => r.periodo);
      let puntaje = 0;
      let evaluadas = 0;
      let mejoras = 0;
      let retrocesos = 0;
      let sobrePares = 0;
      let volatilidadAlta = 0;

      const detalle = metricas.map(([label, key, color]) => {{
        const vals = items.map(r => Number(r[key])).filter(Number.isFinite);
        if (!vals.length) return {{ label, color, estado: 'Sin data', resumen: 'Sin registros en el rango.' }};
        const promedio = vals.reduce((a, b) => a + b, 0) / vals.length;
        const delta = vals.length >= 2 ? vals[vals.length - 1] - vals[0] : 0;
        const desv = desviacionValores(vals) || 0;
        const promedioPares = promedioParesPeriodo(periodos, key);
        const percentil = percentilPromedioAsesor(periodos, key, promedio);
        let score = 0;

        if (delta > 1.5) {{ score += 1; mejoras++; }}
        else if (delta < -1.5) {{ score -= 1; retrocesos++; }}

        if (Number.isFinite(promedioPares)) {{
          if (promedio >= promedioPares) {{ score += 1; sobrePares++; }}
          else if (promedio < promedioPares - 5) score -= 1;
        }}

        if (desv > 12) {{ score -= 0.5; volatilidadAlta++; }}

        puntaje += score;
        evaluadas++;
        const estado = score >= 1 ? 'favorable' : score <= -1 ? 'débil' : 'mixta';
        const posicion = Number.isFinite(percentil) ? `P${{percentil.toFixed(0)}}` : 'sin percentil';
        const paresTxt = Number.isFinite(promedioPares) ? `pares ${{promedioPares.toFixed(1)}}%` : 'pares s/d';
        return {{
          label,
          color,
          estado,
          resumen: `${{formatearPctAsesor(promedio)}} · ${{posicion}} · ${{paresTxt}} · Δ ${{delta.toFixed(1)}} pts`
        }};
      }});

      const promedioScore = evaluadas ? puntaje / evaluadas : 0;
      let diagnostico = 'INESTABLE';
      if (evaluadas >= 3 && promedioScore >= 0.45 && mejoras >= 2 && sobrePares >= 3 && volatilidadAlta <= 2) {{
        diagnostico = 'PRODUCTIVO';
      }} else if (evaluadas >= 3 && (promedioScore <= -0.35 || (retrocesos >= 3 && sobrePares <= 2))) {{
        diagnostico = 'DECADENTE';
      }}

      return {{
        diagnostico,
        detalle,
        resumen: `${{items.length}} meses evaluados · mejoras ${{mejoras}} · retrocesos ${{retrocesos}} · sobre pares ${{sobrePares}}/${{evaluadas}}`
      }};
    }}

    function formatearPctAsesor(valor) {{
      const num = Number(valor);
      return Number.isFinite(num) ? `${{num.toFixed(1)}}%` : '—';
    }}

    function renderizarFichaAsesor(registros) {{
      const cont = document.getElementById('asesorFicha');
      if (!cont) return;
      if (!registros.length) {{
        cont.innerHTML = '<div class="asesor-empty">No se encontró el asesor en la hoja INF.</div>';
        return;
      }}
      const actual = registros[0];
      const vigenciaActual = actual.vigencia || actual.vigente || '?';
      const cesadoActual = String(vigenciaActual || '').toUpperCase() === 'SI'
        ? 'NO'
        : (actual.fecha_salida || (String(actual.cesado || '').toUpperCase() === 'SI' ? 'SI' : (actual.cesado || '?')));
      cont.innerHTML = `
        <div class="asesor-ficha-grid">
          <div class="asesor-ficha-card"><span>Nombre completo</span><strong>${{actual.nombre_completo || '—'}}</strong></div>
          <div class="asesor-ficha-card"><span>Documento</span><strong>${{actual.documento || '—'}}</strong></div>
          <div class="asesor-ficha-card"><span>Fecha ingreso</span><strong>${{actual.fecha_ingreso || '—'}}</strong></div>
          <div class="asesor-ficha-card"><span>Fecha nacimiento</span><strong>${{actual.fecha_nacimiento || '—'}}</strong></div>
          <div class="asesor-ficha-card"><span>Vigencia</span><strong>${{vigenciaActual}}</strong></div>
          <div class="asesor-ficha-card"><span>Cesado</span><strong>${{cesadoActual}}</strong></div>
        </div>
        ${{registros.length > 1 ? `
          <div class="asesor-tabla-wrap" style="margin-bottom:14px;">
            <table class="asesor-tabla">
              <thead><tr><th>Alias</th><th>Ingreso</th><th>Nacimiento</th><th>Vigencia</th><th>Cesado</th></tr></thead>
              <tbody>
                ${{registros.map(r => `
                  <tr>
                    <td>${{r.alias || '—'}}</td>
                    <td>${{r.fecha_ingreso || '—'}}</td>
                    <td>${{r.fecha_nacimiento || '—'}}</td>
                    <td>${{r.vigencia || r.vigente || '—'}}</td>
                    <td>${{String(r.vigencia || r.vigente || '').toUpperCase() === 'SI' ? 'NO' : (r.fecha_salida || (String(r.cesado || '').toUpperCase() === 'SI' ? 'SI' : (r.cesado || '?')))}}</td>
                  </tr>`).join('')}}
              </tbody>
            </table>
          </div>` : ''}}
      `;
    }}

    function renderizarKpisAsesor(items) {{
      const cont = document.getElementById('asesorKpis');
      if (!cont) return;
      const periodos = items.map(r => r.periodo);
      const kpis = [
        ['Alcance', 'alcance', promedioValores(items, 'alcance')],
        ['Cierre', 'cierre', promedioValores(items, 'cierre')],
        ['Condonacion', 'condonacion', promedioValores(items, 'condonacion')],
        ['Puntualidad', 'puntualidad', promedioValores(items, 'puntualidad')],
        ['Calidad', 'calidad', promedioValores(items, 'calidad')]
      ];
      cont.innerHTML = `
        <div class="asesor-kpi-grid">
          ${{kpis.map(([label, key, value]) => `
            <div class="asesor-kpi-card">
              <div class="asesor-kpi-main">
                <span>${{label}} promedio</span>
                <strong>${{formatearPctAsesor(value)}}</strong>
              </div>
              ${{renderPercentilKpi(periodos, key, value)}}
            </div>
          `).join('')}}
        </div>
      `;
    }}

    function renderizarTendenciasAsesor(items) {{
      const cont = document.getElementById('asesorTendencias');
      if (!cont) return;
      const rows = [
        ['Alcance', 'alcance', '#f1c40f'],
        ['Cierre', 'cierre', '#155bb5'],
        ['Condonacion', 'condonacion', '#8e44ad'],
        ['Puntualidad', 'puntualidad', '#219653'],
        ['Calidad', 'calidad', '#008c8c']
      ];
      const analisis = analizarDesempenoAsesor(items);
      cont.innerHTML = `
        <h3>Tendencia del rango</h3>
        <div class="asesor-tendencia-list">
          ${{rows.map(([label, key, color]) => {{
            const texto = tendenciaTexto(items, key);
            return `<div class="asesor-tendencia-item"><span class="asesor-tendencia-label"><span class="asesor-metrica-dot" style="background:${{color}}"></span>${{label}}</span><strong class="${{tendenciaClase(texto)}}">${{texto}}</strong></div>`;
          }}).join('')}}
        </div>
        <div class="asesor-analisis-final asesor-analisis-${{analisis.diagnostico.toLowerCase()}}" onclick="toggleAnalisisAsesor(this)">
          <div class="asesor-analisis-head">
            <div class="asesor-analisis-title">
              <span>Análisis final</span>
              <strong>${{analisis.diagnostico}}</strong>
            </div>
            <div class="asesor-analisis-toggle">+</div>
          </div>
          <div class="asesor-analisis-detalle" onclick="event.stopPropagation()">
            <small>${{analisis.resumen}}</small>
            <div class="asesor-analisis-list">
              ${{analisis.detalle.map(item => `<div><b>${{item.label}}</b><em>${{item.resumen}}</em></div>`).join('')}}
            </div>
          </div>
        </div>
      `;
    }}

    function toggleAnalisisAsesor(card) {{
      if (!card) return;
      card.classList.toggle('abierto');
      const toggle = card.querySelector('.asesor-analisis-toggle');
      if (toggle) toggle.textContent = card.classList.contains('abierto') ? '-' : '+';
      setTimeout(() => {{
        if (chartAsesorHistorico) chartAsesorHistorico.update('none');
        Object.values(chartsMetricasAsesor || {{}}).forEach(chart => chart && chart.update('none'));
      }}, 60);
    }}

    function renderizarTablaHistoricoAsesor(items) {{
      const cont = document.getElementById('asesorHistoricoTabla');
      if (!cont) return;
      if (!items.length) {{
        cont.innerHTML = '<div class="asesor-empty">No hay histórico mensual para el asesor seleccionado.</div>';
        return;
      }}
      const metricas = [
        {{ key: 'alcance', label: 'Alcance', color: '#f1c40f' }},
        {{ key: 'condonacion', label: 'Condonación', color: '#8e44ad' }},
        {{ key: 'cierre', label: 'Cierre', color: '#155bb5' }},
        {{ key: 'calidad', label: 'Calidad', color: '#008c8c' }},
        {{ key: 'puntualidad', label: 'Puntualidad', color: '#219653' }}
      ];
      cont.innerHTML = `
        <div class="seccion-busqueda asesor-comparacion-historico">
          <div class="asesor-metricas-grid">
            ${{metricas.map(metrica => {{
              const promedioMetrica = promedioValores(items, metrica.key);
              return `
              <div class="asesor-metrica-card">
                <div class="asesor-metrica-head">
                  <h3>${{metrica.label}}</h3>
                  <strong>${{formatearPctAsesor(promedioMetrica)}}</strong>
                </div>
                <div class="asesor-metrica-layout">
                  <div class="asesor-tabla-wrap">
                    <table class="asesor-tabla">
                      <thead>
                        <tr><th>Periodo</th><th>${{metrica.label}}</th><th>Supervisor</th><th>Segmento</th></tr>
                      </thead>
                      <tbody>
                        ${{items.slice().reverse().map(r => `
                          <tr>
                            <td>${{r.periodo.replace('_', ' ')}}</td>
                            <td>${{renderValorComparado(r[metrica.key], promedioMetrica)}}</td>
                            <td>${{r.supervisor || '—'}}</td>
                            <td>${{r.segmento || '—'}}</td>
                          </tr>
                        `).join('')}}
                      </tbody>
                    </table>
                  </div>
                  <div class="asesor-metrica-chart">
                    <canvas id="asesorMetricaChart_${{metrica.key}}"></canvas>
                  </div>
                </div>
              </div>
            `}}).join('')}}
          </div>
        </div>
      `;
      renderizarGraficosMetricasAsesor(items, metricas);
    }}

    function renderizarGraficosMetricasAsesor(items, metricas) {{
      Object.values(chartsMetricasAsesor || {{}}).forEach(chart => {{
        if (chart) chart.destroy();
      }});
      chartsMetricasAsesor = {{}};
      if (typeof Chart === 'undefined') return;

      const labels = items.map(r => r.periodo.replace('_', ' '));
      metricas.forEach(metrica => {{
        const canvas = document.getElementById(`asesorMetricaChart_${{metrica.key}}`);
        if (!canvas) return;
        chartsMetricasAsesor[metrica.key] = new Chart(canvas.getContext('2d'), {{
          type: 'line',
          data: {{
            labels,
            datasets: [{{
              label: metrica.label,
              data: items.map(r => Number.isFinite(r[metrica.key]) ? r[metrica.key] : null),
              borderColor: metrica.color,
              backgroundColor: `${{metrica.color}}22`,
              borderWidth: 3,
              fill: true,
              tension: 0.28,
              spanGaps: true,
              pointRadius: 4,
              pointHoverRadius: 6
            }}]
          }},
          options: {{
            responsive: true,
            maintainAspectRatio: false,
            interaction: {{ mode: 'nearest', intersect: false, axis: 'x' }},
            plugins: {{
              legend: {{ display: false }},
              tooltip: {{
                position: 'nearest',
                backgroundColor: 'rgba(35, 49, 66, 0.96)',
                titleFont: {{ size: 15, weight: '900', lineHeight: 1.25 }},
                bodyFont: {{ size: 15, weight: '800', lineHeight: 1.25 }},
                padding: 14,
                caretPadding: 8,
                boxPadding: 6,
                boxWidth: 12,
                boxHeight: 12,
                cornerRadius: 8,
                multiKeyBackground: '#ffffff',
                displayColors: true,
                callbacks: {{
                  label: context => `${{metrica.label}}: ${{Number(context.parsed.y || 0).toFixed(2)}}%`
                }}
              }}
            }},
            scales: {{
              y: {{
                beginAtZero: true,
                ticks: {{ callback: value => value + '%' }}
              }}
            }}
          }}
        }});
      }});
    }}

    function renderizarGraficoAsesor(items) {{
      const canvas = document.getElementById('asesorHistoricoChart');
      if (!canvas || typeof Chart === 'undefined') return;
      if (chartAsesorHistorico) {{
        chartAsesorHistorico.destroy();
        chartAsesorHistorico = null;
      }}
      const labels = items.map(r => r.periodo.replace('_', ' '));
      chartAsesorHistorico = new Chart(canvas.getContext('2d'), {{
        type: 'line',
        data: {{
          labels,
          datasets: [
            {{ label: 'Alcance', data: items.map(r => Number.isFinite(r.alcance) ? r.alcance : null), borderColor: '#f1c40f', backgroundColor: 'rgba(241,196,15,0.12)', tension: 0.28, spanGaps: true }},
            {{ label: 'Cierre', data: items.map(r => Number.isFinite(r.cierre) ? r.cierre : null), borderColor: '#155bb5', backgroundColor: 'rgba(21,91,181,0.10)', tension: 0.28, spanGaps: true }},
            {{ label: 'Condonación', data: items.map(r => Number.isFinite(r.condonacion) ? r.condonacion : null), borderColor: '#8e44ad', backgroundColor: 'rgba(142,68,173,0.10)', tension: 0.28, spanGaps: true }},
            {{ label: 'Puntualidad', data: items.map(r => Number.isFinite(r.puntualidad) ? r.puntualidad : null), borderColor: '#219653', backgroundColor: 'rgba(33,150,83,0.10)', tension: 0.28, spanGaps: true }},
            {{ label: 'Calidad', data: items.map(r => Number.isFinite(r.calidad) ? r.calidad : null), borderColor: '#008c8c', backgroundColor: 'rgba(0,140,140,0.10)', tension: 0.28, spanGaps: true }}
          ]
        }},
        options: {{
          responsive: true,
          maintainAspectRatio: false,
          interaction: {{ mode: 'nearest', intersect: false, axis: 'x' }},
          plugins: {{
            legend: {{
              position: 'bottom',
              labels: {{
                font: {{ size: 14, weight: '800' }},
                boxWidth: 18,
                padding: 16
              }}
            }},
            tooltip: {{
              position: 'nearest',
              backgroundColor: 'rgba(35, 49, 66, 0.96)',
              titleFont: {{ size: 15, weight: '900', lineHeight: 1.25 }},
              bodyFont: {{ size: 15, weight: '800', lineHeight: 1.25 }},
              padding: 14,
              caretPadding: 8,
              boxPadding: 6,
              boxWidth: 12,
              boxHeight: 12,
              cornerRadius: 8,
              multiKeyBackground: '#ffffff',
              displayColors: true
            }}
          }},
          scales: {{ y: {{ beginAtZero: true, ticks: {{ callback: value => value + '%' }} }} }}
        }}
      }});
    }}

    function renderizarVistaAsesorHistorico() {{
      document.querySelectorAll('[data-rango-asesor]').forEach(btn => {{
        btn.classList.toggle('activo', btn.dataset.rangoAsesor === asesorRangoHistorico);
      }});
      if (!asesorSeleccionadoHistorico) {{
        const ficha = document.getElementById('asesorFicha');
        ['asesorKpis', 'asesorTendencias', 'asesorHistoricoTabla'].forEach(id => {{
          const el = document.getElementById(id);
          if (el) el.innerHTML = '';
        }});
        if (chartAsesorHistorico) {{
          chartAsesorHistorico.destroy();
          chartAsesorHistorico = null;
        }}
        Object.values(chartsMetricasAsesor || {{}}).forEach(chart => chart && chart.destroy());
        chartsMetricasAsesor = {{}};
        if (ficha) ficha.innerHTML = '<div class="asesor-empty">Selecciona un asesor para ver su registro histórico.</div>';
        return;
      }}
      const registrosInf = obtenerRegistrosInfAsesor(asesorSeleccionadoHistorico);
      renderizarFichaAsesor(registrosInf);
      const historicoCompleto = obtenerHistoricoAsesor(asesorSeleccionadoHistorico);
      const historico = filtrarHistoricoAsesor(historicoCompleto);
      renderizarKpisAsesor(historico);
      renderizarTendenciasAsesor(historico);
      renderizarTablaHistoricoAsesor(historico);
      renderizarGraficoAsesor(historico);
    }}

    // ========== FUNCIÓN ÚNICA PARA ACTUALIZAR FILTROS ==========
    function actualizarFiltrosGlobales() {{
      const mesSeleccionado = document.getElementById('selectorMes').value;
      const añoSeleccionado = document.getElementById('selectorAño').value;
      const periodoCompleto = `${{mesSeleccionado}}_${{añoSeleccionado}}`;
    
      if (!datosMeses[periodoCompleto]) return;
    
      const asesores = filtrarAsesoresPorCanal(datosMeses[periodoCompleto]);
      const supervisores = [...new Set(asesores.map(a => a.supervisor).filter(s => s))].sort();
      const supervisorActivoValido = (
        supervisorFiltroActual &&
        supervisorFiltroActual !== 'TODOS' &&
        supervisores.includes(supervisorFiltroActual)
      ) ? supervisorFiltroActual : 'TODOS';
      supervisorFiltroActual = supervisorActivoValido;
    
      let html = `
        <button class="filtro-supervisor ${{supervisorActivoValido === 'TODOS' ? 'activo' : ''}}" data-supervisor="TODOS">
          👥 TODOS LOS EQUIPOS
        </button>
      `;
    
      supervisores.forEach(supervisor => {{
        html += `
          <button class="filtro-supervisor ${{supervisor === supervisorActivoValido ? 'activo' : ''}}" data-supervisor="${{supervisor}}">
            ${{supervisor}}
          </button>
        `;
      }});
    
      html += `
    
        <button class="filtro-supervisor accion-global" type="button" onclick="mostrarTop10()">
          🏅 TOP 10
        </button>
        
        <button class="filtro-supervisor accion-global" type="button" onclick="mostrarModalCumpleaños()">
          🎂 CUMPLEAÑOS
        </button>
      `;
    
      document.getElementById('barraFiltrosSupervisoresGlobal').innerHTML = html;
      actualizarSelectorSupervisorAsesores();
    
      // Event listeners únicos SOLO para filtros (los que tienen data-supervisor)
      document.querySelectorAll('.filtro-supervisor[data-supervisor]').forEach(btn => {{
        btn.addEventListener('click', function() {{
          supervisorFiltroActual = this.getAttribute('data-supervisor');
          aplicarFiltroSupervisor(supervisorFiltroActual);
        }});
      }});
    }}
    
    function actualizarSelectorSupervisorAsesores() {{
      const selector = document.getElementById('selectorSupervisorAsesores');
      if (!selector) return;

      const mesSeleccionado = document.getElementById('selectorMes')?.value || '';
      const anioSeleccionado = document.getElementById('selectorA\u00f1o')?.value || '';
      const periodoCompleto = (mesSeleccionado && anioSeleccionado) ? `${{mesSeleccionado}}_${{anioSeleccionado}}` : '';
      const asesoresPeriodo = filtrarAsesoresPorCanal(Array.isArray(datosMeses?.[periodoCompleto]) ? datosMeses[periodoCompleto] : []);
      const supervisoresPeriodo = [...new Set(
        asesoresPeriodo
          .map(a => String(a.supervisor || '').trim())
          .filter(s => s && s !== 'Sin Supervisor')
      )].sort((a, b) => a.localeCompare(b, 'es'));

      const valorActual = selector.value;
      selector.innerHTML = '<option value="">Seleccionar supervisor...</option>' +
        supervisoresPeriodo.map(supervisor => `<option value="${{supervisor}}">${{supervisor}}</option>`).join('');

      if (valorActual && supervisoresPeriodo.includes(valorActual)) {{
        selector.value = valorActual;
      }} else {{
        selector.value = '';
      }}
    }}

    // ========== FUNCIÓN ÚNICA PARA APLICAR FILTRO ==========
    function aplicarFiltroSupervisor(supervisor) {{
        supervisorFiltroActual = supervisor || 'TODOS';
        window.supervisorFiltroActual = supervisorFiltroActual;
        supervisor = supervisorFiltroActual;

        // Actualizar botón activo
        document.querySelectorAll('.filtro-supervisor').forEach(btn => {{
            btn.classList.remove('activo');
            if (btn.getAttribute('data-supervisor') === supervisor) {{
                btn.classList.add('activo');
            }}
        }});
        
        // Determinar sección activa
        const seccionActiva = document.querySelector('.seccion-contenido.activa');
        if (!seccionActiva) return;
        
        if (seccionActiva.id === 'seccion-ranking') {{
            aplicarFiltroRanking(supervisor);
        }} else if (seccionActiva.id === 'seccion-evaluacion') {{
            actualizarTarjetasEvaluacionRapida(supervisor);
            calcularPeriodoPrueba();
        }} else if (seccionActiva.id === 'seccion-calidad') {{
            actualizarCalidad(supervisor);
        }} else if (seccionActiva.id === 'seccion-gestiones') {{
            inicializarVistaAsesor();
        }}
    }}
    
    // ========== FUNCIONES DE NAVEGACIÓN ==========
    function mostrarSeccion(seccion) {{
      // Ocultar todas las secciones
      document.querySelectorAll('.seccion-contenido').forEach(sec => {{
        sec.classList.remove('activa');
      }});
      
      // Remover activo de todos los accesos
      document.querySelectorAll('.boton-modulo, .boton-acceso-secundario').forEach(boton => {{
        boton.classList.remove('activo');
      }});
      
      // Mostrar la seccion seleccionada
      const seccionObjetivo = document.getElementById(`seccion-${{seccion}}`);
      if (!seccionObjetivo) return;
      seccionObjetivo.classList.add('activa');
      const botonPrincipal = document.querySelector(`.boton-modulo.${{seccion}}`);
      const botonSecundario = document.querySelector(`.boton-acceso-secundario.${{seccion}}`);
      if (botonPrincipal) botonPrincipal.classList.add('activo');
      if (botonSecundario) botonSecundario.classList.add('activo');
      if (seccion === 'gestiones') {{
        inicializarVistaAsesor();
      }}
      
      // Aplicar filtro actual a la nueva sección
      setTimeout(() => {{
        aplicarFiltroSupervisor(supervisorFiltroActual);
      }}, 100);
    }}

    function mostrarMensajeSinDatos(canvas, periodoCompleto, supervisor = null) {{
        const ctx = canvas.getContext('2d');
        ctx.clearRect(0, 0, canvas.width, canvas.height);
        
        // Configurar el canvas para mensaje
        canvas.width = 1300;
        canvas.height = 200;
        canvas.style.display = 'block';
        canvas.style.margin = '0 auto';
        canvas.style.maxWidth = '100%';
        
        // Dibujar mensaje
        ctx.fillStyle = '#95a5a6';
        ctx.font = 'bold 24px Arial';
        ctx.textAlign = 'center';
        
        let mensaje = '📊 No hay datos de recupero acumulado disponibles';
        if (supervisor) {{
            mensaje = `📊 No hay datos de recupero para ${{supervisor}}`;
        }}
        
        ctx.fillText(mensaje, canvas.width / 2, canvas.height / 2 - 20);
        
        ctx.fillStyle = '#7f8c8d';
        ctx.font = '18px Arial';
        ctx.fillText(`Periodo: ${{periodoCompleto.replace('_', ' ')}}`, canvas.width / 2, canvas.height / 2 + 20);
    }}
    
    function forzarActualizacionPeriodo(seccionDestino) {{
        // Obtener valores actuales del header
        const mesSeleccionado = document.getElementById('selectorMes').value;
        const añoSeleccionado = document.getElementById('selectorAño').value;
        const periodoCompleto = `${{mesSeleccionado}}_${{añoSeleccionado}}`;
        
        console.log(`🔄 Forzando actualización para ${{seccionDestino}}: ${{periodoCompleto}}`);
        
        if (!datosMeses[periodoCompleto]) {{
            console.warn(`⚠️ No hay datos para ${{periodoCompleto}}`);
            return;
        }}
        
        const asesores = filtrarAsesoresPorCanal(datosMeses[periodoCompleto] || []);
        
        // Aplicar según la sección destino
        if (seccionDestino === 'ranking') {{
            actualizarFiltrosGlobales();
            aplicarFiltroRanking(supervisorFiltroActual);
        }}
        
    }}

    function aplicarFiltroAlcances(supervisor) {{
        const mesSeleccionado = document.getElementById('selectorMes').value;
        const añoSeleccionado = document.getElementById('selectorAño').value;
        const periodoCompleto = `${{mesSeleccionado}}_${{añoSeleccionado}}`;
    
        if (!supervisor || supervisor === 'TODOS') {{
            window.supervisorSeleccionado = '';
        }} else {{
            window.supervisorSeleccionado = String(supervisor).trim();
        }}
    
        if (supervisor === 'TODOS') {{
            calcularEstadisticasRecuperos(periodoCompleto);
            generarGraficasRecuperos(periodoCompleto);
        }} else {{
            calcularEstadisticasRecuperosPorSupervisor(periodoCompleto, supervisor);
            generarGraficasRecuperosSupervisor(periodoCompleto, supervisor);
        }}
    }}

    // ===================== HELPERS GLOBALES =====================
    window.__chartsByCanvasId = window.__chartsByCanvasId || {{}};
    
    function _destroyChartByCanvasId(canvasId) {{
      try {{
        const prev = window.__chartsByCanvasId[canvasId];
        if (prev) {{
          prev.destroy();
          window.__chartsByCanvasId[canvasId] = null;
        }}
      }} catch (e) {{}}
    }}
    
    function _storeChart(canvasId, chartInstance) {{
      window.__chartsByCanvasId[canvasId] = chartInstance;
    }}
    
    function _applyCanvasSize(canvas, view = 'full') {{
      if (!canvas) return;
    
      if (view === 'mini') {{
        canvas.width = 340;
        canvas.height = 220;
      }} else if (view === 'modal') {{
        canvas.width = 1100;
        canvas.height = 520;
      }} else {{
        canvas.width = 1200;
        canvas.height = 700;
      }}
    
      canvas.style.display = 'block';
      canvas.style.margin = '0 auto';
      canvas.style.maxWidth = '100%';
    }}
    
    function _tuneOptionsForView(options, view = 'full') {{
      const o = options || {{}};
    
      if (view === 'mini') {{
        if (o.plugins && o.plugins.title) o.plugins.title.display = false;
        if (o.plugins && o.plugins.legend) o.plugins.legend.display = false;
        if (o.plugins && o.plugins.tooltip) o.plugins.tooltip.enabled = false;
    
        if (o.scales) {{
          Object.keys(o.scales).forEach((k) => {{
            const sc = o.scales[k];
            if (sc && sc.ticks) {{
              sc.ticks.maxTicksLimit = 4;
              sc.ticks.font = sc.ticks.font || {{}};
              sc.ticks.font.size = 10;
            }}
          }});
        }}
      }}
    
      if (view === 'modal') {{
        if (o.plugins && o.plugins.tooltip) o.plugins.tooltip.enabled = true;
      }}
    
      return o;
    }}

    function _buildDataLabelsPluginTotal(view = 'full') {{
      return {{
        id: 'dataLabelsTotal',
        afterDatasetsDraw(chart, args, options) {{
          // ✅ En mini no dibujar etiquetas
          if (view === 'mini') return;
    
          const {{ ctx }} = chart;
          const meta = chart.getDatasetMeta(0);
    
          meta.data.forEach((point, index) => {{
            const value = chart.data.datasets[0].data[index];
            if (value === null || value === undefined) return;
    
            if (value > 0) {{
              ctx.save();
    
              const x = point.x;
              const y = point.y - 20;
    
              const porcentajeAlcance = value;
              const formattedValue = porcentajeAlcance.toFixed(1) + '%';
    
              // ✅ En modal puedes subir un poco el tamaño si quieres
              const fontSize = (view === 'modal') ? 16 : 15;
              ctx.font = `bold ${{fontSize}}px Arial`;
    
              ctx.textAlign = 'center';
              ctx.textBaseline = 'bottom';
    
              let textColor = '#2c3e50';
              if (porcentajeAlcance === 0) textColor = '#95a5a6';
              else if (porcentajeAlcance < 40) textColor = '#e74c3c';
              else if (porcentajeAlcance < 70) textColor = '#f39c12';
              else if (porcentajeAlcance < 100) textColor = '#27ae60';
              else textColor = '#2ecc71';
    
              ctx.fillStyle = textColor;
    
              ctx.shadowColor = 'rgba(255, 255, 255, 0.95)';
              ctx.shadowBlur = 8;
              ctx.shadowOffsetX = 0;
              ctx.shadowOffsetY = 0;
    
              ctx.fillText(formattedValue, x, y);
              ctx.restore();
            }}
          }});
        }}
      }};
    }}
    

    // ===================== TABLA COMPARATIVA 4M (ALCANCES) =====================
    function _mesTitulo(periodo) {{
      const p = String(periodo || '').split('_');
      const mes = (p[0] || '').toLowerCase();
      return mes ? (mes.charAt(0).toUpperCase() + mes.slice(1)) : String(periodo || '');
    }}
    
    function _anioDePeriodo(periodo) {{
      const p = String(periodo || '').split('_');
      return p[1] || '';
    }}
    
    function _mesIndex(mesUpper) {{
      const meses = ['ENERO','FEBRERO','MARZO','ABRIL','MAYO','JUNIO','JULIO','AGOSTO','SETIEMBRE','OCTUBRE','NOVIEMBRE','DICIEMBRE'];
      return Math.max(0, meses.indexOf(String(mesUpper || '').toUpperCase()));
    }}
    
    function _sortPeriodosDesc(periodos) {{
      return (periodos || []).slice().sort((a, b) => {{
        const pa = String(a).split('_');
        const pb = String(b).split('_');
        const ya = Number(pa[1] || 0), yb = Number(pb[1] || 0);
        if (ya !== yb) return yb - ya;
        return _mesIndex(pb[0]) - _mesIndex(pa[0]);
      }});
    }}
    
    window.canalAlcancesSeleccionado = window.canalSeleccionadoGlobal || window.canalAlcancesSeleccionado || 'SURCO';

    function generarControlesCanalesAlcances() {{
      const canalActivo = window.canalAlcancesSeleccionado || 'SURCO';
      const canales = ['TODOS LOS CANALES', 'SURCO', 'BPO'];
      return `
        <div class="canales-alcances" aria-label="Filtro de canal">
          ${{canales.map(canal => `
            <button type="button"
                    class="btn-canal-alcances ${{canal === canalActivo ? 'activo' : ''}}"
                    data-canal-alcances="${{canal}}"
                    onclick="seleccionarCanalAlcances('${{canal}}')">
              ${{canal}}
            </button>
          `).join('')}}
        </div>
      `;
    }}

    function seleccionarCanalAlcances(canal) {{
      window.canalSeleccionadoGlobal = normalizarCanalDashboard(canal || 'SURCO');
      window.canalAlcancesSeleccionado = window.canalSeleccionadoGlobal;
      supervisorFiltroActual = 'TODOS';
      window.supervisorFiltroActual = 'TODOS';
      window.supervisorSeleccionado = '';
      sincronizarBotonesCanalAlcances();
      actualizarFiltrosGlobales();
      aplicarFiltroSupervisor('TODOS');
    }}

    function _getSupervisorActivo() {{
      // Reutiliza tu lógica: barra de filtros global
      let sup = '';
      const barra = document.getElementById('barraFiltrosSupervisoresGlobal');
      if (barra) {{
        const activo = barra.querySelector('.filtro-supervisor.activo[data-supervisor]');
        if (activo) sup = activo.getAttribute('data-supervisor') || '';
      }}
      return (sup || '').trim();
    }}
    
    function _getFuenteSupervisores() {{
      return (typeof datosSupervisores !== 'undefined' && datosSupervisores) ? datosSupervisores :
             (typeof supervisoresData !== 'undefined' && supervisoresData) ? supervisoresData :
             (window && window.supervisoresData) ? window.supervisoresData :
             null;
    }}
    
    function _toNumber(v) {{
      if (v === null || v === undefined) return 0;
      if (typeof v === 'number') return isFinite(v) ? v : 0;
      const s = String(v).replace(/[%,$ ]/g, '').replace(/,/g, '').trim();
      const n = parseFloat(s);
      return isFinite(n) ? n : 0;
    }}
    
    function _diaDeKey(key) {{
      const d = Number(key);
      return Number.isInteger(d) && d >= 1 && d <= 31 ? d : null;
    }}
    
    function _forwardFill31(mapDiaToValor, limiteDia = 31) {{
      const out = new Array(32).fill(null);
    
      for (let d = 1; d <= limiteDia; d++) {{
        const v = mapDiaToValor[d];
        if (v === undefined || v === null) {{
          out[d] = (d === 1) ? null : out[d - 1];
        }} else {{
          out[d] = Number(v);
        }}
      }}
    
      // Forzar día 1 a 0 si quedó vacío
      if (out[1] === null) out[1] = 0;
    
      for (let d = limiteDia + 1; d <= 31; d++) out[d] = null;
    
      return out;
    }}
    
    function _listarPeriodosDisponibles(fuenteSupervisores) {{
      const set = new Set();
      Object.keys(fuenteSupervisores || {{}}).forEach((sup) => {{
        const byPeriodo = fuenteSupervisores?.[sup] || {{}};
        Object.keys(byPeriodo).forEach((periodo) => {{
          if (periodo && String(periodo).includes('_')) set.add(periodo);
        }});
      }});
      return _sortPeriodosDesc(Array.from(set));
    }}
    
    function _periodoHoy() {{
      const fechaHoy = new Date();
      const meses = ['ENERO','FEBRERO','MARZO','ABRIL','MAYO','JUNIO','JULIO','AGOSTO','SETIEMBRE','OCTUBRE','NOVIEMBRE','DICIEMBRE'];
      return `${{meses[fechaHoy.getMonth()]}}_${{fechaHoy.getFullYear()}}`;
    }}
    
    function _limiteHoy() {{
      const fechaHoy = new Date();
      return Math.max(1, fechaHoy.getDate() - 1);
    }}
    
    function _getPeriodoSeleccionadoUI() {{
      const mesSeleccionado = document.getElementById('selectorMes')?.value;
      const añoSeleccionado = document.getElementById('selectorAño')?.value;
      if (!mesSeleccionado || !añoSeleccionado) return '';
      return `${{mesSeleccionado}}_${{añoSeleccionado}}`;
    }}
    
    function _getPeriodosDefault4(mesAñoActual) {{
      if (typeof _getPeriodosAtras === 'function') {{
        const p = String(mesAñoActual || '').split('_');
        const mes = p[0], anio = p[1];
        const arr = _getPeriodosAtras(mes, anio, 4, true);
        return (arr && arr.length === 4) ? arr : [];
      }}
      return [];
    }}
    
    function _normalizarSupervisor4M(v) {{
      return String(v || '').trim().toUpperCase().replace(/\s+/g, ' ');
    }}

    function _getSupervisorKey4M(fuenteSupervisores, supervisor) {{
      const objetivo = _normalizarSupervisor4M(supervisor);
      if (!objetivo || objetivo === 'TODOS') return supervisor || 'TODOS';
      const keys = Object.keys(fuenteSupervisores || {{}});
      return keys.find(k => _normalizarSupervisor4M(k) === objetivo) || supervisor;
    }}

    function _listarSupervisoresActivos4M(fuenteSupervisores, periodo) {{
      return Object.keys(fuenteSupervisores || {{}})
        .filter(sup => sup !== '__vista_orden_4m__')
        .filter(sup => !!fuenteSupervisores?.[sup]?.[periodo])
        .sort((a, b) => a.localeCompare(b, 'es'));
    }}

    function _getMetaMesDesdeSupervisores(fuenteSupervisores, supervisorFiltro, periodo) {{
      if (supervisorFiltro === 'TODOS') {{
        let suma = 0;
        Object.keys(fuenteSupervisores || {{}}).forEach((sup) => {{
          if (sup === '__vista_orden_4m__') return;
          suma += _toNumber(fuenteSupervisores?.[sup]?.[periodo]?.meta_super);
        }});
        return suma;
      }}
      const supKey = _getSupervisorKey4M(fuenteSupervisores, supervisorFiltro);
      return _toNumber(fuenteSupervisores?.[supKey]?.[periodo]?.meta_super);
    }}
    
    function _fmtMontoTabla4M(v) {{
      if (v === null || v === undefined || !isFinite(Number(v))) return '-';
      return 'S/ ' + Number(v).toLocaleString('es-PE', {{ maximumFractionDigits: 0 }});
    }}

    function _getCarteraTabla4M(fuenteSupervisores, supervisorFiltro, periodo) {{
      if (supervisorFiltro === 'TODOS') return 'CONSOLIDADO';
      const supKey = _getSupervisorKey4M(fuenteSupervisores, supervisorFiltro);
      return String(fuenteSupervisores?.[supKey]?.[periodo]?.cartera || '-').trim() || '-';
    }}

    function _bindClicksTabla4M(tbody) {{
      if (!tbody) return;
      try {{
        const rows = tbody.querySelectorAll('tr');
        rows.forEach((tr) => {{
          tr.querySelectorAll('td').forEach(td => td.classList.add('is-clickable'));
          tr.addEventListener('click', () => {{
            rows.forEach(r => r.classList.remove('is-selected'));
            tr.classList.add('is-selected');
          }});
        }});
      }} catch (e) {{
        console.warn('Tabla4M click init error:', e);
      }}
    }}

    function _setTabla4MEmpty(msg) {{
      const pares = [
        [document.getElementById('theadTabla4MRecupero'), document.getElementById('tbodyTabla4MRecupero')],
        [document.getElementById('theadTabla4MAlcance'), document.getElementById('tbodyTabla4MAlcance')]
      ];
      pares.forEach(([thead, tbody]) => {{
        if (thead) thead.innerHTML = '';
        if (tbody) tbody.innerHTML = `<tr><td style="padding:14px; color:#666;" colspan="5">${{msg}}</td></tr>`;
      }});
    }}

    function _renderTabla4MDoble(cols, notaTexto) {{
      const fuenteSupervisores = _getFuenteSupervisores();
      const theadRec = document.getElementById('theadTabla4MRecupero');
      const tbodyRec = document.getElementById('tbodyTabla4MRecupero');
      const theadPct = document.getElementById('theadTabla4MAlcance');
      const tbodyPct = document.getElementById('tbodyTabla4MAlcance');
      const nota = document.getElementById('notaTabla4M');
      if (!theadRec || !tbodyRec || !theadPct || !tbodyPct) return;
      if (!fuenteSupervisores) {{
        _setTabla4MEmpty('No hay datos de supervisores disponibles.');
        if (nota) nota.textContent = '-';
        return;
      }}

      cols = (Array.isArray(cols) ? cols : []).filter(c => c && c.periodo).slice(0, 4);
      if (cols.length < 2) {{
        _setTabla4MEmpty('Selecciona 4 periodos para comparar.');
        if (nota) nota.textContent = '-';
        return;
      }}

      const hoy = _periodoHoy();
      const limiteHoy = _limiteHoy();
      const seriesRec = {{}};
      const seriesPct = {{}};

      cols.forEach((c) => {{
        const supervisorCol = (c.supervisor || 'TODOS');
        const periodo = c.periodo;
        const metaMes = _getMetaMesDesdeSupervisores(fuenteSupervisores, supervisorCol, periodo);
        const mapRec = _getRecuperoDiarioAcum(fuenteSupervisores, supervisorCol, periodo);
        const limite = (periodo === hoy) ? limiteHoy : 31;
        const recFF = _forwardFill31(mapRec, limite);
        const recSerie = new Array(32).fill(null);
        const pctSerie = new Array(32).fill(null);
        for (let d = 1; d <= 31; d++) {{
          const rec = recFF[d];
          recSerie[d] = rec;
          pctSerie[d] = (rec === null || metaMes <= 0) ? null : (Number(rec) / Number(metaMes)) * 100;
        }}
        seriesRec[c.idx] = recSerie;
        seriesPct[c.idx] = pctSerie;
      }});

      let h1 = '<tr><th style="text-align:center;">D&Iacute;A</th>';
      cols.forEach((c) => {{ h1 += `<th>${{_mesTitulo(c.periodo)}} ${{_anioDePeriodo(c.periodo)}}</th>`; }});
      h1 += '</tr>';

      let h2 = '<tr><th style="text-align:center;">Cartera</th>';
      cols.forEach((c) => {{ h2 += `<th>${{_getCarteraTabla4M(fuenteSupervisores, c.supervisor || 'TODOS', c.periodo)}}</th>`; }});
      h2 += '</tr>';

      let h3 = '<tr><th style="text-align:center;">Supervisor</th>';
      cols.forEach((c) => {{
        const sup = String(c.supervisor || 'TODOS').trim();
        h3 += `<th style="font-size:0.78rem; color:#ffffff;">${{sup || '-'}}</th>`;
      }});
      h3 += '</tr>';

      const headHtml = h1 + h2 + h3;
      theadRec.innerHTML = headHtml;
      theadPct.innerHTML = headHtml;

      let bodyRec = '';
      let bodyPct = '';
      for (let d = 1; d <= 31; d++) {{
        bodyRec += `<tr><td>${{d}}</td>`;
        bodyPct += `<tr><td>${{d}}</td>`;
        cols.forEach((c) => {{
          const rec = seriesRec?.[c.idx]?.[d];
          const pct = seriesPct?.[c.idx]?.[d];
          const recEmpty = (rec === null || rec === undefined);
          const pctEmpty = (pct === null || pct === undefined);
          bodyRec += `<td class="${{recEmpty ? 'valor-vacio' : ''}}">${{recEmpty ? '-' : _fmtMontoTabla4M(rec)}}</td>`;
          const clsAlto = (!pctEmpty && Number(pct) >= 70) ? 'valor-alto' : '';
          const clsEmpty = pctEmpty ? 'valor-vacio' : '';
          bodyPct += `<td class="${{clsAlto}} ${{clsEmpty}}">${{pctEmpty ? '-' : Number(pct).toFixed(2) + '%'}}</td>`;
        }});
        bodyRec += '</tr>';
        bodyPct += '</tr>';
      }}
      tbodyRec.innerHTML = bodyRec;
      tbodyPct.innerHTML = bodyPct;
      _bindClicksTabla4M(tbodyRec);
      _bindClicksTabla4M(tbodyPct);
      if (nota) nota.textContent = notaTexto || 'Fuente: recupero acumulado y alcance acumulado por d&iacute;a (1..31). Mes actual se completa hasta ayer.';
    }}

    function _buildComparativoSupervisoresMesHTML(supervisores, periodo) {{
      const fuenteSupervisores = _getFuenteSupervisores();
      if (!fuenteSupervisores || !supervisores.length || !periodo) return '';

      const hoy = _periodoHoy();
      const limite = (periodo === hoy) ? _limiteHoy() : 31;
      const seriesRec = {{}};
      const seriesPct = {{}};

      supervisores.forEach((sup, idx) => {{
        const metaMes = _getMetaMesDesdeSupervisores(fuenteSupervisores, sup, periodo);
        const mapRec = _getRecuperoDiarioAcum(fuenteSupervisores, sup, periodo);
        const recFF = _forwardFill31(mapRec, limite);
        const recSerie = new Array(32).fill(null);
        const pctSerie = new Array(32).fill(null);
        for (let d = 1; d <= 31; d++) {{
          const rec = recFF[d];
          recSerie[d] = rec;
          pctSerie[d] = (rec === null || metaMes <= 0) ? null : (Number(rec) / Number(metaMes)) * 100;
        }}
        seriesRec[idx] = recSerie;
        seriesPct[idx] = pctSerie;
      }});

      let h1 = '<tr><th style="text-align:center;">D&Iacute;A</th>';
      supervisores.forEach(() => {{ h1 += `<th>${{_mesTitulo(periodo)}} ${{_anioDePeriodo(periodo)}}</th>`; }});
      h1 += '</tr>';

      let h2 = '<tr><th style="text-align:center;">Cartera</th>';
      supervisores.forEach((sup) => {{ h2 += `<th>${{_getCarteraTabla4M(fuenteSupervisores, sup, periodo)}}</th>`; }});
      h2 += '</tr>';

      let h3 = '<tr><th style="text-align:center;">Supervisor</th>';
      supervisores.forEach((sup) => {{ h3 += `<th style="font-size:0.78rem; color:#ffffff;">${{sup}}</th>`; }});
      h3 += '</tr>';

      let bodyRec = '';
      let bodyPct = '';
      for (let d = 1; d <= 31; d++) {{
        bodyRec += `<tr><td>${{d}}</td>`;
        bodyPct += `<tr><td>${{d}}</td>`;
        supervisores.forEach((sup, idx) => {{
          const rec = seriesRec?.[idx]?.[d];
          const pct = seriesPct?.[idx]?.[d];
          const recEmpty = (rec === null || rec === undefined);
          const pctEmpty = (pct === null || pct === undefined);
          bodyRec += `<td class="${{recEmpty ? 'valor-vacio' : ''}}">${{recEmpty ? '-' : _fmtMontoTabla4M(rec)}}</td>`;
          const clsAlto = (!pctEmpty && Number(pct) >= 70) ? 'valor-alto' : '';
          const clsEmpty = pctEmpty ? 'valor-vacio' : '';
          bodyPct += `<td class="${{clsAlto}} ${{clsEmpty}}">${{pctEmpty ? '-' : Number(pct).toFixed(2) + '%'}}</td>`;
        }});
        bodyRec += '</tr>';
        bodyPct += '</tr>';
      }}

      const head = h1 + h2 + h3;
      return `
        <div class="tabla4m-doble">
          <div>
            <div class="tabla4m-titulo">RECUPERO POR SUPERVISOR - ${{_mesTitulo(periodo)}} ${{_anioDePeriodo(periodo)}}</div>
            <div class="tabla4m-wrap" style="justify-self:stretch; width:100%; max-width:100%;">
              <table class="tabla-comparativa-4m" style="width:100%;">
                <thead>${{head}}</thead>
                <tbody>${{bodyRec}}</tbody>
              </table>
            </div>
          </div>
          <div>
            <div class="tabla4m-titulo">ALCANCE POR SUPERVISOR - ${{_mesTitulo(periodo)}} ${{_anioDePeriodo(periodo)}}</div>
            <div class="tabla4m-wrap" style="justify-self:stretch; width:100%; max-width:100%;">
              <table class="tabla-comparativa-4m" style="width:100%;">
                <thead>${{head}}</thead>
                <tbody>${{bodyPct}}</tbody>
              </table>
            </div>
          </div>
        </div>
      `;
    }}

    function _renderComparativoSupervisoresMes4M(periodoHeader) {{
      const cont = document.getElementById('tablas4MComparativoSupervisores');
      const fuenteSupervisores = _getFuenteSupervisores();
      if (!cont || !fuenteSupervisores || !periodoHeader) return;
      const supervisores = _listarSupervisoresActivos4M(fuenteSupervisores, periodoHeader);
      if (!supervisores.length) {{
        cont.innerHTML = '';
        return;
      }}
      cont.innerHTML = _buildComparativoSupervisoresMesHTML(supervisores, periodoHeader);
      cont.querySelectorAll('tbody').forEach(tb => _bindClicksTabla4M(tb));
    }}

    function _limpiarComparativoSupervisoresMes4M() {{
      const cont = document.getElementById('tablas4MComparativoSupervisores');
      if (cont) cont.innerHTML = '';
    }}

    function _buildTablaAlcance4MHTML(cols, titulo) {{
      const fuenteSupervisores = _getFuenteSupervisores();
      if (!fuenteSupervisores || !Array.isArray(cols) || !cols.length) return '';

      const hoy = _periodoHoy();
      const limiteHoy = _limiteHoy();
      const seriesPct = {{}};

      cols.forEach((c) => {{
        const supervisorCol = c.supervisor || 'TODOS';
        const periodo = c.periodo;
        const metaMes = _getMetaMesDesdeSupervisores(fuenteSupervisores, supervisorCol, periodo);
        const mapRec = _getRecuperoDiarioAcum(fuenteSupervisores, supervisorCol, periodo);
        const limite = (periodo === hoy) ? limiteHoy : 31;
        const recFF = _forwardFill31(mapRec, limite);
        const pctSerie = new Array(32).fill(null);
        for (let d = 1; d <= 31; d++) {{
          const rec = recFF[d];
          pctSerie[d] = (rec === null || metaMes <= 0) ? null : (Number(rec) / Number(metaMes)) * 100;
        }}
        seriesPct[c.idx] = pctSerie;
      }});

      let h1 = '<tr><th style="text-align:center;">D&Iacute;A</th>';
      cols.forEach((c) => {{ h1 += `<th>${{_mesTitulo(c.periodo)}} ${{_anioDePeriodo(c.periodo)}}</th>`; }});
      h1 += '</tr>';

      let h2 = '<tr><th style="text-align:center;">Cartera</th>';
      cols.forEach((c) => {{ h2 += `<th>${{_getCarteraTabla4M(fuenteSupervisores, c.supervisor || 'TODOS', c.periodo)}}</th>`; }});
      h2 += '</tr>';

      let h3 = '<tr><th style="text-align:center;">Supervisor</th>';
      cols.forEach((c) => {{
        const sup = String(c.supervisor || 'TODOS').trim();
        h3 += `<th style="font-size:0.78rem; color:#ffffff;">${{sup || '-'}}</th>`;
      }});
      h3 += '</tr>';

      let body = '';
      for (let d = 1; d <= 31; d++) {{
        body += `<tr><td>${{d}}</td>`;
        cols.forEach((c) => {{
          const pct = seriesPct?.[c.idx]?.[d];
          const empty = (pct === null || pct === undefined);
          const clsAlto = (!empty && Number(pct) >= 70) ? 'valor-alto' : '';
          const clsEmpty = empty ? 'valor-vacio' : '';
          body += `<td class="${{clsAlto}} ${{clsEmpty}}">${{empty ? '-' : Number(pct).toFixed(2) + '%'}}</td>`;
        }});
        body += '</tr>';
      }}

      return `
        <div class="tabla4m-supervisor-card">
          <div class="tabla4m-titulo">${{titulo}}</div>
          <div class="tabla4m-wrap" style="justify-self:stretch; width:100%; max-width:100%;">
            <table class="tabla-comparativa-4m" style="width:100%;">
              <thead>${{h1 + h2 + h3}}</thead>
              <tbody>${{body}}</tbody>
            </table>
          </div>
        </div>
      `;
    }}

    function _renderTablasSupervisoresTodos4M(periodos) {{
      const cont = document.getElementById('tablas4MSupervisoresTodos');
      const fuenteSupervisores = _getFuenteSupervisores();
      if (!cont || !fuenteSupervisores) return;

      const periodos4 = (periodos || []).filter(Boolean).slice(0, 4);
      const periodoHeader = periodos4[3] || _getPeriodoSeleccionadoUI();
      if (periodos4.length < 4 || !periodoHeader) {{
        cont.innerHTML = '';
        return;
      }}

      const supervisores = _listarSupervisoresActivos4M(fuenteSupervisores, periodoHeader);
      if (!supervisores.length) {{
        cont.innerHTML = '';
        return;
      }}

      cont.innerHTML = supervisores.map((sup) => {{
        const cols = periodos4.map((periodo, i) => ({{
          idx: i + 1,
          periodo,
          supervisor: _getSupervisorVista4M(i + 1, sup, periodoHeader)
        }}));
        return _buildTablaAlcance4MHTML(cols, `ALCANCES - ${{sup}}`);
      }}).join('');

      cont.querySelectorAll('tbody').forEach(tb => _bindClicksTabla4M(tb));
    }}

    function _limpiarTablasSupervisoresTodos4M() {{
      const cont = document.getElementById('tablas4MSupervisoresTodos');
      if (cont) cont.innerHTML = '';
    }}

    function _getRecuperoDiarioAcum(fuenteSupervisores, supervisorFiltro, periodo) {{
      const map = {{}};
    
      if (supervisorFiltro === 'TODOS') {{
        Object.keys(fuenteSupervisores || {{}}).forEach((sup) => {{
          if (sup === '__vista_orden_4m__') return;
          const diarios = fuenteSupervisores?.[sup]?.[periodo]?.datos_diarios_supervisor || {{}};
          Object.keys(diarios).forEach((k) => {{
            const d = _diaDeKey(k);
            if (!d) return;
            map[d] = (Number(map[d]) || 0) + (Number(diarios[k]) || 0);
          }});
        }});
        return map;
      }}
    
      const supKey = _getSupervisorKey4M(fuenteSupervisores, supervisorFiltro);
      const diarios = fuenteSupervisores?.[supKey]?.[periodo]?.datos_diarios_supervisor || {{}};
      Object.keys(diarios).forEach((k) => {{
        const d = _diaDeKey(k);
        if (!d) return;
        map[d] = Number(diarios[k]) || 0;
      }});
      return map;
    }}
    
    function _renderTabla4M(periodosElegidos, supervisorFiltro) {{
      const periodos = (periodosElegidos || []).filter(Boolean).slice(0, 4);
      const nota = document.getElementById('notaTabla4M');
      if (periodos.length < 2) {{
        _setTabla4MEmpty('Selecciona 4 periodos para comparar.');
        if (nota) nota.textContent = '-';
        return;
      }}

      const sup = supervisorFiltro || 'TODOS';
      const cols = periodos.map((periodo, i) => ({{
        idx: i + 1,
        periodo,
        supervisor: sup
      }}));

      const supTxt = (sup === 'TODOS') ? 'CONSOLIDADO (TODOS)' : sup;
      _renderTabla4MDoble(
        cols,
        `Vista: ${{supTxt}} | Fuente: recupero acumulado y alcance acumulado por d&iacute;a (1..31). Mes actual se completa hasta ayer.`
      );

      if (sup === 'TODOS') {{
        const periodoHeader = periodos[3] || _getPeriodoSeleccionadoUI();
        _renderComparativoSupervisoresMes4M(periodoHeader);
        _renderTablasSupervisoresTodos4M(periodos);
      }} else {{
        _limpiarComparativoSupervisoresMes4M();
        _limpiarTablasSupervisoresTodos4M();
      }}
    }}

    function _renderTabla4MConSupervisores(conf) {{
      const cols = (Array.isArray(conf) ? conf : [])
        .filter(c => c && c.periodo)
        .slice(0, 4);

      if (cols.length < 2) {{
        _setTabla4MEmpty('Selecciona 4 periodos para comparar.');
        const nota = document.getElementById('notaTabla4M');
        if (nota) nota.textContent = '-';
        return;
      }}

      const ordenTxt = cols.map(c => `${{c.idx}}:${{c.supervisor || '-'}}`).join(' | ');
      _renderTabla4MDoble(
        cols,
        `Vista por supervisor: ${{ordenTxt}} | Fuente: recupero acumulado y alcance acumulado por d&iacute;a (1..31). Mes actual se completa hasta ayer.`
      );
      _limpiarComparativoSupervisoresMes4M();
      _limpiarTablasSupervisoresTodos4M();
    }}

    // HELPERS 4M

    function _listarSupervisoresDisponibles4M(fuenteSupervisores) {{
      // fuenteSupervisores suele ser datosPorSupervisor o similar
      // Queremos los keys (nombres)
      try {{
        const keys = Object.keys(fuenteSupervisores || {{}})
          .map(s => String(s || '').trim())
          .filter(Boolean);
        keys.sort((a,b) => a.localeCompare(b, 'es'));
        return keys;
      }} catch (e) {{
        return [];
      }}
    }}
    
    function _fillSelectSupervisores4M(sel, supervisores) {{
      if (!sel) return;
      sel.innerHTML = '';
    
      // vacío = usar supervisor del header
      const opt0 = document.createElement('option');
      opt0.value = '';
      opt0.textContent = '(mismo del header)';
      sel.appendChild(opt0);
    
      supervisores.forEach((s) => {{
        const opt = document.createElement('option');
        opt.value = s;
        opt.textContent = s;
        sel.appendChild(opt);
      }});
    
      sel.value = '';
    }}

    function _getSupervisorHeader4M() {{
      // ✅ Prioridad máxima: variable global seteada por la vista supervisor
      if (window.supervisorSeleccionado && window.supervisorSeleccionado !== 'TODOS') {{
        return String(window.supervisorSeleccionado).trim();
      }}
    
      // fallback: select (si existe en alguna vista)
      const el = document.getElementById('selectorSupervisor');
      if (el && el.value && el.value !== 'TODOS') return String(el.value).trim();
    
      return '';
    }}
    
    function _isVistaSupervisor4M() {{
      const s = _getSupervisorHeader4M();
      return !!s;
    }}
    
    function _valSupSelectOrHeader(i, supHeader) {{
      if (i === 4) return supHeader; // P4 fijo
      const el = document.getElementById(`selTabla4M_Sup_${{i}}`);
      const v = (el && el.value) ? String(el.value).trim() : '';
      return v || supHeader;
    }}

    function _getOrdenVista4M(supHeader, periodoHeader) {{
      const fuenteSupervisores = _getFuenteSupervisores();
      const supKey = _getSupervisorKey4M(fuenteSupervisores, supHeader);
      const orden = fuenteSupervisores?.[supKey]?.[periodoHeader]?.vista_orden_4m;
      if (!Array.isArray(orden) || orden.length < 4) return null;
      const limpio = orden.slice(0, 4).map(s => String(s || '').trim());
      return limpio.some(Boolean) ? limpio : null;
    }}

    function _getSupervisorVista4M(i, supHeader, periodoHeader) {{
      const orden = _getOrdenVista4M(supHeader, periodoHeader);
      if (!orden) return supHeader;
      return orden[i - 1] || supHeader;
    }}
    
    function initTablaComparativa4M() {{
      const fuenteSupervisores = _getFuenteSupervisores();
      if (!fuenteSupervisores) return;
    
      const periodosDisponibles = _listarPeriodosDisponibles(fuenteSupervisores);
      const s1 = document.getElementById('selTabla4M_1');
      const s2 = document.getElementById('selTabla4M_2');
      const s3 = document.getElementById('selTabla4M_3');
      const s4 = document.getElementById('selTabla4M_4');
      const btn = document.getElementById('btnTabla4MRefrescar');
      const filaSup = document.getElementById('filaTabla4MSupervisores');
      const sup1 = document.getElementById('selTabla4M_Sup_1');
      const sup2 = document.getElementById('selTabla4M_Sup_2');
      const sup3 = document.getElementById('selTabla4M_Sup_3');
      const sup4Fixed = document.getElementById('selTabla4M_Sup_4_fixed');
    
      if (!s1 || !s2 || !s3 || !s4) return;
    
        const _fillSelect = (sel, permitirVacio) => {{
          sel.innerHTML = '';
        
          if (permitirVacio) {{
            const opt0 = document.createElement('option');
            opt0.value = '';
            opt0.textContent = '-';
            sel.appendChild(opt0);
          }}
        
          periodosDisponibles.forEach((p) => {{
            const opt = document.createElement('option');
            opt.value = p;
            opt.textContent = `${{_mesTitulo(p)}} ${{_anioDePeriodo(p)}}`;
            sel.appendChild(opt);
          }});
        }};
    
      _fillSelect(s1, true); _fillSelect(s2, true); _fillSelect(s3, true); _fillSelect(s4, false);
    
      // Seteo de periodos
      const periodoUI = _getPeriodoSeleccionadoUI();
      let def = periodoUI ? _getPeriodosDefault4(periodoUI) : [];
      if (!def || def.length !== 4) {{
        // fallback: top 4 más recientes disponibles
        def = periodosDisponibles.slice(0, 4);
      }}
    
      if (def.length === 4) {{
        s1.value = def[0];
        s2.value = def[1];
        s3.value = def[2];
        s4.value = def[3];
      }}

      // Seteo de supervisores 4m
      const esVistaSup = _isVistaSupervisor4M();
      const supHeader = _getSupervisorHeader4M();
    
      if (filaSup) {{
        filaSup.style.display = 'none';
      }}
    
      if (esVistaSup) {{
        if (sup4Fixed) sup4Fixed.value = supHeader || '—';
    
        const supervisoresDisponibles = _listarSupervisoresDisponibles4M(fuenteSupervisores);
        _fillSelectSupervisores4M(sup1, supervisoresDisponibles);
        _fillSelectSupervisores4M(sup2, supervisoresDisponibles);
        _fillSelectSupervisores4M(sup3, supervisoresDisponibles);
    
        // Por defecto: vacío => toma el del header
        if (sup1) sup1.value = '';
        if (sup2) sup2.value = '';
        if (sup3) sup3.value = '';
      }}
    
        const _refrescar = () => {{
          let supervisorFiltro = _getSupervisorActivo();
          if (!supervisorFiltro) supervisorFiltro = 'TODOS';
        
          const periodos = [s1.value, s2.value, s3.value, s4.value];
        
          // Vista general (TODOS) -> render normal
          if (!_isVistaSupervisor4M() || supervisorFiltro === 'TODOS') {{
            _renderTabla4M(periodos, supervisorFiltro);
            return;
          }}
        
          // Vista supervisor -> render con supervisor por columna (P4 fijo)
          const supHeader = _getSupervisorHeader4M() || supervisorFiltro;
        
          const periodoHeader = s4.value || _getPeriodoSeleccionadoUI();
          const conf = [
            {{ idx: 1, periodo: s1.value, supervisor: _getSupervisorVista4M(1, supHeader, periodoHeader) }},
            {{ idx: 2, periodo: s2.value, supervisor: _getSupervisorVista4M(2, supHeader, periodoHeader) }},
            {{ idx: 3, periodo: s3.value, supervisor: _getSupervisorVista4M(3, supHeader, periodoHeader) }},
            {{ idx: 4, periodo: s4.value, supervisor: _getSupervisorVista4M(4, supHeader, periodoHeader) }},
          ];
        
          _renderTabla4MConSupervisores(conf);
        }};

        // Eventos
        s1.addEventListener('change', _refrescar);
        s2.addEventListener('change', _refrescar);
        s3.addEventListener('change', _refrescar);
        s4.addEventListener('change', _refrescar);
        if (btn) btn.addEventListener('click', _refrescar);
        
        // Nuevos eventos supervisores
        if (sup1) sup1.addEventListener('change', _refrescar);
        if (sup2) sup2.addEventListener('change', _refrescar);
        if (sup3) sup3.addEventListener('change', _refrescar);
        
        // Primer render
        _refrescar();
    }}
    
    function generarGraficasRecuperosSupervisor(periodoCompleto, supervisor) {{
        window.supervisorSeleccionado = (supervisor && supervisor !== 'TODOS') ? supervisor : '';
        const graficasContainer = document.getElementById('graficas-recuperos');
        if (!graficasContainer) return;
        
        let tituloGráfica = 'Recupero y Alcance';
        if (supervisor && supervisor !== 'TODOS') {{
            tituloGráfica = `Recupero y Alcance - ${{supervisor}}`;
        }}
        
            graficasContainer.innerHTML = `
                <div style="text-align: center; margin-bottom: 30px; display:flex; justify-content:center; gap:12px; flex-wrap:wrap;">
                  <button
                    class="boton-exportar-excel"
                    onclick="exportarRecupero4MesesDiaSupervisor('${{supervisor || ''}}')">
                      Recuperos y Alcances Mensual ${{supervisor && supervisor !== 'TODOS' ? `- ${{supervisor}}` : ''}}
                  </button>
                  <button type="button" class="boton-exportar-excel btn-mundial" onclick="abrirModalMundialEquipos()">Mundial por equipos</button>
                  <button type="button" class="boton-exportar-excel btn-mundial-sec" onclick="abrirModalMundialPaises('SURCO')">Mundial de paises - SURCO</button>
                  <button type="button" class="boton-exportar-excel btn-mundial-sec" onclick="abrirModalMundialPaises('BPO')">Mundial de paises - BPO</button>
                  <button type="button" class="boton-exportar-excel btn-mundial" onclick="mostrarTop10Paises()">TOP10-PAISES</button>
                  <button type="button" class="boton-exportar-excel btn-mundial" onclick="mostrarTop10MundialAsesor()">TOP10-MUNDIAL-ASESOR</button>
                </div>
                
                <!-- ===================== TABLA COMPARATIVA 4 MESES (ALCANCES) ===================== -->
                <div id="cardTablaComparativa4M" style="margin-top: 16px; background:#fff; border-radius:14px; box-shadow: 0 5px 15px rgba(0,0,0,0.08);">
                  <div style="display:flex; justify-content:space-between; align-items:flex-end; gap:12px; flex-wrap:wrap;">
                    <div>
                      <h3 style="margin:0; color:#2c3e50; font-size: 1.25rem; font-weight: 800;">Comparación diaria (4 periodos)</h3>
                      <div style="margin-top:6px; color:#666; font-size:0.92rem;">
                        Selecciona 4 periodos y compara el <b>alcance acumulado (%)</b> por día (1..31).
                      </div>
                    </div>
                
                    <button type="button" class="boton-busqueda" id="btnTabla4MRefrescar" style="padding:10px 14px;">
                      🔄 Actualizar
                    </button>
                      <button
                          type="button"
                          class="boton-busqueda"
                          onclick="copiarVistaAlcances4M()"
                          style="padding:10px 14px; background: linear-gradient(135deg, #34495e, #2c3e50);">
                          Copiar vista
                      </button>
                  </div>
                
                  <div class="tabla4m-selectores">
                    <div>
                      <label style="font-weight:700; color:#2c3e50;">Periodo 1</label>
                      <select id="selTabla4M_1" style="width:100%; padding:10px 12px; border-radius:10px; border:1px solid #ddd;"></select>
                    </div>
                    <div>
                      <label style="font-weight:700; color:#2c3e50;">Periodo 2</label>
                      <select id="selTabla4M_2" style="width:100%; padding:10px 12px; border-radius:10px; border:1px solid #ddd;"></select>
                    </div>
                    <div>
                      <label style="font-weight:700; color:#2c3e50;">Periodo 3</label>
                      <select id="selTabla4M_3" style="width:100%; padding:10px 12px; border-radius:10px; border:1px solid #ddd;"></select>
                    </div>
                    <div>
                      <label style="font-weight:700; color:#2c3e50;">Periodo 4</label>
                      <select id="selTabla4M_4" style="width:100%; padding:10px 12px; border-radius:10px; border:1px solid #ddd;"></select>
                    </div>
                  </div>
    
                  <div class="tabla4m-selectores" id="filaTabla4MSupervisores" style="margin-top:10px; display:none;">
                      <div>
                        <label style="font-weight:700; color:#2c3e50;">Supervisor P1</label>
                        <select id="selTabla4M_Sup_1" style="width:100%; padding:10px 12px; border-radius:10px; border:1px solid #ddd;"></select>
                      </div>
                      <div>
                        <label style="font-weight:700; color:#2c3e50;">Supervisor P2</label>
                        <select id="selTabla4M_Sup_2" style="width:100%; padding:10px 12px; border-radius:10px; border:1px solid #ddd;"></select>
                      </div>
                      <div>
                        <label style="font-weight:700; color:#2c3e50;">Supervisor P3</label>
                        <select id="selTabla4M_Sup_3" style="width:100%; padding:10px 12px; border-radius:10px; border:1px solid #ddd;"></select>
                      </div>
                      <div>
                        <label style="font-weight:700; color:#2c3e50;">Supervisor P4 (fijo)</label>
                        <input id="selTabla4M_Sup_4_fixed" type="text" disabled
                               style="width:100%; padding:10px 12px; border-radius:10px; border:1px solid #ddd; background:#f3f5f7;"
                               value="—" />
                      </div>
                  </div>
    
                  <!-- MINI GRÁFICAS -->
                  <div style="margin-top:12px; display:grid; grid-template-columns: 1fr; gap:12px; align-items:start;">
                      <div id="tablas4MComparativoSupervisores" class="tabla4m-comparativo"></div>
                  <div id="tablas4MSupervisoresTodos" class="tabla4m-supervisores-grid"></div>
                  <div id="tablas4MPrincipales" class="tabla4m-doble">
                    <div>
                      <div class="tabla4m-titulo">RECUPEROS</div>
                      <div class="tabla4m-wrap" style="justify-self:stretch; width:100%; max-width:100%;">
                        <table id="tablaComparativa4MRecupero" class="tabla-comparativa-4m" style="width:100%;">
                          <thead id="theadTabla4MRecupero"></thead>
                          <tbody id="tbodyTabla4MRecupero"></tbody>
                        </table>
                      </div>
                    </div>
                    <div>
                      <div class="tabla4m-titulo">ALCANCES</div>
                      <div class="tabla4m-wrap" style="justify-self:stretch; width:100%; max-width:100%;">
                        <table id="tablaComparativa4MAlcance" class="tabla-comparativa-4m" style="width:100%;">
                          <thead id="theadTabla4MAlcance"></thead>
                          <tbody id="tbodyTabla4MAlcance"></tbody>
                        </table>
                      </div>
                    </div>
                  </div>
                    
                      <div id="miniAlcancesRight" style="display:grid; grid-template-columns: repeat(4, minmax(180px, 1fr)); gap:10px; align-content:start;">
                        <div class="mini-chart-card" data-mini="acumulado">
                          <div class="mini-title">ACUMULADO</div>
                          <canvas id="miniGraficaIncrementoTotal"></canvas>
                        </div>
                    
                        <div class="mini-chart-card" data-mini="diario">
                          <div class="mini-title">DIARIO</div>
                          <canvas id="miniGraficaRecuperoDiario"></canvas>
                        </div>
                    
                        <div class="mini-chart-card" data-mini="equipos">
                          <div class="mini-title">EQUIPOS</div>
                          <canvas id="miniGraficaAlcanceEquipos"></canvas>
                        </div>
                    
                        <div class="mini-chart-card" data-mini="anual">
                          <div class="mini-title">ANUAL</div>
                          <canvas id="miniGraficaEvolucionAnualRecuperos"></canvas>
                        </div>
                      </div>
                  </div>
                
                  <!-- MODAL ALCANCES -->
                  <div id="modalGraficaAlcances" style="display:none;">
                      <div id="modalGraficaAlcancesBackdrop"
                           style="position:fixed; inset:0; background:rgba(0,0,0,.55); z-index:9999;"></div>
                    
                        <div id="modalGraficaAlcancesOverlay"
                             style="
                                position:fixed; inset:0; z-index:10000;
                                display:flex; align-items:center; justify-content:center;
                                padding:18px;">
                        <div style="
                            width:min(1200px, 96vw);
                            max-height:92vh;
                            background:#fff;
                            border-radius:16px;
                            box-shadow:0 18px 50px rgba(0,0,0,.25);
                            overflow:hidden;">
                          
                          <div style="display:flex; justify-content:space-between; align-items:center;
                                      padding:12px 14px; border-bottom:1px solid rgba(0,0,0,.08);">
                            <div id="modalGraficaAlcancesTitulo" style="font-weight:800; font-size:1.5rem; text-align:center; width:100%; color:#2c3e50;">
                              Gráfica de Alcances
                            </div>
                          </div>
                    
                          <div style="padding:12px;">
                            <canvas id="modalCanvasGraficaAlcances" style="display:block; margin:0 auto; width:100%; height: 70vh;"></canvas>
                          </div>
                        </div>
                      </div>
                  </div>
                
                  <div id="notaTabla4M" style="margin-top:10px; color:#666; font-size:0.9rem;">
                    —
                  </div>
                </div>
            `;

        const card4M = document.getElementById('cardTablaComparativa4M');
        const mini4M = document.getElementById('miniAlcancesRight');
        if (card4M && mini4M) {{
          card4M.insertBefore(mini4M, card4M.firstElementChild);
          mini4M.style.marginBottom = '14px';
          mini4M.style.marginTop = '0';
        }}

        try {{
          initTablaComparativa4M();
        }} catch (e) {{
          console.warn('initTablaComparativa4M error:', e);
        }}
        
        setTimeout(() => {{
          if (supervisor && supervisor !== 'TODOS') {{
            generarGraficaIncrementoTotalSupervisor(periodoCompleto, supervisor, 'miniGraficaIncrementoTotal', 'mini');
            generarGraficaRecuperoDiarioSupervisor(periodoCompleto, supervisor, 'miniGraficaRecuperoDiario', 'mini');
          }} else {{
            generarGraficaIncrementoTotal(periodoCompleto, 'miniGraficaIncrementoTotal', 'mini');
            generarGraficaRecuperoDiario(periodoCompleto, 'miniGraficaRecuperoDiario', 'mini');
          }}
        
          generarGraficaAlcanceEquiposRecuperos(periodoCompleto, 'miniGraficaAlcanceEquipos', 'mini');
          actualizarGraficaEvolucionAnualRecuperos('miniGraficaEvolucionAnualRecuperos', 'mini');
        
          _bindMiniAlcancesClicks(periodoCompleto);
        }}, 100);
    }}

    function actualizarGraficaEvolucionAnualRecuperos(canvasId = 'graficaEvolucionAnualRecuperos', view = 'full') {{
        const añoSeleccionado = document.getElementById('selectorAño').value;
        const contenedorGrafica = document.getElementById('graficaAnualContainer');
        const añoElement = document.getElementById('año-grafica-evolucion-recuperos');
        
        if (!añoSeleccionado) {{
            // Mostrar mensaje si no hay año seleccionado
            const canvas = document.getElementById('graficaEvolucionAnualRecuperos');
            if (canvas) {{
                const ctx = canvas.getContext('2d');
                ctx.clearRect(0, 0, canvas.width, canvas.height);
                
                canvas.width = 1200;
                canvas.height = 200;
                canvas.style.display = 'block';
                canvas.style.margin = '0 auto';
                canvas.style.maxWidth = '100%';
                
                ctx.fillStyle = '#95a5a6';
                ctx.font = 'bold 20px Arial';
                ctx.textAlign = 'center';
                ctx.fillText('📊 Seleccione un año en los filtros superiores', canvas.width / 2, canvas.height / 2);
            }}
            return;
        }}

        if (añoElement) {{
            añoElement.textContent = añoSeleccionado;
        }}
        
        // Generar gráfica de tendencia
        generarGraficaEvolucionAnualTendenciaRecuperos(añoSeleccionado, canvasId, view);
    }}

    function generarGraficaEvolucionAnualTendenciaRecuperos(año, canvasId = 'graficaEvolucionAnualRecuperos', view = 'full') {{
        const mesesDelAño = obtenerMesesDelAño(año);
        if (mesesDelAño.length === 0) {{
            mostrarMensajeSinDatosAnualRecuperos('graficaEvolucionAnualRecuperos', año);
            return;
        }}
        
        // Metricas
        const alcancesPorMes = mesesDelAño.map(mes => {{
            const periodoCompleto = `${{mes}}_${{año}}`;
            return calcularAlcanceTotalMes(periodoCompleto);
        }});

        const metasPorMes = mesesDelAño.map(mes => {{
          const periodoCompleto = `${{mes}}_${{año}}`;
          const lista = filtrarAsesoresPorCanal((datosMeses && datosMeses[periodoCompleto]) ? datosMeses[periodoCompleto] : []);
          return lista.reduce((acc, a) => acc + (Number(a.meta) || 0), 0);
        }});
        
        const recuperosPorMes = mesesDelAño.map(mes => {{
          const periodoCompleto = `${{mes}}_${{año}}`;
          const lista = filtrarAsesoresPorCanal((datosMeses && datosMeses[periodoCompleto]) ? datosMeses[periodoCompleto] : []);
          return lista.reduce((acc, a) => acc + (Number(a.recupero) || 0), 0);
        }});
        
        const cantAsesoresPorMes = mesesDelAño.map(mes => {{
          const periodoCompleto = `${{mes}}_${{año}}`;
          const lista = filtrarAsesoresPorCanal((datosMeses && datosMeses[periodoCompleto]) ? datosMeses[periodoCompleto] : []);
          return Array.isArray(lista) ? lista.length : 0;
        }});
        
        const cantSupervisoresPorMes = mesesDelAño.map(mes => {{
          const periodoCompleto = `${{mes}}_${{año}}`;
          const lista = filtrarAsesoresPorCanal((datosMeses && datosMeses[periodoCompleto]) ? datosMeses[periodoCompleto] : []);
          if (!Array.isArray(lista) || lista.length === 0) return 0;
          const set = new Set(lista.map(a => (a.supervisor || '').toString().trim()).filter(Boolean));
          return set.size;
        }});
        
        const canvas = document.getElementById(canvasId)
        if (!canvas) return;
        
        const ctx = canvas.getContext('2d');
        
        // Destruir gráfica anterior si existe
        _destroyChartByCanvasId(canvasId);
        
        // Limpiar canvas
        ctx.clearRect(0, 0, canvas.width, canvas.height);
        
        _applyCanvasSize(canvas, view);
        canvas.style.display = 'block';
        canvas.style.margin = '0 auto';
        canvas.style.maxWidth = '100%';
        
        // Abreviar nombres de meses
        const mesesAbreviados = mesesDelAño.map(mes => {{
            return mes.substring(0, 3).toUpperCase();
        }});
        
        // Calcular línea de tendencia (regresión lineal simple)
        const tendencia = calcularLineaTendencia(alcancesPorMes);

        let options = {{
            responsive: true,
            maintainAspectRatio: false,
            plugins: {{
                tooltip: {{
                  mode: 'index',
                  intersect: false,
                  backgroundColor: 'rgba(0, 0, 0, 0.9)',
                  titleFont: {{ size: 18, weight: 'bold' }},
                  bodyFont: {{ size: 17 }},
                  padding: 12,
                  cornerRadius: 8,
                  callbacks: {{
                    label: function(context) {{
                      const label = context.dataset.label || '';
                
                      // Ocultar solo Meta 100%
                      if (label === 'Meta 100%') {{
                        return null;
                      }}
                
                      // Línea de Tendencia
                      if (label === 'Línea de Tendencia') {{
                        if (!context.parsed || context.parsed.y === null || !isFinite(context.parsed.y)) {{
                          return null;
                        }}
                        return `Línea de Tendencia: ${{context.parsed.y.toFixed(2)}}%`;
                      }}
                
                      // tooltip en el dataset principal
                      if (label !== 'Alcance Real') {{
                        return null;
                      }}
                
                      const i = context.dataIndex;
                
                      const alcance = (context.parsed && context.parsed.y != null && isFinite(context.parsed.y))
                        ? context.parsed.y
                        : null;
                
                      const metaMes = Number(metasPorMes[i] || 0);
                      const recuperoMes = Number(recuperosPorMes[i] || 0);
                      const cantAse = Number(cantAsesoresPorMes[i] || 0);
                      const cantSup = Number(cantSupervisoresPorMes[i] || 0);
                
                      const fmtNum = (n) => {{
                        try {{
                          return Number(n || 0).toLocaleString('es-PE');
                        }} catch (e) {{
                          return String(n || 0);
                        }}
                      }};
                
                      const lineas = [];
                
                      if (alcance === null) {{
                        lineas.push('Alcance Real: Sin datos');
                      }} else {{
                        lineas.push(`Alcance Real: ${{alcance.toFixed(2)}}%`);
                      }}
                
                      lineas.push(`Meta del mes: ${{fmtNum(metaMes)}}`);
                      lineas.push(`Recupero del mes: ${{fmtNum(recuperoMes)}}`);
                      lineas.push(`Cantidad de asesores: ${{fmtNum(cantAse)}}`);
                      lineas.push(`Cantidad de supervisores: ${{fmtNum(cantSup)}}`);
                
                      return lineas;
                    }}
                  }}
                }},
                legend: {{
                    position: 'top',
                    labels: {{ font: {{ size: 12 }} }}
                }}
            }},
            scales: {{
                x: {{
                    title: {{ 
                        display: true, 
                        text: 'Meses del Año',
                        font: {{ size: 14, weight: 'bold' }}
                    }},
                    ticks: {{ 
                        font: {{ size: 13 }}
                    }}
                }},
                y: {{
                    beginAtZero: true,
                    title: {{ 
                        display: true, 
                        text: 'Alcance Total (%)',
                        font: {{ size: 14, weight: 'bold' }}
                    }},
                    ticks: {{
                        callback: function(value) {{ return value + '%'; }},
                        stepSize: 10,
                        font: {{ size: 13 }}
                    }},
                    min: 0,
                    suggestedMax: function() {{
                        const valoresValidos = alcancesPorMes.filter(v => v !== null);
                        if (valoresValidos.length === 0) return 100;
                        const maxValor = Math.max(...valoresValidos);
                        return Math.max(100, Math.ceil(maxValor / 10) * 10);
                    }}()
                }}
            }},
            interaction: {{
                mode: 'nearest',
                axis: 'x',
                intersect: false
            }}
        }};
        
        options = _tuneOptionsForView(options, view);
        
        try {{
            const chart = new Chart(ctx, {{
                type: 'line',
                data: {{
                    labels: mesesAbreviados,
                    datasets: [
                        {{
                            label: 'Alcance Real',
                            data: alcancesPorMes,
                            borderColor: '#3498db',
                            backgroundColor: 'rgba(52, 152, 219, 0.1)',
                            borderWidth: 3,
                            fill: true,
                            tension: 0.4,
                            pointRadius: 6,
                            pointHoverRadius: 10,
                            pointBackgroundColor: '#3498db',
                            pointBorderColor: '#ffffff',
                            pointBorderWidth: 2
                        }},
                        {{
                            label: 'Línea de Tendencia',
                            data: tendencia,
                            borderColor: '#e74c3c',
                            backgroundColor: 'transparent',
                            borderWidth: 2,
                            borderDash: [5, 5],
                            fill: false,
                            tension: 0,
                            pointRadius: 0
                        }},
                        {{
                            label: 'Meta 100%',
                            data: Array(mesesDelAño.length).fill(100),
                            borderColor: '#2ecc71',
                            backgroundColor: 'transparent',
                            borderWidth: 1,
                            borderDash: [3, 3],
                            fill: false,
                            tension: 0,
                            pointRadius: 0
                        }}
                    ]
                }},
                options,
            }});
            _storeChart(canvasId, chart);
            
        }} catch (error) {{
            console.error("❌ Error al crear la gráfica de tendencia para recuperos:", error);
            mostrarMensajeSinDatosAnualRecuperos('graficaEvolucionAnualRecuperos', año);
        }}
    }}
    
    function generarGraficaRecuperoDiarioSupervisor(periodoCompleto, supervisor, canvasId = 'graficaRecuperoDiario', view = 'full') {{
        const canvas = document.getElementById(canvasId);
        if (!canvas) return;
        const dataLabelsPluginTotal = _buildDataLabelsPluginTotal(view);
        const ctx = canvas.getContext('2d');
        
        _destroyChartByCanvasId(canvasId);
        _applyCanvasSize(canvas, view);
        canvas.style.display = 'block';
        canvas.style.margin = '0 auto';
        canvas.style.maxWidth = '100%';
        ctx.clearRect(0, 0, canvas.width, canvas.height);
        
        const supervisorData = datosSupervisores?.[supervisor];
        const datosMes = supervisorData ? supervisorData[periodoCompleto] : null;
        const datosDiarios = datosMes?.datos_diarios_supervisor || {{}};
        const fechasConDatos = Object.keys(datosDiarios).sort((a, b) =>
            convertirFechaDiaria(a) - convertirFechaDiaria(b)
        );
        const fechasOrdenadas = obtenerFechasPeriodoCompleto(periodoCompleto);
        
        if (!datosMes || fechasConDatos.length === 0) {{
            mostrarMensajeSinDatos(canvas, periodoCompleto, supervisor);
            return;
        }}
        
        const metaSupervisor = Number(datosMes.meta_super || 0);
        const diasLaborables = fechasOrdenadas.length;
        const metaDiaria = diasLaborables > 0 ? metaSupervisor / diasLaborables : 0;
        
        const alcanceDiario = fechasOrdenadas.map(fecha => {{
            const tieneDato = Object.prototype.hasOwnProperty.call(datosDiarios, fecha);
            const totalRecuperoDia = tieneDato ? Number(datosDiarios[fecha] || 0) : null;
            const porcentajeDiario = (totalRecuperoDia !== null && metaDiaria > 0) ? (totalRecuperoDia / metaDiaria) * 100 : null;
            return {{
                fecha,
                recupero: totalRecuperoDia,
                porcentaje: porcentajeDiario
            }};
        }});
        
        const porcentajes = alcanceDiario.map(dia => dia.porcentaje);
        const coloresBarras = porcentajes.map(porcentaje => {{
            if (porcentaje === null || porcentaje === undefined) return 'rgba(226, 232, 240, 0.45)';
            if (porcentaje === null || porcentaje === undefined) return 'rgba(226, 232, 240, 0.45)';
            if (porcentaje <= 0) return '#F3E5F5';
            if (porcentaje <= 20) return '#E1BEE7';
            if (porcentaje <= 40) return '#BA68C8';
            if (porcentaje <= 60) return '#8E24AA';
            if (porcentaje <= 80) return '#6A1B9A';
            if (porcentaje <= 100) return '#4A148C';
            return '#38006B';
        }});
        
        let options = {{
            responsive: true,
            maintainAspectRatio: false,
            interaction: {{ mode: 'index', intersect: false }},
            plugins: {{
                title: {{
                    display: true,
                    text: ['            ', '            '],
                    font: {{ size: 24, weight: 'bold' }}
                }},
                tooltip: {{
                    mode: 'nearest',
                    intersect: false,
                    backgroundColor: 'rgba(0, 0, 0, 0.8)',
                    titleFont: {{ size: 18, weight: 'bold' }},
                    bodyFont: {{ size: 17 }},
                    padding: 15,
                    cornerRadius: 10,
                    callbacks: {{
                        title: function(context) {{
                            const index = context[0].dataIndex;
                            return `Día ${{index + 1}}: ${{fechasOrdenadas[index]}}`;
                        }},
                        label: function(context) {{
                            const index = context.dataIndex;
                            const dia = alcanceDiario[index];
                            return [
                                dia.porcentaje === null ? 'Sin datos cargados' : `Alcance: ${{dia.porcentaje.toFixed(2)}}%`,
                                dia.recupero === null ? 'Recupero acumulado: sin data' : `Recupero acumulado: S/ ${{dia.recupero.toLocaleString('es-PE', {{ minimumFractionDigits: 2, maximumFractionDigits: 2 }})}}`,
                                `Meta diaria referencial: S/ ${{metaDiaria.toLocaleString('es-PE', {{ minimumFractionDigits: 2, maximumFractionDigits: 2 }})}}`
                            ];
                        }}
                    }}
                }},
                legend: {{ display: false }}
            }},
            scales: {{
                x: {{
                    ticks: {{ font: {{ size: 16 }}, maxRotation: 0, minRotation: 0 }},
                    grid: {{ display: false }}
                }},
                y: {{
                    beginAtZero: true,
                    ticks: {{
                        callback: function(value) {{ return value + '%'; }},
                        font: {{ size: 16 }},
                        stepSize: 10
                    }},
                    min: 0,
                    suggestedMax: Math.max(100, Math.ceil(Math.max(...porcentajes.filter(v => v !== null && v !== undefined), 0) / 10) * 10)
                }}
            }}
        }};
        
        try {{
            options = _tuneOptionsForView(options, view);
        }} catch (e) {{}}
        
        const chart = new Chart(ctx, {{
            type: 'bar',
            data: {{
                labels: fechasOrdenadas.map(fecha => {{
                    if (fecha.includes('-')) return fecha.split('-')[0];
                    return fecha;
                }}),
                datasets: [{{
                    label: `Alcance Diario - ${{supervisor}}`,
                    data: porcentajes,
                    backgroundColor: coloresBarras,
                    borderColor: coloresBarras.map(color => color + 'CC'),
                    borderWidth: 2,
                    borderRadius: 6,
                    borderSkipped: false
                }}]
            }},
            options,
            plugins: [dataLabelsPluginTotal]
        }});
        
        _storeChart(canvasId, chart);
    }}

    function generarGraficaIncrementoTotalSupervisor(periodoCompleto, supervisor, canvasId = 'graficaIncrementoTotal', view = 'full') {{
    
        const canvas = document.getElementById(canvasId);
        if (!canvas) return;
        const dataLabelsPluginTotal = _buildDataLabelsPluginTotal(view);
        const ctx = canvas.getContext('2d');
    
        // Destruir gráfica anterior si existe (mejor por canvasId)
        _destroyChartByCanvasId(canvasId);
    
        // Limpiar canvas
        ctx.clearRect(0, 0, canvas.width, canvas.height);
    
        // Obtener datos del supervisor
        const supervisorData = datosSupervisores[supervisor];
        if (!supervisorData || !supervisorData[periodoCompleto]) {{
            mostrarMensajeSinDatos(canvas, periodoCompleto, supervisor);
            return;
        }}
    
        const datosMes = supervisorData[periodoCompleto];
        const metaSupervisor = datosMes.meta_super || 0;
        const datosDiarios = datosMes.datos_diarios_supervisor || {{}};
    
        if (Object.keys(datosDiarios).length === 0) {{
            mostrarMensajeSinDatos(canvas, periodoCompleto, supervisor);
            return;
        }}
    
        // Ordenar fechas
        const todasFechas = Object.keys(datosDiarios).sort((a, b) =>
            convertirFechaDiaria(a) - convertirFechaDiaria(b)
        );
    
        const recuperoAcumuladoPorFecha = {{}};
        const datosValidosParaGrafica = [];
    
        todasFechas.forEach(fecha => {{
            const recuperoAcumulado = datosDiarios[fecha] || 0;
            recuperoAcumuladoPorFecha[fecha] = recuperoAcumulado;
    
            const alcanceDia = metaSupervisor > 0 ? (recuperoAcumulado / metaSupervisor) * 100 : 0;
    
            if (recuperoAcumulado > 0) {{
                datosValidosParaGrafica.push(alcanceDia);
            }} else {{
                datosValidosParaGrafica.push(null);
            }}
        }});
    
        // Último día con datos
        let ultimoDiaConDatos = -1;
        for (let i = todasFechas.length - 1; i >= 0; i--) {{
            const fecha = todasFechas[i];
            const recuperoDia = recuperoAcumuladoPorFecha[fecha] || 0;
            if (recuperoDia > 0) {{
                ultimoDiaConDatos = i;
                break;
            }}
        }}
        if (ultimoDiaConDatos === -1) ultimoDiaConDatos = todasFechas.length - 1;
    
        // 🔸 NO fuerces 1200x700 para mini/modal; deja que _tuneOptionsForView + CSS manejen
        canvas.style.display = 'block';
        canvas.style.margin = '0 auto';
        canvas.style.maxWidth = '100%';
    
        const labelsParaGrafica = todasFechas.map(fecha => {{
            if (fecha.includes('-')) return fecha.split('-')[0];
            return fecha;
        }});
    
        const colorLinea = '#9b59b6';
        const colorArea = 'rgba(155, 89, 182, 0.2)';
    
        let options = {{
            responsive: true,
            maintainAspectRatio: false,
            plugins: {{
                title: {{
                    display: true,
                    text: [`            `,`            `],
                    font: {{ size: 22, weight: 'bold' }},
                    padding: {{ bottom: 25 }}
                }},
                tooltip: {{
                    mode: 'nearest',
                    intersect: false,
                    backgroundColor: 'rgba(0, 0, 0, 0.9)',
                    titleFont: {{ size: 18, weight: 'bold' }},
                    bodyFont: {{ size: 17 }},
                    padding: 15,
                    cornerRadius: 10,
                    callbacks: {{
                        title: function(context) {{
                            const index = context[0].dataIndex;
                            return 'Día ' + (index + 1) + ': ' + todasFechas[index];
                        }},
                        label: function(context) {{
                            const index = context.dataIndex;
                            const alcance = context.parsed.y;
                            const recuperoAcumulado = recuperoAcumuladoPorFecha[todasFechas[index]] || 0;
    
                            if (index > ultimoDiaConDatos || alcance === null) {{
                                return '📭 Sin datos de recupero';
                            }}
    
                            let incrementoDia = 0;
                            if (index > 0) {{
                                const alcanceAnterior = context.chart.data.datasets[0].data[index - 1] || 0;
                                incrementoDia = (alcance - alcanceAnterior);
                            }} else {{
                                incrementoDia = alcance;
                            }}
    
                            return [
                                '🎯 Alcance Acumulado: ' + alcance.toFixed(2) + '%',
                                '📈 Alcance del día: ' + incrementoDia.toFixed(2) + '%',
                                '💰 Recupero Acumulado: ' + recuperoAcumulado.toLocaleString('es-PE', {{
                                    minimumFractionDigits: 0,
                                    maximumFractionDigits: 0
                                }})
                            ];
                        }}
                    }}
                }},
                legend: {{ display: false }}
            }},
            scales: {{
                x: {{
                    ticks: {{
                        font: {{ size: 16 }},
                        maxRotation: 0,
                        minRotation: 0,
                        autoSkip: false
                    }},
                    grid: {{
                        display: true,
                        color: 'rgba(0, 0, 0, 0.05)'
                    }}
                }},
                y: {{
                    beginAtZero: true,
                    ticks: {{
                        callback: function(value) {{ return value + '%'; }},
                        stepSize: 10,
                        font: {{ size: 16 }}
                    }},
                    min: 0,
                    suggestedMax: (function() {{
                        const valoresValidos = datosValidosParaGrafica.filter(v => v !== null && v > 0);
                        if (valoresValidos.length === 0) return 100;
                        const maxValor = Math.max(...valoresValidos);
                        return Math.max(100, Math.ceil(maxValor / 10) * 10);
                    }})(),
                    grid: {{
                        display: true,
                        color: 'rgba(0, 0, 0, 0.05)'
                    }}
                }}
            }},
            interaction: {{
                mode: 'nearest',
                axis: 'x',
                intersect: false
            }},
            elements: {{
                line: {{ tension: 0.4 }}
            }}
        }};
    
        try {{
          options = _tuneOptionsForView(options, view);
        }} catch (e) {{}}
        
        
        try {{
          const chart = new Chart(ctx, {{
            type: 'line',
            data: {{
              labels: labelsParaGrafica,
              datasets: [{{
                label: `Alcance Acumulado - ${{supervisor}}`,
                data: datosValidosParaGrafica,
                backgroundColor: colorArea,
                borderColor: colorLinea,
                borderWidth: 3,
                fill: true,
                tension: 0.4,
                pointRadius: function(context) {{
                  const value = context.dataset.data[context.dataIndex];
                  return value === null || value === 0 ? 0 : 6;
                }},
                pointHoverRadius: function(context) {{
                  const value = context.dataset.data[context.dataIndex];
                  return value === null || value === 0 ? 0 : 8;
                }},
                pointBackgroundColor: function(context) {{
                  const value = context.dataset.data[context.dataIndex];
                  return value === null || value === 0 ? 'transparent' : colorLinea;
                }},
                pointBorderColor: function(context) {{
                  const value = context.dataset.data[context.dataIndex];
                  return value === null || value === 0 ? 'transparent' : '#ffffff';
                }},
                pointBorderWidth: 2,
                spanGaps: false
              }}]
            }},
            options,
            plugins: [dataLabelsPluginTotal]
          }});
        
          _storeChart(canvasId, chart);
        
        }} catch (error) {{
          console.error("Error al crear la gráfica:", error);
          ctx.fillStyle = '#e74c3c';
          ctx.font = 'bold 20px Arial';
          ctx.textAlign = 'center';
          ctx.fillText('Error al generar gráfica', canvas.width / 2, canvas.height / 2);
        }}
    }}
    
    function calcularEstadisticasRecuperosPorSupervisor(periodoCompleto, supervisor) {{
        // 1. OBTENER DATOS DEL SUPERVISOR ESPECÍFICO
        const supervisorData = datosSupervisores[supervisor];
        
        if (!supervisorData || !supervisorData[periodoCompleto]) {{
            console.log(`⚠️ No hay datos para el supervisor: ${{supervisor}} en ${{periodoCompleto}}`);
            mostrarEstadisticasVacias(periodoCompleto, supervisor);
            return;
        }}
        
        const datosMes = supervisorData[periodoCompleto];
        
        // 2. DATOS BÁSICOS DEL SUPERVISOR
        const metaTotalMes = datosMes.meta_super || 0;
        const recuperoTotalActual = datosMes.total_recupero || 0;
        
        // 3. OBTENER DÍAS LABORABLES
        const datosDiarios = datosMes.datos_diarios_supervisor || {{}};
        let fechas = Object.keys(datosDiarios);
        
        if (fechas.length === 0) {{

            const alcanceDiario = datosMes.alcance_acumulado_diario || {{}};
            fechas = Object.keys(alcanceDiario);
        }}
        
        const fechasOrdenadas = fechas.sort((a, b) => 
            convertirFechaDiaria(a) - convertirFechaDiaria(b)
        );
        
        // Días registrados
        const diasRegistradosBD = fechasOrdenadas.length;
        
        // Días laborables
        const diasLaborables = 0;
        
        // 4. CALCULAR DÍAS TRABAJADOS
        let diasTrabajados = 0;
        let totalRecuperoDiasTrabajados = 0;
        
        if (fechasOrdenadas.length > 0) {{
            for (let i = 0; i < fechasOrdenadas.length; i++) {{
                const fecha = fechasOrdenadas[i];
                let recuperoDia = 0;
        
                if (datosDiarios[fecha] !== undefined) {{
                    if (i === 0) {{
                        recuperoDia = datosDiarios[fecha] || 0;
                    }} else {{
                        const fechaAnterior = fechasOrdenadas[i-1];
                        const recuperoAnterior = datosDiarios[fechaAnterior] || 0;
                        const recuperoActual = datosDiarios[fecha] || 0;
                        recuperoDia = recuperoActual - recuperoAnterior;
                    }}
                }}
        
                if (recuperoDia > 0) {{
                    diasTrabajados++;
                    totalRecuperoDiasTrabajados += recuperoDia;
                }}
            }}
        }}
        
        // 5. CALCULAR ALCANCE ACTUAL
        const alcanceActual = metaTotalMes > 0 ? 
            parseFloat(((recuperoTotalActual / metaTotalMes) * 100).toFixed(2)) : 0;
        
        // 6. CALCULAR META DIARIA (basada en días laborables)
        const metaDiaria = diasLaborables > 0 ? metaTotalMes / diasLaborables : 0;
        
        // 7. CALCULAR PROMEDIO DIARIO REAL
        const promedioDiarioReal = diasTrabajados > 0 ? totalRecuperoDiasTrabajados / diasTrabajados : 0;
        
        // 8. CALCULAR PROYECCIONES
        const diasRestantes = Math.max(0, diasLaborables - diasTrabajados);
        const recuperoProyectado = promedioDiarioReal * diasRestantes;
        const recuperoTotalProyectado = recuperoTotalActual + recuperoProyectado;
        const eficienciaDiaria = metaTotalMes > 0 ? (recuperoTotalProyectado / metaTotalMes) * 100 : 0;
        
        // 9. Calcular eficiencia vs meta diaria
        const eficienciaVsMetaDiaria = metaDiaria > 0 ? (promedioDiarioReal / metaDiaria) * 100 : 0;
        
        // 10. INTERFAZ
        mostrarEstadisticasUnificada(
          periodoCompleto,
          {{
            metaTotalMes,
            recuperoTotalActual,
            alcanceActual,
            eficienciaDiaria,
            diasLaborables,
            diasRegistradosBD,
            diasTrabajados,
            metaDiaria,
            promedioDiarioReal,
            eficienciaVsMetaDiaria,
            recuperoProyectado,
            recuperoTotalProyectado,
            diasRestantes
          }},
          'supervisor',
          supervisor
        );
    }}

    function mostrarEstadisticasVacias(periodoCompleto, supervisor = null) {{
        const statsElement = document.getElementById('estadisticas-recuperos');
        if (!statsElement) return;
        statsElement.innerHTML = '';
    }}

    function mostrarEstadisticasUnificada(
        periodoCompleto,
        estadisticas,
        contexto = 'global',
        supervisorNombre = null
    ) {{
        const {{
          metaTotalMes,
          recuperoTotalActual,
          alcanceActual,
          eficienciaDiaria,
          diasLaborables,
          diasRegistradosBD,
          diasTrabajados,
          metaDiaria,
          promedioDiarioReal,
          eficienciaVsMetaDiaria,
          recuperoProyectado,
          recuperoTotalProyectado,
          diasRestantes
        }} = estadisticas;

        const dl = Number(diasLaborables || 0);
        const dt = Number(diasTrabajados || 0);
        const dtCap = (dl > 0) ? Math.min(dt, dl) : dt;
        
        const ratioTrabajados = (dl > 0) ? `${{dtCap}}/${{dl}}` : '0/0';
        const porcentajeTiempo = (dl > 0) ? ((dtCap / dl) * 100).toFixed(0) : 0;
        const progresoTiempoPorcentaje = (dl > 0) ? (dtCap / dl) * 100 : 0;
        const formatoMoneda = (valor) => 'S/ ' + valor.toLocaleString('es-PE', {{ minimumFractionDigits: 0 }});
        const [mes, año] = periodoCompleto.split('_');
        const colorAlcanceActual = alcanceActual >= 100 ? '#27ae60' : 
                                  alcanceActual >= 70 ? '#f39c12' : 
                                  alcanceActual >= 40 ? '#e67e22' : '#e74c3c';
        
        const colorEficiencia = eficienciaDiaria >= 100 ? '#27ae60' : 
                               eficienciaDiaria >= 70 ? '#f39c12' : 
                               eficienciaDiaria >= 40 ? '#e67e22' : '#e74c3c';
        
        const textoMeta = contexto === 'supervisor' ? 'Meta del supervisor' : 'Meta total del período';
        const textoRecupero = contexto === 'supervisor' ? 'Total recuperado' : 'Total recuperado por todos';
        const html = `
            <div style="display:grid; grid-template-columns: repeat(4, minmax(180px, 1fr)); gap:15px; margin-bottom:18px;">
                <div style="text-align:center; padding:18px; background:#E8F4FD; border-radius:10px; border:2px solid #3498db;">
                    <div style="font-size:0.95rem; color:#3498db; margin-bottom:6px; font-weight:700;">DIAS TRABAJADOS</div>
                    <div style="font-size:2rem; font-weight:800; color:#3498db;">${{ratioTrabajados}}</div>
                    <div style="font-size:0.85rem; color:#666; margin-top:5px;">${{porcentajeTiempo}}% del mes</div>
                </div>

                <div style="text-align:center; padding:18px; background:#F4ECF7; border-radius:10px; border:2px solid #9b59b6;">
                    <div style="font-size:0.95rem; color:#9b59b6; margin-bottom:6px; font-weight:700;">META DEL MES</div>
                    <div style="font-size:1.55rem; font-weight:800; color:#9b59b6;">${{formatoMoneda(metaTotalMes)}}</div>
                    <div style="font-size:0.85rem; color:#666; margin-top:5px;">${{textoMeta}}</div>
                </div>

                <div style="text-align:center; padding:18px; background:#E8F6F3; border-radius:10px; border:2px solid #27ae60;">
                    <div style="font-size:0.95rem; color:#27ae60; margin-bottom:6px; font-weight:700;">RECUPERO ACTUAL</div>
                    <div style="font-size:1.55rem; font-weight:800; color:#27ae60;">${{formatoMoneda(recuperoTotalActual)}}</div>
                    <div style="font-size:0.85rem; color:#666; margin-top:5px;">${{textoRecupero}}</div>
                </div>

                <div style="text-align:center; padding:18px; background:#D1E9FB; border-radius:10px; border:2px solid #3498db;">
                    <div style="font-size:0.95rem; color:#3498db; margin-bottom:6px; font-weight:700;">?? ALCANCE ACTUAL</div>
                    <div style="font-size:2rem; font-weight:800; color:${{colorAlcanceActual}};">${{alcanceActual.toFixed(2)}}%</div>
                    <div style="font-size:0.85rem; color:#2c3e50; margin-top:5px;">${{formatoMoneda(recuperoTotalActual)}} / ${{formatoMoneda(metaTotalMes)}}</div>
                </div>
            </div>
        `;

        const statsElement = document.getElementById('estadisticas-recuperos');
        if (statsElement) {{
            statsElement.innerHTML = '';
        }}
    }}

    window.__simAlcancesState = window.__simAlcancesState || {{}};
    
    // MODAL: SIMULACIÓN
    function _asegurarModalSimulacionAlcances() {{
        if (document.getElementById('modalSimAlcancesOverlay')) return;

        const overlay = document.createElement('div');
        overlay.id = 'modalSimAlcancesOverlay';
        overlay.style.cssText = `
            position: fixed;
            inset: 0;
            background: rgba(0,0,0,0.55);
            display: none;
            align-items: center;
            justify-content: center;
            z-index: 99999;
            padding: 18px;
        `;

        overlay.innerHTML = `
            <style id="simRangeStyles">
              /* Solo afecta sliders dentro del modal */
              #modalSimAlcancesOverlay .sim-range {{
                width: 100%;
                height: 14px;
                border-radius: 999px;
                cursor: pointer;
                -webkit-appearance: none;
                appearance: none;
                background: #e9e9ef;
                outline: none;
              }}
            
              /* WebKit (Chrome/Edge) */
              #modalSimAlcancesOverlay .sim-range::-webkit-slider-thumb {{
                -webkit-appearance: none;
                appearance: none;
                width: 22px;
                height: 22px;
                border-radius: 50%;
                background: #9b59b6;
                border: 3px solid #fff;
                box-shadow: 0 4px 10px rgba(0,0,0,0.2);
                margin-top: -4px; /* centra el thumb */
              }}
            
              #modalSimAlcancesOverlay .sim-range::-webkit-slider-runnable-track {{
                height: 14px;
                border-radius: 999px;
              }}
            
              /* Firefox */
              #modalSimAlcancesOverlay .sim-range::-moz-range-track {{
                height: 14px;
                border-radius: 999px;
                background: #e9e9ef;
              }}
            
              #modalSimAlcancesOverlay .sim-range::-moz-range-thumb {{
                width: 22px;
                height: 22px;
                border-radius: 50%;
                background: #9b59b6;
                border: 3px solid #fff;
                box-shadow: 0 4px 10px rgba(0,0,0,0.2);
              }}
            </style>
            <div
              id="modalSimAlcancesCard"
              style="
                width: min(1320px, 98vw);
                max-height: 90vh;
                overflow: auto;
                background: #fff;
                border-radius: 16px;
                box-shadow: 0 18px 50px rgba(0,0,0,0.25);
                border: 1px solid rgba(0,0,0,0.08);
              "
            >

                <div style="padding: 16px 18px;">
                    <div id="simAlcancesPanelMes"></div>
                    <div id="simAlcancesTabla"></div>
                </div>
            </div>
        `;
        overlay.addEventListener('click', (e) => {{
            if (e.target && e.target.id === 'modalSimAlcancesOverlay') {{
                cerrarModalSimulacionAlcances();
            }}
        }});

        document.body.appendChild(overlay);
    }}

    function _keyMes(periodoCompleto) {{
        return `MES__${{periodoCompleto}}`;
    }}

    function _getTotalesMes(periodoCompleto) {{
        const supervisores = _getSupervisoresDePeriodo(periodoCompleto);

        let metaMes = 0;
        let recMes = 0;

        supervisores.forEach((sup) => {{
            const d = datosSupervisores[sup][periodoCompleto] || {{}};
            metaMes += Number(d.meta_super || 0);
            recMes  += Number(d.total_recupero || 0);
        }});

        const alcanceMes = (metaMes > 0) ? (recMes / metaMes) * 100 : 0;
        return {{ metaMes, recMes, alcanceMes }};
    }}

    function _getSimMes(periodoCompleto) {{
        const supervisores = _getSupervisoresDePeriodo(periodoCompleto);

        let metaMes = 0;
        let recActualMes = 0;
        let recSimMes = 0;

        supervisores.forEach((sup) => {{
            const d = datosSupervisores[sup][periodoCompleto] || {{}};
            const meta = Number(d.meta_super || 0);
            const rec  = Number(d.total_recupero || 0);

            metaMes += meta;
            recActualMes += rec;

            const key = `${{periodoCompleto}}__${{sup}}`;
            const st = window.__simAlcancesState[key] || {{}};
            const add = Math.max(0, Number(st.montoAdd || 0));

            recSimMes += (rec + add);
        }});

        const alcanceSimMes = (metaMes > 0) ? (recSimMes / metaMes) * 100 : 0;

        return {{
            metaMes,
            recActualMes,
            recSimMes,
            alcanceSimMes
        }};
    }}

    function abrirModalSimulacionAlcances(periodoCompleto) {{
        _asegurarModalSimulacionAlcances();

        const overlay = document.getElementById('modalSimAlcancesOverlay');
        overlay.style.display = 'flex';
        _renderModalSimulacionAlcances(periodoCompleto);
    }}

    function cerrarModalSimulacionAlcances() {{
        const overlay = document.getElementById('modalSimAlcancesOverlay');
        if (overlay) overlay.style.display = 'none';
    }}

    function _getSupervisoresDePeriodo(periodoCompleto) {{
        const supervisoresData = datosSupervisores || {{}};
        return Object.keys(supervisoresData).filter(sup =>
            supervisoresData[sup] && supervisoresData[sup][periodoCompleto]
        );
    }}

    function _fmtMoneda(valor) {{
        const n = Number(valor || 0);
        return 'S/ ' + n.toLocaleString('es-PE', {{ minimumFractionDigits: 0 }});
    }}

    function _clamp(n, a, b) {{
        n = Number(n);
        if (!isFinite(n)) n = 0;
        return Math.max(a, Math.min(b, n));
    }}

    function _colorPorAlcance(pct) {{
        const p = Number(pct || 0);
        return p >= 100 ? '#27ae60' : p >= 70 ? '#f39c12' : p >= 40 ? '#e67e22' : '#e74c3c';
    }}

    function _montoParaAlcance(meta, recuperoActual, objetivoPct) {{
        const m = Number(meta || 0);
        const r = Number(recuperoActual || 0);
        const obj = Number(objetivoPct || 0);

        if (m <= 0) return 0;
        const totalRequerido = (m * obj) / 100;
        return Math.max(0, totalRequerido - r);
    }}

    function _renderModalSimulacionAlcances(periodoCompleto) {{
        const supervisores = _getSupervisoresDePeriodo(periodoCompleto);

        const sub = document.getElementById('simAlcancesSubtitulo');
        if (sub) sub.textContent = `Periodo: ${{String(periodoCompleto).replace('_', ' ')}}`;

        const cont = document.getElementById('simAlcancesTabla');
        if (!cont) return;

        if (!supervisores.length) {{
            cont.innerHTML = `
                <div style="padding: 14px; border: 2px dashed #cfd8dc; border-radius: 12px; text-align:center; color:#7f8c8d;">
                    No hay supervisores con data para este periodo.
                </div>
            `;
            return;
        }}
        let html = `
          <div
            style="
              display:grid;
              grid-template-columns: repeat(2, minmax(520px, 1fr));
              gap: 12px;
              align-items: start;
            "
            id="simGridSupervisores"
          >
        `;
        const panelMes = document.getElementById('simAlcancesPanelMes');
        if (panelMes) {{
            const t = _getTotalesMes(periodoCompleto);
            const kMes = _keyMes(periodoCompleto);
            if (!window.__simAlcancesState[kMes]) {{
                window.__simAlcancesState[kMes] = {{ objetivoPctMes: 100 }};
            }}

            const objMes = _clamp(window.__simAlcancesState[kMes].objetivoPctMes, 0, 200);
            const sim = _getSimMes(periodoCompleto);

            const cAct = _colorPorAlcance(t.alcanceMes);
            const cSim = _colorPorAlcance(sim.alcanceSimMes);

            panelMes.innerHTML = `
                <div style="
                    border: 2px solid rgba(155,89,182,0.25);
                    background: linear-gradient(180deg, rgba(155,89,182,0.08), rgba(155,89,182,0.03));
                    border-radius: 16px;
                    padding: 14px;
                    margin-bottom: 12px;
                ">
                    <div style="display:flex; justify-content:space-between; align-items:flex-start; gap:12px; flex-wrap:wrap;">
                        <div>
                            <div style="font-weight: 1000; color:#2c3e50; font-size: 1.05rem;">
                                📌 Simulación de ${{periodoCompleto.replace('_', ' ')}}
                            </div>
                            <div style="margin-top:6px; font-size:0.92rem; color:#7f8c8d;">
                                Meta mes: <b style="color:#2c3e50;">${{_fmtMoneda(t.metaMes)}}</b>
                                · Recupero actual mes: <b style="color:#2c3e50;">${{_fmtMoneda(t.recMes)}}</b>
                                · Recupero simulado mes: <b id="simMesRec__${{periodoCompleto}}" style="color:#2c3e50;">${{_fmtMoneda(sim.recSimMes)}}</b>
                            </div>
                        </div>

                        <div style="display:flex; gap:10px; flex-wrap:wrap;">
                            <div style="padding:10px 12px; border-radius:12px; background:${{cAct}}14; border:1px solid ${{cAct}}55; min-width: 190px;">
                                <div style="font-size:0.82rem; color:#7f8c8d;">Alcance actual mes</div>
                                <div style="font-size:1.25rem; font-weight:1000; color:${{cAct}};">${{t.alcanceMes.toFixed(2)}}%</div>
                            </div>

                            <div style="padding:10px 12px; border-radius:12px; background:${{cSim}}14; border:1px solid ${{cSim}}55; min-width: 200px;">
                                <div style="font-size:0.82rem; color:#7f8c8d;">Alcance simulado mes</div>
                                <div id="simMesPct__${{periodoCompleto}}" style="font-size:1.25rem; font-weight:1000; color:${{cSim}};">
                                    ${{sim.alcanceSimMes.toFixed(2)}}%
                                </div>
                            </div>
                        </div>
                    </div>

                    <div style="margin-top: 12px;">     
                        <div style="padding: 6px 0;">
                          <input
                            id="simMesSlider__${{periodoCompleto}}"
                            type="range"
                            min="0"
                            max="200"
                            value="${{objMes}}"
                            class="sim-range"
                            oninput="onSimMesObjetivoInput('${{periodoCompleto}}', this.value)"
                            style="width:100%;"
                          />
                        </div>
                    </div>
                </div>
            `;
        }}

        window.__simSupIndexMap = window.__simSupIndexMap || {{}};
        window.__simSupIndexMap[periodoCompleto] = {{}};

        supervisores.forEach((sup, idx) => {{
            window.__simSupIndexMap[periodoCompleto][sup] = idx;
            const d = datosSupervisores[sup][periodoCompleto] || {{}};
            const meta = Number(d.meta_super || 0);
            const rec = Number(d.total_recupero || 0);
            const alcanceActual = (meta > 0) ? (rec / meta) * 100 : 0;
            const key = `${{periodoCompleto}}__${{sup}}`;

            if (!window.__simAlcancesState[key]) {{
                const objetivoInicial = 100;
                const addInicial = _montoParaAlcance(meta, rec, objetivoInicial);
                window.__simAlcancesState[key] = {{
                    objetivoPct: objetivoInicial,
                    montoAdd: addInicial
                }};
            }}

            const st = window.__simAlcancesState[key];
            const objetivoPct = _clamp(st.objetivoPct, 0, 200);
            const montoAdd = Math.max(0, Number(st.montoAdd || 0));

            const recSim = rec + montoAdd;
            const alcanceSim = (meta > 0) ? (recSim / meta) * 100 : 0;

            const cAct = _colorPorAlcance(alcanceActual);
            const cSim = _colorPorAlcance(alcanceSim);

            html += `
                <div style="border:1px solid #eee; border-radius: 14px; padding: 10px; background:#fff;">
                    <div style="display:flex; justify-content:space-between; gap:12px; align-items:flex-start; flex-wrap:wrap;">
                        <div style="min-width: 240px;">
                            <div style="font-weight: 900; color:#2c3e50; font-size: 1.02rem;">👤 ${{sup}}</div>
                            <div style="margin-top:6px; font-size:0.9rem; color:#7f8c8d;">
                              Meta: <b style="color:#2c3e50;">${{_fmtMoneda(meta)}}</b>
                              · Recupero actual: <b style="color:#2c3e50;">${{_fmtMoneda(rec)}}</b>
                              · Recupero simulado: <b id="simRec__${{idx}}" style="color:#2c3e50;">${{_fmtMoneda(recSim)}}</b>
                            </div>
                        </div>

                        <!-- FILA COMPACTA: Actual + Simulado + Monto adicional -->
                        <div style="display:flex; gap:10px; flex-wrap:wrap; align-items:stretch;">
                            <div style="padding:8px 10px; border-radius:12px; background:${{cAct}}14; border:1px solid ${{cAct}}55; min-width: 155px;">
                                <div style="font-size:0.8rem; color:#7f8c8d;">Alcance actual</div>
                                <div style="font-size:1.15rem; font-weight:900; color:${{cAct}};">${{alcanceActual.toFixed(2)}}%</div>
                            </div>

                            <div style="padding:8px 10px; border-radius:12px; background:${{cSim}}14; border:1px solid ${{cSim}}55; min-width: 165px;">
                                <div style="font-size:0.8rem; color:#7f8c8d;">Alcance simulado</div>
                                <div id="simPct__${{idx}}" style="font-size:1.15rem; font-weight:900; color:${{cSim}};">
                                    ${{alcanceSim.toFixed(2)}}%
                                </div>
                            </div>

                            <div style="padding:8px 10px; border-radius:12px; background:#f7f7fb; border:1px solid #e6e6f0; min-width: 220px;">
                                <div style="font-size:0.8rem; color:#7f8c8d;">Monto adicional</div>
                                <input
                                    id="simMonto__${{idx}}"
                                    type="number"
                                    min="0"
                                    step="1"
                                    value="${{Math.round(montoAdd)}}"
                                    oninput="onSimAlcancesMontoInput('${{periodoCompleto}}','${{sup}}', ${{idx}}, this)"
                                    onblur="onSimAlcancesMontoCommit('${{periodoCompleto}}','${{sup}}', ${{idx}}, this)"
                                    style="
                                        width:100%;
                                        margin-top:6px;
                                        padding: 9px 10px;
                                        border-radius: 10px;
                                        border: 1px solid #ddd;
                                        font-weight:900;
                                        color:#2c3e50;
                                    "
                                />
                            </div>
                        </div>
                    </div>

                    <!-- Slider abajo (más bajo, solo una fila) -->
                    <div style="margin-top: 10px;">
                        <input
                          id="simSlider__${{idx}}"
                          type="range"
                          min="0"
                          max="200"
                          value="${{objetivoPct}}"
                          class="sim-range"
                          oninput="onSimAlcancesSliderInput('${{periodoCompleto}}','${{sup}}', ${{idx}}, this.value)"
                          style="width:100%;"
                        />
                    </div>
                </div>
            `;
        }});

        html += `</div>`;
        cont.innerHTML = html;
        const grid = document.getElementById('simGridSupervisores');
        if (grid) {{
            const w = window.innerWidth || 0;
            if (w < 1100) {{
                grid.style.gridTemplateColumns = '1fr';
            }}
        }}
    }}

    // HELPERS ALCANCES

    function _updatePanelMesUI(periodoCompleto) {{
        const sim = _getSimMes(periodoCompleto);

        const elRec = document.getElementById(`simMesRec__${{periodoCompleto}}`);
        if (elRec) elRec.textContent = _fmtMoneda(sim.recSimMes);

        const elPct = document.getElementById(`simMesPct__${{periodoCompleto}}`);
        if (elPct) {{
            elPct.textContent = `${{sim.alcanceSimMes.toFixed(2)}}%`;
            const c = _colorPorAlcance(sim.alcanceSimMes);
            elPct.style.color = c;
        }}
    }}

    function onSimMesObjetivoInput(periodoCompleto, objetivoPctMes) {{
      const kMes = _keyMes(periodoCompleto);
      window.__simAlcancesState[kMes] = window.__simAlcancesState[kMes] || {{}};
    
      const obj = _clamp(objetivoPctMes, 0, 200);
      window.__simAlcancesState[kMes].objetivoPctMes = obj;
    
      const elObj = document.getElementById(`simMesObj__${{periodoCompleto}}`);
      if (elObj) elObj.textContent = `${{obj.toFixed(0)}}%`;
    
      const supervisores = _getSupervisoresDePeriodo(periodoCompleto);
      const mapIdx = (window.__simSupIndexMap && window.__simSupIndexMap[periodoCompleto]) ? window.__simSupIndexMap[periodoCompleto] : {{}};
    
      supervisores.forEach((sup) => {{
        const d = datosSupervisores[sup][periodoCompleto] || {{}};
        const meta = Number(d.meta_super || 0);
        const rec  = Number(d.total_recupero || 0);
    
        const key = `${{periodoCompleto}}__${{sup}}`;
        window.__simAlcancesState[key] = window.__simAlcancesState[key] || {{}};
    
        const add = _montoParaAlcance(meta, rec, obj);
        window.__simAlcancesState[key].objetivoPct = obj;
        window.__simAlcancesState[key].montoAdd = add;
    
        const idx = mapIdx[sup];
        if (idx === undefined) return;
    
        const elSupObj = document.getElementById(`simObj__${{idx}}`);
        if (elSupObj) elSupObj.textContent = `${{obj.toFixed(0)}}%`;
    
        const elSlider = document.getElementById(`simSlider__${{idx}}`);
        if (elSlider && document.activeElement !== elSlider) elSlider.value = String(obj);
    
        const elMonto = document.getElementById(`simMonto__${{idx}}`);
        if (elMonto && document.activeElement !== elMonto) elMonto.value = String(Math.round(add));
    
        _updateSimCardUI(idx, meta, rec, add);
      }});
    
      _updatePanelMesUI(periodoCompleto);
    }}

    function _safeNum(v) {{
        const n = Number(v);
        return (isFinite(n) ? n : 0);
    }}

    function _updateSimCardUI(idx, meta, rec, add) {{
        const recSim = rec + add;
        const pctSim = (meta > 0) ? (recSim / meta) * 100 : 0;

        const elRec = document.getElementById(`simRec__${{idx}}`);
        if (elRec) elRec.textContent = _fmtMoneda(recSim);

        const elPct = document.getElementById(`simPct__${{idx}}`);
        if (elPct) {{
            elPct.textContent = `${{pctSim.toFixed(2)}}%`;
            const c = _colorPorAlcance(pctSim);
            elPct.style.color = c;
        }}
    }}

    // ===== Slider: recalcula monto adicional
    function onSimAlcancesSliderInput(periodoCompleto, supervisor, idx, objetivoPct) {{
        const key = `${{periodoCompleto}}__${{supervisor}}`;
        window.__simAlcancesState[key] = window.__simAlcancesState[key] || {{}};

        const d = (datosSupervisores[supervisor] && datosSupervisores[supervisor][periodoCompleto])
            ? datosSupervisores[supervisor][periodoCompleto]
            : {{}};

        const meta = _safeNum(d.meta_super);
        const rec  = _safeNum(d.total_recupero);

        const obj = _clamp(objetivoPct, 0, 200);
        const add = _montoParaAlcance(meta, rec, obj);

        window.__simAlcancesState[key].objetivoPct = obj;
        window.__simAlcancesState[key].montoAdd = add;

        const elObj = document.getElementById(`simObj__${{idx}}`);
        if (elObj) elObj.textContent = `${{obj.toFixed(0)}}%`;

        const elMonto = document.getElementById(`simMonto__${{idx}}`);
        if (elMonto && document.activeElement !== elMonto) {{
            elMonto.value = String(Math.round(add));
        }}

        _updateSimCardUI(idx, meta, rec, add);
        _updatePanelMesUI(periodoCompleto);
    }}

    // ===== Monto: mientras escribes, NO re-render; solo recalcula UI
    function onSimAlcancesMontoInput(periodoCompleto, supervisor, idx, inputEl) {{
        const key = `${{periodoCompleto}}__${{supervisor}}`;
        window.__simAlcancesState[key] = window.__simAlcancesState[key] || {{}};

        const d = (datosSupervisores[supervisor] && datosSupervisores[supervisor][periodoCompleto])
            ? datosSupervisores[supervisor][periodoCompleto]
            : {{}};

        const meta = _safeNum(d.meta_super);
        const rec  = _safeNum(d.total_recupero);

        const add = Math.max(0, _safeNum(inputEl.value));
        const obj = (meta > 0) ? ((rec + add) / meta) * 100 : 0;

        window.__simAlcancesState[key].montoAdd = add;
        window.__simAlcancesState[key].objetivoPct = _clamp(obj, 0, 200);
        const elObj = document.getElementById(`simObj__${{idx}}`);
        if (elObj) elObj.textContent = `${{window.__simAlcancesState[key].objetivoPct.toFixed(0)}}%`;

        _updateSimCardUI(idx, meta, rec, add);
        _updatePanelMesUI(periodoCompleto);
    }}

    // ===== Commit al salir del input
    function onSimAlcancesMontoCommit(periodoCompleto, supervisor, idx, inputEl) {{
        const key = `${{periodoCompleto}}__${{supervisor}}`;
        const st = window.__simAlcancesState[key] || {{}};
        const add = Math.max(0, Math.round(_safeNum(st.montoAdd)));
        st.montoAdd = add;
        inputEl.value = String(add);
    }}
    
    // FUNCIÓN PARA MOSTRAR ESTADÍSTICAS VACÍAS (CASO ESPECIAL)
    function mostrarEstadisticasVacias(periodoCompleto, supervisor = null) {{
        const statsElement = document.getElementById('estadisticas-recuperos');
        if (!statsElement) return;
        statsElement.innerHTML = '';
    }}

    function actualizarPeriodo() {{
      if (window.actualizacionPeriodoEnCurso) return;
      window.actualizacionPeriodoEnCurso = true;

      try {{
        const mesSeleccionado = document.getElementById('selectorMes')?.value || '';
        const añoSeleccionado = document.getElementById('selectorAño')?.value || '';
        const periodoCompleto = `${{mesSeleccionado}}_${{añoSeleccionado}}`;

        if (!mesSeleccionado || !añoSeleccionado || !datosMeses[periodoCompleto]) return;

        estadoGlobal.mesActual = mesSeleccionado;
        estadoGlobal.añoActual = añoSeleccionado;
        estadoGlobal.periodoActual = periodoCompleto;

        // Una sola ruta de actualización conserva canal y supervisor al cambiar MES_AÑO.
        sincronizarBotonesCanalAlcances();
        actualizarFiltrosGlobales();
        aplicarFiltroSupervisor(supervisorFiltroActual || 'TODOS');

        const seccionRanking = document.getElementById('seccion-ranking');
        if (seccionRanking?.classList.contains('activa')) {{
          if (asesoresSeleccionados.size > 0) {{
            actualizarTablaComparacionAsesores();
            actualizarGraficaComparacionAsesores();
          }}
          if (supervisoresSeleccionados.size > 0) {{
            actualizarTablaComparacionSupervisores();
            actualizarGraficaComparacionSupervisores();
          }}
        }}
      }} finally {{
        window.actualizacionPeriodoEnCurso = false;
      }}
    }}

    function calcularStepSize(valorMaximo) {{
        if (valorMaximo <= 10) return 1;
        if (valorMaximo <= 50) return 5;
        if (valorMaximo <= 100) return 10;
        if (valorMaximo <= 200) return 20;
        if (valorMaximo <= 500) return 50;
        return 100;
    }}
    
    // ========== FUNCIONES PARA RANKING ==========


    // ========== ACTIVIDAD MUNDIALISTA ===========
    function _normMundial(txt) {{ return String(txt || '').normalize('NFD').replace(/[\u0300-\u036f]/g, '').toUpperCase().trim().replace(/\s+/g, ' '); }}
    function _fmtPctMundial(v) {{ const n = Number(v); return Number.isFinite(n) ? n.toFixed(2) + '%' : '0.00%'; }}
    function _colorMundial(v) {{ const n = Number(v) || 0; if (n >= 100) return '#0f8f5f'; if (n >= 70) return '#d89a12'; if (n >= 40) return '#d86820'; return '#c93636'; }}
    function _banderaHTML(src, pais, clase = 'mundial-flag') {{ if (!src) return `<div class="${{clase}} mundial-flag-empty"></div>`; return `<img class="${{clase}}" src="${{src}}" alt="${{pais || 'Bandera'}}" loading="lazy" />`; }}
    function _ensureMundialStyles() {{ if (document.getElementById('mundial-modal-styles')) return; const style = document.createElement('style'); style.id = 'mundial-modal-styles'; style.textContent = `.mundial-modal{{position:fixed;inset:0;z-index:10020;display:none}}.mundial-modal.is-open{{display:block}}.mundial-backdrop{{position:absolute;inset:0;background:rgba(8,17,28,.78);backdrop-filter:blur(6px)}}.mundial-card{{position:absolute;inset:32px;max-width:1720px;margin:auto;background:#f8fafc;border-radius:18px;overflow:hidden;box-shadow:0 24px 70px rgba(0,0,0,.42);display:flex;flex-direction:column;animation:mundialIn .26s ease-out}}.mundial-header{{position:relative;display:flex;justify-content:space-between;gap:16px;align-items:center;padding:18px 22px;color:#fff;background:linear-gradient(135deg,#123b2d,#0f766e 52%,#b91c1c)}}.mundial-header h2{{margin:0;font-size:clamp(2rem,3.2vw,3.4rem);letter-spacing:0;text-transform:uppercase;line-height:1;font-weight:950;text-shadow:0 3px 12px rgba(0,0,0,.28)}}.mundial-header p{{display:none}}.mundial-close,.mundial-nav-btn{{border:0;cursor:pointer;font-weight:900;border-radius:8px}}.mundial-close{{width:40px;height:40px;color:#123;background:#fff;font-size:1.2rem}}.mundial-body{{position:relative;padding:18px;overflow:auto}}.mundial-toolbar{{display:flex;justify-content:space-between;align-items:center;gap:12px;margin-bottom:14px}}.mundial-nav{{display:flex;gap:8px}}.mundial-nav-btn{{min-width:42px;height:38px;background:#0f766e;color:#fff;font-size:1.15rem}}.mundial-page{{font-weight:800;color:#334155}}.mundial-grid-3{{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:14px}}.mundial-team{{background:#fff;border:1px solid #e2e8f0;border-radius:8px;overflow:hidden;box-shadow:0 10px 24px rgba(15,23,42,.08);animation:mundialRise .35s ease both}}.mundial-team-top{{display:grid;grid-template-columns:76px 1fr;gap:12px;align-items:center;padding:14px;background:linear-gradient(180deg,#fff,#f1f5f9);border-bottom:1px solid #e2e8f0}}.mundial-flag{{width:64px;height:44px;object-fit:cover;border-radius:6px;border:1px solid rgba(15,23,42,.18);box-shadow:0 4px 12px rgba(15,23,42,.16);background:#fff}}.mundial-flag-large{{width:82px;height:56px;object-fit:cover;border-radius:7px;border:1px solid rgba(15,23,42,.18)}}.mundial-flag-empty{{background:repeating-linear-gradient(45deg,#e2e8f0,#e2e8f0 6px,#f8fafc 6px,#f8fafc 12px)}}.mundial-team-name{{font-size:1.05rem;font-weight:900;color:#0f172a}}.mundial-country{{color:#64748b;font-weight:800;margin-top:2px}}.mundial-team-score{{margin-top:8px;height:9px;background:#e2e8f0;border-radius:999px;overflow:hidden}}.mundial-team-score span{{display:block;height:100%;border-radius:999px;animation:mundialBar .8s ease-out both}}.mundial-team-pct{{font-size:1.35rem;font-weight:950;margin-top:6px}}.mundial-player{{display:grid;grid-template-columns:1fr auto;gap:10px;align-items:center;padding:10px 14px;border-top:1px solid #eef2f7}}.mundial-player-name{{font-weight:850;color:#1e293b}}.mundial-player-sub{{color:#64748b;font-size:.82rem;font-weight:700}}.mundial-chip{{font-weight:950;border-radius:999px;color:#fff;padding:5px 9px;min-width:74px;text-align:center}}.mundial-paises-grid{{display:grid;grid-template-columns:repeat(6,minmax(190px,1fr));gap:12px}}.mundial-paises-grid-bpo{{grid-template-columns:repeat(42,minmax(0,1fr))}}.mundial-paises-grid-bpo .mundial-pais-card{{grid-column:span 7}}.mundial-paises-grid-bpo .mundial-pais-card:nth-child(n+7){{grid-column:span 6}}.mundial-pais-card{{background:#fff;border:1px solid #e2e8f0;border-radius:8px;overflow:hidden;box-shadow:0 8px 20px rgba(15,23,42,.07)}}.mundial-pais-head{{display:flex;align-items:center;gap:12px;padding:12px;background:#f8fafc;border-bottom:1px solid #e2e8f0}}.mundial-pais-title{{font-weight:950;color:#0f172a}}.mundial-pais-count{{color:#64748b;font-weight:800;font-size:.85rem}}.mundial-pais-list{{padding:8px 12px 12px;display:grid;gap:8px}}.mundial-pais-row{{display:block;font-weight:800;color:#334155;line-height:1.25}}.btn-mundial{{background:linear-gradient(135deg,#0f766e,#15803d)!important}}.btn-mundial-sec{{background:linear-gradient(135deg,#b91c1c,#dc2626)!important}}@keyframes mundialIn{{from{{opacity:0;transform:scale(.97) translateY(12px)}}to{{opacity:1;transform:scale(1) translateY(0)}}}}@keyframes mundialRise{{from{{opacity:0;transform:translateY(12px)}}to{{opacity:1;transform:translateY(0)}}}}@keyframes mundialBar{{from{{width:0}}}}@media(max-width:1200px){{.mundial-paises-grid,.mundial-paises-grid-bpo{{grid-template-columns:repeat(3,minmax(220px,1fr))}}.mundial-paises-grid-bpo .mundial-pais-card{{grid-column:auto}}}}@media(max-width:900px){{.mundial-card{{inset:12px}}.mundial-grid-3{{grid-template-columns:1fr}}.mundial-paises-grid,.mundial-paises-grid-bpo{{grid-template-columns:1fr}}.mundial-paises-grid-bpo .mundial-pais-card{{grid-column:auto}}.mundial-toolbar{{align-items:flex-start;flex-direction:column}}}}`; document.head.appendChild(style); }}
    function abrirModalMundialEquipos() {{ _ensureMundialStyles(); if (!datosMundial || !Array.isArray(datosMundial.equipos) || datosMundial.equipos.length === 0) {{ alert('No hay datos en la hoja ME.'); return; }} mundialPaginaEquipos = Math.max(0, Math.min(mundialPaginaEquipos, Math.ceil(datosMundial.equipos.length / 3) - 1)); let modal = document.getElementById('modalMundialEquipos'); if (!modal) {{ modal = document.createElement('div'); modal.id = 'modalMundialEquipos'; modal.className = 'mundial-modal'; modal.innerHTML = `<div class="mundial-backdrop" onclick="cerrarModalMundial('modalMundialEquipos')"></div><div class="mundial-card" role="dialog" aria-modal="true"><div class="mundial-header"><div><h2>PROGRESO DE LA FASE DE GRUPOS</h2></div><button class="mundial-close" onclick="cerrarModalMundial('modalMundialEquipos')">x</button></div><div class="mundial-body"><div class="mundial-toolbar"><div class="mundial-page" id="mundialEquiposPagina"></div><div class="mundial-nav"><button class="mundial-nav-btn" onclick="cambiarPaginaMundial(-1)">?</button><button class="mundial-nav-btn" onclick="cambiarPaginaMundial(1)">?</button></div></div><div id="mundialEquiposGrid" class="mundial-grid-3"></div></div></div>`; document.body.appendChild(modal); }} document.body.style.overflow = 'hidden'; modal.classList.add('is-open'); renderMundialEquipos(); }}
    function renderMundialEquipos() {{ const equipos = [...(datosMundial.equipos || [])].sort((a,b) => String(a.equipo).localeCompare(String(b.equipo), 'es', {{numeric:true}})); const totalPaginas = Math.max(1, Math.ceil(equipos.length / 3)); mundialPaginaEquipos = Math.max(0, Math.min(mundialPaginaEquipos, totalPaginas - 1)); const pageItems = equipos.slice(mundialPaginaEquipos * 3, mundialPaginaEquipos * 3 + 3); const page = document.getElementById('mundialEquiposPagina'); const grid = document.getElementById('mundialEquiposGrid'); if (page) page.textContent = `Grupo ${{mundialPaginaEquipos + 1}} de ${{totalPaginas}}`; if (!grid) return; grid.innerHTML = pageItems.map((eq, idx) => {{ const color = _colorMundial(eq.alcance); const asesores = [...(eq.asesores || [])].sort((a,b) => Number(b.alcance || 0) - Number(a.alcance || 0)); const paises = (eq.paises || []).join(', '); return `<article class="mundial-team" style="animation-delay:${{idx * 60}}ms"><div class="mundial-team-top">${{_banderaHTML(eq.bandera, paises)}}<div><div class="mundial-team-name">${{eq.equipo}}</div><div class="mundial-country">${{paises || 'Pais no asignado'}}</div><div class="mundial-team-pct" style="color:${{color}}">${{_fmtPctMundial(eq.alcance)}}</div><div class="mundial-team-score"><span style="background:${{color}};width:${{Math.min(100, Number(eq.alcance || 0))}}%"></span></div></div></div><div>${{asesores.map(a => `<div class="mundial-player"><div><div class="mundial-player-name">${{a.asesor}}</div><div class="mundial-player-sub">${{a.pais}}</div></div><div class="mundial-chip" style="background:${{_colorMundial(a.alcance)}}">${{_fmtPctMundial(a.alcance)}}</div></div>`).join('')}}</div></article>`; }}).join(''); }}
    function cambiarPaginaMundial(delta) {{ const total = Math.max(1, Math.ceil(((datosMundial.equipos || []).length) / 3)); mundialPaginaEquipos = (mundialPaginaEquipos + delta + total) % total; renderMundialEquipos(); }}
    function _paisesMundialPorSede(sede) {{ const objetivo = _normMundial(sede || 'SURCO'); const porSede = datosMundial.paises_por_sede || {{}}; const key = Object.keys(porSede).find(k => _normMundial(k) === objetivo); if (key) return porSede[key] || []; return [...(datosMundial.paises || [])].map(pais => {{ const asesores = [...(pais.asesores || [])].filter(a => _normMundial(a.sede || 'SURCO') === objetivo); return {{...pais, asesores}}; }}).filter(pais => (pais.asesores || []).length > 0); }}
    function abrirModalMundialPaises(sede = 'SURCO') {{ _ensureMundialStyles(); const sedeLabel = String(sede || 'SURCO').trim().toUpperCase(); const paises = _paisesMundialPorSede(sedeLabel); if (!datosMundial || !Array.isArray(paises) || paises.length === 0) {{ alert(`No hay datos por pais para la sede ${{sedeLabel}} en la hoja ME.`); return; }} const suffix = sedeLabel.replace(/[^A-Z0-9]/g, ''); const modalId = `modalMundialPaises${{suffix}}`; const gridId = `mundialPaisesGrid${{suffix}}`; let modal = document.getElementById(modalId); if (!modal) {{ modal = document.createElement('div'); modal.id = modalId; modal.className = 'mundial-modal'; modal.innerHTML = `<div class="mundial-backdrop" onclick="cerrarModalMundial('${{modalId}}')"></div><div class="mundial-card" role="dialog" aria-modal="true"><div class="mundial-header"><div><h2>DISTRIBUCI&Oacute;N DE EQUIPOS - ${{sedeLabel}}</h2></div><button class="mundial-close" onclick="cerrarModalMundial('${{modalId}}')">x</button></div><div class="mundial-body"><div id="${{gridId}}" class="mundial-paises-grid${{sedeLabel === 'BPO' ? ' mundial-paises-grid-bpo' : ''}}"></div></div></div>`; document.body.appendChild(modal); }} const grid = modal.querySelector(`#${{gridId}}`); grid.innerHTML = [...paises].map(pais => {{ const asesores = [...(pais.asesores || [])].sort((a,b) => String(a.equipo).localeCompare(String(b.equipo), 'es', {{numeric:true}}) || String(a.asesor).localeCompare(String(b.asesor), 'es', {{numeric:true}})); return `<section class="mundial-pais-card"><div class="mundial-pais-head">${{_banderaHTML(pais.bandera, pais.pais, 'mundial-flag-large')}}<div><div class="mundial-pais-title">${{pais.pais}}</div><div class="mundial-pais-count">${{asesores.length}} asesores</div></div></div><div class="mundial-pais-list">${{asesores.map(a => `<div class="mundial-pais-row"><span>${{a.asesor}}<br><small style="color:#64748b;font-weight:800;">${{a.equipo}}</small></span></div>`).join('')}}</div></section>`; }}).join(''); document.body.style.overflow = 'hidden'; modal.classList.add('is-open'); }}
    function cerrarModalMundial(id) {{ const modal = document.getElementById(id); if (modal) modal.classList.remove('is-open'); document.body.style.overflow = 'auto'; }}
    function _getMundialAsesor(nombre) {{ const objetivo = _normMundial(nombre); return (datosMundial.asesores || []).find(a => _normMundial(a.asesor) === objetivo) || null; }}

    // ========== FUNCIONES TOP 10 ==========
    function mostrarTop10() {{
        const mesSeleccionado = document.getElementById('selectorMes').value;
        const añoSeleccionado = document.getElementById('selectorAño').value;
        const periodoCompleto = `${{mesSeleccionado}}_${{añoSeleccionado}}`;
        
        // Mostrar el modal directamente
        mostrarModalTop10(periodoCompleto);
    }}
    
    function mostrarModalTop10(periodoCompleto) {{
        // Cerrar modal existente si hay uno
        const modalExistente = document.getElementById('modalTop10');
        if (modalExistente) {{
            cerrarModalTop10();
        }}
        
        // Crear el modal
        const modal = document.createElement('div');
        modal.id = 'modalTop10';
        modal.className = 'modal-top10';
        modal.innerHTML = `
            <div class="modal-overlay" onclick="cerrarModalTop10()"></div>
            <div class="modal-content">
                <canvas id="graficaTop10Modal" width="1400" height="850"></canvas>
            </div>
        `;
        
        document.body.appendChild(modal);
        document.body.style.overflow = 'hidden';  // Bloquear scroll
        document.addEventListener('keydown', handleModalEscape);  // Listener ESC
        
        // Agregar estilos si no existen
        if (!document.getElementById('modal-styles-top10')) {{
            agregarEstilosModalMinimalista();
        }}
        
        // Bloquear scroll del body
        document.body.style.overflow = 'hidden';
        
        // Agregar event listener para ESC
        document.addEventListener('keydown', handleModalEscape);
        
        // Mostrar modal
        modal.style.display = 'block';
        
        // Generar la gráfica
        setTimeout(() => {{
            generarGraficaTop10Modal(periodoCompleto);
        }}, 50);
    }}
    
    function agregarEstilosModalMinimalista() {{
        const style = document.createElement('style');
        style.textContent = `
            /* MODAL MINIMALISTA */
            .modal-top10 {{
                display: none;
                position: fixed;
                top: 0;
                left: 0;
                width: 100%;
                height: 100%;
                z-index: 9999;
            }}
            
            .modal-overlay {{
                position: absolute;
                top: 0;
                left: 0;
                width: 100%;
                height: 100%;
                background: rgba(0, 0, 0, 0.92);
                backdrop-filter: blur(5px);
                -webkit-backdrop-filter: blur(5px);
                cursor: pointer;
                animation: fadeIn 0.2s ease-out;
            }}
            
            .modal-content {{
                position: absolute;
                top: 50%;
                left: 50%;
                transform: translate(-50%, -50%);
                background: transparent;
                width: 95%;
                max-width: 1400px;
                max-height: 95vh;
                animation: slideIn 0.3s ease-out;
                display: flex;
                justify-content: center;
                align-items: center;
                padding: 0;
                border: none;
                box-shadow: none;
            }}
            
            #graficaTop10Modal {{
                max-width: 100%;
                max-height: 95vh;
                height: auto !important;
                background: white;
                border-radius: 12px;
                padding: 40px 30px 30px 30px; /* Más padding superior para título */
                box-shadow: 0 20px 60px rgba(0, 0, 0, 0.4);
                cursor: default;
            }}
            
            /* ANIMACIONES SUAVES */
            @keyframes fadeIn {{
                from {{ opacity: 0; }}
                to {{ opacity: 1; }}
            }}
            
            @keyframes slideIn {{
                from {{
                    opacity: 0;
                    transform: translate(-50%, -55%) scale(0.95);
                }}
                to {{
                    opacity: 1;
                    transform: translate(-50%, -50%) scale(1);
                }}
            }}
            
            @keyframes fadeOut {{
                from {{ opacity: 1; }}
                to {{ opacity: 0; }}
            }}
            
            @keyframes slideOut {{
                from {{
                    opacity: 1;
                    transform: translate(-50%, -50%) scale(1);
                }}
                to {{
                    opacity: 0;
                    transform: translate(-50%, -45%) scale(0.95);
                }}
            }}
            
            /* EFECTO DE CIERRE */
            .modal-top10.closing .modal-overlay {{
                animation: fadeOut 0.2s ease-in forwards;
            }}
            
            .modal-top10.closing .modal-content {{
                animation: slideOut 0.2s ease-in forwards;
            }}
            
            /* RESPONSIVE */
            @media (max-width: 1600px) {{
                #graficaTop10Modal {{
                    width: 1300px !important;
                    height: 800px !important;
                    padding: 35px 25px 25px 25px;
                }}
            }}
            
            @media (max-width: 1400px) {{
                #graficaTop10Modal {{
                    width: 1200px !important;
                    height: 750px !important;
                    padding: 30px 20px 20px 20px;
                }}
            }}
            
            @media (max-width: 1200px) {{
                #graficaTop10Modal {{
                    width: 1100px !important;
                    height: 700px !important;
                    padding: 25px 15px 15px 15px;
                }}
            }}
            
            @media (max-width: 992px) {{
                .modal-content {{
                    width: 98%;
                }}
                
                #graficaTop10Modal {{
                    width: 1000px !important;
                    height: 650px !important;
                    padding: 20px 15px 15px 15px;
                }}
            }}
            
            @media (max-width: 768px) {{
                #graficaTop10Modal {{
                    width: 95% !important;
                    height: 550px !important;
                    padding: 15px 10px 10px 10px;
                    border-radius: 8px;
                }}
            }}
            
            @media (max-width: 576px) {{
                #graficaTop10Modal {{
                    width: 98% !important;
                    height: 500px !important;
                    padding: 12px 8px 8px 8px;
                    border-radius: 6px;
                }}
            }}
            
            /* MEJORAR TEXTO EN GRÁFICA */
            .chartjs-render-monitor text {{
                font-family: 'Segoe UI', -apple-system, BlinkMacSystemFont, sans-serif !important;
            }}
        `;
        document.head.appendChild(style);
    }}
    
    function cerrarModalTop10() {{
        const modals = ['modalTop10', 'modalTop10Paises', 'modalTop10MundialAsesor']
            .map(id => document.getElementById(id))
            .filter(Boolean);
        
        if (modals.length === 0) return;
        
        // Destruir gráfica si existe
        if (chartTop10Modal) {{
            try {{
                chartTop10Modal.destroy();
                chartTop10Modal = null;
            }} catch (e) {{
                // Ignorar error
            }}
        }}

        document.body.style.overflow = 'auto';

        document.removeEventListener('keydown', handleModalEscape);

        modals.forEach(modal => {{
            if (modal.parentNode) {{
                modal.parentNode.removeChild(modal);
            }}
        }});

        document.body.focus();
    }}

    function handleModalEscape(event) {{
        if (event.key === 'Escape' || event.keyCode === 27) {{
            cerrarModalTop10();
        }}
    }}
    
    function generarGraficaTop10Modal(periodoCompleto) {{
        if (!datosMeses[periodoCompleto]) {{
            alert('No hay datos disponibles para este periodo.');
            cerrarModalTop10();
            return;
        }}
        
        const asesores = filtrarAsesoresPorCanal(datosMeses[periodoCompleto] || []);
        const asesoresOrdenados = [...asesores].sort((a, b) => b.porcentaje - a.porcentaje);
        const top10 = asesoresOrdenados.slice(0, 10);
        
        if (top10.length === 0) {{
            alert('No hay asesores para mostrar en el Top 10.');
            cerrarModalTop10();
            return;
        }}
        
        // Nombres más cortos
        const nombres = top10.map((asesor, index) => {{
            let nombre = asesor.nombre;
            if (nombre.length > 30) {{
                nombre = nombre.substring(0, 28) + '...';
            }}
            return nombre;
        }});
        
        const porcentajes = top10.map(asesor => asesor.porcentaje);
        const clasificaciones = top10.map(asesor => asesor.clasificacion);
        
        const coloresPorClasificacion = {{
            '>100%': '#27ae60',
            '>70%': '#f39c12',
            '>40%': '#e67e22',
            '>0%': '#e74c3c'
        }};
        
        const coloresBarras = clasificaciones.map(clas => coloresPorClasificacion[clas] || '#95a5a6');
        const coloresNombres = top10.map((asesor, index) => {{
            if (index === 0) return '#B8860B';
            if (index === 1) return '#696969';
            if (index === 2) return '#8B4513';
            return '#2c3e50';
        }});
        
        const mundialTop10 = top10.map(asesor => _getMundialAsesor(asesor.nombre));
        const flagImagesTop10 = mundialTop10.map(item => {{
            if (!item || !item.bandera) return null;
            const img = new Image();
            img.onload = () => {{ if (chartTop10Modal) chartTop10Modal.draw(); }};
            img.src = item.bandera;
            return img;
        }});

        const canvas = document.getElementById('graficaTop10Modal');
        const ctx = canvas.getContext('2d');
        
        // Destruir gráfica anterior
        if (chartTop10Modal) {{
            chartTop10Modal.destroy();
            chartTop10Modal = null;
        }}
        
        // Plugin para etiquetas de porcentaje (LOCAL)
        const dataLabelsPlugin = {{
            id: 'dataLabelsModal',
            afterDatasetsDraw(chart, args, options) {{
                const {{ctx}} = chart;
                const meta = chart.getDatasetMeta(0);
                
                meta.data.forEach((bar, index) => {{
                    const value = chart.data.datasets[0].data[index];
                    if (value === null || value === undefined) return;
                    
                    ctx.save();
                    
                    const x = bar.x - 25;
                    const y = bar.y;
                    const formattedValue = value.toFixed(2) + '%';
                    
                    ctx.font = 'bold 22px Segoe UI';
                    ctx.textAlign = 'right';
                    ctx.textBaseline = 'middle';
                    ctx.fillStyle = '#ffffff';
                    
                    ctx.shadowColor = 'rgba(0, 0, 0, 0.8)';
                    ctx.shadowBlur = 5;
                    ctx.shadowOffsetX = 2;
                    ctx.shadowOffsetY = 2;
                    
                    ctx.fillText(formattedValue, x, y);
                    
                    ctx.restore();
                }});
            }}
        }};
        
        // Plugin para números de posición (LOCAL)
        const positionNumbersPlugin = {{
            id: 'positionNumbersModal',
            afterDatasetsDraw(chart, args, options) {{
                const {{ctx}} = chart;
                const meta = chart.getDatasetMeta(0);
                
                meta.data.forEach((bar, index) => {{
                    ctx.save();
                    
                    const x = 15;
                    const y = bar.y;
                    
                    let textColor;
                    if (index === 0) textColor = '#B8860B';
                    else if (index === 1) textColor = '#696969';
                    else if (index === 2) textColor = '#8B4513';
                    else textColor = '#2c3e50';
                    
                    ctx.font = `bold 20px Segoe UI`;
                    ctx.textAlign = 'left';
                    ctx.textBaseline = 'middle';
                    ctx.fillStyle = textColor;
                    
                    let positionText = (index + 1) + '.';
                    if (index === 0) positionText = '🥇 ' + positionText;
                    else if (index === 1) positionText = '🥈 ' + positionText;
                    else if (index === 2) positionText = '🥉 ' + positionText;
                    
                    ctx.fillText(positionText, x, y);
                    
                    ctx.restore();
                }});
            }}
        }};
        
        // Crear la gráfica con los plugins locales
        const flagsPluginModal = {{
            id: 'flagsPluginModal',
            afterDatasetsDraw(chart) {{
                const {{ctx, chartArea}} = chart;
                const meta = chart.getDatasetMeta(0);
                meta.data.forEach((bar, index) => {{
                    const img = flagImagesTop10[index];
                    if (!img || !img.complete || !img.naturalWidth) return;
                    const w = 54;
                    const h = 36;
                    const x = Math.min(bar.x + 14, chartArea.right - w - 4);
                    const y = bar.y - h / 2;
                    ctx.save();
                    ctx.shadowColor = 'rgba(0,0,0,.25)';
                    ctx.shadowBlur = 8;
                    ctx.drawImage(img, x, y, w, h);
                    ctx.restore();
                }});
            }}
        }};

        chartTop10Modal = new Chart(ctx, {{
            type: 'bar',
            data: {{
                labels: nombres,
                datasets: [{{
                    label: 'Porcentaje de Alcance',
                    data: porcentajes,
                    backgroundColor: coloresBarras,
                    borderColor: coloresBarras.map(color => color + 'CC'),
                    borderWidth: 3,
                    borderRadius: 10,
                    borderSkipped: false
                }}]
            }},
            options: {{
                indexAxis: 'y',
                responsive: false,
                maintainAspectRatio: false,
                
                plugins: {{
                    title: {{
                        display: true,
                        text: `🏆 TOP 10 ASESORES - ${{periodoCompleto.replace('_', ' ')}}`,
                        font: {{
                            size: 32,
                            weight: 'bold',
                            family: 'Segoe UI'
                        }},
                        color: '#2c3e50',
                        padding: {{
                            top: 10,
                            bottom: 30
                        }}
                    }},
                    
                    tooltip: {{
                        backgroundColor: 'rgba(0, 0, 0, 0.9)',
                        titleFont: {{
                            size: 20,
                            weight: 'bold',
                            family: 'Segoe UI'
                        }},
                        bodyFont: {{
                            size: 18,
                            family: 'Segoe UI'
                        }},
                        padding: 20,
                        cornerRadius: 10,
                        displayColors: false,
                        callbacks: {{
                            title: function(context) {{
                                const index = context[0].dataIndex;
                                const asesor = top10[index];
                                const icon = index === 0 ? '🥇 ' : index === 1 ? '🥈 ' : index === 2 ? '🥉 ' : '';
                                return icon + asesor.nombre;
                            }},
                            label: function(context) {{
                              const index = context.dataIndex;
                              const asesor = top10[index] || {{}};
                            
                              const pct = Number(context.parsed.x);
                              const porcentajeTxt = Number.isFinite(pct) ? pct.toFixed(2) : '0.00';
                            
                              const fmtMoneda = (v) => {{
                                const n = Number(v);
                                const val = Number.isFinite(n) ? n : 0;
                                return 'S/ ' + val.toLocaleString('es-PE', {{
                                  minimumFractionDigits: 2,
                                  maximumFractionDigits: 2
                                }});
                              }};
                            
                              return [
                                `Porcentaje: ${{porcentajeTxt}}%`,
                                `Meta: ${{fmtMoneda(asesor.meta)}}`,
                                `Recupero: ${{fmtMoneda(asesor.recupero)}}`,
                                `Cartera: ${{asesor.cartera || 'No definida'}}`,
                                `Supervisor: ${{asesor.supervisor || 'Sin Supervisor'}}`
                              ];
                            }}
                        }}
                    }},
                    legend: {{ display: false }}
                }},
                
                scales: {{
                    x: {{
                        beginAtZero: true,
                        grid: {{
                            display: false,
                            drawBorder: false
                        }},
                        ticks: {{
                            callback: function(value) {{
                                return value + '%';
                            }},
                            font: {{
                                size: 18,
                                family: 'Segoe UI',
                                weight: 'bold'
                            }},
                            color: '#34495e'
                        }},
                        // LIMITAR EJE X
                        min: 0,
                        max: Math.ceil((Math.max(...porcentajes) + 18) / 10) * 10,
                        suggestedMax: 118
                    }},
                    y: {{
                        afterFit: function(scale) {{
                            scale.width = 420;
                            scale.left = 60;
                        }},
                        grid: {{
                            display: false,
                            drawBorder: false
                        }},
                        ticks: {{
                            color: function(context) {{
                                return coloresNombres[context.index];
                            }},
                            font: function(context) {{
                                const index = context.index;
                                return {{ 
                                    weight: index <= 2 ? 'bold' : '600',
                                    size: index <= 2 ? 30 : 27,
                                    family: 'Segoe UI'
                                }};
                            }},
                            autoSkip: false,
                            maxRotation: 0,
                            padding: 15
                        }}
                    }}
                }},
                
                animation: {{
                    duration: 1200,
                    easing: 'easeOutQuart'
                }}
            }},
            // Pasar plugins directamente a esta instancia
            plugins: [dataLabelsPlugin, positionNumbersPlugin]
        }});
        
        chartTop10Modal.update();
    }}

    // ========== FUNCIONES TOP 10 MUNDIAL ==========
    function _clasificacionMundialTop10(valor) {{
        const n = Number(valor) || 0;
        if (n > 100) return '>100%';
        if (n > 70) return '>70%';
        if (n > 40) return '>40%';
        return '>0%';
    }}

    function _banderaPaisMundial(pais) {{
        const key = String(pais || '').trim().toUpperCase();
        return (datosMundial.banderas || {{}})[key] || '';
    }}

    function _top10PaisesMundialData() {{
        const paises = [];
        const vistos = new Set();
        (datosMundial.equipos || []).forEach(eq => {{
            const alcance = Number(eq.alcance || 0);
            const listaPaises = Array.isArray(eq.paises) && eq.paises.length ? eq.paises : ['Sin pais'];
            listaPaises.forEach(pais => {{
                const key = _normMundial(pais);
                if (vistos.has(key)) return;
                vistos.add(key);
                paises.push({{
                    nombre: pais,
                    porcentaje: alcance,
                    clasificacion: _clasificacionMundialTop10(alcance),
                    bandera: eq.bandera || _banderaPaisMundial(pais),
                    detalle: eq.equipo || 'Equipo no definido',
                    tooltip: [`Equipo: ${{eq.equipo || 'No definido'}}`]
                }});
            }});
        }});
        return paises.sort((a, b) => Number(b.porcentaje || 0) - Number(a.porcentaje || 0)).slice(0, 10);
    }}

    function _top10MundialAsesorData() {{
        return [...(datosMundial.asesores || [])]
            .map(a => {{
                const alcance = Number(a.alcance || 0);
                return {{
                    nombre: a.asesor || 'Sin asesor',
                    porcentaje: alcance,
                    clasificacion: _clasificacionMundialTop10(alcance),
                    bandera: a.bandera || _banderaPaisMundial(a.pais),
                    detalle: a.pais || 'Pais no asignado',
                    tooltip: [
                        `Pais: ${{a.pais || 'No asignado'}}`,
                        `Equipo: ${{a.equipo || 'No definido'}}`,
                        `Sede: ${{a.sede || 'No definida'}}`
                    ]
                }};
            }})
            .sort((a, b) => Number(b.porcentaje || 0) - Number(a.porcentaje || 0))
            .slice(0, 10);
    }}

    function mostrarTop10Paises() {{
        mostrarModalTop10Mundial('modalTop10Paises', '🏆 TOP10-PAISES', _top10PaisesMundialData());
    }}

    function mostrarTop10MundialAsesor() {{
        mostrarModalTop10Mundial('modalTop10MundialAsesor', '🌎 TOP10-MUNDIAL-ASESOR', _top10MundialAsesorData());
    }}

    function mostrarModalTop10Mundial(modalId, titulo, top10) {{
        cerrarModalTop10();
        if (!top10 || top10.length === 0) {{
            alert('No hay datos disponibles en la hoja ME para este ranking.');
            return;
        }}

        const modal = document.createElement('div');
        modal.id = modalId;
        modal.className = 'modal-top10';
        modal.innerHTML = `
            <div class="modal-overlay" onclick="cerrarModalTop10()"></div>
            <div class="modal-content">
                <canvas id="graficaTop10Modal" width="1400" height="850"></canvas>
            </div>
        `;

        document.body.appendChild(modal);
        document.body.style.overflow = 'hidden';
        document.addEventListener('keydown', handleModalEscape);
        if (!document.getElementById('modal-styles-top10')) {{
            agregarEstilosModalMinimalista();
        }}
        modal.style.display = 'block';
        setTimeout(() => generarGraficaTop10Mundial(titulo, top10), 50);
    }}

    function generarGraficaTop10Mundial(titulo, top10) {{
        const nombres = top10.map(item => {{
            let nombre = String(item.nombre || 'Sin nombre');
            if (nombre.length > 30) nombre = nombre.substring(0, 28) + '...';
            return nombre;
        }});
        const porcentajes = top10.map(item => Number(item.porcentaje || 0));
        const clasificaciones = top10.map(item => item.clasificacion || _clasificacionMundialTop10(item.porcentaje));

        const coloresPorClasificacion = {{
            '>100%': '#27ae60',
            '>70%': '#f39c12',
            '>40%': '#e67e22',
            '>0%': '#e74c3c'
        }};
        const coloresBarras = clasificaciones.map(clas => coloresPorClasificacion[clas] || '#95a5a6');
        const barWidthsTop10 = top10.map((item, index) => Math.max(68, Math.round(132 * Math.pow(0.93, index))));
        const flagSizesTop10 = top10.map((item, index) => {{
            const w = Math.max(54, Math.round(96 * Math.pow(0.94, index)));
            return {{ w, h: Math.round(w * 0.66) }};
        }});
        const coloresNombres = top10.map((item, index) => {{
            if (index === 0) return '#B8860B';
            if (index === 1) return '#696969';
            if (index === 2) return '#8B4513';
            return '#2c3e50';
        }});
        const flagImagesTop10 = top10.map(item => {{
            if (!item || !item.bandera) return null;
            const img = new Image();
            img.onload = () => {{ if (chartTop10Modal) chartTop10Modal.draw(); }};
            img.src = item.bandera;
            return img;
        }});

        const canvas = document.getElementById('graficaTop10Modal');
        const ctx = canvas.getContext('2d');
        if (chartTop10Modal) {{
            chartTop10Modal.destroy();
            chartTop10Modal = null;
        }}

        const dataLabelsPlugin = {{
            id: 'dataLabelsModalMundial',
            afterDatasetsDraw(chart) {{
                const {{ctx}} = chart;
                const meta = chart.getDatasetMeta(0);
                meta.data.forEach((bar, index) => {{
                    const value = chart.data.datasets[0].data[index];
                    if (value === null || value === undefined) return;
                    ctx.save();
                    ctx.font = 'bold 22px Segoe UI';
                    ctx.fillStyle = esTop10Paises ? '#2c3e50' : '#ffffff';
                    ctx.shadowColor = esTop10Paises ? 'rgba(255, 255, 255, 0.85)' : 'rgba(0, 0, 0, 0.8)';
                    ctx.shadowBlur = esTop10Paises ? 4 : 5;
                    ctx.shadowOffsetX = esTop10Paises ? 0 : 2;
                    ctx.shadowOffsetY = esTop10Paises ? 1 : 2;
                    if (esTop10Paises) {{
                        ctx.textAlign = 'center';
                        ctx.textBaseline = 'top';
                        ctx.fillStyle = '#ffffff';
                        ctx.shadowColor = 'rgba(0, 0, 0, 0.85)';
                        ctx.shadowBlur = 6;
                        ctx.shadowOffsetX = 1;
                        ctx.shadowOffsetY = 2;
                        ctx.fillText(Number(value || 0).toFixed(2) + '%', bar.x, bar.y + 10);
                    }} else {{
                        ctx.textAlign = 'right';
                        ctx.textBaseline = 'middle';
                        ctx.fillText(Number(value || 0).toFixed(2) + '%', Math.max(bar.x - 25, 125), bar.y);
                    }}
                    ctx.restore();
                }});
            }}
        }};

        const positionNumbersPlugin = {{
            id: 'positionNumbersModalMundial',
            afterDatasetsDraw(chart) {{
                const {{ctx}} = chart;
                const meta = chart.getDatasetMeta(0);
                meta.data.forEach((bar, index) => {{
                    if (esTop10Paises) return;
                    ctx.save();
                    const x = 15;
                    const y = bar.y;
                    let textColor;
                    if (index === 0) textColor = '#B8860B';
                    else if (index === 1) textColor = '#696969';
                    else if (index === 2) textColor = '#8B4513';
                    else textColor = '#2c3e50';
                    ctx.font = 'bold 20px Segoe UI';
                    ctx.textAlign = esTop10Paises ? 'center' : 'left';
                    ctx.textBaseline = 'middle';
                    ctx.fillStyle = textColor;
                    let positionText = (index + 1) + '.';
                    if (index === 0) positionText = '🥇 ' + positionText;
                    else if (index === 1) positionText = '🥈 ' + positionText;
                    else if (index === 2) positionText = '🥉 ' + positionText;
                    ctx.fillText(positionText, x, y);
                    ctx.restore();
                }});
            }}
        }};

        const flagsPluginModal = {{
            id: 'flagsPluginModalMundial',
            afterDatasetsDraw(chart) {{
                const {{ctx, chartArea}} = chart;
                const meta = chart.getDatasetMeta(0);
                meta.data.forEach((bar, index) => {{
                    const img = flagImagesTop10[index];
                    if (!img || !img.complete || !img.naturalWidth) return;
                    const size = esTop10Paises ? flagSizesTop10[index] : {{ w: 54, h: 36 }};
                    const w = size.w;
                    const h = size.h;
                    const x = esTop10Paises ? Math.min(Math.max(bar.x - w / 2, chartArea.left + 4), chartArea.right - w - 4) : Math.min(bar.x + 14, chartArea.right - w - 4);
                    const y = esTop10Paises ? Math.max(chartArea.top + 4, bar.y - h - 14) : bar.y - h / 2;
                    ctx.save();
                    ctx.shadowColor = 'rgba(0,0,0,.25)';
                    ctx.shadowBlur = 8;
                    ctx.drawImage(img, x, y, w, h);
                    ctx.restore();
                }});
            }}
        }};

        const variableBarsPlugin = {{
            id: 'variableBarsTop10Paises',
            beforeDatasetsDraw(chart) {{
                if (!esTop10Paises) return;
                const {{ctx, chartArea}} = chart;
                const meta = chart.getDatasetMeta(0);
                const yScale = chart.scales.y;
                meta.data.forEach((bar, index) => {{
                    const value = Number(chart.data.datasets[0].data[index] || 0);
                    if (!Number.isFinite(value)) return;
                    const baseY = yScale.getPixelForValue(0);
                    const topY = yScale.getPixelForValue(value);
                    const width = barWidthsTop10[index] || 30;
                    const left = bar.x - width / 2;
                    const height = Math.max(0, baseY - topY);
                    const radius = Math.min(14, width / 3, height / 2);
                    const color = coloresBarras[index] || '#95a5a6';
                    ctx.save();
                    ctx.fillStyle = color;
                    ctx.strokeStyle = color + 'CC';
                    ctx.lineWidth = 3;
                    ctx.shadowColor = 'rgba(0, 0, 0, 0.16)';
                    ctx.shadowBlur = 10;
                    ctx.beginPath();
                    ctx.moveTo(left, baseY);
                    ctx.lineTo(left, topY + radius);
                    ctx.quadraticCurveTo(left, topY, left + radius, topY);
                    ctx.lineTo(left + width - radius, topY);
                    ctx.quadraticCurveTo(left + width, topY, left + width, topY + radius);
                    ctx.lineTo(left + width, baseY);
                    ctx.closePath();
                    ctx.fill();
                    ctx.shadowBlur = 0;
                    ctx.stroke();
                    ctx.restore();
                }});
            }}
        }};

        const valoresValidosTop10 = porcentajes.filter(value => Number.isFinite(value));
        const maxPct = Math.max(...valoresValidosTop10, 0);
        const esTop10Paises = String(titulo || '').includes('TOP10-PAISES');
        const xAxisMax = esTop10Paises
            ? Math.max(5, Math.ceil(((maxPct * 1.04) + 1.5) / 2.5) * 2.5)
            : Math.max(10, Math.ceil((maxPct + 18) / 10) * 10);
        const xAxisSuggestedMax = xAxisMax;
        const chartIndexAxis = esTop10Paises ? 'x' : 'y';
        const chartScales = esTop10Paises ? {{
            x: {{
                grid: {{ display: false, drawBorder: false }},
                ticks: {{
                    callback: function(value, index) {{
                        const label = this.getLabelForValue(value);
                        const puesto = (index + 1) + '.';
                        return [puesto, label];
                    }},
                    color: function(context) {{ return coloresNombres[context.index]; }},
                    font: function(context) {{
                        const index = context.index;
                        return {{ weight: index <= 2 ? 'bold' : '700', size: index <= 2 ? 18 : 16, family: 'Segoe UI' }};
                    }},
                    autoSkip: false,
                    maxRotation: 0,
                    minRotation: 0,
                    padding: 14
                }}
            }},
            y: {{
                beginAtZero: true,
                grid: {{ display: true, drawBorder: false, color: 'rgba(44, 62, 80, 0.08)' }},
                ticks: {{
                    callback: function(value) {{ return value + '%'; }},
                    font: {{ size: 16, family: 'Segoe UI', weight: 'bold' }},
                    color: '#34495e'
                }},
                min: 0,
                max: xAxisMax,
                suggestedMax: xAxisSuggestedMax
            }}
        }} : {{
            x: {{
                beginAtZero: true,
                grid: {{ display: false, drawBorder: false }},
                ticks: {{
                    callback: function(value) {{ return value + '%'; }},
                    font: {{ size: 18, family: 'Segoe UI', weight: 'bold' }},
                    color: '#34495e'
                }},
                min: 0,
                max: xAxisMax,
                suggestedMax: xAxisSuggestedMax
            }},
            y: {{
                afterFit: function(scale) {{ scale.width = 420; scale.left = 60; }},
                grid: {{ display: false, drawBorder: false }},
                ticks: {{
                    color: function(context) {{ return coloresNombres[context.index]; }},
                    font: function(context) {{
                        const index = context.index;
                        return {{ weight: index <= 2 ? 'bold' : '600', size: index <= 2 ? 30 : 27, family: 'Segoe UI' }};
                    }},
                    autoSkip: false,
                    maxRotation: 0,
                    padding: 15
                }}
            }}
        }};
        chartTop10Modal = new Chart(ctx, {{
            type: 'bar',
            data: {{
                labels: nombres,
                datasets: [{{
                    label: 'Porcentaje de Alcance',
                    data: porcentajes,
                    backgroundColor: esTop10Paises ? coloresBarras.map(() => 'rgba(0,0,0,0)') : coloresBarras,
                    borderColor: esTop10Paises ? coloresBarras.map(() => 'rgba(0,0,0,0)') : coloresBarras.map(color => color + 'CC'),
                    borderWidth: esTop10Paises ? 0 : 3,
                    borderRadius: 10,
                    borderSkipped: false,
                    maxBarThickness: esTop10Paises ? 132 : undefined,
                    categoryPercentage: esTop10Paises ? 0.98 : 0.9,
                    barPercentage: esTop10Paises ? 1 : 0.9
                }}]
            }},
            options: {{
                indexAxis: chartIndexAxis,
                responsive: false,
                maintainAspectRatio: false,
                plugins: {{
                    title: {{
                        display: true,
                        text: titulo,
                        font: {{ size: 32, weight: 'bold', family: 'Segoe UI' }},
                        color: '#2c3e50',
                        padding: {{ top: 10, bottom: 30 }}
                    }},
                    tooltip: {{
                        backgroundColor: 'rgba(0, 0, 0, 0.9)',
                        titleFont: {{ size: 20, weight: 'bold', family: 'Segoe UI' }},
                        bodyFont: {{ size: 18, family: 'Segoe UI' }},
                        padding: 20,
                        cornerRadius: 10,
                        displayColors: false,
                        callbacks: {{
                            title: function(context) {{
                                const index = context[0].dataIndex;
                                const item = top10[index];
                                const icon = index === 0 ? '🥇 ' : index === 1 ? '🥈 ' : index === 2 ? '🥉 ' : '';
                                return icon + (item.nombre || 'Sin nombre');
                            }},
                            label: function(context) {{
                                const item = top10[context.dataIndex] || {{}};
                                const pct = Number(esTop10Paises ? context.parsed.y : context.parsed.x);
                                const base = [`Porcentaje: ${{Number.isFinite(pct) ? pct.toFixed(2) : '0.00'}}%`];
                                return base.concat(item.tooltip || []);
                            }}
                        }}
                    }},
                    legend: {{ display: false }}
                }},
                scales: chartScales,
                animation: {{
                    duration: 1200,
                    easing: 'easeOutQuart'
                }}
            }},
            plugins: esTop10Paises
                ? [variableBarsPlugin, dataLabelsPlugin, positionNumbersPlugin, flagsPluginModal]
                : [dataLabelsPlugin, positionNumbersPlugin, flagsPluginModal]
        }});
        chartTop10Modal.update();
    }}

    // ========== FUNCIONES PARA MODAL DE CUMPLEAÑOS SIMPLIFICADO ==========
    function mostrarModalCumpleaños() {{
        const mesSeleccionado = document.getElementById('selectorMes').value;
        const añoSeleccionado = document.getElementById('selectorAño').value;
        const periodoCompleto = `${{mesSeleccionado}}_${{añoSeleccionado}}`;
        
        // Mostrar el modal directamente
        mostrarModalCumpleañosSimple(periodoCompleto, mesSeleccionado);
    }}
    
    function mostrarModalCumpleañosSimple(periodoCompleto, mesNombre) {{
        // Cerrar modal existente si hay uno
        const modalExistente = document.getElementById('modalCumpleaños');
        if (modalExistente) {{
            cerrarModalCumpleaños();
        }}
        
        // Cerrar también el modal de Top 10 si está abierto
        const modalTop10 = document.getElementById('modalTop10');
        if (modalTop10) {{
            cerrarModalTop10();
        }}
        
        // Obtener datos de cumpleaños para este periodo
        const todosLosCumpleaños = datosCumpleaños[periodoCompleto] || [];
        
        // Filtrar por mes actual (coincidencia exacta)
        const mesFiltroUpper = mesNombre.toUpperCase();
        const cumpleañosDelMes = todosLosCumpleaños.filter(persona => {{
            const vigente = String(persona.vigencia || '').trim().toUpperCase();
            const puesto = String(persona.puesto || 'ASESOR').trim().toUpperCase();
            return persona.mes === mesFiltroUpper && puesto === 'ASESOR' && vigente === 'SI';
        }});
        
        // Ordenar por día del mes
        cumpleañosDelMes.sort((a, b) => a.dia - b.dia);
        
        // Guardar datos globalmente para las funciones de descarga
        window.cumpleañosData = {{
            asesores: cumpleañosDelMes.sort((a, b) => a.dia - b.dia),
            mes: mesNombre
        }};
        
        // Asegurar que los estilos estén cargados (FORZAR recarga)
        if (document.getElementById('modal-cumpleaños-styles')) {{
            // Si ya existe, removerlo y recrearlo
            document.getElementById('modal-cumpleaños-styles').remove();
        }}
        agregarEstilosModalCumpleañosSimple();
        
        // SOLO UNA DECLARACIÓN DE MODAL - Con botones de descarga
        const modal = document.createElement('div');
        modal.id = 'modalCumpleaños';
        modal.innerHTML = `
            <div class="modal-overlay-cumple" onclick="cerrarModalCumpleaños()"></div>
            <div class="modal-content-cumple">
                <div class="modal-header-cumple">
                    <div style="display: flex; justify-content: space-between; align-items: center; width: 100%;">
                        <h2>🎂 CUMPLEAÑOS - ${{mesNombre.toUpperCase()}}</h2>
                        <div style="display: flex; gap: 10px;">
                            <button class="btn-descargar-cumple" onclick="descargarTarjetasAsesores()" 
                                    style="background: linear-gradient(135deg, #2ecc71, #27ae60);">
                                📥 Asesores
                            </button>
                        </div>
                    </div>
                </div>
                <div class="modal-body-cumple">
                    ${{generarContenidoCumpleañosSimple(cumpleañosDelMes, mesNombre)}}
                </div>
            </div>
        `;
        
        document.body.appendChild(modal);
        document.body.style.overflow = 'hidden';
        document.addEventListener('keydown', handleModalCumpleañosEscape);
        
        // Mostrar modal después de un pequeño delay
        setTimeout(() => {{
            modal.style.display = 'block';
        }}, 10);
    }}
    
    function generarContenidoCumpleañosSimple(cumpleañosData, mesNombre) {{
        if (cumpleañosData.length === 0) {{
            return `
                <div style="text-align: center; padding: 50px; color: #666;">
                    <div style="font-size: 3rem; margin-bottom: 20px;">🎂</div>
                    <div style="font-size: 1.5rem; font-weight: bold; margin-bottom: 10px;">
                        No hay cumpleaños este mes
                    </div>
                    <div style="font-size: 1rem;">
                        No se encontraron cumpleaños para ${{mesNombre}}
                    </div>
                </div>
            `;
        }}
        
        const asesores = cumpleañosData.slice().sort((a, b) => a.dia - b.dia);
        
        // Función para generar una columna de cumpleañeros
        const generarColumna = (personas, titulo, color) => {{
            if (personas.length === 0) {{
                return `
                    <div style="text-align: center; padding: 30px; color: #95a5a6;">
                        <div style="font-size: 2rem; margin-bottom: 10px;">👥</div>
                        <div style="font-weight: bold;">No hay ${{titulo.toLowerCase()}}</div>
                    </div>
                `;
            }}
            
            let html = `
                <div style="margin-bottom: 20px;">
                    <div style="background: ${{color}}; color: white; padding: 8px 15px; 
                         border-radius: 8px; font-weight: bold; margin-bottom: 15px; 
                         text-align: center; font-size: 1.1rem;">
                        ${{titulo}} (${{personas.length}})
                    </div>
                    
                    <div style="max-height: 50vh; overflow-y: auto; padding: 0 5px;">
            `;
            
        personas.forEach((persona) => {{
            // MODIFICADO: Mostrar puesto_real en lugar de alias
            const infoAdicional = persona.puesto_real || persona.alias || '';
            
            html += `
                <div style="display: flex; align-items: center; padding: 12px; margin-bottom: 8px; 
                            background: white; border-radius: 8px; border-left: 4px solid ${{color}};
                            box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
                    <div style="font-size: 1.2rem; margin-right: 12px; color: ${{color}};">
                        👤
                    </div>
                    <div style="flex: 1;">
                        <div style="font-weight: bold; font-size: 1.5rem; color: #2c3e50;">
                            ${{persona.nombre}}
                        </div>
                        <!-- MODIFICADO: Mostrar PUESTO REAL en lugar de alias -->
                        <div style="color: #666; font-size: 0.85rem; font-style: italic;">
                            ${{infoAdicional}}
                        </div>
                    </div>
                    <div style="text-align: right;">
                        <div style="font-size: 1.5rem; font-weight: bold; color: #e74c3c;">
                            ${{persona.dia}}
                        </div>
                        <div style="color: #666; font-size: 1rem;">
                            de ${{persona.mes}}
                        </div>
                    </div>
                </div>
            `;
        }});
        
        html += `
                </div>
            </div>
        `;
        
        return html;
        }};
        
        let html = `
        <div style="padding: 20px;">
            <div style="display: grid; grid-template-columns: minmax(280px, 760px); justify-content:center; gap: 25px; min-height: 400px;">
                <div>
                    ${{generarColumna(asesores, 'ASESORES', '#2ecc71')}}
                </div>
            </div>
        </div>
        `;
        
        return html;
    }}
        
    function agregarEstilosModalCumpleañosSimple() {{
        const style = document.createElement('style');
        style.id = 'modal-cumpleaños-styles';
        style.textContent = `
            /* MODAL DE CUMPLEAÑOS - CON ANIMACIONES */
            #modalCumpleaños {{
                display: none;
                position: fixed;
                top: 0;
                left: 0;
                width: 100%;
                height: 100%;
                z-index: 9999;
            }}
            
            #modalCumpleaños .modal-overlay-cumple {{
                position: absolute;
                top: 0;
                left: 0;
                width: 100%;
                height: 100%;
                background: rgba(0, 0, 0, 0.8);
                backdrop-filter: blur(4px);
                -webkit-backdrop-filter: blur(4px);
                cursor: pointer;
                opacity: 0;
                animation: fadeInCumple 0.3s ease-out forwards;
            }}
            
            #modalCumpleaños .modal-content-cumple {{
                position: absolute;
                top: 50%;
                left: 50%;
                transform: translate(-50%, -50%) scale(0.95);
                background: white;
                width: 90%;
                max-width: 1000px;
                max-height: 90vh;
                border-radius: 15px;
                overflow: hidden;
                box-shadow: 0 25px 60px rgba(0, 0, 0, 0.35);
                opacity: 0;
                animation: slideInCumple 0.4s ease-out 0.1s forwards;
            }}
            
            /* ANIMACIONES ESPECÍFICAS PARA CUMPLEAÑOS */
            @keyframes fadeInCumple {{
                from {{ 
                    opacity: 0; 
                    backdrop-filter: blur(0px);
                    -webkit-backdrop-filter: blur(0px);
                }}
                to {{ 
                    opacity: 1; 
                    backdrop-filter: blur(4px);
                    -webkit-backdrop-filter: blur(4px);
                }}
            }}
            
            @keyframes slideInCumple {{
                from {{
                    opacity: 0;
                    transform: translate(-50%, -50%) scale(0.9);
                    box-shadow: 0 10px 30px rgba(0, 0, 0, 0.1);
                }}
                50% {{
                    opacity: 1;
                    transform: translate(-50%, -52%) scale(1.02);
                    box-shadow: 0 30px 70px rgba(0, 0, 0, 0.4);
                }}
                to {{
                    opacity: 1;
                    transform: translate(-50%, -50%) scale(1);
                    box-shadow: 0 25px 60px rgba(0, 0, 0, 0.35);
                }}
            }}
            
            /* ANIMACIÓN DE CIERRE */
            @keyframes fadeOutCumple {{
                from {{ 
                    opacity: 1; 
                    backdrop-filter: blur(4px);
                    -webkit-backdrop-filter: blur(4px);
                }}
                to {{ 
                    opacity: 0; 
                    backdrop-filter: blur(0px);
                    -webkit-backdrop-filter: blur(0px);
                }}
            }}
            
            @keyframes slideOutCumple {{
                from {{
                    opacity: 1;
                    transform: translate(-50%, -50%) scale(1);
                }}
                to {{
                    opacity: 0;
                    transform: translate(-50%, -45%) scale(0.95);
                }}
            }}
            
            /* CLASE PARA CIERRE ANIMADO */
            #modalCumpleaños.cerrando .modal-overlay-cumple {{
                animation: fadeOutCumple 0.25s ease-in forwards !important;
            }}
            
            #modalCumpleaños.cerrando .modal-content-cumple {{
                animation: slideOutCumple 0.25s ease-in forwards !important;
            }}
            
            /* ESTILOS DEL CONTENIDO (igual que antes) */
            #modalCumpleaños .modal-header-cumple {{
                background: linear-gradient(135deg, #FF6B6B, #EE5A24);
                padding: 20px 30px;
                display: flex;
                justify-content: space-between;
                align-items: center;
            }}
            
            #modalCumpleaños .modal-header-cumple h2 {{
                margin: 0;
                color: white;
                font-size: 2rem;
                font-weight: 800;
            }}
            
            #modalCumpleaños .btn-cerrar-modal-cumple {{
                background: rgba(255, 255, 255, 0.2);
                border: none;
                color: white;
                font-size: 2rem;
                width: 45px;
                height: 45px;
                border-radius: 50%;
                cursor: pointer;
                display: flex;
                align-items: center;
                justify-content: center;
                transition: all 0.3s ease;
            }}
            
            #modalCumpleaños .btn-cerrar-modal-cumple:hover {{
                background: rgba(255, 255, 255, 0.3);
                transform: rotate(90deg);
            }}
            
            #modalCumpleaños .modal-body-cumple {{
                padding: 0;
                max-height: calc(90vh - 85px);
                overflow-y: auto;
            }}
            
            /* RESPONSIVE */
            @media (max-width: 1100px) {{
                #modalCumpleaños .modal-content-cumple {{
                    width: 95%;
                    max-width: 95%;
                }}
            }}
            
            @media (max-width: 768px) {{
                #modalCumpleaños .modal-content-cumple {{
                    width: 98%;
                    max-width: 98%;
                    animation: slideInCumple 0.35s ease-out 0.1s forwards !important;
                }}
                
                @keyframes slideInCumple {{
                    from {{
                        opacity: 0;
                        transform: translate(-50%, -50%) scale(0.95);
                    }}
                    to {{
                        opacity: 1;
                        transform: translate(-50%, -50%) scale(1);
                    }}
                }}
                
                #modalCumpleaños .modal-body-cumple > div > div {{
                    grid-template-columns: 1fr !important;
                }}
            }}

            /* Botones de descarga */
            .btn-descargar-cumple {{
                padding: 10px 20px;
                color: white;
                border: none;
                border-radius: 8px;
                cursor: pointer;
                font-size: 14px;
                font-weight: 600;
                transition: all 0.3s ease;
                display: flex;
                align-items: center;
                gap: 8px;
                box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
            }}
            
            .btn-descargar-cumple:hover {{
                transform: translateY(-2px);
                box-shadow: 0 6px 12px rgba(0, 0, 0, 0.15);
            }}
            
            .btn-descargar-cumple:active {{
                transform: translateY(0);
            }}
            
            /* Responsive para botones */
            @media (max-width: 768px) {{
                .btn-descargar-cumple {{
                    padding: 8px 15px;
                    font-size: 12px;
                }}
            }}
        `;
        document.head.appendChild(style);
    }}
    
    function cerrarModalCumpleaños() {{
        const modal = document.getElementById('modalCumpleaños');
        
        if (!modal) return;
        
        // RESTAURAR SCROLL DEL BODY
        document.body.style.overflow = 'auto';
        
        // Eliminar event listener de ESC
        document.removeEventListener('keydown', handleModalCumpleañosEscape);
        
        // Eliminar modal
        if (modal.parentNode) {{
            modal.parentNode.removeChild(modal);
        }}
    }}
    
    function handleModalCumpleañosEscape(event) {{
        if (event.key === 'Escape' || event.keyCode === 27) {{
            cerrarModalCumpleaños();
        }}
    }}
    
    function aplicarFiltroRanking(supervisor) {{
        const mesSeleccionado = document.getElementById('selectorMes').value;
        const añoSeleccionado = document.getElementById('selectorAño').value;
        const periodoCompleto = `${{mesSeleccionado}}_${{añoSeleccionado}}`;
        
        if (!datosMeses[periodoCompleto]) return;
        
        let asesoresBase = filtrarAsesoresPorCanal(datosMeses[periodoCompleto] || []);
        
        if (supervisor !== 'TODOS') {{
        asesoresBase = asesoresBase.filter(a => a.supervisor === supervisor);
        }}
        
        // ✅ Ranking vuelve a su vista normal (mes seleccionado)
        actualizarLista(asesoresBase, '>100%', 'lista-100');
        actualizarLista(asesoresBase, '>70%', 'lista-70');
        actualizarLista(asesoresBase, '>40%', 'lista-40');
        actualizarLista(asesoresBase, '>0%', 'lista-0');
        actualizarEstadisticas(asesoresBase);
        actualizarRankingQuintil(asesoresBase, periodoCompleto, supervisor);
    }}
    
    function actualizarLista(asesores, clasificacion, idLista) {{
        const lista = document.getElementById(idLista);
        const asesoresFiltrados = asesores.filter(a => a.clasificacion === clasificacion);
        
        if (asesoresFiltrados.length === 0) {{
            lista.innerHTML = '<div class="asesor-item">No hay asesores en esta categoría</div>';
            return;
        }}
        
        // Ordenar por porcentaje (mayor a menor)
        asesoresFiltrados.sort((a, b) => b.porcentaje - a.porcentaje);
        
        let html = '';
        asesoresFiltrados.forEach(asesor => {{
            const claseGradiente = `gradiente-${{clasificacion.replace('%', '').replace('>', '')}}`;
            const clasePorcentaje = `porcentaje-${{clasificacion.replace('%', '').replace('>', '')}}`;
            html += `<div class="asesor-item ${{claseGradiente}}">
                <div class="asesor-nombre">${{asesor.nombre}}</div>
                <div class="asesor-porcentaje ${{clasePorcentaje}}">${{asesor.porcentaje}}%</div>
            </div>`;
        }});
        
        lista.innerHTML = html;
    }}
    
    function actualizarEstadisticas(asesores) {{
        const contadores = {{
            ">100%": asesores.filter(a => a.clasificacion === ">100%").length,
            ">70%": asesores.filter(a => a.clasificacion === ">70%").length,
            ">40%": asesores.filter(a => a.clasificacion === ">40%").length,
            ">0%": asesores.filter(a => a.clasificacion === ">0%").length
        }};
        
        // Actualizar estadísticas si los elementos existen
        const elem100 = document.getElementById('cantidad-100');
        const elem70 = document.getElementById('cantidad-70');
        const elem40 = document.getElementById('cantidad-40');
        const elem0 = document.getElementById('cantidad-0');
        
        if (elem100) elem100.textContent = contadores[">100%"];
        if (elem70) elem70.textContent = contadores[">70%"];
        if (elem40) elem40.textContent = contadores[">40%"];
        if (elem0) elem0.textContent = contadores[">0%"];
    }}

    function _setRankingQuintilAbierto(q, abierto) {{
        const lista = document.getElementById(`rankingQuintilQ${{q}}`);
        const toggle = document.getElementById(`rankingQuintilToggleQ${{q}}`);
        const tarjeta = lista?.closest('.eval-quintil-card');
        if (lista) {{
            lista.classList.toggle('colapsada', !abierto);
            lista.setAttribute('aria-hidden', String(!abierto));
        }}
        if (tarjeta) {{
            tarjeta.classList.toggle('quintil-abierto', abierto);
            tarjeta.setAttribute('aria-expanded', String(abierto));
        }}
        if (toggle) toggle.textContent = abierto ? '-' : '+';
    }}

    function toggleRankingQuintil(q) {{
        const lista = document.getElementById(`rankingQuintilQ${{q}}`);
        if (!lista) return;
        _setRankingQuintilAbierto(q, lista.classList.contains('colapsada'));
    }}

    function actualizarRankingQuintil(asesores, periodoCompleto = '', supervisor = 'TODOS') {{
        const grupos = {{ 1: [], 2: [], 3: [], 4: [], 5: [] }};

        (Array.isArray(asesores) ? asesores : []).forEach(asesor => {{
            const q = _normalizarQuintil(asesor?.q_alc);
            const nombre = String(asesor?.nombre || asesor?.alias_crr || '').trim();
            if (!q || !nombre) return;
            const porcentaje = Number(asesor?.porcentaje);
            grupos[q].push({{
                alias: nombre,
                alcance: Number.isFinite(porcentaje) ? porcentaje : null
            }});
        }});

        [5, 4, 3, 2, 1].forEach(q => {{
            const cont = document.getElementById(`rankingQuintilQ${{q}}`);
            const resumen = document.getElementById(`rankingQuintilResumenQ${{q}}`);
            const toggle = document.getElementById(`rankingQuintilToggleQ${{q}}`);
            if (!cont) return;

            const estabaAbierto = !cont.classList.contains('colapsada');
            const items = grupos[q].sort((a, b) => (b.alcance ?? -Infinity) - (a.alcance ?? -Infinity));
            const valores = items.map(it => it.alcance).filter(Number.isFinite);
            const promedio = valores.length ? valores.reduce((acc, valor) => acc + valor, 0) / valores.length : null;

            if (resumen) {{
                resumen.innerHTML = `
                    <div class="eval-quintil-metric">
                        <span class="eval-quintil-metric-label">Asesores</span>
                        <span class="eval-quintil-metric-value">${{items.length}}</span>
                    </div>
                    <div class="eval-quintil-metric">
                        <span class="eval-quintil-metric-label">Promedio</span>
                        <span class="eval-quintil-metric-value">${{promedio === null ? '-' : `${{promedio.toFixed(2)}}%`}}</span>
                    </div>`;
            }}

            cont.innerHTML = items.map((it, index) => `
                <div class="eval-quintil-row" style="--quintil-delay:${{Math.min(index, 10) * 32}}ms">
                    <div class="eval-quintil-asesor" title="${{it.alias}}">${{it.alias}}</div>
                    <div class="eval-quintil-alcance">${{it.alcance === null ? '-' : `${{it.alcance.toFixed(2)}}%`}}</div>
                </div>`).join('') || '<div style="padding:14px 8px; text-align:center; color:#666; font-size:0.85rem;">Sin registros</div>';

            _setRankingQuintilAbierto(q, estabaAbierto);
            if (toggle && !estabaAbierto) toggle.textContent = '+';
        }});

        const nota = document.getElementById('rankingQuintilNota');
        if (nota) {{
            const total = Object.values(grupos).reduce((acc, arr) => acc + arr.length, 0);
            const etiquetaSupervisor = supervisor && supervisor !== 'TODOS' ? ` · ${{supervisor}}` : ' · Todos los equipos';
            nota.textContent = total
                ? `Quintiles mensuales Q_ALC de ${{String(periodoCompleto).replace('_', ' ')}}${{etiquetaSupervisor}}.`
                : `No hay asesores con Q_ALC para ${{String(periodoCompleto || 'este periodo').replace('_', ' ')}}${{etiquetaSupervisor}}.`;
        }}
    }}

    // ========== FUNCIONES PARA BUSQUEDA Y COMPARACIÓN DE ASESORES ==========

    function ordenarTablaComparacionAsesores(colIndex, tipo) {{
      const table = document.getElementById('tablaComparacionAsesoresTabla');
      if (!table) return;
    
      const tbody = table.querySelector('tbody');
      if (!tbody) return;
    
      const keyDir = 'sortDir_' + colIndex;
      const asc = table.dataset[keyDir] !== 'asc';
      table.dataset[keyDir] = asc ? 'asc' : 'desc';
    
      const rows = Array.from(tbody.querySelectorAll('tr'));
    
      // 1) Agrupar en bloques: [fila principal] + [subfilas...]
      const bloques = [];
      for (let i = 0; i < rows.length; i++) {{
        const tr = rows[i];
        if (tr.dataset.subfila === '1') continue;
    
        const bloque = [tr];
        let j = i + 1;
        while (j < rows.length && rows[j].dataset.subfila === '1') {{
          bloque.push(rows[j]);
          j++;
        }}
        bloques.push(bloque);
        i = j - 1;
      }}
    
      function getVal(tr) {{
        const td = tr.children[colIndex];
        if (!td) return (tipo === 'txt') ? '' : -Infinity;
    
        const raw = td.dataset.value;
        if ((tipo || 'num') === 'txt') {{
          return String(raw || '').toUpperCase();
        }}
        const n = parseFloat(raw);
        return Number.isFinite(n) ? n : -Infinity;
      }}
    
      bloques.sort((A, B) => {{
        const a = getVal(A[0]);
        const b = getVal(B[0]);
    
        if ((tipo || 'num') === 'txt') {{
          return asc ? a.localeCompare(b) : b.localeCompare(a);
        }}
    
        // asc=true => mayor a menor (más útil en KPIs)
        return asc ? (b - a) : (a - b);
      }});
    
      const frag = document.createDocumentFragment();
      bloques.forEach(b => b.forEach(tr => frag.appendChild(tr)));
      tbody.appendChild(frag);
    }}

    function _getAnchorDateFromHeader() {{
      const mesSel = document.getElementById('selectorMes')?.value;
      const anioSel = document.getElementById('selectorAño')?.value;
    
      if (!mesSel || !anioSel) return null;
    
      // obtenerNumeroMes(mes) ya lo usas en tu sort; mantenemos el mismo criterio
      return new Date(parseInt(anioSel, 10), obtenerNumeroMes(mesSel));
    }}
    
    function _filtrarHastaAncla(mesesOrdenadosDesc) {{
      const anchor = _getAnchorDateFromHeader();
      if (!anchor) return mesesOrdenadosDesc;
    
      const filtrados = mesesOrdenadosDesc.filter(periodo => {{
        const [mes, anio] = String(periodo).split('_');
        if (!mes || !anio) return false;
        const d = new Date(parseInt(anio, 10), obtenerNumeroMes(mes));
        return d <= anchor;
      }});
    
      // Si por algún motivo quedó vacío (data incompleta / selector no calza), no rompemos nada:
      return (filtrados.length > 0) ? filtrados : mesesOrdenadosDesc;
    }}
    
    function _filtrarMesesComparacionAsesores(mesesOrdenadosDesc) {{
      const anchor = _getAnchorDateFromHeader();
      if (!anchor) return mesesOrdenadosDesc;

      const chkIncluir = document.getElementById('chkAsesIncluirMesSeleccionado');
      const incluirMesSeleccionado = !!(chkIncluir && chkIncluir.checked);

      const filtrados = mesesOrdenadosDesc.filter(periodo => {{
        const [mes, anio] = String(periodo).split('_');
        if (!mes || !anio) return false;
        const d = new Date(parseInt(anio, 10), obtenerNumeroMes(mes));
        return incluirMesSeleccionado ? d <= anchor : d < anchor;
      }});

      return filtrados;
    }}
    
    function _filtrarMesesComparacionSupervisores(mesesOrdenadosDesc) {{
      const anchor = _getAnchorDateFromHeader();
      if (!anchor) return mesesOrdenadosDesc;

      const chkIncluir = document.getElementById('chkSupIncluirMesSeleccionado');
      const incluirMesSeleccionado = !!(chkIncluir && chkIncluir.checked);

      const filtrados = mesesOrdenadosDesc.filter(periodo => {{
        const [mes, anio] = String(periodo).split('_');
        if (!mes || !anio) return false;
        const d = new Date(parseInt(anio, 10), obtenerNumeroMes(mes));
        return incluirMesSeleccionado ? d <= anchor : d < anchor;
      }});

      return filtrados;
    }}
    
    function resolverNombreAsesorParaComparacion(valor) {{
        const clave = normalizarTextoAsesor(valor);
        if (!clave) return '';

        const periodos = Object.keys(datosMeses || {{}}).sort((a, b) => {{
            const [mesA, anioA] = a.split('_');
            const [mesB, anioB] = b.split('_');
            return new Date(anioB, obtenerNumeroMes(mesB)) - new Date(anioA, obtenerNumeroMes(mesA));
        }});

        for (const periodo of periodos) {{
            const match = (datosMeses[periodo] || []).find(a =>
                normalizarTextoAsesor(a.nombre) === clave ||
                normalizarTextoAsesor(a.alias_crr) === clave ||
                normalizarTextoAsesor(a.indicadores_calidad?.alias) === clave
            );
            if (match) return String(match.nombre || valor).trim();
        }}
        return String(valor || '').trim();
    }}

    function agregarAsesorComparacionHistorico(reemplazar = false, mostrarResultados = true) {{
        const input = document.getElementById('asesorSearchInput');
        const valor = String(input?.value || asesorSeleccionadoHistorico || '').trim();
        if (!valor) return;

        const nombreAsesor = resolverNombreAsesorParaComparacion(valor);
        if (!nombreAsesor) return;

        if (reemplazar) asesoresSeleccionados.clear();
        asesoresSeleccionados.add(nombreAsesor);
        actualizarAsesoresSeleccionados();

        if (mostrarResultados) {{
            compararAsesores();
        }} else {{
            const resultadosDiv = document.getElementById('resultadosComparacion');
            if (resultadosDiv && resultadosDiv.style.display !== 'none') {{
                actualizarTablaComparacionAsesores();
                actualizarGraficaComparacionAsesores();
            }}
        }}
    }}

    function agregarAsesor() {{
        agregarAsesorComparacionHistorico(false);
    }}
    
    function _getPeriodoSeleccionadoRanking() {{
        const mesSeleccionado = document.getElementById('selectorMes')?.value || '';
        const anioSeleccionado = document.getElementById('selectorA\u00f1o')?.value || '';
        if (!mesSeleccionado || !anioSeleccionado) return '';
        return `${{mesSeleccionado}}_${{anioSeleccionado}}`;
    }}

    function seleccionarAsesoresPorSupervisor() {{
        const selector = document.getElementById('selectorSupervisorAsesores');
        if (!selector) return;

        const supervisor = String(selector.value || '').trim();
        if (!supervisor) return;

        const periodo = _getPeriodoSeleccionadoRanking();
        const asesoresPeriodo = filtrarAsesoresPorCanal(
            (periodo && Array.isArray(datosMeses[periodo]))
                ? datosMeses[periodo]
                : Object.values(datosMeses).flat()
        );

        const asesoresSupervisor = asesoresPeriodo
            .filter(a => String(a.supervisor || '').trim() === supervisor)
            .map(a => String(a.nombre || '').trim())
            .filter(Boolean)
            .sort((a, b) => a.localeCompare(b, 'es'));

        asesoresSeleccionados.clear();
        asesoresSupervisor.forEach(nombre => asesoresSeleccionados.add(nombre));
        actualizarAsesoresSeleccionados();

        const resultadosDiv = document.getElementById('resultadosComparacion');
        if (resultadosDiv && resultadosDiv.style.display !== 'none') {{
            compararAsesores();
        }}

        if (asesoresSupervisor.length === 0) {{
            alert(`No se encontraron asesores para ${{supervisor}} en el periodo seleccionado.`);
        }}
    }}
    
    function eliminarAsesor(nombre) {{
        asesoresSeleccionados.delete(nombre);
        actualizarAsesoresSeleccionados();
    }}
    
    function actualizarAsesoresSeleccionados() {{
        const container = document.getElementById('asesoresSeleccionados');
        if (!container) return;
        
        container.innerHTML = '';
        
        asesoresSeleccionados.forEach(asesor => {{
            const tag = document.createElement('div');
            tag.className = 'tag-elemento';
            tag.innerHTML = `
                ${{asesor}}
                <button class="eliminar-elemento" onclick="eliminarAsesor('${{asesor}}')">×</button>
            `;
            container.appendChild(tag);
        }});
    }}
    
    function compararAsesores() {{
        if (asesoresSeleccionados.size === 0) {{
            alert('Por favor selecciona al menos un asesor para comparar.');
            return;
        }}
        
        const resultadosDiv = document.getElementById('resultadosComparacion');
        const tablaDiv = document.getElementById('tablaComparacion');
        
        if (!resultadosDiv || !tablaDiv) return;
        
        // Mostrar los controles de meses
        const controlesHTML = `
            <div style="margin: 20px 0; text-align: center;">
                <div class="controles-grafica" style="display:flex; justify-content:center; align-items:center; gap:14px; flex-wrap:wrap;">
                    
                    <div style="display:flex; gap:8px; align-items:center; flex-wrap:wrap;">
                        <button class="btn-comparacion" onclick="cambiarTipoGrafica(3)">
                            3 meses
                        </button>
                        <button class="btn-comparacion activo" onclick="cambiarTipoGrafica(6)">
                            6 meses
                        </button>
                        <button class="btn-comparacion" onclick="cambiarTipoGrafica(12)">
                            12 meses
                        </button>
                    </div>
        
                    <label style="display:flex; gap:8px; align-items:center; cursor:pointer; user-select:none; font-weight:700; color:#2c3e50;">
                        <input type="checkbox" id="chkAsesMostrarFilaSupervisor" onchange="actualizarTablaComparacionAsesores()" />
                        <span>Ver Supervisor</span>
                    </label>

                    <label style="display:flex; gap:8px; align-items:center; cursor:pointer; user-select:none; font-weight:700; color:#2c3e50; text-transform:uppercase;">
                        <input type="checkbox" id="chkAsesIncluirMesSeleccionado" onchange="actualizarTablaComparacionAsesores(); actualizarGraficaComparacionAsesores();" />
                        <span>Incluir mes seleccionado</span>
                    </label>
        
                </div>
            </div>
        `;
        
        // Insertar controles ANTES de la tabla
        resultadosDiv.innerHTML = controlesHTML;
        
        // Crear div para tabla
        const tablaContainer = document.createElement('div');
        tablaContainer.id = 'tablaComparacion';
        resultadosDiv.appendChild(tablaContainer);
        
        // Crear div para gráfica (se creará dinámicamente después)
        const graficaContainer = document.createElement('div');
        graficaContainer.id = 'graficaContainerAsesores';
        resultadosDiv.appendChild(graficaContainer);
        
        // Actualizar tabla y gráfica
        actualizarTablaComparacionAsesores();
        actualizarGraficaComparacionAsesores();
        
        resultadosDiv.style.display = 'block';
    }}
    
    function generarGraficaComparacionAsesores(meses, asesores) {{
        // Crear contenedor para el gráfico si no existe
        let graficaDiv = document.getElementById('graficaComparacion');
        if (!graficaDiv) {{
            const resultadosDiv = document.getElementById('resultadosComparacion');
            if (resultadosDiv) {{
                graficaDiv = document.createElement('div');
                graficaDiv.id = 'graficaComparacion';
                graficaDiv.className = 'grafica-container';
                graficaDiv.innerHTML = `
                    <h3>📈 Evolución de Desempeño</h3>
                    <canvas id="graficaDesempenio" width="1200" height="400"></canvas>
                `;
                resultadosDiv.appendChild(graficaDiv);
            }}
        }}
        
        const canvas = document.getElementById('graficaDesempenio');
        if (!canvas) return;
        
        const ctx = canvas.getContext('2d');
        
        if (chartInstance) {{
            chartInstance.destroy();
            chartInstance = null;
        }}
        
        // LIMPIAR EL CANVAS COMPLETAMENTE
        ctx.clearRect(0, 0, canvas.width, canvas.height);
        const datasets = [];
        const colores = [
            '#FF6384', '#36A2EB', '#FFCE56', '#8AC926', 
            '#9966FF', '#FF9F40', '#4BC0C0', '#1982C4'
        ];
        
        let maxValor = 0;
        let minValor = 0;
        
        asesores.forEach((asesor, index) => {{
            const datosAsesor = [];
            const borderColor = colores[index % colores.length];
            const backgroundColor = borderColor + '40'; // 40% de opacidad para área
            
            meses.forEach(mes => {{
                const datosMes = datosMeses[mes];
                const asesorData = datosMes.find(a => a.nombre === asesor);
                const valor = asesorData ? asesorData.porcentaje : null;
                datosAsesor.push(valor);
                
                // Calcular máximo y mínimo
                if (valor !== null && valor !== undefined && !isNaN(valor)) {{
                    maxValor = Math.max(maxValor, valor);
                    minValor = Math.min(minValor, valor);
                }}
            }});
            
            // GRÁFICA DE LÍNEAS CON ÁREA Y LÍNEAS DISCONTINUAS
            datasets.push({{
                label: asesor,
                data: datosAsesor,
                borderColor: borderColor,
                backgroundColor: backgroundColor,
                borderWidth: 3,
                fill: true,
                tension: 0.4,
                pointRadius: 6,
                pointHoverRadius: 8,
                type: 'line',
                spanGaps: true,
                // LÍNEAS DISCONTINUAS PARA DATOS FALTANTES
                segment: {{
                    borderColor: ctx => {{
                        const p0Null = ctx.p0.parsed.y === null;
                        const p1Null = ctx.p1.parsed.y === null;
                        return (p0Null || p1Null) ? borderColor + '60' : borderColor;
                    }},
                    borderDash: ctx => {{
                        const p0Null = ctx.p0.parsed.y === null;
                        const p1Null = ctx.p1.parsed.y === null;
                        return (p0Null || p1Null) ? [5, 3] : []; // Discontinuo para datos faltantes
                    }}
                }},
                // PUNTOS DIFERENCIADOS PARA DATOS VÁLIDOS VS NULOS
                pointBackgroundColor: datosAsesor.map(valor => 
                    valor === null ? '#95a5a6' : borderColor
                ),
                pointBorderColor: datosAsesor.map(valor => 
                    valor === null ? '#95a5a6' : '#ffffff'
                ),
                pointBorderWidth: 2
            }});
        }});
        
        // Calcular límites inteligentes para el eje Y
        let yMin = 0;
        let yMax = 100; // Por defecto 100%
        
        if (maxValor > 0) {{
            // Redondear al múltiplo de 10 más cercano hacia arriba
            yMax = Math.max(100, Math.ceil(maxValor / 10) * 10);
        }}
        
        // Si hay valores negativos (raro, pero por si acaso)
        if (minValor < 0) {{
            yMin = Math.floor(minValor / 10) * 10;
        }}
        
        // Formatear labels de meses para mejor visualización
        const labels = meses.map(mes => {{
            const [mesNombre, año] = mes.split('_');
            return `${{mesNombre.substring(0, 3)}} ${{año}}`;
        }});
        
        // Crear la gráfica SOLO DE LÍNEAS
        chartInstance = new Chart(ctx, {{
            type: 'line',
            data: {{
                labels: labels,
                datasets: datasets
            }},
            options: {{
                responsive: false,
                maintainAspectRatio: false,
                plugins: {{
                    title: {{
                        display: false,
                        text: 'Evolución del Porcentaje de Alcance - Asesores',
                        font: {{ size: 16 }}
                    }},
                    tooltip: {{
                        mode: 'index',
                        intersect: false,
                        backgroundColor: 'rgba(0, 0, 0, 0.8)',
                        titleFont: {{ size: 14, weight: 'bold' }},
                        bodyFont: {{ size: 13, weight: 'normal' }},
                        padding: 12,
                        cornerRadius: 8,
                        itemSort: function(a, b) {{
                            const ay = Number(a.parsed?.y);
                            const by = Number(b.parsed?.y);
                            const aVal = Number.isFinite(ay) ? ay : -Infinity;
                            const bVal = Number.isFinite(by) ? by : -Infinity;
                            return bVal - aVal;
                        }},
                        callbacks: {{
                            label: function(context) {{
                                const asesor = context.dataset.label;
                                const porcentaje = context.parsed.y;
                                
                                if (porcentaje === null) return `${{asesor}}: No participó`;
                                
                                const label = context.chart.data.labels[context.dataIndex];
                                const [mesAbrev, año] = label.split(' ');
                                
                                // Mapeo de meses en MAYÚSCULAS
                                const mesesMap = {{
                                    'ENE':'ENERO','FEB':'FEBRERO','MAR':'MARZO','ABR':'ABRIL',
                                    'MAY':'MAYO','JUN':'JUNIO','JUL':'JULIO','AGO':'AGOSTO',
                                    'SEP':'SETIEMBRE','OCT':'OCTUBRE','NOV':'NOVIEMBRE','DIC':'DICIEMBRE'
                                }};
                                
                                const mesKey = `${{mesesMap[mesAbrev]}}_${{año}}`;
                                const datos = datosMeses[mesKey]?.find(a => a.nombre === asesor);
                                
                                // Función para formatear con separador de miles
                                const formatoMiles = (valor) => {{
                                    return 'S/ ' + valor.toLocaleString('es-PE', {{
                                        minimumFractionDigits: 2,
                                        maximumFractionDigits: 2
                                    }});
                                }};
                                
                                return [
                                    `${{asesor}}: ${{porcentaje.toFixed(2)}}%`,
                                    `Meta: ${{formatoMiles(datos?.meta || 0)}}`,
                                    `Recupero: ${{formatoMiles(datos?.recupero || 0)}}`
                                ];
                            }}
                        }}
                    }},
                    legend: {{
                        position: 'top',
                        labels: {{ font: {{ size: 14 }} }},
                        onClick: function (e, legendItem, legend) {{
                            const index = legendItem.datasetIndex;
                            const ci = legend.chart;
                            if (ci.isDatasetVisible(index)) {{
                                ci.hide(index);
                            }} else {{
                                ci.show(index);
                            }}
                        }}
                    }}
                }},
                scales: {{
                    y: {{
                        beginAtZero: true,
                        title: {{
                            display: false,
                            text: 'Porcentaje de Alcance (%)'
                        }},
                        ticks: {{
                            font: {{ size: 16 }},
                            callback: function(value) {{
                                return value + '%';
                            }},
                            stepSize: calcularStepSize(yMax)
                        }},
                        // ¡LIMITES EXPLÍCITOS Y FIJO!
                        min: yMin,
                        max: yMax,
                        suggestedMin: yMin,
                        suggestedMax: yMax
                    }},
                    x: {{
                        title: {{
                            display: false,
                            text: 'Meses'
                        }},
                        ticks: {{
                            font: {{ size: 15 }}
                        }}
                    }}
                }},
                interaction: {{
                    mode: 'nearest',
                    axis: 'x',
                    intersect: false
                }}
            }}
        }});
    }}

    function cambiarTipoGrafica(cantidadMeses) {{
        mesesAMostrar = cantidadMeses;
        
        // Actualizar botones activos
        document.querySelectorAll('#resultadosComparacion .btn-comparacion').forEach(btn => {{
            btn.classList.remove('activo');
        }});
        event.target.classList.add('activo');
        
        // Regenerar tabla Y gráfica si hay datos
        if (asesoresSeleccionados.size > 0) {{
            // Primero regenerar tabla
            actualizarTablaComparacionAsesores();
            // Luego regenerar gráfica
            actualizarGraficaComparacionAsesores();
        }}
    }}

    function actualizarGraficaComparacionAsesores() {{
        if (asesoresSeleccionados.size === 0) return;
        
        let mesesOrdenados = Object.keys(datosMeses).sort((a, b) => {{
            const [mesA, añoA] = a.split('_');
            const [mesB, añoB] = b.split('_');
            const fechaA = new Date(añoA, obtenerNumeroMes(mesA));
            const fechaB = new Date(añoB, obtenerNumeroMes(mesB));
            return fechaB - fechaA;
        }});
        mesesOrdenados = _filtrarMesesComparacionAsesores(mesesOrdenados);
        
        if (mesesAMostrar > 0 && mesesOrdenados.length > mesesAMostrar) {{
          mesesOrdenados = mesesOrdenados.slice(0, mesesAMostrar);
        }}
        
        const mesesParaGrafica = [...mesesOrdenados].reverse();
        generarGraficaComparacionAsesores(mesesParaGrafica, Array.from(asesoresSeleccionados));
    }}

    // ========== FUNCION PARA ACTUALIZAR TABLA ASESORES ==========
    function actualizarTablaComparacionAsesores() {{
        if (asesoresSeleccionados.size === 0) return;
    
        const tablaDiv = document.getElementById('tablaComparacion');
        if (!tablaDiv) return;
    
        // Obtener meses ordenados (MÁS RECIENTE PRIMERO)
        let mesesOrdenados = Object.keys(datosMeses).sort((a, b) => {{
            const [mesA, añoA] = a.split('_');
            const [mesB, añoB] = b.split('_');
            const fechaA = new Date(añoA, obtenerNumeroMes(mesA));
            const fechaB = new Date(añoB, obtenerNumeroMes(mesB));
            return fechaB - fechaA;
        }});
        mesesOrdenados = _filtrarMesesComparacionAsesores(mesesOrdenados);
        
        if (mesesAMostrar > 0 && mesesOrdenados.length > mesesAMostrar) {{
          mesesOrdenados = mesesOrdenados.slice(0, mesesAMostrar);
        }}
    
        // Invertir para mostrar de más antiguo a más reciente
        const mesesParaTabla = [...mesesOrdenados].reverse();
    
        // Checkbox mostrar/ocultar
        const chkSup = document.getElementById('chkAsesMostrarFilaSupervisor');
        const mostrarFilaSupervisor = chkSup ? chkSup.checked : false;
    
        // Crear tabla de comparación
        let html = '<div class="contenedor-tabla-scroll"><table id="tablaComparacionAsesoresTabla" class="tabla-comparacion">';
    
        // Encabezado (clickeable)
        html += '<thead><tr>';
        html += '<th style="cursor:pointer;" onclick="ordenarTablaComparacionAsesores(0, `txt`)">Asesor</th>';
        
        mesesParaTabla.forEach((mes, i) => {{
          const col = i + 1;
          html += `<th style="cursor:pointer;" onclick="ordenarTablaComparacionAsesores(${{col}}, \`num\`)">${{mes.replace('_',' ')}}</th>`;
        }});
        
        const colProm = mesesParaTabla.length + 1;
        html += `<th style="cursor:pointer;" onclick="ordenarTablaComparacionAsesores(${{colProm}}, \`num\`)">Promedio</th>`;
        html += '</tr></thead><tbody>';
    
        // Filas por cada asesor
        asesoresSeleccionados.forEach(asesor => {{
            // Fila 1
            html += `<tr><td data-value="${{asesor}}"><strong>${{asesor}}</strong></td>`;
    
            let suma = 0;
            let mesesConData = 0;
            
            mesesParaTabla.forEach(mes => {{
                const datosMes = datosMeses[mes];
                const asesorData = datosMes.find(a => a.nombre === asesor);
            
                if (asesorData) {{
                    const val = Number(asesorData.porcentaje);
                    const v = (isFinite(val)) ? val : 0;
            
                    suma += v;
                    mesesConData += 1;
            
                    html += `<td data-value="${{v}}">${{v.toFixed(2)}}%</td>`;
                }} else {{
                    html += '<td data-value="-1" style="color:#999; font-style: italic;">No participó</td>';
                }}
            }});
            
            const promedio = (mesesConData > 0) ? (suma / mesesConData) : 0;
            const promTxt = (mesesConData > 0) ? `${{promedio.toFixed(2)}}%` : 'Sin datos';
            html += `<td data-value="${{mesesConData > 0 ? promedio : -1}}"><strong>${{promTxt}}</strong></td>`;
            html += '</tr>';
    
            // Fila 2
            if (mostrarFilaSupervisor) {{
                html += `<tr data-subfila="1" class="fila-secundaria-asesor"><td class="label-secundario-asesor">Supervisor:</td>`;
    
                mesesParaTabla.forEach(mes => {{
                    const datosMes = datosMeses[mes];
                    const asesorData = datosMes.find(a => a.nombre === asesor);
    
                    if (asesorData) {{
                        const supervisor = asesorData.supervisor || 'Sin Supervisor';
    
                        // Mantener color/clases de supervisor
                        let supervisorClass = 'supervisor-normal';
                        if (supervisor === 'Sin Supervisor') supervisorClass = 'supervisor-sin';
                        else if (supervisor.toUpperCase().includes('SUPER') || supervisor.toUpperCase().includes('JEFE')) {{
                            supervisorClass = 'supervisor-jefe';
                        }}
    
                        html += `<td class="dato-secundario-asesor"><span class="${{supervisorClass}}">${{supervisor}}</span></td>`;
                    }} else {{
                        html += '<td class="dato-secundario-asesor supervisor-vacio">-</td>';
                    }}
                }});
    
                // Columna extra para Promedio
                html += '<td class="dato-secundario-asesor supervisor-vacio">-</td>';
                html += '</tr>';
            }}
        }});
    
        html += '</tbody></table></div>';
        tablaDiv.innerHTML = html;
    }}
    
    function limpiarBusqueda() {{
        asesoresSeleccionados.clear();
        actualizarAsesoresSeleccionados();

        const selectorSupervisor = document.getElementById('selectorSupervisorAsesores');
        if (selectorSupervisor) selectorSupervisor.value = '';
        
        // Destruir gráfica
        if (chartInstance) {{
            chartInstance.destroy();
            chartInstance = null;
        }}
        
        const resultadosDiv = document.getElementById('resultadosComparacion');
        if (resultadosDiv) resultadosDiv.style.display = 'none';
    }}
    
    // ========== FUNCIONES PARA BUSQUEDA Y COMPARACIÓN DE SUPERVISORES ==========

    function ordenarTablaComparacionSupervisores(colIndex, tipo) {{
      const table = document.getElementById('tablaComparacionSupervisoresTabla');
      if (!table) return;
    
      const tbody = table.querySelector('tbody');
      if (!tbody) return;
    
      // Toggle asc/desc por columna
      const keyDir = 'sortDir_' + colIndex;
      const asc = table.dataset[keyDir] !== 'asc';
      table.dataset[keyDir] = asc ? 'asc' : 'desc';
    
      const rows = Array.from(tbody.querySelectorAll('tr'));
    
      // ===== 1) Agrupar en bloques: [fila principal] + [subfilas...]
      const bloques = [];
      for (let i = 0; i < rows.length; i++) {{
        const tr = rows[i];
    
        // Saltar subfilas sueltas por seguridad
        if (tr.dataset.subfila === '1') continue;
    
        const bloque = [tr];
        let j = i + 1;
        while (j < rows.length && rows[j].dataset.subfila === '1') {{
          bloque.push(rows[j]);
          j++;
        }}
        bloques.push(bloque);
        i = j - 1;
      }}
    
      // ===== 2) Función para leer valor de orden
      function getVal(tr) {{
        const td = tr.children[colIndex];
        if (!td) return (tipo === 'txt') ? '' : -Infinity;
    
        const raw = td.dataset.value;
        if ((tipo || 'num') === 'txt') {{
          return String(raw || '').toUpperCase();
        }}
        const n = parseFloat(raw);
        return Number.isFinite(n) ? n : -Infinity;
      }}
    
      // ===== 3) Ordenar bloques por la fila principal del bloque
      bloques.sort((A, B) => {{
        const a = getVal(A[0]);
        const b = getVal(B[0]);
    
        if ((tipo || 'num') === 'txt') {{
          return asc ? a.localeCompare(b) : b.localeCompare(a);
        }}
    
        // asc=true => mayor a menor (más útil)
        return asc ? (b - a) : (a - b);
      }});
    
      // ===== 4) Volcar orden al DOM
      const frag = document.createDocumentFragment();
      bloques.forEach(b => b.forEach(tr => frag.appendChild(tr)));
      tbody.appendChild(frag);
    }}

    function agregarSupervisor() {{
        const input = document.getElementById('inputBusquedaSupervisor');
        const nombreSupervisor = input.value.trim();
        
        if (nombreSupervisor && !supervisoresSeleccionados.has(nombreSupervisor)) {{
            supervisoresSeleccionados.add(nombreSupervisor);
            actualizarSupervisoresSeleccionados();
            input.value = '';
        }}
    }}

    function _getPeriodoSeleccionadoSupervisores() {{
        const mesSeleccionado = document.getElementById('selectorMes')?.value || '';
        const anioSeleccionado = document.getElementById('selectorA\u00f1o')?.value || '';
        if (!mesSeleccionado || !anioSeleccionado) return '';
        return `${{mesSeleccionado}}_${{anioSeleccionado}}`;
    }}

    function seleccionarSupervisoresPorSegmento() {{
        const selector = document.getElementById('selectorSegmentoSupervisores');
        if (!selector) return;

        const segmento = String(selector.value || '').trim();
        if (!segmento) return;

        const periodo = _getPeriodoSeleccionadoSupervisores();
        const supervisoresSegmento = Object.keys(datosSupervisores || {{}})
            .filter(supervisor => {{
                const datosPeriodo = periodo ? datosSupervisores?.[supervisor]?.[periodo] : null;
                if (datosPeriodo) {{
                    return String(datosPeriodo.cartera || 'No definida').trim() === segmento;
                }}

                return Object.values(datosSupervisores?.[supervisor] || {{}}).some(datosMes =>
                    String(datosMes?.cartera || 'No definida').trim() === segmento
                );
            }})
            .sort((a, b) => a.localeCompare(b, 'es'));

        supervisoresSeleccionados.clear();
        supervisoresSegmento.forEach(supervisor => supervisoresSeleccionados.add(supervisor));
        actualizarSupervisoresSeleccionados();

        const resultadosDiv = document.getElementById('resultadosComparacionSupervisores');
        if (resultadosDiv && resultadosDiv.style.display !== 'none') {{
            compararSupervisores();
        }}

        if (supervisoresSegmento.length === 0) {{
            alert(`No se encontraron supervisores para el segmento ${{segmento}} en el periodo seleccionado.`);
        }}
    }}
    
    function eliminarSupervisor(nombre) {{
        supervisoresSeleccionados.delete(nombre);
        actualizarSupervisoresSeleccionados();
    }}
    
    function actualizarSupervisoresSeleccionados() {{
        const container = document.getElementById('supervisoresSeleccionados');
        if (!container) return;
        
        container.innerHTML = '';
        
        supervisoresSeleccionados.forEach(supervisor => {{
            const tag = document.createElement('div');
            tag.className = 'tag-elemento';
            tag.innerHTML = `
                ${{supervisor}}
                <button class="eliminar-elemento" onclick="eliminarSupervisor('${{supervisor}}')">×</button>
            `;
            container.appendChild(tag);
        }});
    }}
    
    function compararSupervisores() {{
        if (supervisoresSeleccionados.size === 0) {{
            alert('Por favor selecciona al menos un supervisor para comparar.');
            return;
        }}
        
        const resultadosDiv = document.getElementById('resultadosComparacionSupervisores');
        
        if (!resultadosDiv) return;

        // Mostrar los controles de meses
        const controlesHTML = `
          <div style="margin: 20px 0; text-align: center;">
            <div class="controles-grafica" style="display:flex; justify-content:center; align-items:center; gap:14px; flex-wrap:wrap;">
        
              <div style="display:flex; gap:8px; align-items:center; flex-wrap:wrap;">
                <button class="btn-comparacion" onclick="cambiarTipoGraficaSupervisores(3)">
                  3 meses
                </button>
                <button class="btn-comparacion activo" onclick="cambiarTipoGraficaSupervisores(6)">
                  6 meses
                </button>
                <button class="btn-comparacion" onclick="cambiarTipoGraficaSupervisores(12)">
                  12 meses
                </button>
              </div>
        
              <div style="display:flex; gap:12px; align-items:center; flex-wrap:wrap;">
                <label style="display:flex; gap:8px; align-items:center; cursor:pointer; user-select:none; font-weight:700; color:#2c3e50;">
                  <input type="checkbox" id="chkSupMostrarFilaAsesores" onchange="actualizarTablaComparacionSupervisores()" />
                  <span>Ver Asesores</span>
                </label>
        
                <label style="display:flex; gap:8px; align-items:center; cursor:pointer; user-select:none; font-weight:700; color:#2c3e50;">
                  <input type="checkbox" id="chkSupMostrarFilaCartera" onchange="actualizarTablaComparacionSupervisores()" />
                  <span>Ver Cartera</span>
                </label>

                <label style="display:flex; gap:8px; align-items:center; cursor:pointer; user-select:none; font-weight:700; color:#2c3e50; text-transform:uppercase;">
                  <input type="checkbox" id="chkSupIncluirMesSeleccionado" onchange="actualizarTablaComparacionSupervisores(); actualizarGraficaComparacionSupervisores();" />
                  <span>Incluir mes seleccionado</span>
                </label>
              </div>
        
            </div>
          </div>
        `;
        
        // Insertar controles ANTES de la tabla
        resultadosDiv.innerHTML = controlesHTML;
        
        // Crear div para tabla
        const tablaContainer = document.createElement('div');
        tablaContainer.id = 'tablaComparacionSupervisores';
        resultadosDiv.appendChild(tablaContainer);
        
        // Crear div para gráfica
        const graficaContainer = document.createElement('div');
        graficaContainer.id = 'graficaContainerSupervisores';
        resultadosDiv.appendChild(graficaContainer);
        
        // Actualizar tabla y gráfica
        actualizarTablaComparacionSupervisores();
        actualizarGraficaComparacionSupervisores();
        
        resultadosDiv.style.display = 'block';
    }}
    
    function generarGraficaComparacionSupervisores(meses, supervisores) {{
        let graficaDiv = document.getElementById('graficaComparacionSupervisores');
        if (!graficaDiv) {{
            const resultadosDiv = document.getElementById('resultadosComparacionSupervisores');
            if (resultadosDiv) {{
                graficaDiv = document.createElement('div');
                graficaDiv.id = 'graficaComparacionSupervisores';
                graficaDiv.className = 'grafica-container';
                graficaDiv.innerHTML = `
                    <h3>📈 Evolución de Desempeño</h3>
                    <canvas id="graficaDesempenioSupervisores" width="1200" height="400"></canvas>
                `;
                resultadosDiv.appendChild(graficaDiv);
            }}
        }}
        
        const canvas = document.getElementById('graficaDesempenioSupervisores');
        if (!canvas) return;
        
        const ctx = canvas.getContext('2d');

        if (chartInstanceSupervisores) {{
            chartInstanceSupervisores.destroy();
            chartInstanceSupervisores = null;
        }}

        ctx.clearRect(0, 0, canvas.width, canvas.height);

        const datasets = [];
        const colores = [
            '#FF6384', '#36A2EB', '#FFCE56', '#8AC926', 
            '#9966FF', '#FF9F40', '#4BC0C0', '#1982C4'
        ];
        
        let maxValor = 0;
        let minValor = 0;
        
        supervisores.forEach((supervisor, index) => {{
            const datosSupervisor = [];
            const borderColor = colores[index % colores.length];
            const backgroundColor = borderColor + '40'; // 40% de opacidad para área
            
            meses.forEach(mes => {{
                const datosSupervisorMes = datosSupervisores[supervisor];
                if (datosSupervisorMes && datosSupervisorMes[mes]) {{
                    const valor = datosSupervisorMes[mes].porcentaje_vs_meta_super;
                    const valorNum = (valor !== null && valor !== undefined && !isNaN(valor)) 
                        ? parseFloat(valor) 
                        : null;
                    datosSupervisor.push(valorNum);
                    
                    // Calcular máximo y mínimo
                    if (valorNum !== null && !isNaN(valorNum)) {{
                        maxValor = Math.max(maxValor, valorNum);
                        minValor = Math.min(minValor, valorNum);
                    }}
                }} else {{
                    datosSupervisor.push(null);
                }}
            }});
            
            // GRÁFICA DE LÍNEAS CON ÁREA Y LÍNEAS DISCONTINUAS
            datasets.push({{
                label: supervisor,
                data: datosSupervisor,
                borderColor: borderColor,
                backgroundColor: backgroundColor,
                borderWidth: 3,
                fill: true, // ¡ÁREA COLOREA!
                tension: 0.4,
                pointRadius: 6,
                pointHoverRadius: 8,
                type: 'line',
                spanGaps: true,
                // LÍNEAS DISCONTINUAS PARA DATOS FALTANTES
                segment: {{
                    borderColor: ctx => {{
                        const p0Null = ctx.p0.parsed.y === null;
                        const p1Null = ctx.p1.parsed.y === null;
                        return (p0Null || p1Null) ? borderColor + '60' : borderColor;
                    }},
                    borderDash: ctx => {{
                        const p0Null = ctx.p0.parsed.y === null;
                        const p1Null = ctx.p1.parsed.y === null;
                        return (p0Null || p1Null) ? [5, 3] : [];
                    }}
                }},
                // PUNTOS DIFERENCIADOS
                pointBackgroundColor: datosSupervisor.map(valor => 
                    valor === null ? '#95a5a6' : borderColor
                ),
                pointBorderColor: datosSupervisor.map(valor => 
                    valor === null ? '#95a5a6' : '#ffffff'
                ),
                pointBorderWidth: 2
            }});
        }});
        
        // Agregar línea de meta (100%)
        datasets.push({{
            label: 'Meta (100%)',
            data: Array(meses.length).fill(100),
            borderColor: '#2ecc71',
            backgroundColor: 'transparent',
            borderWidth: 2,
            borderDash: [5, 5],
            pointRadius: 0,
            fill: false,
            tension: 0
        }});

        maxValor = Math.max(maxValor, 100);

        let yMin = 0;
        let yMax = 100; // Por defecto 100%
        
        if (maxValor > 0) {{
            yMax = Math.max(100, Math.ceil(maxValor / 10) * 10);
        }}

        if (minValor < 0) {{
            yMin = Math.floor(minValor / 10) * 10;
        }}

        const labels = meses.map(mes => {{
            const [mesNombre, año] = mes.split('_');
            return `${{mesNombre.substring(0, 3)}} ${{año}}`;
        }});

        chartInstanceSupervisores = new Chart(ctx, {{
            type: 'line',
            data: {{
                labels: labels,
                datasets: datasets
            }},
            options: {{
                responsive: false,
                maintainAspectRatio: false,
                plugins: {{
                    title: {{
                        display: false,
                        text: 'Evolución del Porcentaje vs Meta Supervisor - Equipos',
                        font: {{ size: 16 }}
                    }},
                    tooltip: {{
                        mode: 'index',
                        intersect: false,
                        backgroundColor: 'rgba(0, 0, 0, 0.8)',
                        titleFont: {{
                            size: 14,
                            weight: 'bold'
                        }},
                        bodyFont: {{
                            size: 13,
                            weight: 'normal'
                        }},
                        padding: 12,
                        cornerRadius: 8,
                        filter: function(item) {{
                            return item.dataset?.label !== 'Meta (100%)';
                        }},
                        itemSort: function(a, b) {{
                            const ay = Number(a.parsed?.y);
                            const by = Number(b.parsed?.y);
                            const aVal = Number.isFinite(ay) ? ay : -Infinity;
                            const bVal = Number.isFinite(by) ? by : -Infinity;
                            return bVal - aVal;
                        }},
                        callbacks: {{
                            label: function(context) {{
                                const supervisor = context.dataset.label;
                                const valor = context.parsed.y;

                                if (supervisor === 'Meta (100%)') return null;
                                if (valor === null) return `${{supervisor}}: Sin datos`;
                                
                                const [mesAbrev, año] = context.chart.data.labels[context.dataIndex].split(' ');
                                const meses = {{'ENE':'ENERO','FEB':'FEBRERO','MAR':'MARZO','ABR':'ABRIL','MAY':'MAYO','JUN':'JUNIO','JUL':'JULIO','AGO':'AGOSTO','SEP':'SETIEMBRE','OCT':'OCTUBRE','NOV':'NOVIEMBRE','DIC':'DICIEMBRE'}};
                                const mesKey = `${{meses[mesAbrev]}}_${{año}}`;
                                const datos = datosSupervisores[supervisor]?.[mesKey];
                                
                                const formato = (v) => 'S/ ' + (v||0).toLocaleString('es-PE',{{minimumFractionDigits:2}});
                                
                                return [
                                    `${{supervisor}}: ${{valor.toFixed(2)}}%`,
                                    `Meta: ${{formato(datos?.meta_super)}}`,
                                    `Recupero: ${{formato(datos?.total_recupero)}}`
                                ];
                            }}
                        }}
                    }},
                    legend: {{
                        position: 'top',
                        labels: {{ font: {{ size: 14 }} }},
                        onClick: function (e, legendItem, legend) {{
                            const index = legendItem.datasetIndex;
                            const ci = legend.chart;
                            if (ci.isDatasetVisible(index)) {{
                                ci.hide(index);
                            }} else {{
                                ci.show(index);
                            }}
                        }}
                    }}
                }},
                scales: {{
                    y: {{
                        beginAtZero: true,
                        title: {{
                            display: false,
                            text: 'Porcentaje vs Meta Supervisor (%)'
                        }},
                        ticks: {{
                            font: {{ size: 16 }},
                            callback: function(value) {{
                                return value + '%';
                            }},
                            stepSize: calcularStepSize(yMax)
                        }},
                        min: yMin,
                        max: yMax,
                        suggestedMin: yMin,
                        suggestedMax: yMax
                    }},
                    x: {{
                        title: {{
                            display: false,
                            text: 'Meses'
                        }},
                        ticks: {{
                            font: {{ size: 15 }}
                        }}
                    }}
                }},
                interaction: {{
                    mode: 'nearest',
                    axis: 'x',
                    intersect: false
                }}
            }}
        }});
    }}

    function cambiarTipoGraficaSupervisores(cantidadMeses) {{
        mesesAMostrar = cantidadMeses;

        document.querySelectorAll('#resultadosComparacionSupervisores .btn-comparacion').forEach(btn => {{
            btn.classList.remove('activo');
        }});
        event.target.classList.add('activo');

        if (supervisoresSeleccionados.size > 0) {{
            actualizarTablaComparacionSupervisores();
            actualizarGraficaComparacionSupervisores();
        }}
    }}

    function actualizarGraficaComparacionSupervisores() {{
        if (supervisoresSeleccionados.size === 0) return;
        
        // Obtener meses ordenados
        let mesesOrdenados = Object.keys(datosMeses).sort((a, b) => {{
            const [mesA, añoA] = a.split('_');
            const [mesB, añoB] = b.split('_');
            const fechaA = new Date(añoA, obtenerNumeroMes(mesA));
            const fechaB = new Date(añoB, obtenerNumeroMes(mesB));
            return fechaB - fechaA;
        }});
        
        mesesOrdenados = _filtrarMesesComparacionSupervisores(mesesOrdenados);
        
        if (mesesAMostrar > 0 && mesesOrdenados.length > mesesAMostrar) {{
          mesesOrdenados = mesesOrdenados.slice(0, mesesAMostrar);
        }}
        
        const mesesParaGrafica = [...mesesOrdenados].reverse();
        generarGraficaComparacionSupervisores(mesesParaGrafica, Array.from(supervisoresSeleccionados));
    }}

    // ========== FUNCIONES PARA ACTUALIZAR TABLAS ==========
    function actualizarTablaComparacionSupervisores() {{
        if (supervisoresSeleccionados.size === 0) return;
    
        const tablaDiv = document.getElementById('tablaComparacionSupervisores');
        if (!tablaDiv) return;
    
        // Obtener meses ordenados (más reciente primero)
        let mesesOrdenados = Object.keys(datosMeses).sort((a, b) => {{
            const [mesA, añoA] = a.split('_');
            const [mesB, añoB] = b.split('_');
            const fechaA = new Date(añoA, obtenerNumeroMes(mesA));
            const fechaB = new Date(añoB, obtenerNumeroMes(mesB));
            return fechaB - fechaA;
        }});
        mesesOrdenados = _filtrarMesesComparacionSupervisores(mesesOrdenados);
        
        if (mesesAMostrar > 0 && mesesOrdenados.length > mesesAMostrar) {{
          mesesOrdenados = mesesOrdenados.slice(0, mesesAMostrar);
        }}
    
        // Mostrar de más antiguo a más reciente
        const mesesParaTabla = [...mesesOrdenados].reverse();
        const denom = (mesesParaTabla.length > 0) ? mesesParaTabla.length : 1;
    
        // Checkbox
        const chkAses = document.getElementById('chkSupMostrarFilaAsesores');
        const chkCart = document.getElementById('chkSupMostrarFilaCartera');
        const mostrarFilaAsesores = chkAses ? chkAses.checked : false;
        const mostrarFilaCartera  = chkCart ? chkCart.checked : false;
    
        let html = '<div class="contenedor-tabla-scroll"><table id="tablaComparacionSupervisoresTabla" class="tabla-comparacion">';
    
        // Header (clickeable)
        html += '<thead><tr>';
        html += '<th style="cursor:pointer;" onclick="ordenarTablaComparacionSupervisores(0, `txt`)">Supervisor</th>';
        
        mesesParaTabla.forEach((mes, i) => {{
          const col = i + 1;
          html += `<th style="cursor:pointer;" onclick="ordenarTablaComparacionSupervisores(${{col}}, \`num\`)">${{mes.replace('_',' ')}}</th>`;
        }});
        
        const colProm = mesesParaTabla.length + 1;
        html += `<th style="cursor:pointer;" onclick="ordenarTablaComparacionSupervisores(${{colProm}}, \`num\`)">Promedio</th>`;
        html += '</tr></thead><tbody>';
    
        // Filas por supervisor seleccionado
        supervisoresSeleccionados.forEach(supervisor => {{
            // ===== FILA 1: % vs Meta Supervisor (SIN COLORES) + Promedio =====
            html += `<tr data-grupo="sup"><td data-value="${{supervisor}}"><strong>${{supervisor}}</strong></td>`;
    
            let suma = 0;
            let mesesConData = 0;
            
            mesesParaTabla.forEach(mes => {{
                const datosSupervisor = datosSupervisores[supervisor];
                const datosMes = (datosSupervisor && datosSupervisor[mes]) ? datosSupervisor[mes] : null;
            
                if (datosMes) {{
                    const val = Number(datosMes.porcentaje_vs_meta_super);
                    const v = (isFinite(val)) ? val : 0;
            
                    suma += v;
                    mesesConData += 1;
            
                    html += `<td data-value="${{v}}">${{v.toFixed(2)}}%</td>`;
                }} else {{
                    html += '<td style="color: #999; font-style: italic;">Sin datos</td>';
                }}
            }});
            
            const promedio = (mesesConData > 0) ? (suma / mesesConData) : 0;
            const promTxt = (mesesConData > 0) ? `${{promedio.toFixed(2)}}%` : 'Sin datos';
            html += `<td data-value="${{mesesConData > 0 ? promedio : -1}}"><strong>${{promTxt}}</strong></td>`;
            html += '</tr>';
    
            // ===== FILA 2: Total de Asesores + celda extra Promedio =====
            if (mostrarFilaAsesores) {{
                html += `<tr data-subfila="1" class="fila-equipo fila-asesores"><td class="equipo-label">Asesores:</td>`;
                mesesParaTabla.forEach(mes => {{
                    const datosSupervisor = datosSupervisores[supervisor];
                    const datosMes = (datosSupervisor && datosSupervisor[mes]) ? datosSupervisor[mes] : null;
    
                    if (datosMes) {{
                        const cantidad = Number(datosMes.cantidad_asesores) || 0;
                        html += `<td class="equipo-supervisor">${{cantidad}}</td>`;
                    }} else {{
                        html += '<td class="sin-equipo">-</td>';
                    }}
                }});
                html += '<td class="sin-equipo">-</td>';
                html += '</tr>';
            }}
    
            // ===== FILA 3: Cartera + celda extra Promedio =====
            if (mostrarFilaCartera) {{
                html += `<tr data-subfila="1" class="fila-equipo fila-cartera"><td class="equipo-label">Cartera:</td>`;
                mesesParaTabla.forEach(mes => {{
                    const datosSupervisor = datosSupervisores[supervisor];
                    const datosMes = (datosSupervisor && datosSupervisor[mes]) ? datosSupervisor[mes] : null;
    
                    if (datosMes) {{
                        const cartera = datosMes.cartera || 'No definida';
    
                        let segmentoClass = '';
                        if (cartera.includes('PREMIER') || cartera.includes('PRIME') || cartera.includes('PREMIUM')) {{
                            segmentoClass = 'segmento-premier';
                        }} else if (cartera.includes('EMPRESARIAL') || cartera.includes('BUSINESS')) {{
                            segmentoClass = 'segmento-empresarial';
                        }} else if (cartera.includes('MASIVO') || cartera.includes('MASS')) {{
                            segmentoClass = 'segmento-masivo';
                        }} else if (cartera.includes('PYME') || cartera.includes('SME')) {{
                            segmentoClass = 'segmento-pyme';
                        }}
    
                        html += `<td class="equipo-supervisor ${{segmentoClass}}">${{cartera}}</td>`;
                    }} else {{
                        html += '<td class="sin-equipo">-</td>';
                    }}
                }});
                html += '<td class="sin-equipo">-</td>';
                html += '</tr>';
            }}
        }});
    
        html += '</table></div>';
        tablaDiv.innerHTML = html;
    }}

    function limpiarBusquedaSupervisores() {{
        supervisoresSeleccionados.clear();
        actualizarSupervisoresSeleccionados();

        const selectorSegmento = document.getElementById('selectorSegmentoSupervisores');
        if (selectorSegmento) selectorSegmento.value = '';
        
        // Destruir gráfica si existe
        if (chartInstanceSupervisores) {{
            chartInstanceSupervisores.destroy();
            chartInstanceSupervisores = null;
        }}
        
        // Ocultar resultados
        const resultadosDiv = document.getElementById('resultadosComparacionSupervisores');
        if (resultadosDiv) resultadosDiv.style.display = 'none';
    }}
    


    function generarGraficaIncrementoTotal(periodoCompleto, canvasId = 'graficaIncrementoTotal', view = 'full') {{
        const canvas = document.getElementById(canvasId);
        const dataLabelsPluginTotal = _buildDataLabelsPluginTotal(view);
        if (!canvas) {{
            console.log('Canvas no encontrado');
            return;
        }}
        
        const ctx = canvas.getContext('2d');
        
        // Destruir gráfica anterior si existe
        _destroyChartByCanvasId(canvasId);
        
        // Limpiar canvas
        ctx.clearRect(0, 0, canvas.width, canvas.height);
        
        // ========== 1. CALCULAR META TOTAL DEL MES ==========
        let metaTotalMes = 0;
        const supervisoresUnicos = new Set();
        
        Object.keys(datosSupervisores).forEach(supervisor => {{
            const datosMes = datosSupervisores[supervisor][periodoCompleto];
            if (datosMes && datosMes.meta_super !== undefined && !supervisoresUnicos.has(supervisor)) {{
                const metaSupervisor = parseFloat(datosMes.meta_super) || 0;
                metaTotalMes += metaSupervisor;
                supervisoresUnicos.add(supervisor);
            }}
        }});
        
        // Si no hay meta total, no hay datos para graficar
        if (metaTotalMes === 0) {{
            console.log('No hay meta total para el periodo:', periodoCompleto);
            mostrarMensajeSinDatos(canvas, periodoCompleto);
            return;
        }}
        
        // ========== 2. RECOPILAR TODAS LAS FECHAS ÚNICAS ==========
        const todasFechas = new Set();
        const recuperoAcumuladoPorFecha = {{}};
        
        Object.keys(datosSupervisores).forEach(supervisor => {{
            const datosMes = datosSupervisores[supervisor][periodoCompleto];
            if (!datosMes) return;
            
            const datosDiarios = datosMes.datos_diarios_supervisor || {{}};
            
            Object.keys(datosDiarios).forEach(fecha => {{
                todasFechas.add(fecha);
                
                const recuperoSupervisor = parseFloat(datosDiarios[fecha]) || 0;
                
                if (!recuperoAcumuladoPorFecha[fecha]) {{
                    recuperoAcumuladoPorFecha[fecha] = 0;
                }}
                
                recuperoAcumuladoPorFecha[fecha] += recuperoSupervisor;
            }});
        }});
        
        // Si no hay fechas con datos, no hay nada que graficar
        if (todasFechas.size === 0) {{
            console.log('No hay datos diarios para el periodo:', periodoCompleto);
            mostrarMensajeSinDatos(canvas, periodoCompleto);
            return;
        }}
        
        // ========== 3. ORDENAR FECHAS CRONOLÓGICAMENTE ==========
        const fechasOrdenadas = Array.from(todasFechas).sort((a, b) => 
            convertirFechaDiaria(a) - convertirFechaDiaria(b)
        );
        
        console.log(`📊 Total fechas disponibles: ${{fechasOrdenadas.length}}`);
        
        // ========== 4. CALCULAR ALCANCE ACUMULADO ==========
        const datosValidosParaGrafica = [];  // Datos para la línea
        const alcancesPorcentaje = [];
        
        // Inicializar todas las fechas
        fechasOrdenadas.forEach(fecha => {{
            const recuperoAcumulado = recuperoAcumuladoPorFecha[fecha] || 0;
            
            const alcanceDia = metaTotalMes > 0 ? (recuperoAcumulado / metaTotalMes) * 100 : 0;
            
            // SOLO agregar a datos válidos si hay datos reales (recupero > 0)
            if (recuperoAcumulado > 0) {{
                datosValidosParaGrafica.push(alcanceDia);
                alcancesPorcentaje.push(alcanceDia);
            }} else {{
                // Para días sin datos, poner null (la línea no se dibujará aquí)
                datosValidosParaGrafica.push(null);
                alcancesPorcentaje.push(0);
            }}
        }});
        
        // ========== 5. ENCONTRAR EL ÚLTIMO DÍA CON DATOS REALES ==========
        let ultimoDiaConDatos = -1;
        for (let i = fechasOrdenadas.length - 1; i >= 0; i--) {{
            const fecha = fechasOrdenadas[i];
            const recuperoDia = recuperoAcumuladoPorFecha[fecha] || 0;
            if (recuperoDia > 0) {{
                ultimoDiaConDatos = i;
                break;
            }}
        }}
        
        if (ultimoDiaConDatos === -1) {{
            ultimoDiaConDatos = fechasOrdenadas.length - 1;
        }}
        
        console.log(`📊 Fechas con datos > 0: hasta día ${{ultimoDiaConDatos + 1}}`);
        
        // ========== 6. CREAR LA GRÁFICA DE LÍNEA CON ÁREA ==========
        _applyCanvasSize(canvas, view);
        canvas.style.display = 'block';
        canvas.style.margin = '0 auto';
        canvas.style.maxWidth = '100%';
        
        // USAR TODAS LAS FECHAS para las etiquetas (CAMBIO PRINCIPAL)
        const labelsParaGrafica = fechasOrdenadas.map(fecha => {{
            if (fecha.includes('-')) {{
                return fecha.split('-')[0];
            }}
            return fecha;
        }});
        
        // COLORES PARA GRÁFICA DE LÍNEA (MORADO)
        const colorLinea = '#9b59b6';
        const colorArea = 'rgba(155, 89, 182, 0.2)';

        let options = {{
            responsive: true,
            maintainAspectRatio: false,
            plugins: {{
                title: {{
                    display: true,
                    text: `            `,
                    font: {{ size: 22, weight: 'bold' }},
                    padding: {{ bottom: 25 }}
                }},
                tooltip: {{
                    mode: 'nearest',
                    intersect: false,
                    backgroundColor: 'rgba(0, 0, 0, 0.9)',
                    titleFont: {{ size: 18, weight: 'bold' }},
                    bodyFont: {{ size: 17 }},
                    padding: 15,
                    cornerRadius: 10,
                    callbacks: {{
                        title: function(context) {{
                            const index = context[0].dataIndex;
                            return 'Día ' + (index + 1) + ': ' + fechasOrdenadas[index];
                        }},
                        label: function(context) {{
                            const index = context.dataIndex;
                            const alcance = context.parsed.y;
                            const recuperoAcumulado = recuperoAcumuladoPorFecha[fechasOrdenadas[index]] || 0;
                            
                            // Si no hay datos para este día
                            if (index > ultimoDiaConDatos || alcance === null) {{
                                return '📭 Sin datos de recupero';
                            }}
                            
                            // Calcular incremento del día
                            let incrementoDia = 0;
                            if (index > 0) {{
                                const alcanceAnterior = context.chart.data.datasets[0].data[index - 1] || 0;
                                incrementoDia = (alcance - alcanceAnterior);
                            }} else {{
                                incrementoDia = alcance;
                            }}
                            
                            return [
                                '🎯 Alcance Acumulado: ' + alcance.toFixed(2) + '%',
                                '📈 Alcance del día: ' + incrementoDia.toFixed(2) + '%',
                                '💰 Recupero Acumulado: ' + recuperoAcumulado.toLocaleString('es-PE', {{ 
                                    minimumFractionDigits: 0,
                                    maximumFractionDigits: 0 
                                }})
                            ];
                        }}
                    }}
                }},
                legend: {{ display: false }}
            }},
            scales: {{
                x: {{
                    title: {{ 
                        display: false, 
                        text: 'Días del Mes',
                        font: {{ size: 14, weight: 'bold' }}
                    }},
                    ticks: {{ 
                        font: {{ size: 16 }},
                        maxRotation: 0,
                        minRotation: 0,
                        autoSkip: false,
                        callback: function(value, index, values) {{
                            // Colores diferentes para días con/sin datos
                            const tieneDatos = datosValidosParaGrafica[index] !== null && 
                                              datosValidosParaGrafica[index] > 0;
                            this.fontColor = tieneDatos ? '#2c3e50' : '#bdc3c7';
                            return this.getLabelForValue(value);
                        }}
                    }},
                    grid: {{ 
                        display: true,
                        color: 'rgba(0, 0, 0, 0.05)'
                    }}
                }},
                y: {{
                    beginAtZero: true,
                    title: {{ 
                        display: false, 
                        text: 'Alcance Acumulado (%)',
                        font: {{ size: 16, weight: 'bold' }}
                    }},
                    ticks: {{
                        callback: function(value) {{ return value + '%'; }},
                        stepSize: 10,
                        font: {{ size: 16 }}
                    }},
                    min: 0,
                    suggestedMax: function() {{
                        // Calcular máximo solo de los datos válidos
                        const valoresValidos = datosValidosParaGrafica.filter(v => v !== null && v > 0);
                        if (valoresValidos.length === 0) return 100;
                        const maxValor = Math.max(...valoresValidos);
                        return Math.max(100, Math.ceil(maxValor / 10) * 10);
                    }}(),
                    grid: {{
                        display: true,
                        color: 'rgba(0, 0, 0, 0.05)'
                    }}
                }}
            }},
            interaction: {{
                mode: 'nearest',
                axis: 'x',
                intersect: false
            }},
            elements: {{
                line: {{
                    tension: 0.4  // Suavizado de la línea
                }},
                point: {{
                    hoverBackgroundColor: function(context) {{
                        const value = context.dataset.data[context.dataIndex];
                        return value === null || value === 0 ? 'transparent' : colorLinea;
                    }}
                }}
            }}
        }};

        options = _tuneOptionsForView(options, view);

        try {{
            const chart = new Chart(ctx, {{
                type: 'line',
                data: {{
                    labels: labelsParaGrafica,  // TODAS las fechas
                    datasets: [{{
                        label: 'Alcance Acumulado (%)',
                        data: datosValidosParaGrafica,  // Datos: válidos donde hay > 0, null donde no
                        backgroundColor: colorArea,
                        borderColor: colorLinea,
                        borderWidth: 3,
                        fill: true,
                        tension: 0.4,
                        pointRadius: function(context) {{
                            // Puntos más grandes para datos válidos, invisibles para nulos
                            const value = context.dataset.data[context.dataIndex];
                            return value === null || value === 0 ? 0 : 6;
                        }},
                        pointHoverRadius: function(context) {{
                            const value = context.dataset.data[context.dataIndex];
                            return value === null || value === 0 ? 0 : 10;
                        }},
                        pointBackgroundColor: function(context) {{
                            const value = context.dataset.data[context.dataIndex];
                            return value === null || value === 0 ? 'transparent' : colorLinea;
                        }},
                        pointBorderColor: function(context) {{
                            const value = context.dataset.data[context.dataIndex];
                            return value === null || value === 0 ? 'transparent' : '#ffffff';
                        }},
                        pointBorderWidth: 2,
                        spanGaps: false
                    }}]
                }},
                options,
                plugins: [dataLabelsPluginTotal]
            }});
            _storeChart(canvasId, chart);
            
        }} catch (error) {{
            console.error("Error al crear la gráfica:", error);
            // Mostrar mensaje de error en el canvas
            ctx.fillStyle = '#e74c3c';
            ctx.font = 'bold 20px Arial';
            ctx.textAlign = 'center';
            ctx.fillText('Error al generar gráfica', canvas.width / 2, canvas.height / 2);
        }}
    }}
    
    // Función auxiliar para mostrar mensaje cuando no hay datos
    function mostrarMensajeSinDatos(canvas, periodoCompleto) {{
        const ctx = canvas.getContext('2d');
        ctx.clearRect(0, 0, canvas.width, canvas.height);
        
        // Configurar el canvas para mensaje
        canvas.width = 1300;
        canvas.height = 200;
        canvas.style.display = 'block';
        canvas.style.margin = '0 auto';
        canvas.style.maxWidth = '100%';
        
        // Dibujar mensaje
        ctx.fillStyle = '#95a5a6';
        ctx.font = 'bold 24px Arial';
        ctx.textAlign = 'center';
        ctx.fillText('📊 No hay datos de recupero acumulado disponibles', canvas.width / 2, canvas.height / 2 - 20);
        
        ctx.fillStyle = '#7f8c8d';
        ctx.font = '18px Arial';
        ctx.fillText(`Periodo: ${{periodoCompleto.replace('_', ' ')}}`, canvas.width / 2, canvas.height / 2 + 20);
    }}

    function actualizarVistaRecuperos() {{
        const mesSeleccionado = document.getElementById('selectorMes').value;
        const añoSeleccionado = document.getElementById('selectorAño').value;
        const periodoCompleto = `${{mesSeleccionado}}_${{añoSeleccionado}}`;
        
        if (!datosMeses[periodoCompleto]) {{
            document.getElementById('graficas-recuperos').innerHTML = 
                '<div class="seccion-vacia"><h3>No hay datos disponibles para este periodo</h3></div>';
            return;
        }}
        
        // Calcular y mostrar estadísticas
        calcularEstadisticasRecuperos(periodoCompleto);
        
        // Generar gráficos
        generarGraficasRecuperos(periodoCompleto);
    }}

    // CALCULO DE DATA ESTADÍSTICA PARA ALCANCES
    function calcularEstadisticasRecuperos(periodoCompleto) {{
        // 1. OBTENER DATOS DE SUPERVISORES
        const supervisoresData = datosSupervisores;
        
        // Filtrar supervisores que tienen datos para este periodo
        const supervisoresConDatos = Object.keys(supervisoresData).filter(supervisor => 
            supervisorEnCanal(supervisor, periodoCompleto) && supervisoresData[supervisor] && supervisoresData[supervisor][periodoCompleto]
        );
        
        if (supervisoresConDatos.length === 0) {{
            console.log(`⚠️ No hay datos de supervisores para: ${{periodoCompleto}}`);
            const statsElement = document.getElementById('estadisticas-recuperos');
            if (!statsElement) return;
            statsElement.innerHTML = '';
            return;
        }}
        
        // 2. CALCULAR META TOTAL DEL MES (SUMA DE METAS DE SUPERVISORES)
        let metaTotalMes = 0;
        let recuperoTotalActual = 0;
        
        supervisoresConDatos.forEach(supervisor => {{
            const datosSupervisor = supervisoresData[supervisor][periodoCompleto];
            if (datosSupervisor) {{
                metaTotalMes += datosSupervisor.meta_super || 0;
                recuperoTotalActual += datosSupervisor.total_recupero || 0;
            }}
        }});
        
        // 3. OBTENER DÍAS LABORABLES (de los datos diarios de SUPERVISORES)
        const todasFechas = new Set();
        
        // Recorrer todos los supervisores para obtener fechas únicas
        supervisoresConDatos.forEach(supervisor => {{
            const datosSupervisor = supervisoresData[supervisor][periodoCompleto];
            if (datosSupervisor && datosSupervisor.datos_diarios_supervisor) {{
                Object.keys(datosSupervisor.datos_diarios_supervisor).forEach(fecha => {{
                    // Solo agregar fechas que tengan valor > 0 o que sean relevantes
                    const valor = datosSupervisor.datos_diarios_supervisor[fecha];
                    if (valor !== null && valor !== undefined) {{
                        todasFechas.add(fecha);
                    }}
                }});
            }}
        }});
        
        // Si no hay fechas en datos diarios, intentar con alcance acumulado
        if (todasFechas.size === 0) {{
            supervisoresConDatos.forEach(supervisor => {{
                const datosSupervisor = supervisoresData[supervisor][periodoCompleto];
                if (datosSupervisor && datosSupervisor.alcance_acumulado_diario) {{
                    Object.keys(datosSupervisor.alcance_acumulado_diario).forEach(fecha => {{
                        todasFechas.add(fecha);
                    }});
                }}
            }});
        }}
        
        const fechasOrdenadas = Array.from(todasFechas).sort((a, b) => 
            convertirFechaDiaria(a) - convertirFechaDiaria(b)
        );
        
        const diasRegistradosBD = fechasOrdenadas.length;
        const diasLaborables = 0;
        
        // 4. CALCULAR DÍAS TRABAJADOS (con recupero > 0 en datos diarios de supervisores)
        let diasTrabajados = 0;
        
        if (diasLaborables > 0) {{
            // Para cada fecha, verificar si hubo recupero en algún supervisor
            fechasOrdenadas.forEach(fecha => {{
                let totalRecuperoDia = 0;
                
                // Sumar recupero de todos los supervisores para esta fecha
                supervisoresConDatos.forEach(supervisor => {{
                    const datosSupervisor = supervisoresData[supervisor][periodoCompleto];
                    if (datosSupervisor && datosSupervisor.datos_diarios_supervisor) {{
                        // Obtener el INCREMENTO del día, no el acumulado
                        // Para esto necesitamos calcular la diferencia
                        const recuperoDia = datosSupervisor.datos_diarios_supervisor[fecha] || 0;
                        
                        // Si es acumulado, calcular diferencia con el día anterior
                        if (fechasOrdenadas.indexOf(fecha) > 0) {{
                            const fechaAnterior = fechasOrdenadas[fechasOrdenadas.indexOf(fecha) - 1];
                            const recuperoAnterior = datosSupervisor.datos_diarios_supervisor[fechaAnterior] || 0;
                            totalRecuperoDia += (recuperoDia - recuperoAnterior);
                        }} else {{
                            // Primer día
                            totalRecuperoDia += recuperoDia;
                        }}
                    }}
                }});
                
                // Si hubo recupero en algún supervisor, contar como día trabajado
                if (totalRecuperoDia > 0) {{
                    diasTrabajados++;
                }}
            }});
        }}
        
        // 5. CALCULAR ALCANCE ACTUAL
        const alcanceActual = metaTotalMes > 0 ? 
            parseFloat(((recuperoTotalActual / metaTotalMes) * 100).toFixed(2)) : 0;
        
        // 6. CALCULAR META DIARIA
        const metaDiaria = diasLaborables > 0 ? metaTotalMes / diasLaborables : 0;
        
        // 7. CALCULAR PROMEDIO DIARIO REAL
        const promedioDiarioReal = diasTrabajados > 0 ? recuperoTotalActual / diasTrabajados : 0;
        
        // 8. CALCULAR PROYECCIONES
        const diasRestantes = Math.max(0, diasLaborables - diasTrabajados);
        const recuperoProyectado = promedioDiarioReal * diasRestantes;
        const recuperoTotalProyectado = recuperoTotalActual + recuperoProyectado;
        const eficienciaDiaria = metaTotalMes > 0 ? (recuperoTotalProyectado / metaTotalMes) * 100 : 0;
        
        // 9. Calcular eficiencia vs meta diaria
        const eficienciaVsMetaDiaria = metaDiaria > 0 ? (promedioDiarioReal / metaDiaria) * 100 : 0;
        
        // 10. MOSTRAR EN INTERFAZ
        mostrarEstadisticasUnificada(
            periodoCompleto,
            {{
                metaTotalMes,
                recuperoTotalActual,
                alcanceActual,
                eficienciaDiaria,
                diasLaborables,
                diasRegistradosBD,
                diasTrabajados,
                metaDiaria,
                promedioDiarioReal,
                eficienciaVsMetaDiaria,
                recuperoProyectado,
                recuperoTotalProyectado,
                diasRestantes
            }},
            'global'
        );
    }}

    function generarGraficasRecuperos(periodoCompleto) {{
        const graficasContainer = document.getElementById('graficas-recuperos');
        if (!graficasContainer) return;
        
            graficasContainer.innerHTML = `
              <div style="text-align: center; margin-bottom: 30px; display:flex; justify-content:center; gap:12px; flex-wrap:wrap;">
                  <button class="boton-exportar-excel" onclick="exportarRecupero4MesesDiaSupervisor('TODOS')">
                      📈 Exportar Recupero 4M
                  </button>
              </div>

            <!-- ===================== TABLA COMPARATIVA 4 MESES (ALCANCES) ===================== -->
            <div id="cardTablaComparativa4M" style="margin-top: 16px; background:#fff; border-radius:14px; padding:16px; box-shadow: 0 5px 15px rgba(0,0,0,0.08);">
              <div style="display:flex; justify-content:space-between; align-items:flex-end; gap:12px; flex-wrap:wrap;">
                  <div>
                    <h3 style="margin:0; color:#2c3e50; font-size: 1.25rem; font-weight: 800;">ANALISIS DE ALCANCES</h3>
                  </div>

                  <div style="display:flex; gap:10px; flex-wrap:wrap; align-items:center;">

                      <button
                          type="button"
                          class="boton-busqueda"
                          onclick="abrirModalSimulacionAlcances('${{periodoCompleto}}')"
                          style="padding:10px 14px;">
                          Simular
                      </button>

                      <button
                          type="button"
                          class="boton-busqueda"
                          id="btnTabla4MRefrescar"
                          style="padding:10px 14px;">
                          🔄 Actualizar
                      </button>

                      <button
                          type="button"
                          class="boton-busqueda"
                          onclick="copiarVistaAlcances4M()"
                          style="padding:10px 14px; background: linear-gradient(135deg, #34495e, #2c3e50);">
                          Copiar vista
                      </button>
                  </div>
              </div>
            
              <div class="tabla4m-selectores">
                <div>
                  <label style="font-weight:700; color:#2c3e50;">Periodo 1</label>
                  <select id="selTabla4M_1" style="width:100%; padding:10px 12px; border-radius:10px; border:1px solid #ddd;"></select>
                </div>
                <div>
                  <label style="font-weight:700; color:#2c3e50;">Periodo 2</label>
                  <select id="selTabla4M_2" style="width:100%; padding:10px 12px; border-radius:10px; border:1px solid #ddd;"></select>
                </div>
                <div>
                  <label style="font-weight:700; color:#2c3e50;">Periodo 3</label>
                  <select id="selTabla4M_3" style="width:100%; padding:10px 12px; border-radius:10px; border:1px solid #ddd;"></select>
                </div>
                <div>
                  <label style="font-weight:700; color:#2c3e50;">Periodo 4</label>
                  <select id="selTabla4M_4" style="width:100%; padding:10px 12px; border-radius:10px; border:1px solid #ddd;"></select>
                </div>
              </div>

              <div class="tabla4m-selectores" id="filaTabla4MSupervisores" style="margin-top:10px; display:none;">
                  <div>
                    <label style="font-weight:700; color:#2c3e50;">Supervisor P1</label>
                    <select id="selTabla4M_Sup_1" style="width:100%; padding:10px 12px; border-radius:10px; border:1px solid #ddd;"></select>
                  </div>
                  <div>
                    <label style="font-weight:700; color:#2c3e50;">Supervisor P2</label>
                    <select id="selTabla4M_Sup_2" style="width:100%; padding:10px 12px; border-radius:10px; border:1px solid #ddd;"></select>
                  </div>
                  <div>
                    <label style="font-weight:700; color:#2c3e50;">Supervisor P3</label>
                    <select id="selTabla4M_Sup_3" style="width:100%; padding:10px 12px; border-radius:10px; border:1px solid #ddd;"></select>
                  </div>
                  <div>
                    <label style="font-weight:700; color:#2c3e50;">Supervisor P4 (fijo)</label>
                    <input id="selTabla4M_Sup_4_fixed" type="text" disabled
                           style="width:100%; padding:10px 12px; border-radius:10px; border:1px solid #ddd; background:#f3f5f7;"
                           value="—" />
                  </div>
              </div>

              <!-- MINI GRÁFICAS -->
              <div style="margin-top:12px; display:grid; grid-template-columns: 1fr; gap:12px; align-items:start;">
                  <div id="tablas4MComparativoSupervisores" class="tabla4m-comparativo"></div>
                  <div id="tablas4MSupervisoresTodos" class="tabla4m-supervisores-grid"></div>
                  <div id="tablas4MPrincipales" class="tabla4m-doble">
                    <div>
                      <div class="tabla4m-titulo">RECUPEROS</div>
                      <div class="tabla4m-wrap" style="justify-self:stretch; width:100%; max-width:100%;">
                        <table id="tablaComparativa4MRecupero" class="tabla-comparativa-4m" style="width:100%;">
                          <thead id="theadTabla4MRecupero"></thead>
                          <tbody id="tbodyTabla4MRecupero"></tbody>
                        </table>
                      </div>
                    </div>
                    <div>
                      <div class="tabla4m-titulo">ALCANCES</div>
                      <div class="tabla4m-wrap" style="justify-self:stretch; width:100%; max-width:100%;">
                        <table id="tablaComparativa4MAlcance" class="tabla-comparativa-4m" style="width:100%;">
                          <thead id="theadTabla4MAlcance"></thead>
                          <tbody id="tbodyTabla4MAlcance"></tbody>
                        </table>
                      </div>
                    </div>
                  </div>
                
                  <div id="miniAlcancesRight" style="display:grid; grid-template-columns: repeat(4, minmax(180px, 1fr)); gap:10px; align-content:start;">
                    <div class="mini-chart-card" data-mini="acumulado">
                      <div class="mini-title">ACUMULADO</div>
                      <canvas id="miniGraficaIncrementoTotal"></canvas>
                    </div>
                
                    <div class="mini-chart-card" data-mini="diario">
                      <div class="mini-title">DIARIO</div>
                      <canvas id="miniGraficaRecuperoDiario"></canvas>
                    </div>
                
                    <div class="mini-chart-card" data-mini="equipos">
                      <div class="mini-title">EQUIPOS</div>
                      <canvas id="miniGraficaAlcanceEquipos"></canvas>
                    </div>
                
                    <div class="mini-chart-card" data-mini="anual">
                      <div class="mini-title">ANUAL</div>
                      <canvas id="miniGraficaEvolucionAnualRecuperos"></canvas>
                    </div>
                  </div>
              </div>
                
              <!-- MODAL ALCANCES -->
              <div id="modalGraficaAlcances" style="display:none;">
                  <div id="modalGraficaAlcancesBackdrop"
                       style="position:fixed; inset:0; background:rgba(0,0,0,.55); z-index:9999;"></div>
                
                    <div id="modalGraficaAlcancesOverlay"
                         style="
                            position:fixed; inset:0; z-index:10000;
                            display:flex; align-items:center; justify-content:center;
                            padding:18px;">
                    <div style="
                        width:min(1200px, 96vw);
                        max-height:92vh;
                        background:#fff;
                        border-radius:16px;
                        box-shadow:0 18px 50px rgba(0,0,0,.25);
                        overflow:hidden;">
                      
                      <div style="display:flex; justify-content:space-between; align-items:center;
                                  padding:12px 14px; border-bottom:1px solid rgba(0,0,0,.08);">
                        <div id="modalGraficaAlcancesTitulo" style="font-weight:800; font-size:1.5rem; text-align:center; width:100%; color:#2c3e50;">
                          Gráfica de Alcances
                        </div>
                      </div>
                
                      <div style="padding:12px;">
                        <canvas id="modalCanvasGraficaAlcances" style="display:block; margin:0 auto; width:100%; height: 70vh;"></canvas>
                      </div>
                    </div>
                  </div>
              </div>
            
              <div id="notaTabla4M" style="margin-top:10px; color:#666; font-size:0.9rem;">
                —
              </div>
            </div>
        `;

        const card4M = document.getElementById('cardTablaComparativa4M');
        const mini4M = document.getElementById('miniAlcancesRight');
        if (card4M && mini4M) {{
          card4M.insertBefore(mini4M, card4M.firstElementChild);
          mini4M.style.marginBottom = '14px';
          mini4M.style.marginTop = '0';
        }}

        try {{
          initTablaComparativa4M();
        }} catch (e) {{
          console.warn('initTablaComparativa4M error:', e);
        }}
        
        // Generar gráficos después de un breve delay
        setTimeout(() => {{
          generarGraficaIncrementoTotal(periodoCompleto, 'miniGraficaIncrementoTotal', 'mini');
          generarGraficaRecuperoDiario(periodoCompleto, 'miniGraficaRecuperoDiario', 'mini');
          generarGraficaAlcanceEquiposRecuperos(periodoCompleto, 'miniGraficaAlcanceEquipos', 'mini');
          actualizarGraficaEvolucionAnualRecuperos('miniGraficaEvolucionAnualRecuperos', 'mini');

          _bindMiniAlcancesClicks(periodoCompleto);
        }}, 100);
    }}

    function _cloneHtmlClipboard4M(node) {{
      if (!node) return '';
      const clone = node.cloneNode(true);
      clone.querySelectorAll('[id]').forEach(el => el.removeAttribute('id'));
      clone.querySelectorAll('.is-selected').forEach(el => el.classList.remove('is-selected'));
      clone.querySelectorAll('.is-clickable').forEach(el => el.classList.remove('is-clickable'));
      clone.querySelectorAll('.tabla4m-wrap').forEach(el => {{
        el.style.overflow = 'visible';
        el.style.maxHeight = 'none';
        el.style.maxWidth = 'none';
        el.style.width = 'auto';
      }});
      clone.querySelectorAll('table').forEach(tb => {{
        tb.style.borderCollapse = 'collapse';
        tb.style.width = '100%';
      }});
      return clone.outerHTML;
    }}

    function _tdClipboard4M(html) {{
      return `<td style="vertical-align:top; padding:8px; border:1px solid #d9e2ec;">${{html}}</td>`;
    }}

    function _filaClipboard4M(htmls) {{
      const celdas = (htmls || []).filter(Boolean).map(_tdClipboard4M).join('');
      return celdas ? `<tr>${{celdas}}</tr>` : '';
    }}

    async function copiarVistaAlcances4M() {{
      const comparativo = document.getElementById('tablas4MComparativoSupervisores');
      const individuales = document.getElementById('tablas4MSupervisoresTodos');
      const principales = document.getElementById('tablas4MPrincipales');

      const row1Base = comparativo?.querySelector('.tabla4m-doble');
      const row1 = Array.from(row1Base?.children || []).map(_cloneHtmlClipboard4M);
      const row2 = Array.from(individuales?.children || []).map(_cloneHtmlClipboard4M);
      const row3 = Array.from(principales?.children || []).map(_cloneHtmlClipboard4M);

      const filas = [
        _filaClipboard4M(row1),
        _filaClipboard4M(row2),
        _filaClipboard4M(row3)
      ].filter(Boolean).join('');

      if (!filas) {{
        alert('No hay tablas de ALCANCES para copiar.');
        return;
      }}

      const estilos = `
        <style>
          table {{ border-collapse: collapse; font-family: Arial, sans-serif; font-size: 11px; }}
          th {{ background:#0d47a1; color:#fff; font-weight:700; border:1px solid #d9e2ec; padding:3px 5px; text-align:center; }}
          td {{ border:1px solid #d9e2ec; padding:3px 5px; text-align:center; white-space:nowrap; }}
          .tabla4m-titulo {{ font-weight:800; color:#2c3e50; margin-bottom:5px; text-align:center; }}
          .valor-vacio {{ color:#9aa4b2; }}
          .valor-alto {{ font-weight:800; }}
        </style>
      `;
      const html = `${{estilos}}<table>${{filas}}</table>`;
      const texto = html.replace(/<[^>]+>/g, ' ').replace(/\s+/g, ' ').trim();

      try {{
        if (navigator.clipboard && window.ClipboardItem) {{
          await navigator.clipboard.write([
            new ClipboardItem({{
              'text/html': new Blob([html], {{ type: 'text/html' }}),
              'text/plain': new Blob([texto], {{ type: 'text/plain' }})
            }})
          ]);
        }} else {{
          const tmp = document.createElement('div');
          tmp.style.position = 'fixed';
          tmp.style.left = '-9999px';
          tmp.innerHTML = html;
          document.body.appendChild(tmp);
          const range = document.createRange();
          range.selectNodeContents(tmp);
          const sel = window.getSelection();
          sel.removeAllRanges();
          sel.addRange(range);
          document.execCommand('copy');
          sel.removeAllRanges();
          tmp.remove();
        }}
        alert('Vista copiada. Puedes pegarla en Excel o correo.');
      }} catch (err) {{
        console.error('No se pudo copiar la vista:', err);
        alert('No se pudo copiar la vista. Intenta nuevamente desde el navegador.');
      }}
    }}

    function _bindMiniAlcancesClicks(periodoCompleto) {{
      const wrap = document.getElementById('miniAlcancesRight');
      if (!wrap) return;
    
      const cards = wrap.querySelectorAll('.mini-chart-card[data-mini]');
      cards.forEach((card) => {{
        card.style.cursor = 'pointer';
    
        // Evitar doble bindeo (si se re-renderiza la sección)
        if (card.dataset.bound === '1') return;
        card.dataset.bound = '1';
    
        card.addEventListener('click', () => {{
          const tipo = (card.getAttribute('data-mini') || '').trim();
    
          const titulo = document.getElementById('modalGraficaAlcancesTitulo');
          if (titulo) {{
            const map = {{
              acumulado: 'Gráfica Acumulada',
              diario: 'Gráfica Diaria',
              equipos: 'Gráfica por Equipos',
              anual: 'Evolución Anual'
            }};
            titulo.textContent = map[tipo] || 'Gráfica de Alcances';
          }}
    
          abrirModalGraficaAlcances(tipo, periodoCompleto);
        }});
      }});
    }}

    // ========== FUNCIÓN PARA CAMBIAR ENTRE GRÁFICAS ==========
    function cambiarGrafica(tipo) {{
        const btnAcumulado = document.getElementById('btnRecuperoAcumulado');
        const btnDiario = document.getElementById('btnRecuperoDiario');
        const btnEquipos = document.getElementById('btnRecuperoEquipos');
        const btnAnual = document.getElementById('btnRecuperoAnual');
        const containerAcumulado = document.getElementById('graficaAcumuladoContainer');
        const containerDiario = document.getElementById('graficaDiarioContainer');
        const containerEquipos = document.getElementById('graficaEquiposContainer');
        const containerAnual = document.getElementById('graficaAnualContainer');
        
        if (!btnAcumulado || !btnDiario || !btnEquipos || !btnAnual || 
            !containerAcumulado || !containerDiario || !containerEquipos || !containerAnual) return;
        
        // Remover clase activa de todos los botones
        btnAcumulado.classList.remove('activo');
        btnDiario.classList.remove('activo');
        btnEquipos.classList.remove('activo');
        btnAnual.classList.remove('activo');
        
        // Ocultar todos los contenedores
        containerAcumulado.style.display = 'none';
        containerDiario.style.display = 'none';
        containerEquipos.style.display = 'none';
        containerAnual.style.display = 'none';
        
        // Restaurar estilos de todos los botones
        const estilosInactivo = `
            background: linear-gradient(135deg, #9b59b6, #8e44ad);
            box-shadow: 0 4px 10px rgba(155, 89, 182, 0.3);
        `;
        
        btnAcumulado.style.cssText = btnAcumulado.style.cssText.replace(/background[^;]+;/, '').replace(/box-shadow[^;]+;/, '') + estilosInactivo;
        btnDiario.style.cssText = btnDiario.style.cssText.replace(/background[^;]+;/, '').replace(/box-shadow[^;]+;/, '') + estilosInactivo;
        btnEquipos.style.cssText = btnEquipos.style.cssText.replace(/background[^;]+;/, '').replace(/box-shadow[^;]+;/, '') + estilosInactivo;
        btnAnual.style.cssText = btnAnual.style.cssText.replace(/background[^;]+;/, '').replace(/box-shadow[^;]+;/, '') + estilosInactivo;
        
        // Mostrar la gráfica seleccionada y activar su botón
        const estilosActivo = `
            background: linear-gradient(135deg, #2ecc71, #27ae60);
            box-shadow: 0 0 0 3px white, 0 0 0 6px #2ecc71;
        `;
        
        if (tipo === 'acumulado') {{
            btnAcumulado.classList.add('activo');
            containerAcumulado.style.display = 'block';
            btnAcumulado.style.cssText = btnAcumulado.style.cssText.replace(/background[^;]+;/, '').replace(/box-shadow[^;]+;/, '') + estilosActivo;
        }} 
        else if (tipo === 'diario') {{
            btnDiario.classList.add('activo');
            containerDiario.style.display = 'block';
            btnDiario.style.cssText = btnDiario.style.cssText.replace(/background[^;]+;/, '').replace(/box-shadow[^;]+;/, '') + estilosActivo;
        }}
        else if (tipo === 'equipos') {{
            btnEquipos.classList.add('activo');
            containerEquipos.style.display = 'block';
            btnEquipos.style.cssText = btnEquipos.style.cssText.replace(/background[^;]+;/, '').replace(/box-shadow[^;]+;/, '') + estilosActivo;
        }}
        
        else if (tipo === 'anual') {{
            btnAnual.classList.add('activo');
            containerAnual.style.display = 'block';
            btnAnual.style.cssText = btnAnual.style.cssText
              .replace(/background[^;]+;/, '')
              .replace(/box-shadow[^;]+;/, '') + estilosActivo;
            if (typeof actualizarGraficaEvolucionAnualRecuperos === 'function') {{
                actualizarGraficaEvolucionAnualRecuperos();
            }} else {{
                console.warn('⚠️ actualizarGraficaEvolucionAnualRecuperos no existe');
            }}
        }}
    }}

    // Función para convertir fecha diaria (ej: "3-Nov") a número para ordenar
    function convertirFechaDiaria(fecha) {{
      const n = Number(fecha);
      return Number.isFinite(n) ? n : 0;
    }}

    function obtenerDiasDelPeriodo(periodoCompleto) {{
      const partes = String(periodoCompleto || '').split('_');
      if (partes.length < 2) return 31;
      const meses = {{
        ENERO: 0, FEBRERO: 1, MARZO: 2, ABRIL: 3, MAYO: 4, JUNIO: 5,
        JULIO: 6, AGOSTO: 7, SETIEMBRE: 8, SEPTIEMBRE: 8,
        OCTUBRE: 9, NOVIEMBRE: 10, DICIEMBRE: 11
      }};
      const mesIdx = meses[String(partes[0] || '').toUpperCase()];
      const anio = Number(partes[1]);
      if (mesIdx === undefined || !Number.isFinite(anio)) return 31;
      return new Date(anio, mesIdx + 1, 0).getDate();
    }}

    function obtenerFechasPeriodoCompleto(periodoCompleto) {{
      const totalDias = obtenerDiasDelPeriodo(periodoCompleto);
      return Array.from({{ length: totalDias }}, (_, idx) => String(idx + 1));
    }}

    function obtenerUltimoDiaConDato(datosDiarios) {{
      return Object.keys(datosDiarios || {{}}).reduce((maxDia, fecha) => {{
        const dia = convertirFechaDiaria(fecha);
        const valor = Number(datosDiarios[fecha] || 0);
        return (dia > maxDia && valor > 0) ? dia : maxDia;
      }}, 0);
    }}
    
    function abrirModalGraficaAlcances(tipo, periodoCompleto) {{
        const modal = document.getElementById('modalGraficaAlcances');
        const modalCanvasId = 'modalCanvasGraficaAlcances';
        if (modal) {{
        modal.style.display = 'block';
        }}
        document.addEventListener('keydown', handleModalGraficaAlcancesEscape);
        const overlay = document.getElementById('modalGraficaAlcancesOverlay');
        if (overlay && overlay.dataset.bound !== '1') {{
        overlay.dataset.bound = '1';
        overlay.addEventListener('click', function(e) {{
          if (e.target === overlay) {{
            cerrarModalGraficaAlcances();
          }}
        }});
        }}
        const supUI = document.getElementById('selectorSupervisor');
        let supervisor = (window.supervisorSeleccionado || '').trim();
        
        // fallback al selector actual si el global está vacío
        if (!supervisor && supUI && supUI.value) {{
            supervisor = String(supUI.value).trim();
        }}
        
        // normalizar TODOS
        if (supervisor === 'TODOS') supervisor = '';
        requestAnimationFrame(() => {{
            if (supervisor && supervisor !== 'TODOS') {{
              if (tipo === 'acumulado') {{
                generarGraficaIncrementoTotalSupervisor(periodoCompleto, supervisor, modalCanvasId, 'modal');
              }} else if (tipo === 'diario') {{
                generarGraficaRecuperoDiarioSupervisor(periodoCompleto, supervisor, modalCanvasId, 'modal');
              }} else if (tipo === 'equipos') {{
                generarGraficaAlcanceEquiposRecuperos(periodoCompleto, modalCanvasId, 'modal');
              }} else if (tipo === 'anual') {{
                actualizarGraficaEvolucionAnualRecuperos(modalCanvasId, 'modal');
              }}
        
            }} else {{
        
              if (tipo === 'acumulado') {{
                generarGraficaIncrementoTotal(periodoCompleto, modalCanvasId, 'modal');
              }} else if (tipo === 'diario') {{
                generarGraficaRecuperoDiario(periodoCompleto, modalCanvasId, 'modal');
              }} else if (tipo === 'equipos') {{
                generarGraficaAlcanceEquiposRecuperos(periodoCompleto, modalCanvasId, 'modal');
              }} else if (tipo === 'anual') {{
                actualizarGraficaEvolucionAnualRecuperos(modalCanvasId, 'modal');
              }}
            }}
        }});
    }}
    
    function handleModalGraficaAlcancesEscape(event) {{
        if (event.key === 'Escape' || event.keyCode === 27) {{
            cerrarModalGraficaAlcances();
        }}
    }}

    function cerrarModalGraficaAlcances() {{
        const modal = document.getElementById('modalGraficaAlcances');
        if (modal) modal.style.display = 'none';
    
        document.removeEventListener('keydown', handleModalGraficaAlcancesEscape);
    }}

    // ========== EXPORTAR EXCEL DIARIO POR ASESOR (DESHABILITADO) ==========
    async function exportarTablaDetalleExcelAvanzado(supervisorFiltro = null) {{
        alert('La exportación diaria por asesor fue deshabilitada: la Vista de Asesor por Día ya no se usa como fuente.');
        return;
    }}

    async function exportarRecupero4MesesDiaSupervisor(supervisorParam = '') {{
      const wb = new ExcelJS.Workbook();
      const mesSeleccionado = document.getElementById('selectorMes')?.value;
      const añoSeleccionado = document.getElementById('selectorAño')?.value;
      const periodoActual = `${{mesSeleccionado}}_${{añoSeleccionado}}`;
      const fechaHoy = new Date();
      const meses = ['ENERO','FEBRERO','MARZO','ABRIL','MAYO','JUNIO','JULIO','AGOSTO','SETIEMBRE','OCTUBRE','NOVIEMBRE','DICIEMBRE'];
      const periodoHoy = `${{meses[fechaHoy.getMonth()]}}_${{fechaHoy.getFullYear()}}`;
      const limiteHoy = Math.max(1, fechaHoy.getDate() - 1);
    
      if (!mesSeleccionado || !añoSeleccionado) {{
        alert('Selecciona mes y año');
        return;
      }}

      const fuenteSupervisores =
        (typeof datosSupervisores !== 'undefined' && datosSupervisores) ? datosSupervisores :
        (typeof supervisoresData !== 'undefined' && supervisoresData) ? supervisoresData :
        (window && window.supervisoresData) ? window.supervisoresData :
        null;
    
      if (!fuenteSupervisores || typeof fuenteSupervisores !== 'object') {{
        alert('No hay datos de supervisores disponibles (datosSupervisores)');
        return;
      }}
    
      let supervisorFiltro = (supervisorParam || '').trim();
    
      if (!supervisorFiltro) {{
        const barraFiltrosGlobal = document.getElementById('barraFiltrosSupervisoresGlobal');
        if (barraFiltrosGlobal) {{
          const botonActivoGlobal = barraFiltrosGlobal.querySelector('.filtro-supervisor.activo[data-supervisor]');
          if (botonActivoGlobal) {{
            supervisorFiltro = botonActivoGlobal.getAttribute('data-supervisor') || '';
          }}
        }}
      }}
    
      if (!supervisorFiltro) supervisorFiltro = 'TODOS';
    
      const periodos = _getPeriodosAtras(mesSeleccionado, añoSeleccionado, 4, true);
      if (!periodos.length) {{
        alert('No se pudieron calcular los 4 periodos');
        return;
      }}
    
      const _tituloMes = (periodo) => {{
        const p = String(periodo).split('_');
        const mes = (p[0] || '').toLowerCase();
        return mes ? (mes.charAt(0).toUpperCase() + mes.slice(1)) : periodo;
      }};
    
      const _anioPeriodo = (periodo) => {{
        const p = String(periodo).split('_');
        return p[1] || '';
      }};

      const _diaDeKey = (key) => {{
          const d = Number(key);
          return Number.isInteger(d) && d >= 1 && d <= 31 ? d : null;
      }};
    
        const _forwardFill31 = (mapDiaToValor, limiteDia = 31) => {{
          const out = new Array(32).fill(null);
        
          for (let d = 1; d <= limiteDia; d++) {{
            const v = mapDiaToValor[d];
        
            if (v === undefined || v === null) {{
              out[d] = (d === 1) ? null : out[d - 1];
            }} else {{
              out[d] = Number(v);
            }}
          }}
        
          for (let d = limiteDia + 1; d <= 31; d++) {{
            out[d] = null;
          }}
        
          return out;
        }};
    
      // =====================
      // Helpers numéricos
      // (si ya existe _toNumber arriba en tu archivo, puedes borrar este y usar el tuyo)
      // =====================
      const _toNumber = (v) => {{
        if (v === null || v === undefined) return 0;
        if (typeof v === 'number') return isFinite(v) ? v : 0;
        const s = String(v).replace(/[%,$ ]/g, '').replace(/,/g, '').trim();
        const n = parseFloat(s);
        return isFinite(n) ? n : 0;
      }};
    
      // =====================
      // META/RECUPERO desde SUPERVISORES (CORRECTO)
      // =====================
      const _getMetaMesDesdeSupervisores = (periodo) => {{
        if (supervisorFiltro === 'TODOS') {{
          let suma = 0;
          Object.keys(fuenteSupervisores || {{}}).forEach((sup) => {{
            suma += _toNumber(fuenteSupervisores?.[sup]?.[periodo]?.meta_super);
          }});
          return suma;
        }}
        return _toNumber(fuenteSupervisores?.[supervisorFiltro]?.[periodo]?.meta_super);
      }};
    
      const _getRecuperoMesDesdeSupervisores = (periodo) => {{
        if (supervisorFiltro === 'TODOS') {{
          let suma = 0;
          Object.keys(fuenteSupervisores || {{}}).forEach((sup) => {{
            // en tu estructura de supervisores: total_recupero por mes
            suma += _toNumber(fuenteSupervisores?.[sup]?.[periodo]?.total_recupero);
          }});
          return suma;
        }}
        return _toNumber(fuenteSupervisores?.[supervisorFiltro]?.[periodo]?.total_recupero);
      }};
    
      // ====== helpers de hoja (para reusar en consolidado y supervisores) ======
      const _estilizarHojaDoble = (ws) => {{
        // Colores header
        const fillRow1 = {{
          type: 'pattern',
          pattern: 'solid',
          fgColor: {{ argb: 'FF0D47A1' }}
        }};
        const fillRow2 = {{
          type: 'pattern',
          pattern: 'solid',
          fgColor: {{ argb: 'FF2C3E50' }}
        }};
    
        // 1) Estilos fila 1 (A..E y G..K)
        const r1 = ws.getRow(1);
        r1.font = {{ bold: true, color: {{ argb: 'FFFFFFFF' }} }};
        r1.alignment = {{ horizontal: 'center', vertical: 'middle' }};
    
        // Tabla 1 (A..E) => col 1..(1+periodos.length)
        for (let c = 1; c <= (1 + periodos.length); c++) {{
          ws.getCell(1, c).fill = fillRow1;
        }}
        // Tabla 2 (G..K) => start 7
        for (let c = 7; c <= (7 + periodos.length); c++) {{
          ws.getCell(1, c).fill = fillRow1;
        }}
    
        // 2) Estilos fila 2 (A..E y G..K)
        const r2 = ws.getRow(2);
        r2.font = {{ bold: true, color: {{ argb: 'FFFFFFFF' }} }};
        r2.alignment = {{ horizontal: 'center', vertical: 'middle' }};
    
        for (let c = 1; c <= (1 + periodos.length); c++) {{
          ws.getCell(2, c).fill = fillRow2;
        }}
        for (let c = 7; c <= (7 + periodos.length); c++) {{
          ws.getCell(2, c).fill = fillRow2;
        }}
    
        // Congelar
        ws.views = [{{ state: 'frozen', ySplit: 2, xSplit: 1 }}];
    
        // Formato numérico
        // Tabla 1 (RECUPERO): columnas B..E
        for (let c = 2; c <= (1 + periodos.length); c++) {{
          ws.getColumn(c).numFmt = '#,##0';
        }}
    
        // Tabla 2 (ALCANCE %): columnas H..K => start 8
        // (Guardaremos valor en % ya multiplicado *100, así que formateamos con % literal)
        for (let c = 8; c <= (7 + periodos.length); c++) {{
          ws.getColumn(c).numFmt = '0.00"%"';
        }}
    
        // Anchos
        ws.getColumn(1).width = 6; // A (Dia)
        for (let c = 2; c <= (1 + periodos.length); c++) ws.getColumn(c).width = 16;
    
        ws.getColumn(6).width = 3; // F (espacio)
        ws.getColumn(7).width = 6; // G (Dia)
        for (let c = 8; c <= (7 + periodos.length); c++) ws.getColumn(c).width = 16;
      }};
    
      // Escribe 2 tablas: Recupero (A..E) y Alcance% (G..K)
      const _poblarHojaRecuperoYAlcance = (ws, colRecuperoPorPeriodo, metaPorPeriodo) => {{
        // Headers tabla 1 (RECUPERO)
        ws.getCell(1, 1).value = '';
        ws.getCell(2, 1).value = 'Dia';
    
        // Headers tabla 2 (ALCANCE)
        ws.getCell(1, 7).value = '';
        ws.getCell(2, 7).value = 'Dia';
    
        // Titulos de meses/años (ambas tablas)
        for (let i = 0; i < periodos.length; i++) {{
          const periodo = periodos[i];
    
          // Tabla 1: B..E (col = 2+i)
          ws.getCell(1, 2 + i).value = _anioPeriodo(periodo);
          ws.getCell(2, 2 + i).value = _tituloMes(periodo);
    
          // Tabla 2: H..K (col = 8+i)
          ws.getCell(1, 8 + i).value = _anioPeriodo(periodo);
          ws.getCell(2, 8 + i).value = _tituloMes(periodo);
        }}
    
        // Datos días 1..31
        for (let d = 1; d <= 31; d++) {{
          const rowIdx = 2 + d; // fila 3..33
    
          // Dia tabla 1 y 2
          ws.getCell(rowIdx, 1).value = d; // A
          ws.getCell(rowIdx, 7).value = d; // G
    
          for (let i = 0; i < periodos.length; i++) {{
            const periodo = periodos[i];
    
            const rec = colRecuperoPorPeriodo?.[periodo]?.[d] ?? null;
            
            // TABLA 1 (RECUPERO)
            ws.getCell(rowIdx, 2 + i).value = (rec === null ? null : Number(rec));
            
            // TABLA 2 (ALCANCE %)
            const metaMes = Number(metaPorPeriodo?.[periodo] ?? 0);
            
            if (rec !== null && metaMes > 0) {{
              const alcancePct = (Number(rec) / metaMes) * 100;
              ws.getCell(rowIdx, 8 + i).value = alcancePct;
            }} else {{
              ws.getCell(rowIdx, 8 + i).value = null;
            }}
          }}
        }}
    
        _estilizarHojaDoble(ws);
      }};
    
      // ===================== CASO 1: FILTRADO POR UN SUPERVISOR =====================
      if (supervisorFiltro !== 'TODOS') {{
        const sup = supervisorFiltro;
        const sheetName = String(sup || 'SUPERVISOR').substring(0, 31);
        const ws = wb.addWorksheet(sheetName);
    
        const colValoresPorPeriodo = {{}};
        const metaPorPeriodo = {{}};
    
        for (const periodo of periodos) {{
          const dataPeriodo = fuenteSupervisores?.[sup]?.[periodo];
    
          // Recupero diario
          const diarios = dataPeriodo?.datos_diarios_supervisor || {{}};
          const mapDia = {{}};
          Object.keys(diarios).forEach((k) => {{
            const d = _diaDeKey(k);
            if (!d || d < 1 || d > 31) return;
            mapDia[d] = Number(diarios[k]) || 0;
          }});
          const limite = (periodo === periodoHoy) ? limiteHoy : 31;
          colValoresPorPeriodo[periodo] = _forwardFill31(mapDia, limite);
    
          // ✅ Meta del supervisor en ese mes (DESDE SUPERVISORES)
          metaPorPeriodo[periodo] = _getMetaMesDesdeSupervisores(periodo);
        }}
    
        _poblarHojaRecuperoYAlcance(ws, colValoresPorPeriodo, metaPorPeriodo);
      }}
      // ===================== CASO 2: TODOS (CONSOLIDADO + HOJA POR SUPERVISOR) =====================
      else {{
        const supervisoresTodos = Object.keys(fuenteSupervisores || {{}});
    
        if (!supervisoresTodos.length) {{
          alert('No hay supervisores para exportar');
          return;
        }}
    
        //supervisores actuales
        const supervisoresMesActual = supervisoresTodos.filter((sup) => {{
          const dataMes = fuenteSupervisores?.[sup]?.[periodoActual];
          const diarios = dataMes?.datos_diarios_supervisor || {{}};
          return Object.keys(diarios).length > 0;
        }});
    
        // (A) CONSOLIDADO TOTAL (usa TODOS)
        {{
          const wsCons = wb.addWorksheet('CONSOLIDADO');
    
          const colValoresPorPeriodo = {{}};
          const metaPorPeriodo = {{}};
    
          for (const periodo of periodos) {{
            const mapDiaConsolidado = {{}};
    
            // Consolidar recupero diario sumando supervisores
            for (const sup of supervisoresTodos) {{
              const dataPeriodo = fuenteSupervisores?.[sup]?.[periodo];
    
              const diarios = dataPeriodo?.datos_diarios_supervisor || {{}};
              Object.keys(diarios).forEach((k) => {{
                const d = _diaDeKey(k);
                if (!d || d < 1 || d > 31) return;
    
                const val = Number(diarios[k]) || 0;
                mapDiaConsolidado[d] = (Number(mapDiaConsolidado[d]) || 0) + val;
              }});
            }}
    
            const limite = (periodo === periodoHoy) ? limiteHoy : 31;
            colValoresPorPeriodo[periodo] = _forwardFill31(mapDiaConsolidado, limite);
    
            // ✅ Meta total del mes (SUMA metas de todos los sup) DESDE SUPERVISORES
            metaPorPeriodo[periodo] = _getMetaMesDesdeSupervisores(periodo);
          }}
    
          _poblarHojaRecuperoYAlcance(wsCons, colValoresPorPeriodo, metaPorPeriodo);
        }}
    
        // (B) HOJAS INDIVIDUALES (solo supervisores del mes filtrado)
        if (!supervisoresMesActual.length) {{
          console.warn('No hay supervisores con data en el periodo actual:', periodoActual);
        }} else {{
          for (const sup of supervisoresMesActual) {{
            const sheetName = String(sup || 'SUP').substring(0, 31);
            const ws = wb.addWorksheet(sheetName);
    
            const colValoresPorPeriodo = {{}};
            const metaPorPeriodo = {{}};
    
            for (const periodo of periodos) {{
              const dataPeriodo = fuenteSupervisores?.[sup]?.[periodo];
    
              // Recupero diario
              const diarios = dataPeriodo?.datos_diarios_supervisor || {{}};
              const mapDia = {{}};
              Object.keys(diarios).forEach((k) => {{
                const d = _diaDeKey(k);
                if (!d || d < 1 || d > 31) return;
                mapDia[d] = Number(diarios[k]) || 0;
              }});
              const limite = (periodo === periodoHoy) ? limiteHoy : 31;
              colValoresPorPeriodo[periodo] = _forwardFill31(mapDia, limite);
    
              // Meta mensual del supervisor
              metaPorPeriodo[periodo] = _toNumber(fuenteSupervisores?.[sup]?.[periodo]?.meta_super);
            }}
    
            _poblarHojaRecuperoYAlcance(ws, colValoresPorPeriodo, metaPorPeriodo);
          }}
        }}
      }}
    
      // Guardar
      const yy = fechaHoy.getFullYear();
      const mm = String(fechaHoy.getMonth() + 1).padStart(2, '0');
      const dd = String(fechaHoy.getDate()).padStart(2, '0');
      const stamp = `${{yy}}${{mm}}${{dd}}`;
    
      const nombreArchivo = `Recupero_4M_Dia_x_Mes_${{supervisorFiltro}}_${{periodoActual}}_${{stamp}}.xlsx`;
      const buffer = await wb.xlsx.writeBuffer();
    
      saveAs(
        new Blob([buffer], {{
          type: "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
        }}),
        nombreArchivo
      );
    }}
    
    function generarGraficaAlcanceEquiposRecuperos(periodoCompleto, canvasId = 'graficaAlcanceEquipos', view = 'full') {{
        const supervisores = Object.keys(datosSupervisores).filter(supervisor => 
            supervisorEnCanal(supervisor, periodoCompleto) && datosSupervisores[supervisor] && datosSupervisores[supervisor][periodoCompleto]
        );
        
        if (supervisores.length === 0) return;
        
        // ========== OBTENER TODOS LOS DÍAS DEL MES ==========
        const todasFechasArray = obtenerFechasPeriodoCompleto(periodoCompleto);
        
        console.log(`📊 Gráfica equipos - Total días del mes: ${{todasFechasArray.length}}`);
        
        // ========== PREPARAR DATOS CON TODAS LAS FECHAS ==========
        const datasets = [];
        const colores = ['#FF6384', '#36A2EB', '#FFCE56', '#8AC926', '#9966FF', '#FF9F40'];
        
        supervisores.forEach((supervisor, index) => {{
            const datosMes = datosSupervisores[supervisor][periodoCompleto];
            const alcanceDiario = datosMes.alcance_acumulado_diario || {{}};
            const datosDiarios = datosMes.datos_diarios_supervisor || {{}};
            
            // USAR TODOS LOS DÍAS DEL MES; días futuros/sin carga quedan vacíos
            const datosAlcance = todasFechasArray.map(fecha => {{
                let valor = alcanceDiario[fecha];
                if ((valor === undefined || valor === null) && Object.prototype.hasOwnProperty.call(datosDiarios, fecha)) {{
                    const recupero = Number(datosDiarios[fecha] || 0);
                    const meta = Number(datosMes.meta_super || 0);
                    valor = meta > 0 ? (recupero / meta) * 100 : null;
                }}
                return (valor !== undefined && valor !== null && valor > 0) ? valor : null;
            }});
            
            // Verificar que haya datos válidos (no todos null)
            const tieneDatos = datosAlcance.some(valor => valor !== null);
            
            if (tieneDatos) {{
                const color = colores[index % colores.length];
                
                datasets.push({{
                    label: supervisor,
                    data: datosAlcance,
                    borderColor: color,
                    backgroundColor: color + '20',
                    borderWidth: 3,
                    fill: false,
                    tension: 0.4,
                    pointRadius: function(context) {{
                        // Puntos solo para datos válidos
                        const value = context.dataset.data[context.dataIndex];
                        return value === null ? 0 : 6;
                    }},
                    pointHoverRadius: function(context) {{
                        const value = context.dataset.data[context.dataIndex];
                        return value === null ? 0 : 10;
                    }},
                    pointBackgroundColor: function(context) {{
                        const value = context.dataset.data[context.dataIndex];
                        return value === null ? 'transparent' : color;
                    }},
                    pointBorderColor: function(context) {{
                        const value = context.dataset.data[context.dataIndex];
                        return value === null ? 'transparent' : '#ffffff';
                    }},
                    pointBorderWidth: 2,
                    spanGaps: false  // IMPORTANTE: No conectar puntos nulos
                }});
            }}
        }});
        
        // ========== VERIFICAR QUE HAYA DATASETS VÁLIDOS ==========
        if (datasets.length === 0) {{
            const canvas = document.getElementById(canvasId);
            if (!canvas) return;
            
            const ctx = canvas.getContext('2d');
            ctx.clearRect(0, 0, canvas.width, canvas.height);
            
            _applyCanvasSize(canvas, view);
            canvas.style.display = 'block';
            canvas.style.margin = '0 auto';
            canvas.style.maxWidth = '100%';
            
            ctx.fillStyle = '#95a5a6';
            ctx.font = 'bold 24px Arial';
            ctx.textAlign = 'center';
            ctx.fillText('📊 No hay datos de equipos disponibles', canvas.width / 2, canvas.height / 2 - 20);
            
            ctx.fillStyle = '#7f8c8d';
            ctx.font = '18px Arial';
            ctx.fillText(`Periodo: ${{periodoCompleto.replace('_', ' ')}}`, canvas.width / 2, canvas.height / 2 + 20);
            
            return;
        }}
        
        // ========== AGREGAR LÍNEA DE META ==========
        datasets.push({{
            label: 'Meta 100%',
            data: Array(todasFechasArray.length).fill(100),
            borderColor: '#2ecc71',
            backgroundColor: 'transparent',
            borderWidth: 2,
            borderDash: [5, 5],
            fill: false,
            tension: 0,
            pointRadius: 0,
            order: 999
        }});
        
        // ========== CONFIGURAR CANVAS ==========
        const canvas = document.getElementById(canvasId || 'graficaAlcanceEquipos');
        if (!canvas) {{
          console.warn(`Canvas no encontrado: ${{canvasId || 'graficaAlcanceEquipos'}}`);
          return;
        }}
        _applyCanvasSize(canvas, view);
        canvas.style.display = 'block';
        canvas.style.margin = '0 auto';
        canvas.style.maxWidth = '100%';
        
        const ctx = canvas.getContext('2d');
        
        // ========== DESTRUIR GRÁFICA ANTERIOR ==========
        _destroyChartByCanvasId(canvasId);
        
        // ========== OBTENER TÍTULO ==========
        const [mes, año] = periodoCompleto.split('_');
        const tituloGrafica = [`        `,`        `];

        let options = {{
            responsive: true,
            maintainAspectRatio: false,
            plugins: {{
                legend: {{
                    display: true,
                    position: 'bottom',
                    labels: {{ 
                        font: {{ 
                            size: 18, 
                            weight: 'bold' 
                        }} 
                    }}
                }},
                tooltip: {{
                  enabled: true,
                  mode: 'nearest',
                  intersect: false,
                  padding: 15,
                  cornerRadius: 10,
                  backgroundColor: 'rgba(0, 0, 0, 0.9)',
                  titleFont: {{ size: 18, weight: 'bold' }},
                  bodyFont: {{ size: 18 }},
                
                  filter: function(tooltipItem) {{
                    const labelDataset = tooltipItem?.dataset?.label || '';
                    return labelDataset !== 'Meta 100%';
                  }},
                
                  itemSort: function(a, b) {{
                    const ay = (a && a.parsed && a.parsed.y != null) ? a.parsed.y : -Infinity;
                    const by = (b && b.parsed && b.parsed.y != null) ? b.parsed.y : -Infinity;
                    return by - ay; // DESC
                  }},
                
                  callbacks: {{
                    title: function(tooltipItems) {{
                      if (!tooltipItems || tooltipItems.length === 0) return '';
                      const index = tooltipItems[0].dataIndex;
                      return todasFechasArray[index] || '';
                    }},
                    label: function(context) {{
                      if (!context || !context.dataset) return null;
                    
                      const labelDataset = context.dataset.label || '';
                    
                      // Ocultar "Meta 100%"
                      if (labelDataset === 'Meta 100%') {{
                        return null;
                      }}
                    
                      if (!context.parsed || context.parsed.y === null || !isFinite(context.parsed.y)) {{
                        return null; // mejor que "Sin datos" para evitar tooltips vacíos con basura
                      }}
                    
                      return `${{labelDataset}}: ${{context.parsed.y.toFixed(2)}}%`;
                    }}
                  }}
                }}
            }},
            interaction: {{
                mode: 'nearest',
                axis: 'x',
                intersect: false
            }},
            scales: {{
                x: {{
                    title: {{
                        display: false,
                        text: 'Días del Mes',
                        font: {{ size: 12, weight: 'bold' }}
                    }},
                    ticks: {{
                        font: {{ size: 16 }},
                        callback: function(value, index, values) {{
                            // Colores diferentes para días con/sin datos en al menos un supervisor
                            const tieneDatosEnAlguno = datasets.some(dataset => 
                                dataset.data[index] !== null && 
                                dataset.label !== 'Meta 100%'
                            );
                            this.fontColor = tieneDatosEnAlguno ? '#2c3e50' : '#bdc3c7';
                            return this.getLabelForValue(value);
                        }}
                    }},
                    grid: {{
                        drawOnChartArea: true
                    }}
                }},
                y: {{
                    type: 'linear',
                    display: true,
                    position: 'left',
                    title: {{
                        display: false,
                        text: '% Alcance Acumulado',
                        font: {{ size: 12, weight: 'bold' }}
                    }},
                    ticks: {{
                        stepSize: 10,
                        callback: function(value) {{
                            return value + '%';
                        }},
                        font: {{ size: 16 }}
                    }},
                    min: 0,
                    suggestedMax: 100
                }}
            }}
        }};
        
        options = _tuneOptionsForView(options, view);
        
        // ========== CREAR GRÁFICA CON TODAS LAS FECHAS ==========
        const chart = new Chart(ctx, {{
            type: 'line',
            data: {{
                labels: todasFechasArray.map(fecha => {{
                    // Mostrar solo el número del día
                    if (fecha.includes('-')) {{
                        return fecha.split('-')[0];
                    }}
                    return fecha;
                }}),
                datasets: datasets
            }},
            options
        }});
        _storeChart(canvasId, chart);
    }}
    
    function generarGraficaRecuperoDiario(periodoCompleto, canvasId = 'graficaRecuperoDiario', view = 'full') {{
        const supervisoresPeriodo = obtenerSupervisoresParaResumen(periodoCompleto, 'TODOS');
        const dataLabelsPluginTotal = _buildDataLabelsPluginTotal(view);
        const fechasConDatos = new Set();
        supervisoresPeriodo.forEach(item => {{
            Object.keys(item.datos?.datos_diarios_supervisor || {{}}).forEach(fecha => fechasConDatos.add(fecha));
        }});
        const fechasOrdenadas = obtenerFechasPeriodoCompleto(periodoCompleto);

        const canvas = document.getElementById(canvasId);
        if (!canvas) return;
        _applyCanvasSize(canvas, view);
        canvas.style.display = 'block';
        canvas.style.margin = '0 auto';
        canvas.style.maxWidth = '100%';
        const ctx = canvas.getContext('2d');
        _destroyChartByCanvasId(canvasId);
        ctx.clearRect(0, 0, canvas.width, canvas.height);
        
        if (fechasConDatos.size === 0) {{
            mostrarMensajeSinDatos(canvas, periodoCompleto);
            return;
        }}
        
        let metaTotalMes = 0;
        supervisoresPeriodo.forEach(item => {{
            metaTotalMes += Number(item.datos?.meta_super || 0);
        }});
        
        const diasLaborables = fechasOrdenadas.length;
        const metaDiaria = diasLaborables > 0 ? metaTotalMes / diasLaborables : 0;
        
        const alcanceDiario = fechasOrdenadas.map(fecha => {{
            let totalRecuperoDia = 0;
            let tieneDato = false;
            supervisoresPeriodo.forEach(item => {{
                const diarios = item.datos?.datos_diarios_supervisor || {{}};
                if (Object.prototype.hasOwnProperty.call(diarios, fecha)) {{
                    tieneDato = true;
                    totalRecuperoDia += Number(diarios[fecha] || 0);
                }}
            }});
            
            const porcentajeDiario = (tieneDato && metaDiaria > 0) ? (totalRecuperoDia / metaDiaria) * 100 : null;
            if (!tieneDato) totalRecuperoDia = null;
            return {{
                fecha: fecha,
                recupero: totalRecuperoDia,
                porcentaje: porcentajeDiario
            }};
        }});
        
        const porcentajes = alcanceDiario.map(dia => dia.porcentaje);
        const coloresBarras = porcentajes.map(porcentaje => {{
            if (porcentaje <= 0) return '#F3E5F5';
            if (porcentaje <= 20) return '#E1BEE7';
            if (porcentaje <= 40) return '#BA68C8';
            if (porcentaje <= 60) return '#8E24AA';
            if (porcentaje <= 80) return '#6A1B9A';
            if (porcentaje <= 100) return '#4A148C';
            return '#38006B';
        }});

        let options = {{
            responsive: true,
            maintainAspectRatio: false,
            interaction: {{ mode: 'index', intersect: false }},
            plugins: {{
                title: {{
                    display: true,
                    text: `            `,
                    font: {{ size: 24, weight: 'bold' }}
                }},
                tooltip: {{
                    mode: 'nearest',
                    intersect: false,
                    backgroundColor: 'rgba(0, 0, 0, 0.8)',
                    titleFont: {{ size: 18, weight: 'bold' }},
                    bodyFont: {{ size: 17 }},
                    padding: 15,
                    cornerRadius: 10,
                    callbacks: {{
                        title: function(context) {{
                            const index = context[0].dataIndex;
                            return `Día ${{index + 1}}: ${{fechasOrdenadas[index]}}`;
                        }},
                        label: function(context) {{
                            const index = context.dataIndex;
                            const dia = alcanceDiario[index];
                            return [
                                dia.porcentaje === null ? 'Sin datos cargados' : `Alcance: ${{dia.porcentaje.toFixed(2)}}%`,
                                dia.recupero === null ? 'Recupero acumulado: sin data' : `Recupero acumulado: S/ ${{dia.recupero.toLocaleString('es-PE', {{ minimumFractionDigits: 2, maximumFractionDigits: 2 }})}}`,
                                `Meta diaria referencial: S/ ${{metaDiaria.toLocaleString('es-PE', {{ minimumFractionDigits: 2, maximumFractionDigits: 2 }})}}`
                            ];
                        }}
                    }}
                }},
                legend: {{ display: false }}
            }},
            scales: {{
                x: {{
                    title: {{ display: false, text: 'Días del Mes', font: {{ size: 14, weight: 'bold' }} }},
                    ticks: {{ font: {{ size: 16 }}, maxRotation: 0, minRotation: 0 }},
                    grid: {{ display: false }}
                }},
                y: {{
                    type: 'linear',
                    display: true,
                    position: 'left',
                    title: {{ display: false, text: 'Alcance Diario (%)', font: {{ size: 14, weight: 'bold' }} }},
                    ticks: {{ callback: function(value) {{ return value + '%'; }}, font: {{ size: 16 }}, stepSize: 10 }},
                    min: 0,
                    suggestedMax: Math.max(100, Math.ceil(Math.max(...porcentajes.filter(v => v !== null && v !== undefined), 0) / 10) * 10)
                }}
            }},
            elements: {{ line: {{ borderWidth: 0 }} }}
        }};
        
        options = _tuneOptionsForView(options, view);

        const chart = new Chart(ctx, {{
            type: 'bar',
            data: {{
                labels: fechasOrdenadas.map(fecha => {{
                    if (fecha.includes('-')) {{
                        return fecha.split('-')[0];
                    }}
                    return fecha;
                }}),
                datasets: [{{
                    label: 'Alcance Diario (%)',
                    data: porcentajes,
                    backgroundColor: coloresBarras,
                    borderColor: coloresBarras.map(color => color + 'CC'),
                    borderWidth: 2,
                    borderRadius: 6,
                    borderSkipped: false
                }}]
            }},
            options,
            plugins: [dataLabelsPluginTotal]
        }});
        _storeChart(canvasId, chart);
    }}

    // FUNCIONES AUXILIARES

    function sincronizarSelectoresGlobal() {{
        // Obtener valores actuales de los selectores del header
        const mesSeleccionado = document.getElementById('selectorMes').value;
        const añoSeleccionado = document.getElementById('selectorAño').value;
        
        // Actualizar estado global
        estadoGlobal.mesActual = mesSeleccionado;
        estadoGlobal.añoActual = añoSeleccionado;
        estadoGlobal.periodoActual = `${{mesSeleccionado}}_${{añoSeleccionado}}`;
        
        console.log(`🔄 Estado global actualizado: ${{estadoGlobal.periodoActual}}`);
    }}
    
    function obtenerNumeroMes(mes) {{
        const meses = {{
            'ENERO': 0, 'FEBRERO': 1, 'MARZO': 2, 'ABRIL': 3, 'MAYO': 4, 'JUNIO': 5,
            'JULIO': 6, 'AGOSTO': 7, 'SETIEMBRE': 8, 'SEPTIEMBRE': 8, 
            'OCTUBRE': 9, 'NOVIEMBRE': 10, 'DICIEMBRE': 11
        }};
        return meses[mes.toUpperCase()] || 0;
    }}
    
    // Event listeners para slider e input (sin llamadas a funciones inexistentes)
    document.addEventListener('DOMContentLoaded', function () {{
        const slider = document.getElementById('alcanceObjetivoSlider');
        const input = document.getElementById('alcanceObjetivoInput');
    
        if (slider && input) {{
            slider.addEventListener('input', function () {{
                input.value = this.value;
            }});
    
            input.addEventListener('change', function () {{
                let valor = parseInt(this.value);
                if (isNaN(valor)) valor = 0;
                if (valor < 0) valor = 0;
                if (valor > 200) valor = 200;
                this.value = valor;
                slider.value = valor;
            }});
        }}
    }});

    function obtenerMesesDelAño(año) {{
        // Versión mejorada que maneja variaciones en nombres de meses
        const mesesDisponibles = Object.keys(datosMeses || {{}});
        
        console.log(`🔍 Buscando meses para año ${{año}}`);
        console.log(`📊 Meses disponibles:`, mesesDisponibles);
        
        // Filtrar meses que corresponden al año seleccionado
        const mesesDelAño = mesesDisponibles
            .filter(periodo => {{
                // Extraer año del periodo (última parte después del _)
                const partes = periodo.split('_');
                if (partes.length < 2) return false;
                
                const añoPeriodo = String(partes[partes.length - 1] || '').trim();
                return añoPeriodo === String(año || '').trim();
            }})
            .map(periodo => {{
                // Extraer nombre del mes (todo antes del último _)
                const partes = periodo.split('_');
                partes.pop(); // Remover el año
                return partes.join('_'); // Unir en caso de que haya más de un _
            }})
            .filter(mes => mes && mes !== '');
        
        console.log(`📈 Meses encontrados para ${{año}}:`, mesesDelAño);
        
        // Ordenar meses cronológicamente
        const ordenMeses = {{
            'ENERO': 1, 'FEBRERO': 2, 'MARZO': 3, 'ABRIL': 4, 
            'MAYO': 5, 'JUNIO': 6, 'JULIO': 7, 'AGOSTO': 8, 
            'SETIEMBRE': 9, 'OCTUBRE': 10, 
            'NOVIEMBRE': 11, 'DICIEMBRE': 12
        }};
        
        return mesesDelAño.sort((a, b) => {{
            const mesA = a.toUpperCase();
            const mesB = b.toUpperCase();
            
            // Manejar sinónimos (SETIEMBRE/SEPTIEMBRE)
            const numA = ordenMeses[mesA] || (mesA.includes('SETIEMBRE') ? 9 : 0);
            const numB = ordenMeses[mesB] || (mesB.includes('SEPTIEMBRE') ? 9 : 0);
            
            return numA - numB;
        }});
    }}

    function actualizarSelectorMesSegunAñoHeader() {{
      const selectorMes = document.getElementById('selectorMes');
      const selectorAño = document.getElementById('selectorAño');
      if (!selectorMes || !selectorAño) return;
    
      const año = String(selectorAño.value || '').trim();
      if (!año) return;
    
      const meses = obtenerMesesDelAño(año) || [];
    
      // Si no hay meses, dejamos el selector vacío para evitar “meses fantasmas”
      if (!Array.isArray(meses) || meses.length === 0) {{
        selectorMes.innerHTML = '';
        return;
      }}
    
      const mesActual = String(selectorMes.value || '').trim().toUpperCase();
    
      let html = '';
      meses.forEach(m => {{
        const mesUp = String(m || '').trim().toUpperCase();
        if (!mesUp) return;
    
        const selected = (mesUp === mesActual) ? 'selected' : '';
        const label = mesUp.charAt(0) + mesUp.slice(1).toLowerCase();
        html += `<option value="${{mesUp}}" ${{selected}}>${{label}}</option>\n`;
      }});
    
      selectorMes.innerHTML = html;
    
      // Si el mes actual ya no existe en el año nuevo, usar el primero disponible
      const setMeses = new Set(meses.map(x => String(x).trim().toUpperCase()));
      if (!setMeses.has(mesActual)) {{
        selectorMes.value = String(meses[0]).trim().toUpperCase();
      }}
    }}
    
    function calcularAlcanceTotalMes(periodoCompleto) {{
        console.log(`📊 Calculando alcance TOTAL para: ${{periodoCompleto}}`);
        
        if (!datosMeses || !datosMeses[periodoCompleto]) {{
            console.warn(`⚠️ No hay datos para: ${{periodoCompleto}}`);
            return null;
        }}
        
        const asesores = filtrarAsesoresPorCanal(datosMeses[periodoCompleto] || []);
        console.log(`👥 ${{asesores.length}} asesores encontrados`);
        
        // USAR SOLO DATOS DE SUPERVISORES (sin duplicados)
        const supervisoresUnicos = new Map(); // Usar Map para evitar duplicados
        
        // 1. RECOLECTAR DATOS ÚNICOS DE SUPERVISORES
        asesores.forEach(asesor => {{
            const supervisor = asesor.supervisor;
            
            if (supervisor && supervisor !== 'Sin Supervisor') {{
                // Solo guardar una vez por supervisor
                if (!supervisoresUnicos.has(supervisor)) {{
                    supervisoresUnicos.set(supervisor, {{
                        meta_super: asesor.meta_super || 0,
                        recupero_supervisor: asesor.recupero_supervisor || 0
                    }});
                }}
            }}
        }});
        
        // 2. SUMAR METAS Y RECUPEROS DE SUPERVISORES
        let metaTotal = 0;
        let recuperoTotal = 0;
        
        supervisoresUnicos.forEach((datos, supervisor) => {{
            metaTotal += datos.meta_super;
            recuperoTotal += datos.recupero_supervisor;
            console.log(`👨‍💼 ${{supervisor}}: Meta S/ ${{datos.meta_super.toLocaleString()}}, Recupero S/ ${{datos.recupero_supervisor.toLocaleString()}}`);
        }});
        
        console.log(`💰 Meta total (supervisores únicos): S/ ${{metaTotal.toLocaleString()}}`);
        console.log(`💰 Recupero total (supervisores): S/ ${{recuperoTotal.toLocaleString()}}`);
        
        // 3. CALCULAR ALCANCE
        if (metaTotal > 0) {{
            const alcance = (recuperoTotal / metaTotal) * 100;
            console.log(`🎯 Alcance calculado: ${{alcance.toFixed(2)}}%`);
            return alcance;
        }}
        
        console.warn(`❌ Meta total es 0 para ${{periodoCompleto}}`);
        return 0;
    }}
    
    function calcularLineaTendencia(datos) {{
        // Regresión lineal simple: y = mx + b
        const n = datos.length;
        const indices = datos.map((_, i) => i);
        
        // Filtrar datos nulos
        const datosValidos = datos.filter((d, i) => d !== null);
        const indicesValidos = indices.filter((_, i) => datos[i] !== null);
        
        if (datosValidos.length < 2) return datos;
        
        // Calcular promedios
        const sumX = indicesValidos.reduce((a, b) => a + b, 0);
        const sumY = datosValidos.reduce((a, b) => a + b, 0);
        const sumXY = indicesValidos.reduce((sum, x, i) => sum + x * datosValidos[i], 0);
        const sumX2 = indicesValidos.reduce((sum, x) => sum + x * x, 0);
        
        const m = (datosValidos.length * sumXY - sumX * sumY) / (datosValidos.length * sumX2 - sumX * sumX);
        const b = (sumY - m * sumX) / datosValidos.length;
        
        // Calcular valores de la línea de tendencia
        return indices.map(x => m * x + b);
    }}
    
    function mostrarMensajeSinDatosAnual(canvasId, año) {{
        const canvas = document.getElementById(canvasId);
        if (!canvas) return;
        
        const ctx = canvas.getContext('2d');
        ctx.clearRect(0, 0, canvas.width, canvas.height);
        
        canvas.width = 1200;
        canvas.height = 200;
        canvas.style.display = 'block';
        canvas.style.margin = '0 auto';
        canvas.style.maxWidth = '100%';
        
        ctx.fillStyle = '#95a5a6';
        ctx.font = 'bold 20px Arial';
        ctx.textAlign = 'center';
        ctx.fillText('📊 No hay datos disponibles para este año', canvas.width / 2, canvas.height / 2 - 20);
        
        ctx.fillStyle = '#7f8c8d';
        ctx.font = '16px Arial';
        ctx.fillText(`Año: ${{año}}`, canvas.width / 2, canvas.height / 2 + 20);
    }}

    // ========== FUNCIONES PARA SECCION EVALUACION ========== 
    function setVistaEvaluacion(tipo) {{
      if (tipo === '3M' || tipo === '6M' || tipo === '12M') {{
        vistaEvaluacion = tipo;
      }}

      // Activo visual: 3/6/12 definen rango; OTROS es un modo independiente.
      const btn3  = document.getElementById('btnEval3Meses');
      const btn6  = document.getElementById('btnEval6Meses');
      const btn12 = document.getElementById('btnEval12Meses');
      const btnOtros = document.getElementById('btnEvalOtros');

      btn3?.classList.toggle('activo', vistaEvaluacion === '3M');
      btn6?.classList.toggle('activo', vistaEvaluacion === '6M');
      btn12?.classList.toggle('activo', vistaEvaluacion === '12M');
      btnOtros?.classList.toggle('activo', modoOtrosEvaluacion);

      actualizarTarjetasEvaluacionRapida();
    }}

    function setModoOtrosEvaluacion(activo) {{
      modoOtrosEvaluacion = !!activo;
      document.getElementById('btnEvalOtros')?.classList.toggle('activo', modoOtrosEvaluacion);
      actualizarTarjetasEvaluacionRapida();
    }}

    // ===================== FUNCIONES DE TARJETAS RAPIDAS (EVALUACIÓN) =====================
    
    // Normaliza porcentaje para trabajar SIEMPRE en fracción (0.70 = 70%)
    function _normPctToFrac(v) {{
      if (v === null || v === undefined || v === '') return null;
      const n = Number(v);
      return isFinite(n) ? n : null;
    }}
    
    function _fmtPct(vFrac) {{
      if (vFrac === null || vFrac === undefined) return 'No participó';
      const n = Number(vFrac);
      if (!isFinite(n)) return 'No participó';
      return `${{(n * 100).toFixed(2)}}%`;
    }}
    


    function _normalizarQuintil(valor) {{
      const raw = String(valor ?? '').trim().toUpperCase();
      if (!raw) return null;
      const limpio = raw.replace(/^Q\s*/, '').replace(/,/, '.');
      const n = Number(limpio);
      if (Number.isFinite(n) && Number.isInteger(n) && n >= 1 && n <= 5) return n;
      const exacto = raw.match(/^Q?\s*([1-5])\s*$/);
      return exacto ? Number(exacto[1]) : null;
    }}

    function _colorQuintil(q) {{
      return {{
        1: '#c62828',
        2: '#ef6c00',
        3: '#f9a825',
        4: '#26a69a',
        5: '#29b6f6'
      }}[q] || '#95a5a6';
    }}

    function _renderBarraQuintil(valor) {{
      const q = _normalizarQuintil(valor);
      if (!q) return '<span style="color:#95a5a6; font-weight:700;">-</span>';
      const width = Math.max(20, q * 20);
      return `
        <div class="quintil-barra" title="Quintil ${{q}}">
          <div class="quintil-barra-track">
            <div class="quintil-barra-fill" style="width:${{width}}%; background:${{_colorQuintil(q)}}; color:${{q >= 3 && q <= 4 ? '#2c3e50' : '#fff'}};">${{q}}</div>
          </div>
        </div>
      `;
    }}
    function _renderBarraQuintilGris(valor) {{
      const q = _normalizarQuintil(valor);
      if (!q) return '<span style="color:#95a5a6; font-weight:700;">-</span>';
      const width = Math.max(20, q * 20);
      return `
        <div class="quintil-barra" title="Quintil ${{q}}">
          <div class="quintil-barra-track">
            <div class="quintil-barra-fill gris" style="width:${{width}}%;">${{q}}</div>
          </div>
        </div>
      `;
    }}

    function _quintilIndicadorEvaluacion(periodo, alias, key) {{
      const clave = normalizarTextoAsesor(alias);
      const items = (datosMeses?.[periodo] || [])
        .map(a => {{
          const nombre = String(a.nombre || a.alias_crr || a.indicadores_calidad?.alias || '').trim();
          const valor = obtenerIndicadorAsesorPeriodo(a, key);
          return {{ nombre, clave: normalizarTextoAsesor(nombre), valor }};
        }})
        .filter(it => it.nombre && Number.isFinite(it.valor))
        .sort((a, b) => b.valor - a.valor);
      const index = items.findIndex(it => it.clave === clave);
      if (index < 0 || !items.length) return null;
      return items.length <= 1 ? 5 : Math.max(1, Math.min(5, 5 - Math.floor((index * 5) / items.length)));
    }}

    function _quintilesExtraEvaluacion(periodo, alias) {{
      return {{
        condonacion: _quintilIndicadorEvaluacion(periodo, alias, 'condonacion'),
        cierre: _quintilIndicadorEvaluacion(periodo, alias, 'cierre'),
        calidad_pdp: _quintilIndicadorEvaluacion(periodo, alias, 'calidad_pdp'),
        puntualidad: _quintilIndicadorEvaluacion(periodo, alias, 'puntualidad')
      }};
    }}

    window.evalQuintilesExtraAbiertos = window.evalQuintilesExtraAbiertos || false;

    function toggleEvalQuintilesExtra() {{
      window.evalQuintilesExtraAbiertos = !window.evalQuintilesExtraAbiertos;
      document.querySelectorAll('.eval-quintil-extra-col').forEach(col => col.classList.toggle('oculta', !window.evalQuintilesExtraAbiertos));
      const btn = document.getElementById('btnEvalQuintilesExtra');
      if (btn) btn.textContent = window.evalQuintilesExtraAbiertos ? '-' : '+';
    }}

    function _renderQuintilesExtraEvaluacion(extras) {{
      const datos = [
        ['Condonacion', extras?.condonacion],
        ['Cierre', extras?.cierre],
        ['Calidad', extras?.calidad_pdp],
        ['Puntualidad', extras?.puntualidad]
      ];
      return `<div class="eval-quintiles-extra-box">${{datos.map(([label, q]) => `
        <div class="eval-quintil-extra-item">
          <span class="eval-quintil-extra-label">${{label}}</span>
          ${{_renderBarraQuintilGris(q)}}
        </div>`).join('')}}</div>`;
    }}

    function _calcularPromedioAlcance(alias, fechaIngreso, periodosProm, mapsPorPeriodo) {{
      const nums = [];
    
      for (const periodo of periodosProm) {{
        // Respeta fecha de inicio de gestion (misma regla que usas en export)
        if (!_mesEsValidoParaIngreso(periodo, fechaIngreso)) {{
          continue;
        }}
    
        const reg = mapsPorPeriodo?.[periodo]?.get(alias);
        const v = _normPctToFrac(reg?.alcance);
    
        if (v !== null) nums.push(v);
      }}
    
      if (!nums.length) return null;
      return _avg(nums);
    }}
    
    function _renderListaEvaluacion(listaId, items, modo) {{
      const cont = document.getElementById(listaId);
      if (!cont) return;
    
      let html = '';
    
      items.forEach(it => {{
        const alias = it.alias || '';
        const dni = it.dni || '';
        const sup = it.supervisor || '';
        const pctTxt = _fmtPct(it.pct);
        const pctClass = (modo === 'ok') ? 'porcentaje-100' : 'porcentaje-0';
        const gradClass = (modo === 'ok') ? 'gradiente-100' : 'gradiente-0';
        const det = window.evalDetallePorAlias?.[alias];
        const estadoActual = it.estadoActual || det?.estadoActual || '';
        const estadoTxt = estadoActual ? ` | Estado: ${{estadoActual}}` : '';
        
        html += `
          <div class="asesor-item ${{gradClass}}" style="gap:12px;">
            <div style="display:flex; flex-direction:column; gap:4px; flex:1;">
              <div class="asesor-nombre">${{alias}}</div>
              <div style="font-size:0.85rem; color:#666;">
                Inicio de Gestión: ${{_fmtFechaIngresoUI(det.fechaIngreso)}} · SUP: ${{sup}}${{estadoTxt}}
              </div>
            </div>
    
            <button
              type="button"
              class="btn-eval-detalle"
              data-alias="${{alias}}"
              title="Ver detalle del cálculo"
              style="
                border:none; cursor:pointer;
                padding:8px 10px; border-radius:10px;
                background:#eef2f7; color:#2c3e50; font-weight:800;
              "
            >ℹ️</button>
    
            <div class="asesor-porcentaje ${{pctClass}}">${{pctTxt}}</div>
          </div>
        `;
      }});
    
      cont.innerHTML = html || `
        <div style="padding:18px; text-align:center; color:#666;">
          Sin registros
        </div>
      `;
    }}

    function _getNDesdeVistaEvaluacion() {{
      if (vistaEvaluacion === '12M') return 12;
      if (vistaEvaluacion === '6M') return 6;
      return 3; // default 3M
    }}

    function actualizarTarjetasEvaluacionRapida(supervisorParam = '') {{
      const mesSeleccionado = document.getElementById('selectorMes')?.value;
      const añoSeleccionado = document.getElementById('selectorAño')?.value;
    
      const periodoFiltrado = `${{mesSeleccionado}}_${{añoSeleccionado}}`;
    
      // Si no hay datos del periodo, limpiamos UI
      if (!mesSeleccionado || !añoSeleccionado || !datosMeses?.[periodoFiltrado]) {{
        document.getElementById('lista-eval-ok')?.replaceChildren();
        document.getElementById('lista-eval-bad')?.replaceChildren();
        const ok = document.getElementById('cantidad-eval-ok');
        const bad = document.getElementById('cantidad-eval-bad');
        if (ok) ok.textContent = '0';
        if (bad) bad.textContent = '0';
        const nota = document.getElementById('evalResumenNota');
        if (nota) nota.textContent = 'Selecciona un mes/año con data.';
        actualizarEvaluacionQuintil([], {{ periodoFiltrado }});
        return;
      }}
    
      // 3M / 6M / 12M según Evaluación
      const modoOtros = !!modoOtrosEvaluacion;
      const n = _getNDesdeVistaEvaluacion();
    
      // OTROS ignora los checkbox: siempre usa meses cerrados y muestra asesores en vacaciones/licencia.
      const incluirActual = modoOtros ? false : !!document.getElementById('chkIncluirMesSeleccionadoEval')?.checked;
      const mostrarNuevos = modoOtros ? true : !!document.getElementById('chkMostrarAsesoresNuevosEval')?.checked;
    
      // Filtro por supervisor (header)
      let supervisorFiltro = (supervisorParam || '').trim();
    
      // Si no viene por parámetro, tomar el activo del header
      if (!supervisorFiltro) {{
        const barra = document.getElementById('barraFiltrosSupervisoresGlobal');
        const btnActivo = barra?.querySelector('.filtro-supervisor.activo[data-supervisor]');
        supervisorFiltro = btnActivo?.getAttribute('data-supervisor') || '';
      }}
    
      // Fallback al estado global
      if (!supervisorFiltro) supervisorFiltro = (window.supervisorFiltroActual || 'TODOS');
    
      // Base desde el periodo filtrado
      let baseAsesores = filtrarAsesoresPorCanal((window.baseAsesoresAnalisis?.[periodoFiltrado] || []).slice());
    
      // Aplicar filtro solo para Evaluación
      if (supervisorFiltro && supervisorFiltro !== 'TODOS') {{
        baseAsesores = baseAsesores.filter(p => String(p.supervisor || '').trim() === supervisorFiltro);
      }}

      baseAsesores = baseAsesores.filter(p => {{
        const estado = String(p.estado || '').toUpperCase().trim();
        const esOtros = estado === 'VACACIONES' || estado === 'LICENCIA';
        return modoOtros ? esOtros : !esOtros;
      }});
    
      if (!baseAsesores.length) {{
        _renderListaEvaluacion('lista-eval-ok', [], 'ok');
        _renderListaEvaluacion('lista-eval-bad', [], 'bad');
        const nota = document.getElementById('evalResumenNota');
        if (nota) nota.textContent = 'No hay base de asesores para este periodo.';
        actualizarEvaluacionQuintil([], {{ periodoFiltrado, n, incluirActual, modoOtros, supervisorFiltro }});
        return;
      }}
    
      const okList = [];
      const badList = [];
    
      // detalle para el modal
      window.evalDetallePorAlias = window.evalDetallePorAlias || {{}};
    
      baseAsesores.forEach(p => {{
        const alias = (p.alias_crr || '').trim();
        if (!alias) return;
    
        const dni = (p.dni || '').toString();
        const supervisor = (p.supervisor || '').toString();
        const fechaIngreso = (p.fecha_inicio || '').toString();
        const estadoActual = String(p.estado || '').toUpperCase().trim();
    
        // N meses efectivos
        const det = _calcDetallePromedioEfectivo(
          alias,
          fechaIngreso,
          mesSeleccionado,
          añoSeleccionado,
          n,
          incluirActual
        );
    
        // no completó N meses efectivos
        const mesesUsados = Number(det?.mesesUsados ?? 0);
        const esNuevo = (mesesUsados >= 0 && mesesUsados < n);
    
        if (!mostrarNuevos && esNuevo) return;
    
        // Guardar para modal (incluimos dni/supervisor por comodidad)
        window.evalDetallePorAlias[alias] = {{
          ...det,
          dni,
          supervisor,
          estadoActual,
          modoOtros
        }};
    
        const prom = det.pct; // fracción 0..1 o null
        const item = {{ alias, dni, supervisor, pct: prom, estadoActual }};
    
        // Regla: >=70% verde, <70% rojo
        if (prom !== null && prom >= 0.70) okList.push(item);
        else badList.push(item);
      }});
    
      okList.sort((a, b) => (b.pct ?? -1) - (a.pct ?? -1));
      badList.sort((a, b) => (a.pct ?? 9) - (b.pct ?? 9));
    
      _renderListaEvaluacion('lista-eval-ok', okList, 'ok');
      _renderListaEvaluacion('lista-eval-bad', badList, 'bad');
      actualizarEvaluacionQuintil([...okList, ...badList], {{
        periodoFiltrado,
        n,
        incluirActual,
        modoOtros,
        supervisorFiltro
      }});
    
      const ok = document.getElementById('cantidad-eval-ok');
      const bad = document.getElementById('cantidad-eval-bad');
      if (ok) ok.textContent = String(okList.length);
      if (bad) bad.textContent = String(badList.length);
    
      const nota = document.getElementById('evalResumenNota');
      if (nota) {{
        const modo = incluirActual ? 'ACTUAL' : 'CERRADO';
        if (modoOtros) {{
          nota.textContent =
            `OTROS: VACACIONES / LICENCIA del mes seleccionado. ` +
            `Promedio calculado con ALCANCE_${{n}}M_CERRADO hasta el ultimo mes con datos.`;
        }} else {{
          nota.textContent =
            `Mostrando ALCANCE_${{n}}M_${{modo}} | ` +
            `Clic en detalle para ver el calculo.`;
        }}
      }}
    }}

    function _setEvalQuintilAbierto(q, abierto) {{
      const lista = document.getElementById(`evalQuintilQ${{q}}`);
      const toggle = document.getElementById(`evalQuintilToggleQ${{q}}`);
      const tarjeta = lista?.closest('.eval-quintil-card');
      if (lista) {{
        lista.classList.toggle('colapsada', !abierto);
        lista.setAttribute('aria-hidden', String(!abierto));
      }}
      if (tarjeta) {{
        tarjeta.classList.toggle('quintil-abierto', abierto);
        tarjeta.setAttribute('aria-expanded', String(abierto));
      }}
      if (toggle) toggle.textContent = abierto ? '-' : '+';
    }}

    function toggleEvalQuintil(q) {{
      const lista = document.getElementById(`evalQuintilQ${{q}}`);
      if (!lista) return;
      _setEvalQuintilAbierto(q, lista.classList.contains('colapsada'));
    }}

    function actualizarEvaluacionQuintil(itemsEvaluacion = [], contexto = {{}}) {{
      const grupos = {{ 1: [], 2: [], 3: [], 4: [], 5: [] }};
      const nota = document.getElementById('evalQuintilNota');

      // La lista ya viene de Evaluacion por Promedios y, por tanto, ya respeta
      // 3/6/12 meses, incluir mes, nuevos, OTROS y supervisor.
      const ordenados = (Array.isArray(itemsEvaluacion) ? itemsEvaluacion : [])
        .map(item => {{
          const rawPct = item?.pct;
          const numeroPct = (rawPct === null || rawPct === undefined || rawPct === '') ? null : Number(rawPct);
          return {{
            alias: String(item?.alias || '').trim(),
            alcance: Number.isFinite(numeroPct) ? numeroPct : null
          }};
        }})
        .filter(item => item.alias)
        .sort((a, b) => {{
          if (a.alcance === null && b.alcance === null) return a.alias.localeCompare(b.alias, 'es');
          if (a.alcance === null) return 1;
          if (b.alcance === null) return -1;
          return (b.alcance - a.alcance) || a.alias.localeCompare(b.alias, 'es');
        }});

      const total = ordenados.length;
      ordenados.forEach((item, indice) => {{
        // Q5 = 20% superior; Q1 = 20% inferior. La formula reparte toda
        // la lista incluso cuando el total no es multiplo de cinco.
        const q = Math.max(1, 5 - Math.floor((indice * 5) / Math.max(total, 1)));
        grupos[q].push(item);
      }});

      window.__evalQuintilExportRows = [5, 4, 3, 2, 1].flatMap(q =>
        grupos[q].map(it => ({{
          asesor: it.alias,
          promedio: it.alcance,
          quintil: `Q${{q}}`
        }}))
      );

      [5, 4, 3, 2, 1].forEach(q => {{
        const cont = document.getElementById(`evalQuintilQ${{q}}`);
        const resumen = document.getElementById(`evalQuintilResumenQ${{q}}`);
        const toggle = document.getElementById(`evalQuintilToggleQ${{q}}`);
        if (!cont) return;

        const estabaAbierto = !cont.classList.contains('colapsada');
        const items = grupos[q];
        const valores = items.map(it => it.alcance).filter(Number.isFinite);
        const promedio = valores.length ? valores.reduce((acc, valor) => acc + valor, 0) / valores.length : null;

        if (resumen) {{
          resumen.innerHTML = `
            <div class="eval-quintil-metric">
              <span class="eval-quintil-metric-label">Asesores</span>
              <span class="eval-quintil-metric-value">${{items.length}}</span>
            </div>
            <div class="eval-quintil-metric">
              <span class="eval-quintil-metric-label">Promedio</span>
              <span class="eval-quintil-metric-value">${{promedio === null ? '-' : _fmtPct(promedio)}}</span>
            </div>`;
        }}

        cont.innerHTML = items.map((it, index) => `
          <div class="eval-quintil-row" style="--quintil-delay:${{Math.min(index, 10) * 32}}ms">
            <div class="eval-quintil-asesor" title="${{it.alias}}">${{it.alias}}</div>
            <div class="eval-quintil-alcance">${{_fmtPct(it.alcance)}}</div>
          </div>`).join('') || '<div style="padding:14px 8px; text-align:center; color:#666; font-size:0.85rem;">Sin registros</div>';

        _setEvalQuintilAbierto(q, estabaAbierto);
        if (toggle && !estabaAbierto) toggle.textContent = '+';
      }});

      if (nota) {{
        const periodo = String(contexto.periodoFiltrado || 'este periodo').replace('_', ' ');
        const n = Number(contexto.n || _getNDesdeVistaEvaluacion());
        const modo = contexto.modoOtros ? 'CERRADO · OTROS' : (contexto.incluirActual ? 'ACTUAL' : 'CERRADO');
        const supervisor = contexto.supervisorFiltro && contexto.supervisorFiltro !== 'TODOS'
          ? ` · ${{contexto.supervisorFiltro}}`
          : ' · Todos los equipos';
        nota.textContent = total
          ? `Quintiles de ALCANCE_${{n}}M_${{modo}} · ${{periodo}}${{supervisor}}. Q5 es el 20% superior y Q1 el inferior.`
          : `No hay promedios para formar quintiles en ${{periodo}}${{supervisor}}.`;
      }}
    }}

    function _getSupervisorFiltroEvaluacion() {{
      let supervisorFiltro = '';
    
      const barra = document.getElementById('barraFiltrosSupervisoresGlobal');
      const btnActivo = barra?.querySelector('.filtro-supervisor.activo[data-supervisor]');
      supervisorFiltro = btnActivo?.getAttribute('data-supervisor') || '';
    
      if (!supervisorFiltro) {{
        supervisorFiltro = window.supervisorFiltroActual || '';
      }}
    
      if (!supervisorFiltro && window.supervisorSeleccionado) {{
        supervisorFiltro = window.supervisorSeleccionado;
      }}
    
      supervisorFiltro = String(supervisorFiltro || '').trim();
    
      if (!supervisorFiltro || supervisorFiltro === 'TODOS') return '';
      return supervisorFiltro;
    }}

    // ===================== HELPERS DE FECHAS / PERIODOS =====================
    
    // MES (nombre) -> número
    function _mesToNum(mesNombre) {{
      const m = String(mesNombre || '').toUpperCase();
      const map = {{
        'ENERO': 1, 'FEBRERO': 2, 'MARZO': 3, 'ABRIL': 4, 'MAYO': 5, 'JUNIO': 6,
        'JULIO': 7, 'AGOSTO': 8, 'SETIEMBRE': 9, 'OCTUBRE': 10, 'NOVIEMBRE': 11, 'DICIEMBRE': 12
      }};
      return map[m] || 0;
    }}
    
    // número -> MES (nombre)
    function _numToMes(n) {{
      const arr = ['', 'ENERO','FEBRERO','MARZO','ABRIL','MAYO','JUNIO',
                   'JULIO','AGOSTO','SETIEMBRE','OCTUBRE','NOVIEMBRE','DICIEMBRE'];
      return arr[n] || '';
    }}
    
    // Retroceder un mes
    function _retroMes(m, y) {{
      let mm = m - 1;
      let yy = y;
      if (mm <= 0) {{ mm = 12; yy = y - 1; }}
      return {{ m: mm, y: yy }};
    }}
    
    // "MES_AÑO" -> {{ y, m }}
    function _periodoToYM(periodo) {{
      const p = String(periodo || '').split('_');
      if (p.length !== 2) return {{ y: 0, m: 0 }};
      const m = _mesToNum(p[0]);
      const y = parseInt(p[1], 10);
      return {{ y: y || 0, m: m || 0 }};
    }}
    
    // y,m -> YYYYMM
    function _ymToNum(y, m) {{
      return (y * 100) + m;
    }}

    function _periodoFromYM(y, m) {{
      return `${{_numToMes(m)}}_${{String(y)}}`;
    }}
    
    function _iterPeriodosAtras(mesNombre, añoStr, incluirActual, limite = 36) {{
      const y0 = parseInt(añoStr, 10);
      const m0 = _mesToNum(mesNombre);
      if (!y0 || !m0) return [];
    
      // si NO incluirActual, empezamos en el mes anterior al seleccionado
      let cur = incluirActual ? {{ m: m0, y: y0 }} : _retroMes(m0, y0);
    
      const out = [];
      for (let k = 0; k < limite; k++) {{
        out.push(_periodoFromYM(cur.y, cur.m));
        cur = _retroMes(cur.m, cur.y);
      }}
      return out;
    }}

    // ===================== CACHE DE MAPAS POR PERIODO =====================
    window._evalMapsCache = window._evalMapsCache || {{}};
    
    function _getMapPeriodoCached(periodo) {{
      if (!periodo) return new Map();
      if (!window._evalMapsCache[periodo]) {{
        window._evalMapsCache[periodo] = _mapBasePeriodoPorAlias(periodo) || new Map();
      }}
      return window._evalMapsCache[periodo];
    }}
    
    // ===================== FECHA CON HORA A SOLO FECHA=====================

    function _fmtFechaIngresoUI(fechaIngresoStr) {{
      const s = String(fechaIngresoStr || '').trim();
      if (!s) return '—';
    
      // yyyy-mm-dd
      const m1 = s.match(/^(\d{{4}})-(\d{{2}})-(\d{{2}})/);
      if (m1) {{
        return `${{m1[3]}}/${{m1[2]}}/${{m1[1]}}`;
      }}
    
      // dd/mm/yyyy (ya viene bien)
      const m2 = s.match(/^(\d{{2}})\/(\d{{2}})\/(\d{{4}})/);
      if (m2) {{
        return s;
      }}
    
      return s; // fallback
    }}
    
    // ===================== FECHA DE INICIO =====================
    
    // "dd/mm/yyyy" o "yyyy-mm-dd" -> YYYYMM
    function _fechaIngresoToYMNum(fechaIngresoStr) {{
      const s = String(fechaIngresoStr || '').trim();
      if (!s) return 0;
    
      // yyyy-mm-dd
      const m1 = s.match(/^(\d{{4}})-(\d{{2}})-(\d{{2}})/);
      if (m1) {{
        const y = parseInt(m1[1], 10);
        const m = parseInt(m1[2], 10);
        return _ymToNum(y, m);
      }}
    
      // dd/mm/yyyy
      const m2 = s.match(/^(\d{{2}})\/(\d{{2}})\/(\d{{4}})/);
      if (m2) {{
        const y = parseInt(m2[3], 10);
        const m = parseInt(m2[2], 10);
        return _ymToNum(y, m);
      }}
    
      return 0;
    }}

    // ===================== NUEVOS ASESORES =====================

    function _getAnclaYMNum(mesSel, añoSel, incluirActual) {{
      const y = parseInt(añoSel, 10);
      const m = _mesToNum(mesSel);
      if (!y || !m) return null;
    
      if (incluirActual) return _ymToNum(y, m);
    
      const prev = _retroMes(m, y);
      return _ymToNum(prev.y, prev.m);
    }}

    function _esAsesorNuevo(fechaIngreso, anclaYMNum, umbralMeses) {{
      const ingresoYMNum = _fechaIngresoToYMNum(fechaIngreso);
      if (!ingresoYMNum || !anclaYMNum) return false;
    
      const mesesAntiguedad = anclaYMNum - ingresoYMNum;
      return mesesAntiguedad < umbralMeses;
    }}
        
    // ===================== PERIODOS HACIA ATRÁS =====================
    
    // Devuelve N periodos hacia atrás
    // incluirActual = true  -> incluye mes seleccionado
    // incluirActual = false -> empieza desde el mes anterior
    function _getPeriodosAtras(mesNombre, anioStr, cantidad, incluirActual) {{
      const m0 = _mesToNum(mesNombre);
      const y0 = parseInt(anioStr, 10);
      if (!m0 || !y0) return [];
    
      let m = m0;
      let y = y0;
    
      if (!incluirActual) {{
        const r = _retroMes(m, y);
        m = r.m;
        y = r.y;
      }}
    
      const out = [];
      while (out.length < cantidad) {{
        // Omitir setiembre (regla del negocio)
        if (m !== 9) {{
          out.push(`${{_numToMes(m)}}_${{y}}`);
        }}
        const r2 = _retroMes(m, y);
        m = r2.m;
        y = r2.y;
      }}
    
      return out.reverse(); // antiguo -> reciente
    }}
    
    // ===================== MAPAS POR ALIAS =====================
    
    // Mapea datosMeses[periodo] por alias_crr
    function _mapPeriodoPorAlias(periodo) {{
      const arr = datosMeses?.[periodo] || [];
      const map = new Map();
      arr.forEach(a => {{
        const alias = (a.alias_crr || '').trim();
        if (alias) map.set(alias, a);
      }});
      return map;
    }}
    
    // Mapea baseAsesoresAnalisis[periodo] por alias_crr
    function _mapBasePeriodoPorAlias(periodo) {{
      const arr = window.baseAsesoresAnalisis?.[periodo] || [];
      const map = new Map();
      arr.forEach(p => {{
        const alias = (p.alias_crr || '').trim();
        if (alias) map.set(alias, p);
      }});
      return map;
    }}
    
    // ===================== CÁLCULOS =====================
    
    // Promedio seguro
    function _avg(nums) {{
      if (!nums || !nums.length) return null;
      const s = nums.reduce((acc, x) => acc + x, 0);
      return s / nums.length;
    }}
    
    // Clasificación por porcentaje
    function _clasificarPorcentaje(pct) {{
      const v = Number(pct || 0);
      if (v > 100) return '>100%';
      if (v > 70) return '>70%';
      if (v > 40) return '>40%';
      return '>0%';
    }}









    // ===================== MODAL DE DETALLE EVALUACIÓN =====================
    function _calcDetallePromedioEfectivo(alias, fechaIngreso, mesSel, añoSel, n, incluirActual) {{
      const ingresoYM = _fechaIngresoToYMNum(fechaIngreso);
    
      const candidatos = _iterPeriodosAtras(mesSel, añoSel, incluirActual, 48);
    
      const incluidos = [];
      const filas = [];
    
      // ✅ ahora acumulamos alcances mensuales (fracción), no sumas
      const fracsMensuales = [];
    
      for (const periodo of candidatos) {{
        if (incluidos.length >= n) break;
    
        const {{ y, m }} = _periodoToYM(periodo);
        const ymNum = _ymToNum(y, m);
    
        // 1) Respeta fecha ingreso
        if (ingresoYM && ymNum < ingresoYM) {{
          filas.push({{
            periodo,
            estado: '—',
            meta: null,
            recupero: null,
            alcanceFrac: null,
            incluido: false,
            motivo: 'Anterior a fecha de inicio',
            q_alc: '',
            q_extra: {{}}
          }});
          break;
        }}
    
        // 2) Debe existir registro en base del periodo
        const mp = _getMapPeriodoCached(periodo);
        const reg = mp.get(alias);
    
        if (!reg) {{
          filas.push({{
            periodo,
            estado: '—',
            meta: null,
            recupero: null,
            alcanceFrac: null,
            incluido: false,
            motivo: 'Sin registro en base del periodo',
            q_alc: '',
            q_extra: {{}}
          }});
          continue;
        }}
    
        const estado = String(reg.estado || 'CALL').toUpperCase().trim();
        const meta = Number(reg.meta ?? 0);
        const recupero = Number(reg.recupero ?? 0);
    
        // 3) Solo CALL
        if (estado !== 'CALL') {{
          filas.push({{
            periodo,
            estado,
            meta,
            recupero,
            alcanceFrac: null,
            incluido: false,
            motivo: 'No activo (estado distinto a CALL)',
            q_alc: reg.q_alc || '',
            q_extra: _quintilesExtraEvaluacion(periodo, alias)
          }});
          continue;
        }}
    
        // 4) Meta > 0
        if (!(meta > 0)) {{
          filas.push({{
            periodo,
            estado,
            meta,
            recupero,
            alcanceFrac: null,
            incluido: false,
            motivo: 'Meta = 0 (no evaluable)',
            q_alc: reg.q_alc || '',
            q_extra: _quintilesExtraEvaluacion(periodo, alias)
          }});
          continue;
        }}
    
        // ✅ Alcance mensual (recupero / meta)
        const fracMes = recupero / meta;
    
        incluidos.push(periodo);
        fracsMensuales.push(fracMes);
    
        filas.push({{
          periodo,
          estado,
          meta,
          recupero,
          alcanceFrac: fracMes,
          incluido: true,
          motivo: 'Mes efectivo (CALL + meta>0)',
          q_alc: reg.q_alc || '',
          q_extra: _quintilesExtraEvaluacion(periodo, alias)
        }});
      }}
    
      const mesesUsados = incluidos.length;
    
      // ✅ PROMEDIO SIMPLE de alcances mensuales
      let pct = null;
      if (fracsMensuales.length > 0) {{
        const sum = fracsMensuales.reduce((a, b) => a + b, 0);
        pct = sum / fracsMensuales.length; // fracción 0..1
      }}
    
      return {{
        alias,
        fechaIngreso,
        nObjetivo: n,
        incluirActual,
        mesesUsados,
        incluidos,
        filas,
        pct
      }};
    }}

    // ===================== EXPORTAR ANALISIS (EVALUACIÓN) =====================
    async function exportarAnalisisEvaluacion() {{
        const mesSeleccionado = document.getElementById('selectorMes')?.value;
        const añoSeleccionado = document.getElementById('selectorAño')?.value;
        const periodoFiltrado = `${{mesSeleccionado}}_${{añoSeleccionado}}`;
        
        if (!datosMeses?.[periodoFiltrado]) {{
        alert('No hay datos para exportar');
        return;
        }}
        
        // Vista Evaluación
        const n = _getNDesdeVistaEvaluacion();
        
        const periodosCols = _getPeriodosAtras(mesSeleccionado, añoSeleccionado, n + 1, true);
        
        const periodosCerrado = _getPeriodosAtras(mesSeleccionado, añoSeleccionado, n, false);
        
        const periodosActual = _getPeriodosAtras(mesSeleccionado, añoSeleccionado, n, true);
        
        const mapsPorPeriodo = {{}};
        [...new Set([...periodosCols, ...periodosCerrado, ...periodosActual])].forEach(p => {{
        mapsPorPeriodo[p] = _mapBasePeriodoPorAlias(p);
        }});
        
        const supervisorFiltro = _getSupervisorFiltroEvaluacion();
        
        let baseAsesores = (window.baseAsesoresAnalisis?.[periodoFiltrado] || []).slice();
        
        if (supervisorFiltro) {{
          baseAsesores = baseAsesores.filter(p =>
            String(p.supervisor || '').trim() === supervisorFiltro
          );
        }}
        
        const COLOR1 = 'FF0D47A1'; // azul oscuro
        const COLOR2 = 'FFFCCF10'; // amarillo
        const COLOR3 = 'FFFFB300'; // ambar
    
        const wb = new ExcelJS.Workbook();
        const ws = wb.addWorksheet('ANALISIS');
        
        // Encabezados
        const headersFijos = ['DNI', 'ALIAS', 'Fecha de Inicio', 'Estado'];
        const headersMeses = periodosCols;
        
        const headersCalc = [
          `ALCANCE_${{n}}M_CERRADO`,
          `ALCANCE_${{n}}M_ACTUAL`
        ];
        
        const headersFinal = ['CARTERA', 'EQUIPO'];
        const encabezados = [...headersFijos, ...headersMeses, ...headersCalc, ...headersFinal];
        
        ws.addRow(encabezados);
        
        const headerRow = ws.getRow(1);
        headerRow.height = 40;
        
        headerRow.alignment = {{
          horizontal: 'center',
          vertical: 'middle',
          wrapText: true
        }};
        
        headerRow.font = {{
          bold: true,
          color: {{ argb: 'FFFFFFFF' }}
        }};
        
        // Fondo 
        headerRow.eachCell((cell) => {{
          cell.fill = {{
            type: 'pattern',
            pattern: 'solid',
            fgColor: {{ argb: 'FF0D47A1' }}
          }};
        }});
        
        // Estilo encabezados por grupo
        headerRow.eachCell((cell, colNumber) => {{
        cell.font = {{ bold: true, color: {{ argb: 'FFFFFFFF' }} }};
        cell.alignment = {{ vertical: 'middle', horizontal: 'center' }};
        cell.border = {{
          top: {{ style: 'thin', color: {{ argb: 'FFD0D0D0' }} }},
          left: {{ style: 'thin', color: {{ argb: 'FFD0D0D0' }} }},
          bottom: {{ style: 'thin', color: {{ argb: 'FFD0D0D0' }} }},
          right: {{ style: 'thin', color: {{ argb: 'FFD0D0D0' }} }}
        }};
        
        const idxMesInicio = 5;
        const idxMesFin = 4 + headersMeses.length;
        const idxCalcInicio = idxMesFin + 1;
        const idxCalcFin = idxMesFin + headersCalc.length;
        const idxFinalInicio = idxCalcFin + 1;
        
        let fillColor = COLOR1;
        if (colNumber >= idxMesInicio && colNumber <= idxMesFin) fillColor = COLOR2;
        else if (colNumber >= idxCalcInicio && colNumber <= idxCalcFin) fillColor = COLOR3;
        else if (colNumber >= idxFinalInicio) fillColor = COLOR1;
        
        const textoNegro = (fillColor === COLOR2);
        cell.font = {{ bold: true, color: {{ argb: textoNegro ? 'FF000000' : 'FFFFFFFF' }} }};
        
        cell.fill = {{
          type: 'pattern',
          pattern: 'solid',
          fgColor: {{ argb: fillColor }}
        }};
        }});
        
        // Filas
        baseAsesores.forEach(asesorBase => {{
        const alias = String(asesorBase.alias_crr || '').trim();
        const dni = String(asesorBase.dni || '').trim();
        const estado = String(asesorBase.estado || 'CALL').toUpperCase().trim();
        
        let fechaIngreso = String(asesorBase.fecha_inicio || '').trim();
        if (fechaIngreso.includes(' ')) fechaIngreso = fechaIngreso.split(' ')[0];
        if (/^\d{{4}}-\d{{2}}-\d{{2}}$/.test(fechaIngreso)) {{
          const [y, m, d] = fechaIngreso.split('-');
          fechaIngreso = `${{d}}/${{m}}/${{y}}`;
        }}
        
        const supervisor = String(asesorBase.supervisor || '').trim();
        
        let cartera = '';
        try {{
          cartera = String(
            datosSupervisores?.[supervisor]?.[periodoFiltrado]?.cartera || ''
          ).trim();
        }} catch (e) {{
          cartera = '';
        }}
        
        const ingresoYM = _fechaIngresoToYMNum(fechaIngreso);
        
        // Valores por mes
        const alcances = [];
        periodosCols.forEach(p => {{
          const {{ y, m }} = _periodoToYM(p);
          const ymNum = _ymToNum(y, m);
        
          if (ingresoYM && ymNum < ingresoYM) {{
            alcances.push("No participó");
            return;
          }}
        
          const reg = mapsPorPeriodo[p]?.get(alias);
          if (!reg) {{
            alcances.push("No participó");
            return;
          }}
        
          const estadoReg = String(reg.estado || '').toUpperCase().trim();
        
          if (estadoReg === 'VACACIONES') {{
            alcances.push("Vacaciones");
            return;
          }}
        
          const val = reg.alcance;
          alcances.push((val === null || val === undefined) ? "No participó" : Number(val));
        }});
        
        // Promedio cerrado
        const detCerrado = _calcDetallePromedioEfectivo(
          alias,
          fechaIngreso,
          mesSeleccionado,
          añoSeleccionado,
          n,
          false
        );
        
        // Promedio actual
        const detActual = _calcDetallePromedioEfectivo(
          alias,
          fechaIngreso,
          mesSeleccionado,
          añoSeleccionado,
          n,
          true
        );

        const promCerrado = detCerrado?.pct ?? null;
        const promActual  = detActual?.pct ?? null;
        
        const row = [
          dni,
          alias,
          fechaIngreso,
          estado,
          ...alcances,
          promCerrado,
          promActual,
          cartera,
          supervisor
        ];
        
        ws.addRow(row);
        }});
        
        // Formatos de columnas
        ws.getColumn(1).numFmt = '@'; // DNI como texto
        
        const colMesInicio = 5;
        const colMesFin = 4 + periodosCols.length;
        for (let c = colMesInicio; c <= colMesFin; c++) {{
        ws.getColumn(c).numFmt = '0.00%';
        }}
        ws.getColumn(colMesFin + 1).numFmt = '0.00%';
        ws.getColumn(colMesFin + 2).numFmt = '0.00%';
        
        // Auto ancho simple
        ws.columns.forEach(col => {{
        col.width = 18;
        }});
        ws.getColumn(2).width = 28; // alias
        ws.getColumn(3).width = 16; // fecha
        ws.getColumn(9 + periodosCols.length).width = 22;
        
        // Guardar
        const fechaHoy = new Date();
        const yy = fechaHoy.getFullYear();
        const mm = String(fechaHoy.getMonth() + 1).padStart(2, '0');
        const dd = String(fechaHoy.getDate()).padStart(2, '0');
        const stamp = `${{yy}}${{mm}}${{dd}}`;
        
        const sufSupervisor = supervisorFiltro
          ? `_${{supervisorFiltro.replace(/[\\/:*?"<>|]/g, '_')}}`
          : '_TODOS';
        
        const nombreArchivo = `Analisis_Evaluacion${{sufSupervisor}}_${{periodoFiltrado}}_${{stamp}}.xlsx`;
        
        const buffer = await wb.xlsx.writeBuffer();
        saveAs(new Blob([buffer], {{
        type: "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
        }}), nombreArchivo);
    }}

    // ===================== EXPORTAR QUINTILES (EVALUACION) =====================
    async function exportarQuintilesEvaluacionPromedios() {{
      const rows = Array.isArray(window.__evalQuintilExportRows)
        ? window.__evalQuintilExportRows.filter(row => row?.asesor)
        : [];

      if (!rows.length) {{
        alert('No hay quintiles de evaluacion para exportar');
        return;
      }}

      const mesSeleccionado = document.getElementById('selectorMes')?.value || '';
      const anioSeleccionado = document.getElementById('selectorAño')?.value || '';
      const periodoFiltrado = mesSeleccionado && anioSeleccionado ? `${{mesSeleccionado}}_${{anioSeleccionado}}` : 'PERIODO';
      const supervisorFiltro = _getSupervisorFiltroEvaluacion();

      const wb = new ExcelJS.Workbook();
      wb.creator = 'Ranking Evaluacion';
      wb.created = new Date();
      const ws = wb.addWorksheet('QUINTILES');

      ws.addRow(['Asesor', 'Promedio', 'Quintil']);
      const header = ws.getRow(1);
      header.height = 28;
      header.eachCell(cell => {{
        cell.font = {{ bold: true, color: {{ argb: 'FFFFFFFF' }} }};
        cell.alignment = {{ horizontal: 'center', vertical: 'middle' }};
        cell.fill = {{ type: 'pattern', pattern: 'solid', fgColor: {{ argb: 'FF2C3E50' }} }};
        cell.border = {{
          top: {{ style: 'thin', color: {{ argb: 'FFD9E2EC' }} }},
          left: {{ style: 'thin', color: {{ argb: 'FFD9E2EC' }} }},
          bottom: {{ style: 'thin', color: {{ argb: 'FFD9E2EC' }} }},
          right: {{ style: 'thin', color: {{ argb: 'FFD9E2EC' }} }}
        }};
      }});

      const colorPorQuintil = {{
        Q5: 'FFD6F1FF',
        Q4: 'FFDDF7F3',
        Q3: 'FFFFF4C2',
        Q2: 'FFFFE2C2',
        Q1: 'FFFFD6D6'
      }};

      rows.forEach(row => {{
        const promedioNum = Number(row.promedio);
        const excelRow = ws.addRow([
          row.asesor,
          Number.isFinite(promedioNum) ? promedioNum : null,
          row.quintil
        ]);
        excelRow.eachCell(cell => {{
          cell.border = {{
            top: {{ style: 'thin', color: {{ argb: 'FFE8EEF5' }} }},
            left: {{ style: 'thin', color: {{ argb: 'FFE8EEF5' }} }},
            bottom: {{ style: 'thin', color: {{ argb: 'FFE8EEF5' }} }},
            right: {{ style: 'thin', color: {{ argb: 'FFE8EEF5' }} }}
          }};
          cell.alignment = {{ vertical: 'middle' }};
        }});
        excelRow.getCell(2).numFmt = '0.00%';
        excelRow.getCell(2).alignment = {{ horizontal: 'right', vertical: 'middle' }};
        excelRow.getCell(3).alignment = {{ horizontal: 'center', vertical: 'middle' }};
        excelRow.getCell(3).font = {{ bold: true }};
        excelRow.eachCell(cell => {{
          cell.fill = {{
            type: 'pattern',
            pattern: 'solid',
            fgColor: {{ argb: colorPorQuintil[row.quintil] || 'FFFFFFFF' }}
          }};
        }});
      }});

      ws.columns = [
        {{ key: 'asesor', width: 34 }},
        {{ key: 'promedio', width: 14 }},
        {{ key: 'quintil', width: 12 }}
      ];
      ws.autoFilter = {{ from: 'A1', to: `C${{Math.max(rows.length + 1, 1)}}` }};
      ws.views = [{{ state: 'frozen', ySplit: 1 }}];

      const fechaHoy = new Date();
      const yy = fechaHoy.getFullYear();
      const mm = String(fechaHoy.getMonth() + 1).padStart(2, '0');
      const dd = String(fechaHoy.getDate()).padStart(2, '0');
      const stamp = `${{yy}}${{mm}}${{dd}}`;
      const sufSupervisor = supervisorFiltro
        ? `_${{supervisorFiltro.replace(/[\\/:*?"<>|]/g, '_')}}`
        : '_TODOS';
      const nombreArchivo = `Quintiles_Evaluacion${{sufSupervisor}}_${{periodoFiltrado}}_${{stamp}}.xlsx`;

      const buffer = await wb.xlsx.writeBuffer();
      saveAs(new Blob([buffer], {{
        type: 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
      }}), nombreArchivo);
    }}

    // ===================== EXPORTAR RyM =====================

    async function exportarRecuperoYMetasEvaluacion() {{
      const mesSeleccionado = document.getElementById('selectorMes')?.value;
      const añoSeleccionado = document.getElementById('selectorAño')?.value;
      const periodoFiltrado = `${{mesSeleccionado}}_${{añoSeleccionado}}`;
    
      if (!datosMeses?.[periodoFiltrado]) {{
        alert('No hay datos para exportar');
        return;
      }}
    
      const n = _getNDesdeVistaEvaluacion();
      const incluirMesSeleccionado = !!document.getElementById('chkIncluirMesSeleccionadoEval')?.checked;
      const cantidad = incluirMesSeleccionado ? (n + 1) : n;
      const incluirMesFlag = incluirMesSeleccionado ? true : false;
      const periodos = _getPeriodosAtras(mesSeleccionado, añoSeleccionado, cantidad, incluirMesFlag);
    
      if (!periodos?.length) {{
        alert('No se pudieron calcular los periodos');
        return;
      }}
    
      // Mapas por periodo para encontrar rápido por alias_crr
      const mapsBase = {{}};
      periodos.forEach(p => {{
        mapsBase[p] = _mapBasePeriodoPorAlias(p);
      }});
    
      // Base (misma lista que EXPORTAR ANALISIS)
        const supervisorFiltro = _getSupervisorFiltroEvaluacion();
        
        let baseAsesores = (window.baseAsesoresAnalisis?.[periodoFiltrado] || []).slice();
        
        if (supervisorFiltro) {{
          baseAsesores = baseAsesores.filter(p =>
            String(p.supervisor || '').trim() === supervisorFiltro
          );
        }}
        
      if (!baseAsesores.length) {{
        alert('No hay base de asesores para este periodo');
        return;
      }}
    
      const wb = new ExcelJS.Workbook();
      const ws = wb.addWorksheet('RECUPERO_METAS');
    
      // Encabezados
      const headersFijos = ['DNI', 'ALIAS', 'Fecha de Inicio'];
      const headersMeses = [];
    
      periodos.forEach(p => {{
        headersMeses.push(`RECUPERO\n${{p}}`); // doble línea
        headersMeses.push(`META\n${{p}}`);     // doble línea
      }});
    
      const headersFinal = ['SUPERVISOR'];
    
      const COLOR_RECUPERO = 'FFFF9800'; // naranja
      const COLOR_META     = 'FFD32F2F'; // rojo
      const COLOR_FIJO     = 'FF0D47A1'; // azul (para fijos + supervisor)
    
      ws.addRow([ ...headersFijos, ...headersMeses, ...headersFinal ]);
    
      const headerRow = ws.getRow(1);
    
      headerRow.height = 40;
      headerRow.alignment = {{ horizontal: 'center', vertical: 'middle', wrapText: true }};
      headerRow.font = {{ bold: true, color: {{ argb: 'FFFFFFFF' }} }};
    
      // encabezados
      headerRow.eachCell((cell) => {{
        const text = String(cell.value || '').toUpperCase();
    
        if (text.startsWith('RECUPERO')) {{
          cell.fill = {{
            type: 'pattern',
            pattern: 'solid',
            fgColor: {{ argb: COLOR_RECUPERO }}
          }};
          return;
        }}
    
        if (text.startsWith('META')) {{
          cell.fill = {{
            type: 'pattern',
            pattern: 'solid',
            fgColor: {{ argb: COLOR_META }}
          }};
          return;
        }}
    
        // Fijos (DNI/ALIAS/Fecha) + SUPERVISOR
        cell.fill = {{
          type: 'pattern',
          pattern: 'solid',
          fgColor: {{ argb: COLOR_FIJO }}
        }};
      }});
    
      // Filas
      baseAsesores.forEach(p => {{
        const alias = (p.alias_crr || '').trim();
        if (!alias) return;
    
        const dni = (p.dni || '').toString();
        const fechaIngreso = (p.fecha_inicio || '').toString();
        const supervisorBase = (p.supervisor || '').toString();
    
        const valoresMeses = [];
        let supervisor = supervisorBase;
    
        periodos.forEach(periodo => {{
          const regBase = mapsBase[periodo]?.get(alias);
    
          // Respeta ingreso
          if (!_mesEsValidoParaIngreso(periodo, fechaIngreso)) {{
            valoresMeses.push('');
            valoresMeses.push('');
            return;
          }}
    
          const rec = regBase ? Number(regBase.recupero || 0) : 0;
          const met = regBase ? Number(regBase.meta || 0) : 0;
    
          valoresMeses.push(rec);
          valoresMeses.push(met);
        }});
    
        const row = [
          dni,
          alias,
          fechaIngreso,
          ...valoresMeses,
          supervisor || ''
        ];
    
        ws.addRow(row);
      }});
    
      // Formatos
      ws.getColumn(1).numFmt = '@'; // DNI texto
      ws.getColumn(2).width = 28;
      ws.getColumn(3).width = 16;
    
      // Columnas numéricas (recupero/meta)
      const colNumInicio = 4;
      const colNumFin = 3 + headersMeses.length;
      for (let c = colNumInicio; c <= colNumFin; c++) {{
        ws.getColumn(c).numFmt = '#,##0.00';
        ws.getColumn(c).width = 16;
      }}
    
      // Supervisor
      ws.getColumn(colNumFin + 1).width = 22;
    
      // Guardar
      const fechaHoy = new Date();
      const yy = fechaHoy.getFullYear();
      const mm = String(fechaHoy.getMonth() + 1).padStart(2, '0');
      const dd = String(fechaHoy.getDate()).padStart(2, '0');
      const stamp = `${{yy}}${{mm}}${{dd}}`;
      const suf = incluirMesSeleccionado ? 'INCLUYE_MES' : 'SOLO_ANTERIORES';
      const sufSupervisor = supervisorFiltro
          ? `_${{supervisorFiltro.replace(/[\\/:*?"<>|]/g, '_')}}`
          : '_TODOS';
        
      const nombreArchivo = `Recupero_Metas${{sufSupervisor}}_${{n}}M_${{suf}}_${{periodoFiltrado}}_${{stamp}}.xlsx`;
    
      const buffer = await wb.xlsx.writeBuffer();
      saveAs(new Blob([buffer], {{
        type: "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
      }}), nombreArchivo);
    }}

    function _parseFechaIngreso(fechaStr) {{
    if (!fechaStr) return null;
    const s = String(fechaStr).trim();
    
    // Caso: "2023-12-21 00:00:00" o "2023-12-21"
    const iso = s.match(/^(\d{{4}})-(\d{{2}})-(\d{{2}})/);
    if (iso) {{
    const y = Number(iso[1]), m = Number(iso[2]), d = Number(iso[3]);
    const dt = new Date(y, m - 1, d);
    return isNaN(dt.getTime()) ? null : dt;
    }}
    
    // Caso: "dd/mm/yyyy"
    const latam = s.match(/^(\d{{1,2}})\/(\d{{1,2}})\/(\d{{4}})$/);
    if (latam) {{
    const d = Number(latam[1]), m = Number(latam[2]), y = Number(latam[3]);
    const dt = new Date(y, m - 1, d);
    return isNaN(dt.getTime()) ? null : dt;
    }}
    
    const dt = new Date(s);
    return isNaN(dt.getTime()) ? null : dt;
    }}
    
    function _finDeMes(periodo) {{
    // periodo: "DICIEMBRE_2025"
    const [mesTxt, anioTxt] = String(periodo || '').split('_');
    const y = Number(anioTxt);
    
    const meses = {{
    'ENERO': 0, 'FEBRERO': 1, 'MARZO': 2, 'ABRIL': 3, 'MAYO': 4, 'JUNIO': 5,
    'JULIO': 6, 'AGOSTO': 7, 'SETIEMBRE': 8, 'SEPTIEMBRE': 8, 'OCTUBRE': 9,
    'NOVIEMBRE': 10, 'DICIEMBRE': 11
    }};
    
    const m = meses[String(mesTxt || '').toUpperCase()];
    if (m === undefined || isNaN(y)) return null;
    
    // último día del mes: (mes+1, día 0)
    return new Date(y, m + 1, 0, 23, 59, 59);
    }}
    
    function _mesEsValidoParaIngreso(periodo, fechaIngresoStr) {{
    const ingreso = _parseFechaIngreso(fechaIngresoStr);
    if (!ingreso) return true; // si no hay fecha, no bloqueamos
    
    const finMes = _finDeMes(periodo);
    if (!finMes) return true;
    
    // “mostrar info si el asesor ya estaba contratado en ese mes”
    return ingreso <= finMes;
    }}
    
    // Hooks (EVALUACIÓN)
    
    document.getElementById('btnEval3Meses')
      ?.addEventListener('click', () => setVistaEvaluacion('3M'));
    
    document.getElementById('btnEval6Meses')
      ?.addEventListener('click', () => setVistaEvaluacion('6M'));

    document.getElementById('btnEval12Meses')
      ?.addEventListener('click', () => setVistaEvaluacion('12M'));

    document.getElementById('btnEvalOtros')
      ?.addEventListener('click', () => setModoOtrosEvaluacion(!modoOtrosEvaluacion));
    
    document.getElementById('chkIncluirMesSeleccionadoEval')
      ?.addEventListener('change', (e) => {{
        incluirMesSeleccionadoEval = !!e.target.checked;
        actualizarTarjetasEvaluacionRapida();
      }});
    
    document.getElementById('chkMostrarAsesoresNuevosEval')
      ?.addEventListener('change', () => {{
        actualizarTarjetasEvaluacionRapida();
      }});
    
    document.getElementById('btnEvalExportarAnalisis')
      ?.addEventListener('click', async () => {{
        await exportarAnalisisEvaluacion();
      }});

    document.getElementById('btnEvalExportarRecuperoMetas')
      ?.addEventListener('click', async () => {{
        await exportarRecuperoYMetasEvaluacion();
      }});

    document.getElementById('btnEvalExportarQuintiles')
      ?.addEventListener('click', async () => {{
        await exportarQuintilesEvaluacionPromedios();
      }});


    // Estado inicial
    setVistaEvaluacion('3M');
    calcularPeriodoPrueba();

    // ====================== FUNCIONES PARA PERIODO DE PRUEBA ======================
    function agregarAsesorPeriodo() {{
        const input = document.getElementById('inputBusquedaPeriodo');
        const nombreAsesor = input.value.trim();
        
        if (nombreAsesor && !asesoresSeleccionadosPeriodo.has(nombreAsesor)) {{
            asesoresSeleccionadosPeriodo.add(nombreAsesor);
            actualizarAsesoresSeleccionadosPeriodo();
            input.value = '';
        }}
    }}

    // SELECCIÓN POR FECHA DE INICIO
    
    function _parseFechaFlexible(s) {{
      if (!s) return null;
      const str = String(s).trim();
      if (!str) return null;
    
      // yyyy-mm-dd (input type="date")
      let m = str.match(/^(\d{{4}})-(\d{{2}})-(\d{{2}})$/);
      if (m) {{
        const y = Number(m[1]), mo = Number(m[2]) - 1, d = Number(m[3]);
        const dt = new Date(y, mo, d);
        return isNaN(dt.getTime()) ? null : dt;
      }}
    
      // dd/mm/yyyy o d/m/yyyy
      m = str.match(/^(\d{{1,2}})\/(\d{{1,2}})\/(\d{{4}})$/);
      if (m) {{
        const d = Number(m[1]), mo = Number(m[2]) - 1, y = Number(m[3]);
        const dt = new Date(y, mo, d);
        return isNaN(dt.getTime()) ? null : dt;
      }}
    
      const dt = new Date(str);
      return isNaN(dt.getTime()) ? null : dt;
    }}
    
    function _getPeriodoHeader() {{
      const mesSeleccionado = document.getElementById('selectorMes')?.value;
      const añoSeleccionado = document.getElementById('selectorAño')?.value;
      if (!mesSeleccionado || !añoSeleccionado) return null;
      return {{ mes: mesSeleccionado, año: añoSeleccionado, periodo: `${{mesSeleccionado}}_${{añoSeleccionado}}` }};
    }}

    function _getFechaIngresoDesdeHeaderBase_GLOBAL(alias) {{
      const al = String(alias || '').trim();
      if (!al) return '';
    
      const headMes = document.getElementById('selectorMes')?.value;
      const headAño = document.getElementById('selectorAño')?.value;
      const periodoHead = (headMes && headAño) ? `${{headMes}}_${{headAño}}` : '';
      if (!periodoHead) return '';
    
      const base = window.baseAsesoresAnalisis?.[periodoHead];
      if (!Array.isArray(base)) return '';
    
      const r = base.find(b => String(b.alias_crr || '').trim() === al);
      if (!r) return '';
    
      // Normalización igual a tu lógica
      let fi = String(r.fecha_inicio || '').trim();
      if (!fi) return '';
      if (fi.includes(' ')) fi = fi.split(' ')[0];
    
      if (/^\d{{4}}-\d{{2}}-\d{{2}}$/.test(fi)) {{
        const [y, m, d] = fi.split('-');
        fi = `${{d}}/${{m}}/${{y}}`;
      }}
      return fi;
    }}
    
    function _getBasePeriodo(periodo) {{
      const base = window.baseAsesoresAnalisis ? window.baseAsesoresAnalisis[periodo] : null;
      return Array.isArray(base) ? base : [];
    }}
    
    function aplicarAutoSeleccionIngresoRango() {{
      calcularPeriodoPrueba();
    }}
    
    function eliminarAsesorPeriodo(nombre) {{
        asesoresSeleccionadosPeriodo.delete(nombre);
        actualizarAsesoresSeleccionadosPeriodo();
    }}
    
    function actualizarAsesoresSeleccionadosPeriodo() {{
        const container = document.getElementById('asesoresSeleccionadosPeriodo');
        if (!container) return;
        
        container.innerHTML = '';
        
        asesoresSeleccionadosPeriodo.forEach(asesor => {{
            const tag = document.createElement('div');
            tag.className = 'tag-elemento';
            tag.innerHTML = `
                ${{asesor}}
                <button class="eliminar-elemento" onclick="eliminarAsesorPeriodo('${{asesor}}')">×</button>
            `;
            container.appendChild(tag);
        }});
    }}
    
    function limpiarPeriodoPrueba() {{
        // Limpiar selects
        const selects = ['mes1', 'año1', 'mes2', 'año2', 'mes3', 'año3'];
        selects.forEach(id => {{
            const select = document.getElementById(id);
            if (select) select.value = '';
        }});
        
        // Limpiar porcentaje mínimo
        const porcentajeInput = document.getElementById('porcentajeMinimo');
        if (porcentajeInput) porcentajeInput.value = '70';
        
        // Limpiar input de búsqueda
        const inputBusqueda = document.getElementById('inputBusquedaPeriodo');
        if (inputBusqueda) inputBusqueda.value = '';
        
        // Limpiar asesores seleccionados
        asesoresSeleccionadosPeriodo.clear();
        actualizarAsesoresSeleccionadosPeriodo();

        // Limpiar selectores de nuevos asesores
        const inpF = document.getElementById('selectorFechaIngresoDesde');
        if (inpF) inpF.value = '';
        
        const inpH = document.getElementById('selectorFechaIngresoHasta');
        if (inpH) inpH.value = '';
        
        // Ocultar resultados
        const resultadosDiv = document.getElementById('resultadosPeriodo');
        if (resultadosDiv) resultadosDiv.style.display = 'none';
    }}

    function activarOrdenamientoTablaPeriodoPonderado() {{
      const tabla = document.getElementById('tablaPeriodoPonderado');
      if (!tabla) return;
    
      const thead = tabla.querySelector('thead');
      const tbody = tabla.querySelector('tbody');
      if (!thead || !tbody) return;
    
      const headers = Array.from(thead.querySelectorAll('th.sortable'));
    
      const _parseNum = (txt) => {{
        const t = String(txt || '').replace(/\s+/g, ' ').trim();
        const cleaned = t
          .replace(/S\/\s*/gi, '')
          .replace(/%/g, '')
          .replace(/,/g, '');
        const m = cleaned.match(/-?\d+(\.\d+)?/);
        return m ? Number(m[0]) : 0;
      }};
    
      const _parseDate = (txt) => {{
        const t = String(txt || '').trim();
        if (!t || t === '—') return null;
    
        // yyyy-mm-dd
        if (/^\d{{4}}-\d{{2}}-\d{{2}}$/.test(t)) {{
          const d = new Date(t + 'T00:00:00');
          return isNaN(d.getTime()) ? null : d;
        }}
    
        // dd/mm/yyyy
        const m = t.match(/^(\d{{2}})\/(\d{{2}})\/(\d{{4}})$/);
        if (m) {{
          const dd = Number(m[1]);
          const mm = Number(m[2]) - 1;
          const yy = Number(m[3]);
          const d = new Date(yy, mm, dd);
          return isNaN(d.getTime()) ? null : d;
        }}
    
        const d = new Date(t);
        return isNaN(d.getTime()) ? null : d;
      }};
    
      headers.forEach((th, colIndex) => {{
        th.style.cursor = 'pointer';
    
        th.addEventListener('click', () => {{
          const sortType = String(th.dataset.sort || 'text').toLowerCase();
          const isDate = sortType === 'date';
    
          // FECHA: siempre asc (menor -> mayor)
          // NO-FECHA: toggle desc/asc (por defecto desc)
          const prev = th.dataset.order || (isDate ? 'asc' : 'desc');
          const next = isDate ? 'asc' : (prev === 'desc' ? 'asc' : 'desc');
    
          // limpiar estado de otros headers
          headers.forEach(h => {{ if (h !== th) h.dataset.order = ''; }});
          th.dataset.order = next;
    
          const rows = Array.from(tbody.querySelectorAll('tr'));
    
          rows.sort((ra, rb) => {{
            const aCell = ra.children[colIndex];
            const bCell = rb.children[colIndex];
            const aTxt = aCell ? aCell.textContent : '';
            const bTxt = bCell ? bCell.textContent : '';
    
            if (isDate) {{
              const da = _parseDate(aTxt);
              const db = _parseDate(bTxt);
              // nulos al final
              if (!da && !db) return 0;
              if (!da) return 1;
              if (!db) return -1;
              return da - db; // asc fijo
            }}
    
            if (sortType === 'num') {{
              const na = _parseNum(aTxt);
              const nb = _parseNum(bTxt);
              return next === 'desc' ? (nb - na) : (na - nb);
            }}
    
            // text
            const ta = String(aTxt || '').trim().toUpperCase();
            const tb = String(bTxt || '').trim().toUpperCase();
            return next === 'desc' ? tb.localeCompare(ta) : ta.localeCompare(tb);
          }});
    
          rows.forEach(r => tbody.appendChild(r));
        }});
      }});
    }}
    
    function calcularPeriodoPrueba() {{
      const mesSeleccionado = document.getElementById('selectorMes')?.value || '';
      const anioSeleccionado = document.getElementById('selectorA\u00f1o')?.value || '';
      const periodoFiltrado = (mesSeleccionado && anioSeleccionado) ? `${{mesSeleccionado}}_${{anioSeleccionado}}` : '';

      const resultadosDiv = document.getElementById('resultadosPeriodo');
      const tablaDiv = document.getElementById('tablaPeriodo');
      if (!resultadosDiv || !tablaDiv) return;

      if (!periodoFiltrado || !Array.isArray(window.baseAsesoresAnalisis?.[periodoFiltrado])) {{
        tablaDiv.innerHTML = '<div style="padding:18px; text-align:center; color:#666;">Selecciona un mes/anio con base disponible.</div>';
        resultadosDiv.style.display = 'block';
        return;
      }}

      const porcentajeMinimo = Number(document.getElementById('porcentajeMinimo')?.value || 70);
      const omitirPrimerMes = !!document.getElementById('chkOmitirPrimerMesPeriodo')?.checked;
      const filtrarPeriodoCumplido = !!document.getElementById('chkFiltrarPeriodoCumplido')?.checked;

      const periodosBase = _getPeriodosAtras(mesSeleccionado, anioSeleccionado, 3, true);
      const periodos = omitirPrimerMes ? periodosBase.slice(1) : periodosBase.slice();

      if (!periodos.length) {{
        tablaDiv.innerHTML = '<div style="padding:18px; text-align:center; color:#666;">No hay periodos disponibles para evaluar.</div>';
        resultadosDiv.style.display = 'block';
        return;
      }}

      const periodoInicio = periodosBase[0];
      const {{ y: yInicio, m: mInicio }} = _periodoToYM(periodoInicio);
      const {{ y: yFin, m: mFin }} = _periodoToYM(periodosBase[periodosBase.length - 1]);
      const inicioVentana = new Date(yInicio, mInicio - 1, 1);
      const finVentana = new Date(yFin, mFin, 0, 23, 59, 59, 999);

      const fechaDesdeUI = document.getElementById('selectorFechaIngresoDesde')?.value;
      const fechaHastaUI = document.getElementById('selectorFechaIngresoHasta')?.value;
      let desde = fechaDesdeUI ? _parseFechaFlexible(fechaDesdeUI) : inicioVentana;
      let hasta = fechaHastaUI ? _parseFechaFlexible(fechaHastaUI) : finVentana;
      if (desde) desde.setHours(0,0,0,0);
      if (hasta) hasta.setHours(23,59,59,999);
      if (desde && hasta && desde > hasta) {{
        const tmp = desde; desde = hasta; hasta = tmp;
      }}

      const supervisorFiltro = _getSupervisorFiltroEvaluacion();
      let baseActual = (window.baseAsesoresAnalisis?.[periodoFiltrado] || []).slice();
      if (supervisorFiltro) {{
        baseActual = baseActual.filter(p => String(p.supervisor || '').trim() === supervisorFiltro);
      }}

      const indexPorPeriodo = {{}};
      periodos.forEach(p => {{
        const idx = {{}};
        const baseP = window.baseAsesoresAnalisis?.[p] || [];
        baseP.forEach(r => {{
          const al = String(r.alias_crr || '').trim().toUpperCase();
          if (!al) return;
          if (!idx[al]) {{
            idx[al] = {{
              alias_crr: al,
              recupero: Number(r.recupero) || 0,
              meta: Number(r.meta) || 0,
              alcance_suma: Number(r.alcance) || 0,
              alcance_count: 1
            }};
          }} else {{
            idx[al].recupero += Number(r.recupero) || 0;
            idx[al].meta += Number(r.meta) || 0;
            idx[al].alcance_suma += Number(r.alcance) || 0;
            idx[al].alcance_count += 1;
          }}
        }});
        Object.values(idx).forEach(d => {{
          d.alcance = d.alcance_count > 0 ? (d.alcance_suma / d.alcance_count) : 0;
        }});
        indexPorPeriodo[p] = idx;
      }});

      const formatoNumero = (numero) => Number(numero || 0).toLocaleString('es-PE', {{
        minimumFractionDigits: 2,
        maximumFractionDigits: 2
      }});

      const _normAliasLocal = (v) => String(v || '').trim().toUpperCase();
      const _getNombreMostrarNuevo = (r) => String(r.nombre || r.staff || r.asesor || r.alias_crr || '').trim() || '?';
      const _getFechaIngresoNormalizada = (r) => {{
        let fi = String(r.fecha_inicio || '').trim();
        if (fi.includes(' ')) fi = fi.split(' ')[0];
        if (/^\d{{4}}-\d{{2}}-\d{{2}}$/.test(fi)) {{
          const [y, m, d] = fi.split('-');
          fi = `${{d}}/${{m}}/${{y}}`;
        }}
        return fi;
      }};

      const _renderPeriodoInfo = (p, alias) => {{
        const d = indexPorPeriodo[p]?.[_normAliasLocal(alias)];
        if (!d) return '&mdash;';
        const rec = Number(d.recupero) || 0;
        const meta = Number(d.meta) || 0;
        const alcance = Number(d.alcance) || 0;
        return `
          <div style="text-align:right;">
            <div><strong>S/ ${{formatoNumero(rec)}}</strong></div>
            <div style="color:#666; font-size:0.9em;">Meta: S/ ${{formatoNumero(meta)}}</div>
            <div style="color:#2980b9; font-size:0.9em; font-weight:700;">Alcance: ${{(alcance * 100).toFixed(2)}}%</div>
          </div>
        `;
      }};

      const vistos = new Set();
      const nuevos = [];

      baseActual.forEach(r => {{
        const alias = String(r.alias_crr || '').trim();
        const aliasKey = _normAliasLocal(alias);
        if (!alias || vistos.has(aliasKey)) return;
        vistos.add(aliasKey);

        const estado = String(r.estado || 'CALL').toUpperCase().trim();
        if (estado !== 'CALL') return;

        const fechaIngresoTxt = _getFechaIngresoNormalizada(r);
        const fechaIngreso = _parseFechaFlexible(fechaIngresoTxt);
        if (!fechaIngreso) return;
        fechaIngreso.setHours(0,0,0,0);

        if (fechaIngreso < inicioVentana || fechaIngreso > finVentana) return;
        if (desde && fechaIngreso < desde) return;
        if (hasta && fechaIngreso > hasta) return;

        const ingresoYMNum = _fechaIngresoToYMNum(fechaIngresoTxt);
        const periodosCumplidos = periodosBase.filter(periodo => {{
          const {{ y, m }} = _periodoToYM(periodo);
          return _ymToNum(y, m) >= ingresoYMNum;
        }}).length;

        if (filtrarPeriodoCumplido && periodosCumplidos < 3) return;

        nuevos.push({{
          alias,
          aliasKey,
          nombre: _getNombreMostrarNuevo(r),
          fechaIngresoTxt,
          supervisor: String(r.supervisor || '').trim(),
          periodosCumplidos
        }});
      }});

      nuevos.sort((a, b) => {{
        const fa = _parseFechaFlexible(a.fechaIngresoTxt)?.getTime() || 0;
        const fb = _parseFechaFlexible(b.fechaIngresoTxt)?.getTime() || 0;
        return fb - fa;
      }});

      let html = '<div class="contenedor-tabla-scroll"><table class="tabla-comparacion" id="tablaPeriodoPonderado">';
      html += '<thead><tr>';
      html += '<th class="sortable" data-sort="text">Asesor</th>';
      html += '<th class="sortable" data-sort="date">Fecha de Inicio</th>';
      periodos.forEach((p, idx) => {{
        html += `<th class="sortable" data-sort="num">PERIODO ${{idx + 1}}<br><span style="font-weight:400; font-size:0.85em;">${{p.replace('_',' ')}}</span></th>`;
      }});
      html += '<th class="sortable" data-sort="num">Total Recupero</th>';
      html += '<th class="sortable" data-sort="num">Total Meta</th>';
      html += '<th class="sortable" data-sort="num">% Calculado</th>';
      html += '<th class="sortable" data-sort="text">Estado</th>';
      html += '<th class="sortable" data-sort="num">Falta Recuperar</th>';
      html += '</tr></thead><tbody>';

      let sumaRecuperoGeneral = 0;
      let sumaMetaGeneral = 0;

      nuevos.forEach(item => {{
        let totalRecupero = 0;
        let totalMeta = 0;

        periodos.forEach(p => {{
          const d = indexPorPeriodo[p]?.[item.aliasKey];
          if (d) {{
            totalRecupero += Number(d.recupero) || 0;
            totalMeta += Number(d.meta) || 0;
          }}
        }});

        const pct = (totalMeta > 0) ? (totalRecupero / totalMeta) * 100 : 0;
        const estado = (pct >= porcentajeMinimo)
          ? '<span style="color:#27ae60; font-weight:700;">&#10003; Supera</span>'
          : '<span style="color:#e74c3c; font-weight:700;">&#10007; No supera</span>';

        const falta = (pct < porcentajeMinimo && totalMeta > 0)
          ? Math.max(0, (totalMeta * (porcentajeMinimo / 100)) - totalRecupero)
          : 0;
        const faltaHtml = `<strong style="color:${{falta > 0 ? '#e74c3c' : '#27ae60'}};">S/ ${{formatoNumero(falta)}}</strong>`;

        sumaRecuperoGeneral += totalRecupero;
        sumaMetaGeneral += totalMeta;

        html += `<tr>
          <td style="text-align:left; padding-left:15px;"><strong>${{item.nombre}}</strong><div style="font-size:0.85em; color:#666;">SUP: ${{item.supervisor || '&mdash;'}}</div></td>
          <td>${{item.fechaIngresoTxt || '&mdash;'}}</td>
          ${{periodos.map(p => `<td>${{_renderPeriodoInfo(p, item.alias)}}</td>`).join('')}}
          <td class="numero-grande"><strong style="color:#27ae60;">S/ ${{formatoNumero(totalRecupero)}}</strong></td>
          <td class="numero-grande"><strong style="color:#2c3e50;">S/ ${{formatoNumero(totalMeta)}}</strong></td>
          <td><strong style="color:#2980b9;">${{pct.toFixed(2)}}%</strong></td>
          <td>${{estado}}</td>
          <td class="numero-grande">${{faltaHtml}}</td>
        </tr>`;
      }});

      if (!nuevos.length) {{
        const colspan = 7 + periodos.length;
        html += `<tr><td colspan="${{colspan}}" style="padding:18px; text-align:center; color:#666;">No hay asesores nuevos para los filtros actuales.</td></tr>`;
      }}

      html += '</tbody></table></div>';
      const pctGeneral = sumaMetaGeneral > 0 ? (sumaRecuperoGeneral / sumaMetaGeneral) * 100 : 0;
      html += `
        <div style="margin-top:12px; text-align:center; color:#2c3e50; font-weight:700;">
          Nuevos detectados: ${{nuevos.length}} &middot; Periodos evaluados: ${{periodos.map(p => p.replace('_',' ')).join(', ')}} &middot; Alcance ponderado general: ${{pctGeneral.toFixed(2)}}%
        </div>
      `;

      tablaDiv.innerHTML = html;
      resultadosDiv.style.display = 'block';
      activarOrdenamientoTablaPeriodoPonderado();
    }}

    function toggleModoPromedioPeriodo() {{
      modoPromedioPeriodo = !modoPromedioPeriodo;
    
      const btn = document.getElementById('btnModoPromedioPeriodo');
    
      if (btn) {{
        btn.textContent = modoPromedioPeriodo
          ? '% Calculado (promedio)'
          : '% Calculado (ponderado)';
    
        btn.style.background = modoPromedioPeriodo ? '#d6eaf8' : '#ecf0f1';
      }}
    
      calcularPeriodoPrueba();
    }}
    
    // ========== NUEVA FUNCIÓN AUXILIAR ==========
    function segundosAHorasMinutosSegundos(segundosTotales) {{
        // Convierte segundos totales a formato HH:MM:SS
        const horas = Math.floor(segundosTotales / 3600);
        const minutos = Math.floor((segundosTotales % 3600) / 60);
        const segundos = Math.floor(segundosTotales % 60);
        
        return `${{String(horas).padStart(2, '0')}}:${{String(minutos).padStart(2, '0')}}:${{String(segundos).padStart(2, '0')}}`;
    }}
    
    // ========== FUNCIONES AUXILIARES EXISTENTES ==========
    
    function convertirFechaExcelAFechaKey(fechaExcel) {{
        // Convierte formato DD/MM/YYYY a YYYY-MM-DD
        try {{
            const [dia, mes, año] = fechaExcel.split('/');
            return `${{año}}-${{mes.padStart(2, '0')}}-${{dia.padStart(2, '0')}}`;
        }} catch {{
            return fechaExcel;
        }}
    }}
    
    function formatearHora(valor) {{
      if (valor === null || valor === undefined || valor === '') return '';
    
      // Si viene como Date
      if (valor instanceof Date && !isNaN(valor)) {{
        const hh = String(valor.getHours()).padStart(2, '0');
        const mm = String(valor.getMinutes()).padStart(2, '0');
        const ss = String(valor.getSeconds()).padStart(2, '0');
        return `${{hh}}:${{mm}}:${{ss}}`;
      }}
    
      const s = String(valor).trim();
    
      // Si ya viene como HH:MM o HH:MM:SS
      if (/^\d{{1,2}}:\d{{2}}(:\d{{2}})?$/.test(s)) {{
        const parts = s.split(':');
        const hh = String(parseInt(parts[0], 10)).padStart(2, '0');
        const mm = String(parseInt(parts[1], 10)).padStart(2, '0');
        const ss = String(parseInt(parts[2] || '0', 10)).padStart(2, '0');
        return `${{hh}}:${{mm}}:${{ss}}`;
      }}
    
      // Si viene como decimal Excel (ej: 0.2917013889)
      const n = Number(s);
      if (Number.isFinite(n)) {{
        const totalSeg = Math.round(n * 24 * 60 * 60);
        const hh = String(Math.floor(totalSeg / 3600)).padStart(2, '0');
        const mm = String(Math.floor((totalSeg % 3600) / 60)).padStart(2, '0');
        const ss = String(totalSeg % 60).padStart(2, '0');
        return `${{hh}}:${{mm}}:${{ss}}`;
      }}
    
      return '';
    }}
    
    function convertirHoraASegundos(horaStr) {{
        // Convierte HH:MM:SS a segundos totales
        try {{
            const [horas, minutos, segundos] = horaStr.split(':').map(Number);
            return horas * 3600 + minutos * 60 + segundos;
        }} catch {{
            return 0;
        }}
    }}
    
    // ========== INICIALIZACIÓN SIMPLIFICADA ==========
    document.addEventListener('DOMContentLoaded', function() {{
        
        // 1. INICIALIZAR FILTROS GLOBALES UNA VEZ
        const mesSeleccionado = document.getElementById('selectorMes').value;
        const añoSeleccionado = document.getElementById('selectorAño').value;
        const periodoCompleto = `${{mesSeleccionado}}_${{añoSeleccionado}}`;
        
        if (datosMeses[periodoCompleto]) {{
            actualizarFiltrosGlobales();
        }}
        
        // Los selectores del periodo se conectan una sola vez en la inicialización principal.

    }});

    // ========== INICIALIZACIÓN ==========
    document.addEventListener('DOMContentLoaded', function() {{
        // Inicializar la sección de ranking por defecto
        const mesSeleccionado = document.getElementById('selectorMes').value;
        const añoSeleccionado = document.getElementById('selectorAño').value;
        actualizarPeriodo();
        actualizarSelectorMesSegunAñoHeader();
    
        // Inicializar la sección de ranking por defecto (primera carga)
        if (datosMeses[estadoGlobal.periodoActual]) {{
            const asesores = filtrarAsesoresPorCanal(datosMeses[estadoGlobal.periodoActual] || []);
            actualizarBotonesSupervisores(asesores);
        }}
    
        // Permitir agregar asesor con Enter
        const inputBusqueda = document.getElementById('inputBusqueda');
        if (inputBusqueda) {{
            inputBusqueda.addEventListener('keypress', function(e) {{
                if (e.key === 'Enter') {{
                    agregarAsesor();
                }}
            }});
        }}
        
        // Permitir agregar supervisor con Enter
        const inputBusquedaSupervisor = document.getElementById('inputBusquedaSupervisor');
        if (inputBusquedaSupervisor) {{
            inputBusquedaSupervisor.addEventListener('keypress', function(e) {{
                if (e.key === 'Enter') {{
                    agregarSupervisor();
                }}
            }});
        }}
        
        // Permitir agregar asesor periodo con Enter
        const inputBusquedaPeriodo = document.getElementById('inputBusquedaPeriodo');
        if (inputBusquedaPeriodo) {{
            inputBusquedaPeriodo.addEventListener('keypress', function(e) {{
                if (e.key === 'Enter') {{
                    agregarAsesorPeriodo();
                }}
            }});
        }}
    
        // Los selectores ejecutan actualizarPeriodo directamente mediante onchange.
        }});

        // ========== FUNCIONES PARA DESCARGAR TARJETAS DE CUMPLEAÑOS ==========
        function descargarTarjetasAsesores() {{
            if (!window.cumpleañosData || !window.cumpleañosData.asesores) {{
                alert('No hay datos de asesores para descargar');
                return;
            }}
            
            descargarTarjetasSeccion('asesores', window.cumpleañosData.asesores, 'Asesores');
        }}
        
        function descargarTarjetasSeccion(tipo, personas, nombreSeccion) {{
            if (!personas || personas.length === 0) {{
                alert(`No hay ${{nombreSeccion.toLowerCase()}} para descargar`);
                return;
            }}
            
            const mensajeProcesando = document.createElement('div');
            mensajeProcesando.style.cssText = `
                position: fixed;
                top: 50%;
                left: 50%;
                transform: translate(-50%, -50%);
                background: rgba(0, 0, 0, 0.9);
                color: white;
                padding: 20px 30px;
                border-radius: 10px;
                z-index: 99999;
                text-align: center;
                font-size: 16px;
            `;
            mensajeProcesando.innerHTML = `🔄 Generando imagen...`;
            document.body.appendChild(mensajeProcesando);
            
            // Crear contenedor mínimo sin fondos externos
            const contenedorTemp = document.createElement('div');
            contenedorTemp.id = 'contenedor-temp-cumple';
            contenedorTemp.style.cssText = `
                position: absolute;
                left: -9999px;
                top: -9999px;
                background: transparent !important;
            `;
            
            // Color según el tipo
            const colorPrincipal = tipo === 'asesores' ? '#2ecc71' : '#3498db';
            const colorPrincipalTransparente = tipo === 'asesores' ? 'rgba(46, 204, 113, 0.95)' : 'rgba(52, 152, 219, 0.95)';
            
            // Crear tarjetas individuales con 70% de opacidad
            personas.forEach(persona => {{
                const tarjeta = document.createElement('div');
                tarjeta.style.cssText = `
                    width: 760px;
                    min-height: 100px;
                    background: rgba(255, 255, 255, 0.95); /* FONDO BLANCO 70% OPACO */
                    border-radius: 10px;
                    padding: 20px;
                    display: flex;
                    align-items: center;
                    box-shadow: 0 4px 12px rgba(0,0,0,0.1);
                    border-left: 6px solid ${{colorPrincipalTransparente}}; /* BORDE 70% OPACO */
                    margin-bottom: 15px;
                    box-sizing: border-box;
                    backdrop-filter: blur(2px); /* Efecto de vidrio opcional */
                `;
                
                tarjeta.innerHTML = `
                    <div style="margin-right: 15px; flex-shrink: 0;">
                        <div style="font-size: 36px; color: ${{colorPrincipalTransparente}}; opacity: 0.7;">👤</div>
                    </div>
                    <div style="flex: 1; min-width: 0;">
                        <div style="font-size: 40px; font-weight: bold; color: rgba(44, 62, 80, 1); margin-bottom: 5px;">
                            ${{persona.nombre}}
                        </div>
                        <div style="color: rgba(102, 102, 102, 0.7); font-size: 22px; font-style: italic;">
                            ${{persona.puesto_real || persona.alias || ''}}
                        </div>
                    </div>
                    <div style="text-align: right; flex-shrink: 0; margin-left: 15px;">
                        <div style="font-size: 48px; font-weight: bold; color: rgba(231, 76, 60, 1); line-height: 1;">
                            ${{persona.dia}}
                        </div>
                        <div style="color: rgba(102, 102, 102, 0.7); font-size: 22px;">
                            ${{persona.mes}}
                        </div>
                    </div>
                `;
                
                contenedorTemp.appendChild(tarjeta);
            }});
            
            document.body.appendChild(contenedorTemp);
            
            setTimeout(() => {{
                html2canvas(contenedorTemp, {{
                    scale: 2,
                    backgroundColor: null, // FONDO TRANSPARENTE
                    logging: false,
                    useCORS: true
                }}).then(canvas => {{
                    // Obtener imagen con fondo transparente
                    const imagen = canvas.toDataURL('image/png');
                    
                    const enlace = document.createElement('a');
                    enlace.href = imagen;
                    enlace.download = `Cumpleaños_${{window.cumpleañosData.mes}}_${{nombreSeccion}}_translucido.png`;
                    document.body.appendChild(enlace);
                    enlace.click();
                    document.body.removeChild(enlace);
                    
                    // Limpiar
                    document.body.removeChild(contenedorTemp);
                    document.body.removeChild(mensajeProcesando);
                    
                }}).catch(error => {{
                    console.error('Error:', error);
                    alert('Error al generar la imagen');
                    document.body.removeChild(contenedorTemp);
                    document.body.removeChild(mensajeProcesando);
                }});
            }}, 500);
        }}
        
        // Verificar si html2canvas está disponible, si no, cargarlo dinámicamente
        function cargarHtml2CanvasSiEsNecesario() {{
            if (typeof html2canvas === 'undefined') {{
                const script = document.createElement('script');
                script.src = 'https://cdnjs.cloudflare.com/ajax/libs/html2canvas/1.4.1/html2canvas.min.js';
                script.integrity = 'sha512-BNaRQnYJYiPSqHHDb58B0yaPfCu+Wgds8Gp/gU33kqBtgNS4tSPHuGibyoeqMV/TJlSKda6FXzoEyYGjTe+vXA==';
                script.crossOrigin = 'anonymous';
                script.referrerPolicy = 'no-referrer';
                
                script.onload = function() {{
                    console.log('html2canvas cargado exitosamente');
                }};
                
                script.onerror = function() {{
                    alert('Error al cargar html2canvas. No se pueden generar imágenes.');
                }};
                
                document.head.appendChild(script);
            }}
        }}
        
        // Cargar html2canvas automáticamente cuando se carga la página
        document.addEventListener('DOMContentLoaded', function() {{
            cargarHtml2CanvasSiEsNecesario();
        }});

        function calcularNecesarioParaObjetivoEval(objetivoDecimal, det, alias) {{
          const resultado = {{
            textoMonto: '—',
            textoDetalle: 'Sin datos del mes objetivo'
          }};
        
            const chkIncluirMes = document.getElementById('chkIncluirMesSeleccionadoEval');
            const incluirMesSeleccionado = !!chkIncluirMes?.checked;
            
            const head = _getPeriodoHeader ? _getPeriodoHeader() : null;
            const periodoHeader = head?.periodo || '';
            
            const ordenMesesEval = {{
              ENERO: 1,
              FEBRERO: 2,
              MARZO: 3,
              ABRIL: 4,
              MAYO: 5,
              JUNIO: 6,
              JULIO: 7,
              AGOSTO: 8,
              SETIEMBRE: 9,
              SEPTIEMBRE: 9,
              OCTUBRE: 10,
              NOVIEMBRE: 11,
              DICIEMBRE: 12
            }};
            
            function periodoToNumeroEval(periodo) {{
              const partes = String(periodo || '').split('_');
              const mes = String(partes[0] || '').toUpperCase();
              const anio = Number(partes[1]) || 0;
              const nroMes = ordenMesesEval[mes] || 0;
              return (anio * 100) + nroMes;
            }}
            
            const filasConDatos = (det.filas || []).filter(r =>
              r.alcanceFrac != null &&
              r.periodo
            );
            
            const periodoAnteriorReal = filasConDatos
              .filter(r => periodoToNumeroEval(r.periodo) < periodoToNumeroEval(periodoHeader))
              .sort((a, b) => periodoToNumeroEval(b.periodo) - periodoToNumeroEval(a.periodo))[0]?.periodo || '';
            
            const periodoObjetivo = incluirMesSeleccionado
              ? periodoHeader
              : periodoAnteriorReal;
        
          const filasBase = (det.filas || []).filter(r =>
            r.incluido &&
            r.alcanceFrac != null &&
            r.periodo !== periodoObjetivo
          );
        
          if (!periodoObjetivo || !datosMeses?.[periodoObjetivo]) {{
            return resultado;
          }}
        
          const datosMesObjetivo = datosMeses[periodoObjetivo] || [];
        
          const asesorMesObjetivo = datosMesObjetivo.find(a => {{
            const aliasA = String(a.alias_crr || '').trim();
            const nombreA = String(a.nombre || '').trim();
            const aliasBuscado = String(alias || '').trim();
        
            return aliasA === aliasBuscado || nombreA === aliasBuscado;
          }});
        
          if (!asesorMesObjetivo) {{
            return resultado;
          }}
        
          const filaObjetivo = (det.filas || []).find(r => r.periodo === periodoObjetivo);
            
          const metaMesObjetivo = Number(filaObjetivo?.meta) || Number(asesorMesObjetivo.meta) || 0;
          const recuperoActualMes = Number(filaObjetivo?.recupero) || Number(asesorMesObjetivo.recupero) || 0;
        
          const sumaAlcancesPrevios = filasBase.reduce((acc, r) => {{
            return acc + (Number(r.alcanceFrac) || 0);
          }}, 0);
        
          const mesesParaPromedio = filasBase.length + 1;
        
          const alcanceFinalNecesarioMes = Math.max(
            0,
            (objetivoDecimal * mesesParaPromedio) - sumaAlcancesPrevios
          );
        
          const montoFinalNecesarioMes =
            alcanceFinalNecesarioMes * metaMesObjetivo;
        
          const montoFaltanteReal = Math.max(
            0,
            montoFinalNecesarioMes - recuperoActualMes
          );
        
          resultado.textoMonto =
            `S/ ${{montoFaltanteReal.toLocaleString('es-PE', {{
              minimumFractionDigits: 2,
              maximumFractionDigits: 2
            }})}}`;
        
          resultado.textoDetalle =
              `${{(alcanceFinalNecesarioMes * 100).toFixed(2)}}% requerido en ${{periodoObjetivo.replace('_', ' ')}} ` +
              `| Total requerido: S/ ${{montoFinalNecesarioMes.toLocaleString('en-US', {{
                minimumFractionDigits: 2,
                maximumFractionDigits: 2
          }})}}`;
        
          return resultado;
        }}

        function _parseObjetivoNecesarioEval(valor, fallbackDecimal) {{
          const limpio = String(valor ?? '').replace('%', '').replace(',', '.').trim();
          let n = Number(limpio);
          if (!isFinite(n)) return fallbackDecimal;
          if (n > 1) n = n / 100;
          if (n < 0) n = 0;
          return n;
        }}

        function _fmtObjetivoNecesarioEval(decimal) {{
          const n = Number(decimal);
          return `${{(isFinite(n) ? n * 100 : 0).toFixed(2)}}%`;
        }}

        function _actualizarNecesarioObjetivoEval(config, det, alias) {{
          const input = document.getElementById(config.inputId);
          const monto = document.getElementById(config.montoId);
          const detalle = document.getElementById(config.detalleId);
          const objetivo = _parseObjetivoNecesarioEval(input?.value, config.fallback);
          const calc = calcularNecesarioParaObjetivoEval(objetivo, det, alias);
          if (monto) monto.textContent = calc.textoMonto;
          if (detalle) detalle.textContent = calc.textoDetalle;
        }}

        function _openModalEval(alias) {{
          const modal = document.getElementById('modalEvalDetalle');
          if (!modal) return;
        
          const det = window.evalDetallePorAlias?.[alias];
          if (!det) return;
        
          const titulo = document.getElementById('modalEvalTitulo');
          const sub = document.getElementById('modalEvalSub');
          const reglas = document.getElementById('modalEvalReglas');
          const tbody = document.getElementById('modalEvalTbody');
          const tot = document.getElementById('modalEvalTotales');
        
          if (titulo) titulo.textContent = `Detalle de Evaluación · ${{alias}}`;
          if (sub) sub.textContent = `Inicio de Gestión: ${{_fmtFechaIngresoUI(det.fechaIngreso)}} · SUP: ${{det.supervisor || '—'}} · Meses usados: ${{det.mesesUsados}}/${{det.nObjetivo}}`;
        
          if (reglas) {{
            reglas.innerHTML = `
              <div style="font-weight:800; margin-bottom:6px;">Reglas aplicadas</div>
              <ul style="margin:0; padding-left:18px;">
                <li>Solo se consideran meses con <b>ESTADO = CALL</b>.</li>
                <li>Se excluye el mes de <b>SETIEMBRE_2025</b> (no evaluable).</li>
                <li>Se reconoce a partir de <b>fecha de Inicio</b> (meses anteriores: “No participó”).</li>
                <li>Para completar <b>${{det.nObjetivo}}</b> meses, se retrocede hasta encontrar meses efectivos.</li>
              </ul>
            `;
          }}
        
          if (tbody) {{
              const rows = det.filas || [];
              tbody.innerHTML = rows.map(r => {{
                const inc = r.incluido
                  ? `<span class="tag-si">SI</span>`
                  : `<span class="tag-no">NO</span>`;
            
                const metaTxt =
                  (r.meta === null || r.meta === undefined)
                    ? '—'
                    : Number(r.meta).toLocaleString('en-US', {{
                        minimumFractionDigits: 2,
                        maximumFractionDigits: 2
                      }});
                
                const recTxt =
                  (r.recupero === null || r.recupero === undefined)
                    ? '—'
                    : Number(r.recupero).toLocaleString('en-US', {{
                        minimumFractionDigits: 2,
                        maximumFractionDigits: 2
                      }});
                const alcTxt  = (r.alcanceFrac == null) ? '—' : `${{(r.alcanceFrac * 100).toFixed(2)}}%`;
            
                const quintilHtml = _renderBarraQuintil(r.q_alc);
                const extraClase = window.evalQuintilesExtraAbiertos ? '' : 'oculta';
                const extras = r.q_extra || {{}};
            
                return `
                  <tr>
                    <td>${{r.periodo}}</td>
                    <td>${{r.estado || '?'}}</td>
                    <td style="text-align:right;">${{metaTxt}}</td>
                    <td style="text-align:right;">${{recTxt}}</td>
                    <td style="text-align:right;">${{alcTxt}}</td>
                    <td>${{inc}}</td>
                    <td>${{quintilHtml}}</td>
                    <td class="eval-quintil-extra-col ${{extraClase}}">${{_renderBarraQuintilGris(extras.condonacion)}}</td>
                    <td class="eval-quintil-extra-col ${{extraClase}}">${{_renderBarraQuintilGris(extras.cierre)}}</td>
                    <td class="eval-quintil-extra-col ${{extraClase}}">${{_renderBarraQuintilGris(extras.calidad_pdp)}}</td>
                    <td class="eval-quintil-extra-col ${{extraClase}}">${{_renderBarraQuintilGris(extras.puntualidad)}}</td>
                  </tr>
                `;
              }}).join('') || `
                <tr>
                  <td colspan="11" style="padding:14px; text-align:center; color:#666;">—</td>
                </tr>
              `;
          }}
        
            const pctTxt = (det.pct === null) ? 'No participó' : `${{(det.pct * 100).toFixed(2)}}%`;
            
            if (tot) {{
              // opcional: promedio ponderado SOLO como referencia
              let metaSum = 0;
              let recSum = 0;
              (det.filas || []).forEach(r => {{
                if (r.incluido && (r.meta ?? 0) > 0) {{
                  metaSum += Number(r.meta) || 0;
                  recSum  += Number(r.recupero) || 0;
                }}
              }});
              const pctPond = (metaSum > 0) ? (recSum / metaSum) : null;
              const pctPondTxt = (pctPond === null) ? '—' : `${{(pctPond * 100).toFixed(2)}}%`;

              const calc70 = calcularNecesarioParaObjetivoEval(0.70, det, alias);
              const calc90 = calcularNecesarioParaObjetivoEval(0.90, det, alias);
            
              tot.innerHTML = `
                <div style="padding:12px 14px; border:1px solid #eef2f7; border-radius:12px;">
                  <div style="font-size:0.85rem; color:#666; font-weight:700;">MESES EFECTIVOS</div>
                  <div style="font-size:1.25rem; font-weight:900; color:#2c3e50;">${{det.mesesUsados}} / ${{det.nObjetivo}}</div>
                </div>
            
                <div style="padding:12px 14px; border:1px solid #eef2f7; border-radius:12px;">
                  <div style="font-size:0.85rem; color:#666; font-weight:700;">PROMEDIO DE ALCANCES</div>
                  <div style="font-size:1.25rem; font-weight:900; color:#2c3e50;">${{pctTxt}}</div>
                </div>
            
                <div style="padding:12px 14px; border:1px solid #eef2f7; border-radius:12px;">
                  <div style="font-size:0.85rem; color:#666; font-weight:700;">ALCANCE PONDERADO (REF)</div>
                  <div style="font-size:1.25rem; font-weight:900; color:#2c3e50;">${{pctPondTxt}}</div>
                </div>

                <div style="padding:12px 14px; border:1px solid #fee2e2; border-radius:12px; background:#fff7f7;">
                  <div style="font-size:0.85rem; color:#b71c1c; font-weight:700; display:flex; align-items:center; gap:6px; flex-wrap:wrap;">
                    <span>NECESARIO PARA</span>
                    <input id="objetivoEval70" type="text" value="70%" aria-label="Objetivo necesario 1" style="width:62px; height:24px; border:1px solid #f3b6b6; border-radius:6px; padding:2px 6px; color:#b71c1c; font-weight:900; background:#fff; font-size:0.85rem;">
                  </div>
                
                  <div id="necesarioEvalMonto70" style="font-size:1.25rem; font-weight:900; color:#b71c1c;">
                    ${{calc70.textoMonto}}
                  </div>
                
                  <div id="necesarioEvalDetalle70" style="font-size:0.95rem; color:#7f8c8d; margin-top:3px;">
                    ${{calc70.textoDetalle}}
                  </div>
                </div>

                <div style="padding:12px 14px; border:1px solid #d6eaf8; border-radius:12px; background:#f4faff;">
                  <div style="font-size:0.85rem; color:#1565c0; font-weight:700; display:flex; align-items:center; gap:6px; flex-wrap:wrap;">
                    <span>NECESARIO PARA</span>
                    <input id="objetivoEval90" type="text" value="90%" aria-label="Objetivo necesario 2" style="width:62px; height:24px; border:1px solid #9fc9ee; border-radius:6px; padding:2px 6px; color:#1565c0; font-weight:900; background:#fff; font-size:0.85rem;">
                  </div>
                
                  <div id="necesarioEvalMonto90" style="font-size:1.25rem; font-weight:900; color:#1565c0;">
                    ${{calc90.textoMonto}}
                  </div>
                
                  <div id="necesarioEvalDetalle90" style="font-size:0.95rem; color:#7f8c8d; margin-top:3px;">
                    ${{calc90.textoDetalle}}
                  </div>
                </div>
              `;

              [
                {{ inputId: 'objetivoEval70', montoId: 'necesarioEvalMonto70', detalleId: 'necesarioEvalDetalle70', fallback: 0.70 }},
                {{ inputId: 'objetivoEval90', montoId: 'necesarioEvalMonto90', detalleId: 'necesarioEvalDetalle90', fallback: 0.90 }}
              ].forEach(config => {{
                const input = document.getElementById(config.inputId);
                if (!input) return;
                const refrescar = () => _actualizarNecesarioObjetivoEval(config, det, alias);
                input.addEventListener('input', refrescar);
                input.addEventListener('change', refrescar);
                input.addEventListener('blur', () => {{
                  input.value = _fmtObjetivoNecesarioEval(_parseObjetivoNecesarioEval(input.value, config.fallback));
                  refrescar();
                }});
                refrescar();
              }});
            }}
            modal.style.display = 'block';
        }}
        
        function _closeModalEval() {{
          const modal = document.getElementById('modalEvalDetalle');
          if (modal) modal.style.display = 'none';
        }}
        document.addEventListener('click', (e) => {{
          const btn = e.target?.closest?.('.btn-eval-detalle');
          if (btn) {{
            const alias = btn.getAttribute('data-alias') || '';
            if (alias) _openModalEval(alias);
            return;
          }}
        
          const close = e.target?.closest?.('[data-close="1"]');
          if (close) {{
            _closeModalEval();
          }}
        }});
        
        // Escape con ESC
        document.addEventListener('keydown', (e) => {{
          if (e.key === 'Escape') _closeModalEval();
        }});

</script>
</body>
</html>'''
    
    letras_codificacion = r'A-Za-z\u00c1\u00c9\u00cd\u00d3\u00da\u00d1\u00dc\u00e1\u00e9\u00ed\u00f3\u00fa\u00f1\u00fc'
    patron_codificacion = '[' + letras_codificacion + ']' + re.escape(chr(63)) + '[' + letras_codificacion + ']'
    patrones_codificacion = re.findall(patron_codificacion, html_content, flags=re.IGNORECASE)
    if patrones_codificacion:
        muestras = ', '.join(sorted(set(patrones_codificacion))[:10])
        raise ValueError(f"Se detectaron posibles caracteres rotos por codificacion antes de guardar el HTML: {muestras}")

    with open(ruta_html, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print(f"HTML modular generado y guardado en: {ruta_html}")

def generar_seccion_ranking(asesores_data, contadores, opciones_asesores, opciones_supervisores, opciones_supervisores_select, opciones_segmentos_select, meses_unicos, años_unicos):
    """Genera el contenido de la sección RANKING"""

    # Listas de asesores por clasificación
    lista_100 = generar_lista_asesores(asesores_data, ">100%")
    lista_70 = generar_lista_asesores(asesores_data, ">70%")
    lista_40 = generar_lista_asesores(asesores_data, ">40%")
    lista_0 = generar_lista_asesores(asesores_data, ">0%")
    
    return f'''
        <div class="clasificaciones">
            <div class="clasificacion">
                <div class="clasificacion-header clasificacion-100">100% – ↑</div>
                <div class="asesores-lista" id="lista-100">
                    {lista_100}
                </div>
            </div>

            <div class="clasificacion">
                <div class="clasificacion-header clasificacion-70">70% – 100%</div>
                <div class="asesores-lista" id="lista-70">
                    {lista_70}
                </div>
            </div>

            <div class="clasificacion">
                <div class="clasificacion-header clasificacion-40">40% – 70%</div>
                <div class="asesores-lista" id="lista-40">
                    {lista_40}
                </div>
            </div>

            <div class="clasificacion">
                <div class="clasificacion-header clasificacion-0">0% – 40%</div>
                <div class="asesores-lista" id="lista-0">
                    {lista_0}
                </div>
            </div>
        </div>
        
        <!-- Estadísticas -->
        <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 20px; margin: 40px 0;">
            <div style="text-align: center; background: #e8f6f3; padding: 25px; border-radius: 12px; border: 2px solid #27ae60;">
                <div style="font-size: 1.1rem; color: #27ae60; margin-bottom: 10px;">Superaron el 100%</div>
                <div id="cantidad-100" style="font-size: 3rem; font-weight: bold; color: #27ae60;">{contadores[">100%"]}</div>
                <div style="font-size: 0.9rem; color: #666; margin-top: 10px;">Meta superada</div>
            </div>
            
            <div style="text-align: center; background: #fef9e7; padding: 25px; border-radius: 12px; border: 2px solid #f39c12;">
                <div style="font-size: 1.1rem; color: #f39c12; margin-bottom: 10px;">Superaron el 70%</div>
                <div id="cantidad-70" style="font-size: 3rem; font-weight: bold; color: #f39c12;">{contadores[">70%"]}</div>
                <div style="font-size: 0.9rem; color: #666; margin-top: 10px;">Alto rendimiento</div>
            </div>
            
            <div style="text-align: center; background: #fdebd0; padding: 25px; border-radius: 12px; border: 2px solid #e67e22;">
                <div style="font-size: 1.1rem; color: #e67e22; margin-bottom: 10px;">Superaron el 40%</div>
                <div id="cantidad-40" style="font-size: 3rem; font-weight: bold; color: #e67e22;">{contadores[">40%"]}</div>
                <div style="font-size: 0.9rem; color: #666; margin-top: 10px;">Rendimiento medio</div>
            </div>
            
            <div style="text-align: center; background: #fadbd8; padding: 25px; border-radius: 12px; border: 2px solid #e74c3c;">
                <div style="font-size: 1.1rem; color: #e74c3c; margin-bottom: 10px;">Entre 0% a 40%</div>
                <div id="cantidad-0" style="font-size: 3rem; font-weight: bold; color: #e74c3c;">{contadores[">0%"]}</div>
                <div style="font-size: 0.9rem; color: #666; margin-top: 10px;">Por mejorar</div>
            </div>
        </div>

    '''
    
def generar_seccion_alcances(mes_actual, año_actual):
    """Genera el contenido de la sección RECUPEROS Y ALCANCES con filtros por supervisor"""
    return f'''
        
        <!-- Estadísticas del mes -->
        <div class="mundial-accesos" style="display:flex; gap:10px; flex-wrap:wrap; justify-content:flex-end; align-items:center; margin:0 0 14px 0;">
            <button type="button" class="boton-exportar-excel btn-mundial" onclick="abrirModalMundialEquipos()">Mundial por equipos</button>
            <button type="button" class="boton-exportar-excel btn-mundial-sec" onclick="abrirModalMundialPaises('SURCO')">Mundial de paises - SURCO</button>
                  <button type="button" class="boton-exportar-excel btn-mundial-sec" onclick="abrirModalMundialPaises('BPO')">Mundial de paises - BPO</button>
                  <button type="button" class="boton-exportar-excel btn-mundial" onclick="mostrarTop10Paises()">TOP10-PAISES</button>
                  <button type="button" class="boton-exportar-excel btn-mundial" onclick="mostrarTop10MundialAsesor()">TOP10-MUNDIAL-ASESOR</button>
        </div>

        <div id="estadisticas-recuperos">
            <!-- Las estadísticas se cargarán dinámicamente -->
        </div>
        
        <!-- Analisis de alcances -->
        <div id="graficas-recuperos">
            <!-- Los gráficos se generarán dinámicamente -->
        </div>
    '''

def generar_lista_asesores(asesores_data, clasificacion):
    """Genera HTML para la lista de asesores de una clasificación específica"""
    asesores_filtrados = [a for a in asesores_data if a['clasificacion'] == clasificacion]
    
    if not asesores_filtrados:
        return '<div class="asesor-item">No hay asesores en esta categoría</div>'
    
    html_items = []
    for asesor in asesores_filtrados:
        clase_gradiente = f"gradiente-{clasificacion.replace('%', '').replace('>', '')}"
        clase_porcentaje = f"porcentaje-{clasificacion.replace('%', '').replace('>', '')}"
        
        item_html = f'''
        <div class="asesor-item {clase_gradiente}">
            <div class="asesor-nombre">{asesor['nombre']}</div>
            <div class="asesor-porcentaje {clase_porcentaje}">
                {asesor['porcentaje']}%
            </div>
        </div>'''
        html_items.append(item_html)
    
    return ''.join(html_items)

def obtener_mes_año_actual(meses_validos):
    """Obtiene el mes y año más reciente disponible"""
    if not meses_validos:
        return "NOVIEMBRE", "2025"
    
    # Ordenar por fecha (más reciente primero)
    meses_ordenados = sorted(meses_validos, key=lambda x: convertir_mes_a_fecha(x), reverse=True)
    mes, año = meses_ordenados[0].split('_')
    return mes, año


def convertir_mes_a_fecha(nombre_mes):
    """Convierte MES_AÑO a una fecha real para ordenar cronológicamente."""
    try:
        mes, año = nombre_mes.rsplit('_', 1)
        meses = {
            'ENERO': 1, 'FEBRERO': 2, 'MARZO': 3, 'ABRIL': 4,
            'MAYO': 5, 'JUNIO': 6, 'JULIO': 7, 'AGOSTO': 8,
            'SETIEMBRE': 9, 'SEPTIEMBRE': 9, 'OCTUBRE': 10,
            'NOVIEMBRE': 11, 'DICIEMBRE': 12
        }
        return datetime(int(año), meses[mes.upper()], 1)
    except (ValueError, KeyError, TypeError):
        return datetime(1900, 1, 1)


if __name__ == "__main__":
    procesar_excel_y_actualizar_html()


Cargando DIM y METAS desde CS_AVANCE_AS.xlsx...
DIM cargada: 5577 filas, 73 periodos, 5216 relaciones exactas
  ⚠️ DIM: 180 relaciones periodo+asesor son contradictorias y serán excluidas
METAS cargada: 231 filas


Usuario SQL:  Usr_Back_Office
Contraseña SQL:  ········


Cargando indicadores históricos desde SQL Server...


C:\Users\Jorge Vasquez\AppData\Local\Temp\ipykernel_30644\3137206838.py:570: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(consulta, conexion)


  ⚠️ SQL indicadores: 14 filas tienen CONCAT duplicado; se conservarán para conciliación.
Indicadores SQL cargados: 68171 filas, 51 periodos
Cargando INF histórico desde SQL Server...
INF SQL cargado: 5156 filas


C:\Users\Jorge Vasquez\AppData\Local\Temp\ipykernel_30644\3137206838.py:642: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(consulta, conexion)


DIM excluye 10 periodos SQL sin cobertura: AGOSTO_2019 a MAYO_2020
Construyendo JUNIO_2020 desde Indicadores + DIM + METAS...
  ⚠️ JUNIO_2020: 12 asesores SQL excluidos por no existir en DIM para ese periodo
  ⚠️ JUNIO_2020: 22 asesores CALL sin Meta_Asesor compatible
Construyendo JULIO_2020 desde Indicadores + DIM + METAS...
  ⚠️ JULIO_2020: 4 asesores SQL excluidos por no existir en DIM para ese periodo
  ⚠️ JULIO_2020: 28 asesores CALL sin Meta_Asesor compatible
Construyendo AGOSTO_2020 desde Indicadores + DIM + METAS...
  ⚠️ AGOSTO_2020: 11 asesores SQL excluidos por no existir en DIM para ese periodo
  ⚠️ AGOSTO_2020: 27 asesores CALL sin Meta_Asesor compatible
Construyendo SETIEMBRE_2020 desde Indicadores + DIM + METAS...
  ⚠️ SETIEMBRE_2020: 15 asesores SQL excluidos por no existir en DIM para ese periodo
  ⚠️ SETIEMBRE_2020: 29 asesores CALL sin Meta_Asesor compatible
Construyendo OCTUBRE_2020 desde Indicadores + DIM + METAS...
  ⚠️ OCTUBRE_2020: 7 asesores SQL excluidos por no

In [2]:
import subprocess
import os
from datetime import datetime

def realizar_git_operations():
    """
    Función separada para operaciones Git.
    Se puede llamar desde otra celda cuando se quiera hacer commit/push.
    """
    try:
        # Cambiar al directorio del proyecto
        base_path = r"C:\Users\Jorge Vasquez\Ranking"
        os.chdir(base_path)
        
        print("=" * 60)
        print("🚀 INICIANDO OPERACIONES GIT")
        print("=" * 60)
        
        # 1. VERIFICAR ESTADO DE GIT
        print("\n📋 Verificando estado del repositorio...")
        try:
            result = subprocess.run(["git", "status"], capture_output=True, text=True, check=True)
            print("   ✅ Repositorio Git detectado")
        except subprocess.CalledProcessError:
            print("   ❌ No se pudo ejecutar 'git status'")
            print("   ℹ️  El directorio podría no ser un repositorio Git")
            return
        
        # 2. AGREGAR ARCHIVO AL STAGING
        print("\n📁 Agregando cambios al staging area...")
        subprocess.run(["git", "add", "."], check=True)
        print("   ✅ Todos los cambios del proyecto fueron agregados al staging")
        
        # 3. VERIFICAR SI HAY CAMBIOS PARA COMMIT
        print("\n🔍 Verificando cambios pendientes...")
        result = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True, check=True)
        
        if not result.stdout.strip():
            print("   ℹ️  No hay cambios para commit (archivo ya actualizado)")
            return
        
        # 4. REALIZAR COMMIT
        print("\n💾 Creando commit...")
        commit_message = f"Actualización automática - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
        print(f"   📝 Mensaje: '{commit_message}'")
        
        commit_res = subprocess.run(
            ["git", "commit", "-m", commit_message],
            capture_output=True,
            text=True
        )
        
        if commit_res.returncode != 0:
            print("❌ Falló el commit")
            print("STDOUT:", commit_res.stdout)
            print("STDERR:", commit_res.stderr)
            return
        print("   ✅ Commit realizado exitosamente")
        
        # 5. REALIZAR PUSH
        print("\n📤 Enviando cambios al repositorio remoto...")
        push_result = subprocess.run(["git", "push"], capture_output=True, text=True)
        
        if push_result.returncode == 0:
            print("   ✅ Push completado exitosamente")
            
            # Mostrar información del push si está disponible
            if push_result.stdout:
                for line in push_result.stdout.strip().split('\n'):
                    if '->' in line or 'branch' in line.lower():
                        print(f"   📊 {line.strip()}")
        else:
            print("   ⚠️  Push encontró problemas")
            print(f"   📄 Salida: {push_result.stdout}")
            if push_result.stderr:
                print(f"   ❌ Error: {push_result.stderr.strip()}")
        
    except subprocess.CalledProcessError as e:
        print(f"\n❌ ERROR en operación Git (Código: {e.returncode})")
        print(f"   Comando: {e.cmd}")
        print(f"   Salida: {e.stdout}")
        print(f"   Error: {e.stderr}")
        
        print("\n💡 SOLUCIÓN RÁPIDA - Configurar Git:")
        print("   1. git config --global user.name 'Tu Nombre'")
        print("   2. git config --global user.email 'tu@email.com'")
        print("   3. git init  # Si no hay repositorio")
        print("   4. git remote add origin [url-del-repositorio]")
        
    except FileNotFoundError:
        print("\n❌ GIT NO ENCONTRADO")
        print("   Git no está instalado o no está en el PATH")
        print("   Descarga Git desde: https://git-scm.com/")
        
    except Exception as e:
        print(f"\n❌ ERROR INESPERADO: {e}")
        import traceback
        traceback.print_exc()

# Función auxiliar para solo hacer commit sin push
def solo_commit():
    """
    Solo hace commit sin push, útil para pruebas locales
    """
    try:
        base_path = r"C:\Users\Jorge Vasquez\Ranking"
        os.chdir(base_path)
        
        print("📁 Agregando archivo al staging...")
        subprocess.run(["git", "add", "."], check=True)
        
        print("💾 Creando commit local...")
        commit_message = f"Prueba local - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
        commit_res = subprocess.run(
            ["git", "commit", "-m", commit_message],
            capture_output=True,
            text=True
        )
        
        if commit_res.returncode != 0:
            print("❌ Falló el commit")
            print("STDOUT:", commit_res.stdout)
            print("STDERR:", commit_res.stderr)
            return
        
        print("✅ Commit local realizado")
        
    except Exception as e:
        print(f"❌ Error en commit local: {e}")

# Función para solo hacer push de commits existentes
def solo_push():
    """
    Solo hace push de commits existentes
    """
    try:
        base_path = r"C:\Users\Jorge Vasquez\Ranking"
        os.chdir(base_path)
        
        print("📤 Haciendo push de commits pendientes...")
        push_result = subprocess.run(["git", "push"], capture_output=True, text=True)
        
        if push_result.returncode == 0:
            print("✅ Push completado")
        else:
            print(f"⚠️  Error en push: {push_result.stderr}")
            
    except Exception as e:
        print(f"❌ Error en push: {e}")

# Función para ver estado sin modificar nada
def ver_estado_git():
    """
    Solo muestra el estado del repositorio
    """
    try:
        base_path = r"C:\Users\Jorge Vasquez\Ranking"
        os.chdir(base_path)
        
        print("📊 ESTADO DEL REPOSITORIO GIT")
        print("-" * 40)
        
        # Estado general
        subprocess.run(["git", "status"])
        
        print("\n📝 ÚLTIMOS COMMITS")
        print("-" * 40)
        subprocess.run(["git", "log", "--oneline", "-5"])
        
    except Exception as e:
        print(f"❌ Error al ver estado: {e}")

if __name__ == "__main__":
    realizar_git_operations()

🚀 INICIANDO OPERACIONES GIT

📋 Verificando estado del repositorio...
   ✅ Repositorio Git detectado

📁 Agregando cambios al staging area...
   ✅ Todos los cambios del proyecto fueron agregados al staging

🔍 Verificando cambios pendientes...

💾 Creando commit...
   📝 Mensaje: 'Actualización automática - 2026-07-14 12:46:17'
   ✅ Commit realizado exitosamente

📤 Enviando cambios al repositorio remoto...
   ✅ Push completado exitosamente
